In [ ]:
'''Kaggriculture | Adaptive Farm Intelligence — Cha22 Route Replay

Свежая независимая ветка для проверки на текущем ladder: frozen route-replay
с runtime router, reactive safety и market layers. Исходный агент и SHA-проверка
сохранены byte-exact; добавлено только наше единое оформление и понятный вводный
блок.

Запуск ячеек создаёт submission_cha22.tar.gz и submission.py. Ноутбук не
отправляет сабмит автоматически. Заявленные локальные win-rate не заменяют
проверку реальным Kaggle-сабмитом; текущий live-score может измениться.

Источник: публичный Kaggriculture cha22 agent. Apache-2.0 notices сохранены
внутри упакованного исходника.
'''


In [ ]:
'''Notebook context

# Kaggriculture cha22 - route-replay agent

A strong general agent for the Kaggriculture simulation competition. It plays a precomputed
**719-step action tape** per shop sequence (a "route", selected at runtime by a router) and
wraps it in reactive safety and market layers.

## Local full-pool regression

Paired A/B, 12 seeds per seat, fresh random worlds, local fast simulator: **57 opponents /
1,368 games - 1,337W / 0T / 31L (97.7% of decided games), mean paired score delta +6,670.**
A same-build self-play check adds 24 ties (shown last).

Rows are sorted by losses, then by mean score delta. A `0` in the loss column means that
matchup was never lost in its 24 games. Local simulator results, not ladder ratings.

| opponent | W | T | L | win% | mean delta |
|---|---|---|---|---|---|
| kaggriculture_market_rhythm_sale_policy | 20 | 0 | 4 | 83.3% | +2728 |
| kaggriculture_pipe18_six_layers | 22 | 0 | 2 | 91.7% | +1013 |
| kaggriculture_v52_lean_flock_yarn_route | 22 | 0 | 2 | 91.7% | +1120 |
| kaggriculture_harvest_ledger | 22 | 0 | 2 | 91.7% | +1301 |
| kaggriculture_v56_smarter_seeds_and_fertilizer | 22 | 0 | 2 | 91.7% | +1500 |
| shop_router_reactive_v7 | 22 | 0 | 2 | 91.7% | +3827 |
| kaggriculture_more_wheat_smarter_sales | 23 | 0 | 1 | 95.8% | +1473 |
| kaggriculture_master_engine_v4 | 23 | 0 | 1 | 95.8% | +1492 |
| kaggriculture_v51_lean_flock | 23 | 0 | 1 | 95.8% | +1494 |
| one_more_wheat | 23 | 0 | 1 | 95.8% | +1599 |
| v54_productive_idle_workers | 23 | 0 | 1 | 95.8% | +1609 |
| kaggriculture_master_engine_v4 | 23 | 0 | 1 | 95.8% | +1646 |
| kaggriculture_v53_opening_signature | 23 | 0 | 1 | 95.8% | +1696 |
| your_market_list_is_an_order_book | 23 | 0 | 1 | 95.8% | +1718 |
| farmer_john_and_the_idle_seller | 23 | 0 | 1 | 95.8% | +1770 |
| the_metav4_farm_submission_v13 | 23 | 0 | 1 | 95.8% | +1949 |
| kaggriculture_v49_funded_sale_timing_and_worker | 23 | 0 | 1 | 95.8% | +2210 |
| kaggriculture_v50_early_yarn_commit | 23 | 0 | 1 | 95.8% | +2461 |
| v44_market_layers_1234 | 23 | 0 | 1 | 95.8% | +3598 |
| market_smart_farming_kaggriculture | 23 | 0 | 1 | 95.8% | +3716 |
| first_in_line_stock_into_income | 23 | 0 | 1 | 95.8% | +3873 |
| v46_first_turn_microstructure | 23 | 0 | 1 | 95.8% | +3914 |
| farming_score_v5_timing_optimized | 23 | 0 | 1 | 95.8% | +4413 |
| kaggriculture_pipe16_idle_workers | 24 | 0 | 0 | 100.0% | +1342 |
| kaggriculture_v57_funding_order_invariant | 24 | 0 | 0 | 100.0% | +1538 |
| god_s_mode_hacked_stores | 24 | 0 | 0 | 100.0% | +1595 |
| v54_productive_wheat_and_patient | 24 | 0 | 0 | 100.0% | +1649 |
| kaggriculture_v55_one_turn_market_race_edge | 24 | 0 | 0 | 100.0% | +1808 |
| kaggriculture_master_engine_v3 | 24 | 0 | 0 | 100.0% | +2125 |
| the_2945_farm_96_vs_the_top_10_public_bots | 24 | 0 | 0 | 100.0% | +2245 |
| v44_winning_the_same_turn_sale_race | 24 | 0 | 0 | 100.0% | +3732 |
| ready_stock_earlier_sales | 24 | 0 | 0 | 100.0% | +4008 |
| shop_router_reactive_v5 | 24 | 0 | 0 | 100.0% | +4028 |
| v45_first_turn_wheat_round_trip | 24 | 0 | 0 | 100.0% | +4063 |
| v44_market_layers_124 | 24 | 0 | 0 | 100.0% | +4134 |
| v47_reactive_market_coordination | 24 | 0 | 0 | 100.0% | +4189 |
| demand_preserving_turn_sale_timing | 24 | 0 | 0 | 100.0% | +4383 |
| pipe_2_agent | 24 | 0 | 0 | 100.0% | +4543 |
| kaggriculture_pipe_7_wheat_microstructure | 24 | 0 | 0 | 100.0% | +4830 |
| v48_clear_the_queue | 24 | 0 | 0 | 100.0% | +5222 |
| v43_recovering_lost_harvests | 24 | 0 | 0 | 100.0% | +5298 |
| v41_review_candidate | 24 | 0 | 0 | 100.0% | +6052 |
| fully_dynamic_autonomous_agent | 24 | 0 | 0 | 100.0% | +6114 |
| beyond_48_0_128_128_worlds_with_95_cis | 24 | 0 | 0 | 100.0% | +6385 |
| more_yield_smarter_labor | 24 | 0 | 0 | 100.0% | +6642 |
| master_engine_v3 | 24 | 0 | 0 | 100.0% | +6762 |
| master_engine_v2 | 24 | 0 | 0 | 100.0% | +6991 |
| observed_timing_r37 | 24 | 0 | 0 | 100.0% | +7033 |
| v36_guarded_4turn_sales | 24 | 0 | 0 | 100.0% | +7088 |
| herd_safe_sale_window | 24 | 0 | 0 | 100.0% | +7456 |
| better_shop_v4 | 24 | 0 | 0 | 100.0% | +7510 |
| market_smart_base | 24 | 0 | 0 | 100.0% | +8220 |
| score_v35 | 24 | 0 | 0 | 100.0% | +8225 |
| smaller_shock | 24 | 0 | 0 | 100.0% | +8427 |
| v31_prod_sale_priority | 24 | 0 | 0 | 100.0% | +8938 |
| most_powerful_route | 24 | 0 | 0 | 100.0% | +9110 |
| pass_bot | 24 | 0 | 0 | 100.0% | +163036 |
| cha22 (same build, self-play) | 0 | 24 | 0 | - | +0 |
'''


In [ ]:
'''Notebook context

## What the ladder losses taught us

We lost games on the live ladder to some of the strongest opponents. Re-running this agent on
those worlds beats the recorded opponent's score in 14 of them. Analysing those games with our
simulator produced the design targets for this lineage:

- **The deficit is late-game conversion.** Our money path leads through day 17, then loses days
  18-29: the opponents turn the same farm into revenue better in the final third.
- **It is not a shop-draw lottery.** When we control for demand - replaying the recorded games'
  own shop schedules into our simulator - the gap widens instead of closing: the top players win
  by execution, not by lucky shops.
- **The edge is the same-turn sell race.** Across 11k public replays the most robust
  winner-vs-loser difference is realised price per unit (+8-10%): the top bots win the market-hour
  pairing race.
- **Capacity is already matched.** On the same worlds our land, crops, herd and money paths track
  theirs within a few thousand points all game - the gap is conversion, not scale.

---

Run all cells to write the SHA-verified `main.py` and build `submission.tar.gz`.
'''


In [ ]:
from pathlib import Path
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORKDIR = OUTPUT_ROOT
print('Building the embedded cha22 agent in', WORKDIR)


In [ ]:
import hashlib

# Frozen R5-era cha22 champion. Exact bytes; the pinned digest includes them.
EXPECTED_MAIN_SHA256 = '127ed3e62988c0474d386db6527ae8ca9de9bb1fe7004128557ddef67126c652'
SOURCE_BYTES = b''.join((
    b'# Kaggriculture submission v9/3: public V39 (Apache-2.0, notices below) plus the v9 layers\r\n# RACEPX gate, RACE (reservation from step 192, horizon 40 / margin 12), COURIER, CARROT and HERD\r\n# appended at the end of this file.\r\n# EXP-173 isolate opening market sequence inspired by yhay81/shop-router-0911-simple (Apache-2.0).\r\n# Kaggriculture EXP-167 candidate. Not submitted automatically.\r\n# Attribution: thomastschinkel, yhay81, destbreso, aurax7, tetsutani,\r\n# prvsiyan and Dmitrii Gluzdov. Apache-2.0 derivations; notices retained below.\r\n# Kaggriculture v31 / EXP-157, Ahmed Berat Ozer, September 9 2026.\r\n# Selected mechanism: crop_public_order. New independent confirmation is required.\r\n# Public V221B/V224C production/timing lineage: prvsiyan, Apache-2.0.\r\n# Original economics and integration; retained upstream licenses follow.\r\n# Kaggriculture v28 / EXP-154, Ahmed Berat Ozer, September 9 2026.\r\n# Changes: aurax7 day-end storage guard; Dmitrii Gluzdov physical terminal rescue\r\n# adapted to v27, with 64 deter',
    b'ministic simulations. Apache-2.0.\r\n# New action tapes and ordered shop-pair map: yhay81/shop-router-0909, Apache-2.0.\r\n# Kaggriculture v25, EXP-149: Shop0908 production, sale lead, terminal cargo rescue.\r\n# Runtime chassis: Apache-2.0; thomastschinkel, yhay81, tetsutani.\r\n# Routing and public action data: yhay81/shop-router-0908, frozen September 8, 2026.\r\n# \r\n#                                  Apache License\r\n#                            Version 2.0, January 2004\r\n#                         http://www.apache.org/licenses/\r\n# \r\n#    TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\r\n# \r\n#    1. Definitions.\r\n# \r\n#       "License" shall mean the terms and conditions for use, reproduction,\r\n#       and distribution as defined by Sections 1 through 9 of this document.\r\n# \r\n#       "Licensor" shall mean the copyright owner or entity authorized by\r\n#       the copyright owner that is granting the License.\r\n# \r\n#       "Legal Entity" shall mean the union of the acting entity and all\r\n#       other entitie',
    b's that control, are controlled by, or are under common\r\n#       control with that entity. For the purposes of this definition,\r\n#       "control" means (i) the power, direct or indirect, to cause the\r\n#       direction or management of such entity, whether by contract or\r\n#       otherwise, or (ii) ownership of fifty percent (50%) or more of the\r\n#       outstanding shares, or (iii) beneficial ownership of such entity.\r\n# \r\n#       "You" (or "Your") shall mean an individual or Legal Entity\r\n#       exercising permissions granted by this License.\r\n# \r\n#       "Source" form shall mean the preferred form for making modifications,\r\n#       including but not limited to software source code, documentation\r\n#       source, and configuration files.\r\n# \r\n#       "Object" form shall mean any form resulting from mechanical\r\n#       transformation or translation of a Source form, including but\r\n#       not limited to compiled object code, generated documentation,\r\n#       and conversions to other media types.\r\n# \r\n#     ',
    b'  "Work" shall mean the work of authorship, whether in Source or\r\n#       Object form, made available under the License, as indicated by a\r\n#       copyright notice that is included in or attached to the work\r\n#       (an example is provided in the Appendix below).\r\n# \r\n#       "Derivative Works" shall mean any work, whether in Source or Object\r\n#       form, that is based on (or derived from) the Work and for which the\r\n#       editorial revisions, annotations, elaborations, or other modifications\r\n#       represent, as a whole, an original work of authorship. For the purposes\r\n#       of this License, Derivative Works shall not include works that remain\r\n#       separable from, or merely link (or bind by name) to the interfaces of,\r\n#       the Work and Derivative Works thereof.\r\n# \r\n#       "Contribution" shall mean any work of authorship, including\r\n#       the original version of the Work and any modifications or additions\r\n#       to that Work or Derivative Works thereof, that is intentionally\r\n#       ',
    b'submitted to Licensor for inclusion in the Work by the copyright owner\r\n#       or by an individual or Legal Entity authorized to submit on behalf of\r\n#       the copyright owner. For the purposes of this definition, "submitted"\r\n#       means any form of electronic, verbal, or written communication sent\r\n#       to the Licensor or its representatives, including but not limited to\r\n#       communication on electronic mailing lists, source code control systems,\r\n#       and issue tracking systems that are managed by, or on behalf of, the\r\n#       Licensor for the purpose of discussing and improving the Work, but\r\n#       excluding communication that is conspicuously marked or otherwise\r\n#       designated in writing by the copyright owner as "Not a Contribution."\r\n# \r\n#       "Contributor" shall mean Licensor and any individual or Legal Entity\r\n#       on behalf of whom a Contribution has been received by Licensor and\r\n#       subsequently incorporated within the Work.\r\n# \r\n#    2. Grant of Copyright License. ',
    b'Subject to the terms and conditions of\r\n#       this License, each Contributor hereby grants to You a perpetual,\r\n#       worldwide, non-exclusive, no-charge, royalty-free, irrevocable\r\n#       copyright license to reproduce, prepare Derivative Works of,\r\n#       publicly display, publicly perform, sublicense, and distribute the\r\n#       Work and such Derivative Works in Source or Object form.\r\n# \r\n#    3. Grant of Patent License. Subject to the terms and conditions of\r\n#       this License, each Contributor hereby grants to You a perpetual,\r\n#       worldwide, non-exclusive, no-charge, royalty-free, irrevocable\r\n#       (except as stated in this section) patent license to make, have made,\r\n#       use, offer to sell, sell, import, and otherwise transfer the Work,\r\n#       where such license applies only to those patent claims licensable\r\n#       by such Contributor that are necessarily infringed by their\r\n#       Contribution(s) alone or by combination of their Contribution(s)\r\n#       with the Work to which',
    b' such Contribution(s) was submitted. If You\r\n#       institute patent litigation against any entity (including a\r\n#       cross-claim or counterclaim in a lawsuit) alleging that the Work\r\n#       or a Contribution incorporated within the Work constitutes direct\r\n#       or contributory patent infringement, then any patent licenses\r\n#       granted to You under this License for that Work shall terminate\r\n#       as of the date such litigation is filed.\r\n# \r\n#    4. Redistribution. You may reproduce and distribute copies of the\r\n#       Work or Derivative Works thereof in any medium, with or without\r\n#       modifications, and in Source or Object form, provided that You\r\n#       meet the following conditions:\r\n# \r\n#       (a) You must give any other recipients of the Work or\r\n#           Derivative Works a copy of this License; and\r\n# \r\n#       (b) You must cause any modified files to carry prominent notices\r\n#           stating that You changed the files; and\r\n# \r\n#       (c) You must retain, in the Source for',
    b'm of any Derivative Works\r\n#           that You distribute, all copyright, patent, trademark, and\r\n#           attribution notices from the Source form of the Work,\r\n#           excluding those notices that do not pertain to any part of\r\n#           the Derivative Works; and\r\n# \r\n#       (d) If the Work includes a "NOTICE" text file as part of its\r\n#           distribution, then any Derivative Works that You distribute must\r\n#           include a readable copy of the attribution notices contained\r\n#           within such NOTICE file, excluding those notices that do not\r\n#           pertain to any part of the Derivative Works, in at least one\r\n#           of the following places: within a NOTICE text file distributed\r\n#           as part of the Derivative Works; within the Source form or\r\n#           documentation, if provided along with the Derivative Works; or,\r\n#           within a display generated by the Derivative Works, if and\r\n#           wherever such third-party notices normally appear. The contents\r',
    b'\n#           of the NOTICE file are for informational purposes only and\r\n#           do not modify the License. You may add Your own attribution\r\n#           notices within Derivative Works that You distribute, alongside\r\n#           or as an addendum to the NOTICE text from the Work, provided\r\n#           that such additional attribution notices cannot be construed\r\n#           as modifying the License.\r\n# \r\n#       You may add Your own copyright statement to Your modifications and\r\n#       may provide additional or different license terms and conditions\r\n#       for use, reproduction, or distribution of Your modifications, or\r\n#       for any such Derivative Works as a whole, provided Your use,\r\n#       reproduction, and distribution of the Work otherwise complies with\r\n#       the conditions stated in this License.\r\n# \r\n#    5. Submission of Contributions. Unless You explicitly state otherwise,\r\n#       any Contribution intentionally submitted for inclusion in the Work\r\n#       by You to the Licensor shall',
    b' be under the terms and conditions of\r\n#       this License, without any additional terms or conditions.\r\n#       Notwithstanding the above, nothing herein shall supersede or modify\r\n#       the terms of any separate license agreement you may have executed\r\n#       with Licensor regarding such Contributions.\r\n# \r\n#    6. Trademarks. This License does not grant permission to use the trade\r\n#       names, trademarks, service marks, or product names of the Licensor,\r\n#       except as required for reasonable and customary use in describing the\r\n#       origin of the Work and reproducing the content of the NOTICE file.\r\n# \r\n#    7. Disclaimer of Warranty. Unless required by applicable law or\r\n#       agreed to in writing, Licensor provides the Work (and each\r\n#       Contributor provides its Contributions) on an "AS IS" BASIS,\r\n#       WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\r\n#       implied, including, without limitation, any warranties or conditions\r\n#       of TITLE, NON-INFRINGEMENT, M',
    b'ERCHANTABILITY, or FITNESS FOR A\r\n#       PARTICULAR PURPOSE. You are solely responsible for determining the\r\n#       appropriateness of using or redistributing the Work and assume any\r\n#       risks associated with Your exercise of permissions under this License.\r\n# \r\n#    8. Limitation of Liability. In no event and under no legal theory,\r\n#       whether in tort (including negligence), contract, or otherwise,\r\n#       unless required by applicable law (such as deliberate and grossly\r\n#       negligent acts) or agreed to in writing, shall any Contributor be\r\n#       liable to You for damages, including any direct, indirect, special,\r\n#       incidental, or consequential damages of any character arising as a\r\n#       result of this License or out of the use or inability to use the\r\n#       Work (including but not limited to damages for loss of goodwill,\r\n#       work stoppage, computer failure or malfunction, or any and all\r\n#       other commercial damages or losses), even if such Contributor\r\n#       has be',
    b'en advised of the possibility of such damages.\r\n# \r\n#    9. Accepting Warranty or Additional Liability. While redistributing\r\n#       the Work or Derivative Works thereof, You may choose to offer,\r\n#       and charge a fee for, acceptance of support, warranty, indemnity,\r\n#       or other liability obligations and/or rights consistent with this\r\n#       License. However, in accepting such obligations, You may act only\r\n#       on Your own behalf and on Your sole responsibility, not on behalf\r\n#       of any other Contributor, and only if You agree to indemnify,\r\n#       defend, and hold each Contributor harmless for any liability\r\n#       incurred by, or claims asserted against, such Contributor by reason\r\n#       of your accepting any such warranty or additional liability.\r\n# \r\n#    END OF TERMS AND CONDITIONS\r\n# \r\n#    APPENDIX: How to apply the Apache License to your work.\r\n# \r\n#       To apply the Apache License to your work, attach the following\r\n#       boilerplate notice, with the fields enclosed by br',
    b'ackets "[]"\r\n#       replaced with your own identifying information. (Don\'t include\r\n#       the brackets!)  The text should be enclosed in the appropriate\r\n#       comment syntax for the file format. We also recommend that a\r\n#       file or class name and description of purpose be included on the\r\n#       same "printed page" as the copyright notice for easier\r\n#       identification within third-party archives.\r\n# \r\n#    Copyright [yyyy] [name of copyright owner]\r\n# \r\n#    Licensed under the Apache License, Version 2.0 (the "License");\r\n#    you may not use this file except in compliance with the License.\r\n#    You may obtain a copy of the License at\r\n# \r\n#        http://www.apache.org/licenses/LICENSE-2.0\r\n# \r\n#    Unless required by applicable law or agreed to in writing, software\r\n#    distributed under the License is distributed on an "AS IS" BASIS,\r\n#    WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\r\n#    See the License for the specific language governing permissions and\r\n# ',
    b'   limitations under the License.\r\n"""Kaggriculture route-replay chassis (pure Python, stdlib only).\r\n\r\nA *route* is a pre-computed tape of 719 Kaggle-format actions\r\n``{"farmer": [op, ...], "hands": [[op, ...], ...], "market": [[order, item, qty], ...]}``.\r\nThe chassis replays the tape chosen by a caller-supplied ``router`` and wraps it\r\nin small reactive layers (each independently switchable via ``settings``):\r\n\r\n    hand_align            pad/truncate hands to the real hand count      (fieldbook_logic)\r\n    weed_repair           DIG a weed that blocks PLANT/BUILD, replay      (tetsutani + task spec)\r\n    sell_lead             sell next step\'s lots one step early            (fieldbook _lead_sale)\r\n    front_run             sell before the opponent\'s scheduled SELL       (hook; opponent_plan)\r\n    budget_guard          fund each 72-step block\'s purchases             (six_day_budget_guard.hpp)\r\n    room_guard            keep shed <= 99 at hour 23                      (tetsutani)\r\n    clamp_sells           trim',
    b' SELL orders to the projected shed          (tetsutani)\r\n    dead_stock            sell stock the route will never sell            (tetsutani)\r\n    terminal_liquidation  step >= 718: sell the whole projected shed      (fieldbook _terminal_sale)\r\n\r\nEngine facts (verified against kaggle_environments 1.32.7, env_1_32_7.py):\r\n  observation["farms"][p] = {"money", "tiles"[y][x], "farmer"[x,y], "hands"[[x,y]..],\r\n                             "unlocked_quadrants", "hires_today"}\r\n  tiles: None (empty) | "LOCKED" | {"kind": WEED|COOP|PASTURE|PLANT, "crop"/"animal", ...}\r\n  observation["private"] = {"shed": {item: n}, "seeds": {crop: n}, "inventories": [{}...]}\r\n  observation["market"] = {"inventory": {...}, "prices": {...}}\r\n  observation["town"] = {"unlocked_shops": [...]}\r\n  Agents act on steps 0..718 (interpreter marks DONE once step >= episodeSteps-2).\r\n"""\r\nfrom __future__ import annotations\r\n\r\nimport copy\r\n\r\nPRODUCTS = ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER")\r\nSE',
    b'ED_PRICE = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}\r\nANIMAL_COST = {"GOOSE": 300, "COW": 400, "SHEEP": 500}\r\nANIMAL_STRUCTURE = {"GOOSE": "COOP", "COW": "PASTURE", "SHEEP": "PASTURE"}\r\nLAND_PRICES = (1000, 2000, 4000)\r\nMOVES = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}\r\nFRONT_RUN_ITEMS = ("MILK", "WOOL", "STRAWBERRY", "MELON")\r\nLAST_ACT_STEP = 718\r\nPASS_ACTION = {"farmer": ["PASS"], "hands": [], "market": []}\r\n\r\nDEFAULT_SETTINGS = {\r\n    "hand_align": True,\r\n    "weed_repair": True,\r\n    "sell_lead": True,\r\n    "front_run": True,\r\n    "budget_guard": True,\r\n    "room_guard": True,\r\n    "clamp_sells": True,\r\n    "dead_stock": True,\r\n    "terminal_liquidation": True,\r\n    # tunables\r\n    "block_turns": 72,\r\n    "shed_capacity": 100,\r\n    "board_size": 10,\r\n    "max_orders": 10,\r\n    "turns_per_day": 24,\r\n    "min_sell_price": 2,\r\n}\r\n\r\n\r\n# --------------------------------------------------------------------------- helpers\r\ndef _get(value, key, default=No',
    b'ne):\r\n    """Field access that works for dicts and Kaggle Struct/attribute objects."""\r\n    if isinstance(value, dict):\r\n        return value.get(key, default)\r\n    getter = getattr(value, "get", None)\r\n    if callable(getter):\r\n        return getter(key, default)\r\n    return getattr(value, key, default)\r\n\r\n\r\ndef _int(value, default=0):\r\n    try:\r\n        return int(value)\r\n    except (TypeError, ValueError):\r\n        return default\r\n\r\n\r\ndef _fib(n):\r\n    a, b = 1, 1\r\n    for _ in range(n):\r\n        a, b = b, a + b\r\n    return a\r\n\r\n\r\ndef _step_of(observation):\r\n    raw = _get(observation, "step")\r\n    if raw is not None:\r\n        return _int(raw)\r\n    return _int(_get(observation, "day", 0)) * 24 + _int(_get(observation, "hour", 0))\r\n\r\n\r\ndef _shed_adjacent(pos, board):\r\n    if not isinstance(pos, (list, tuple)) or len(pos) < 2:\r\n        return False\r\n    half = board // 2\r\n    return pos[0] in (half - 1, half) and pos[1] in (half - 1, half)\r\n\r\n\r\ndef _tile_at(tiles, pos):\r\n    try:\r\n        x, y = int(pos[0]),',
    b' int(pos[1])\r\n        return tiles[y][x]\r\n    except (TypeError, ValueError, IndexError):\r\n        return "LOCKED"\r\n\r\n\r\ndef _is_noop(act, tile, inv, seeds, pos, board):\r\n    """True when the engine will certainly ignore ``act`` (mirrors _apply_unit_action)."""\r\n    if not act:\r\n        return True\r\n    op = act[0]\r\n    x, y = pos[0], pos[1]\r\n    if op in MOVES:\r\n        dx, dy = MOVES[op]\r\n        return not (0 <= x + dx < board and 0 <= y + dy < board)\r\n    if op == "PASS":\r\n        return True\r\n    adjacent = _shed_adjacent(pos, board)\r\n    if op == "DROP":\r\n        return (not adjacent) or (not inv)\r\n    if op == "PICKUP":\r\n        return not adjacent\r\n    if op == "PLACE":\r\n        item = act[1] if len(act) > 1 else None\r\n        if item in ANIMAL_STRUCTURE and isinstance(tile, dict) \\\r\n                and _get(tile, "kind") == ANIMAL_STRUCTURE[item] and _get(tile, "animal") is None:\r\n            return _int(_get(inv, item, 0)) <= 0\r\n        return (not adjacent) or _int(_get(inv, item, 0)) <= 0\r\n    if t',
    b'ile == "LOCKED":\r\n        return True\r\n    is_dict = isinstance(tile, dict)\r\n    kind = _get(tile, "kind") if is_dict else None\r\n    animal = is_dict and _get(tile, "animal") is not None\r\n    if op == "PLANT":\r\n        return tile is not None or _int(_get(seeds, act[1] if len(act) > 1 else None, 0)) <= 0\r\n    if op == "WATER":\r\n        return kind != "PLANT" or bool(_get(tile, "watered_today"))\r\n    if op == "HARVEST":\r\n        return (not is_dict) or _int(_get(tile, "yield_units", 0)) <= 0\r\n    if op == "FERTILIZE":\r\n        return kind != "PLANT" or _int(_get(inv, "FERTILIZER", 0)) <= 0\r\n    if op == "DIG":\r\n        return tile is None or animal\r\n    if op in ("BUILD_COOP", "BUILD_PASTURE"):\r\n        return tile is not None\r\n    if op == "FEED":\r\n        return (not animal) or bool(_get(tile, "fed_today")) or _int(_get(inv, "WHEAT", 0)) <= 0\r\n    if op == "COLLECT_FERTILIZER":\r\n        return (not animal) or (not _get(tile, "fertilizer_available"))\r\n    if op == "CARE":\r\n        return (not animal) or bool(',
    b'_get(tile, "cared_today"))\r\n    return True\r\n\r\n\r\nclass _View:\r\n    """Cheap per-step snapshot of everything the layers read from the observation."""\r\n\r\n    def __init__(self, observation, player, cfg):\r\n        farms = list(_get(observation, "farms", []) or [])\r\n        self.farm = farms[player] if player < len(farms) else {}\r\n        self.rival = farms[1 - player] if len(farms) >= 2 and 1 - player < len(farms) else {}\r\n        private = _get(observation, "private", {}) or {}\r\n        self.shed = {k: max(0, _int(v)) for k, v in dict(_get(private, "shed", {}) or {}).items()}\r\n        self.seeds = dict(_get(private, "seeds", {}) or {})\r\n        self.invs = [dict(i or {}) for i in (_get(private, "inventories", []) or [])]\r\n        market = _get(observation, "market", {}) or {}\r\n        self.prices = {k: _int(v) for k, v in dict(_get(market, "prices", {}) or {}).items()}\r\n        self.money = float(_get(self.farm, "money", 0.0) or 0.0)\r\n        self.tiles = _get(self.farm, "tiles", []) or []\r\n        self.board =',
    b' len(self.tiles) or cfg["board_size"]\r\n        self.positions = [_get(self.farm, "farmer", None)] + [list(p) for p in (_get(self.farm, "hands", []) or [])]\r\n        self.hires_today = _int(_get(self.farm, "hires_today", 0))\r\n        self.quadrants = len(list(_get(self.farm, "unlocked_quadrants", []) or []))\r\n\r\n    def inv(self, idx):\r\n        return self.invs[idx] if idx < len(self.invs) else {}\r\n\r\n    def in_hands(self, item):\r\n        return sum(max(0, _int(_get(inv, item, 0))) for inv in self.invs)\r\n\r\n\r\n# --------------------------------------------------------------------------- chassis\r\nclass Chassis:\r\n    """Replays ``routes[router(...)]`` with reactive safety/market layers.\r\n\r\n    routes         : {route_id: list of >= 719 Kaggle action dicts}\r\n    router         : callable(observation, step, state_dict) -> route_id, called every\r\n                     step; ``state_dict`` is per-player and persists across the game.\r\n    settings       : overrides for DEFAULT_SETTINGS (layer switches + tunables)\r\n    op',
    b'ponent_plan  : optional list of the opponent\'s expected actions (front_run hook)\r\n    """\r\n\r\n    def __init__(self, routes, router=None, settings=None, opponent_plan=None):\r\n        self.routes = {rid: list(tape) for rid, tape in routes.items()}\r\n        self.router = router or (lambda observation, step, state: next(iter(self.routes)))\r\n        self.cfg = dict(DEFAULT_SETTINGS)\r\n        self.cfg.update(settings or {})\r\n        self.opponent_plan = opponent_plan\r\n        self.players = {}\r\n        self.diagnostics = {"layer_fallbacks": 0, "entry_fallbacks": 0}\r\n        self._future_sells = {}   # route id -> {item: [remaining planned SELL qty from step t]}\r\n\r\n    # ---- state -----------------------------------------------------------------\r\n    def _state(self, player, step):\r\n        st = self.players.get(player)\r\n        if st is None or step == 0 or step <= st["last_step"]:\r\n            st = {"last_step": -1, "route": None, "router_state": {},\r\n                  "pending": {}, "sell_state": {"due_step": -1',
    b', "suppress": {}}}\r\n            self.players[player] = st\r\n        st["last_step"] = step\r\n        return st\r\n\r\n    def _route_action(self, route, step):\r\n        tape = self.routes[route]\r\n        if 0 <= step < len(tape) and isinstance(tape[step], dict):\r\n            return copy.deepcopy(tape[step])\r\n        return copy.deepcopy(PASS_ACTION)\r\n\r\n    def future_sells(self, route, item, step):\r\n        """Planned SELL quantity of ``item`` in route steps >= ``step`` (suffix sums)."""\r\n        table = self._future_sells.get(route)\r\n        if table is None:\r\n            tape = self.routes[route]\r\n            n = len(tape)\r\n            table = {p: [0] * (n + 1) for p in PRODUCTS}\r\n            for t in range(n - 1, -1, -1):\r\n                for p in PRODUCTS:\r\n                    table[p][t] = table[p][t + 1]\r\n                for o in (tape[t].get("market") or []) if isinstance(tape[t], dict) else []:\r\n                    if o and o[0] == "SELL" and len(o) >= 3 and o[1] in table:\r\n                        table[o[1',
    b']][t] += max(0, _int(o[2]))\r\n            self._future_sells[route] = table\r\n        col = table.get(item)\r\n        return col[step] if col and 0 <= step < len(col) else 0\r\n\r\n    # ---- main entry -----------------------------------------------------------\r\n    def act(self, observation, configuration=None):\r\n        if len(_get(observation, "farms", []) or []) < 2:\r\n            raise ValueError("incomplete observation")  # factory falls back to tape\r\n        step = _step_of(observation)\r\n        player = _int(_get(observation, "player", 0))\r\n        st = self._state(player, step)\r\n        cfg = self.cfg\r\n        view = _View(observation, player, cfg)\r\n\r\n        route = self.router(observation, step, st["router_state"])\r\n        if route not in self.routes:\r\n            route = st["route"] if st["route"] in self.routes else next(iter(self.routes))\r\n        st["route"] = route\r\n        action = self._route_action(route, step)\r\n        raw = copy.deepcopy(action)\r\n        try:\r\n            if cfg["hand_align"]:\r',
    b'\n                self._hand_align(action, view)\r\n            if cfg["weed_repair"]:\r\n                self._weed_repair(action, view, st, route, step)\r\n            if cfg["sell_lead"] or cfg["front_run"]:\r\n                self._apply_suppression(action, st["sell_state"], step)\r\n            projected = self._projected_shed(action, view)\r\n            lead_available = dict(projected)\r\n            next_sup = {"due_step": -1, "suppress": {}, "r36_debts": st["sell_state"].get("r36_debts", {})}\r\n            if cfg["sell_lead"]:\r\n                self._sell_lead(action, view, lead_available, route, step, next_sup)\r\n            if cfg["front_run"] and self.opponent_plan:\r\n                self._front_run(action, view, lead_available, route, step, next_sup)\r\n            st["sell_state"] = next_sup\r\n            if cfg["budget_guard"]:\r\n                self._budget_guard(action, view, route, step)\r\n            if cfg["room_guard"]:\r\n                self._room_guard(action, view, route, step)\r\n            if cfg["clamp_sells',
    b'"]:\r\n                self._clamp_sells(action, projected)\r\n            if cfg["dead_stock"]:\r\n                self._dead_stock(action, view, projected, route, step)\r\n            if cfg["terminal_liquidation"]:\r\n                self._terminal_liquidation(action, projected, step)\r\n            action["market"] = action["market"][: cfg["max_orders"]]\r\n            return action\r\n        except Exception:\r\n            self.diagnostics["layer_fallbacks"] += 1\r\n            return raw\r\n\r\n    # ---- layer: hand_align ----------------------------------------------------\r\n    def _hand_align(self, action, view):\r\n        """Pad with PASS / truncate the tape\'s hand list to the real number of hands\r\n        (fieldbook_logic.act). Extra hands would be ignored by the engine anyway;\r\n        missing ones just idle, so alignment only tidies the action."""\r\n        expected = max(0, len(view.positions) - 1)\r\n        hands = list(action.get("hands") or [])\r\n        hands.extend([["PASS"] for _ in range(max(0, expected - len(hand',
    b's)))])\r\n        action["hands"] = hands[:expected]\r\n\r\n    # ---- layer: weed_repair ---------------------------------------------------\r\n    def _weed_repair(self, action, view, st, route, step):\r\n        """If a PLANT/BUILD_* target tile is a WEED, DIG now and queue the intended\r\n        action for that unit; the queue replays on a later step when the unit still\r\n        stands there and its tape action would be a no-op (the displaced no-op is\r\n        queued behind it, so PLANT -> WATER chains survive). A PLANT is only replayed\r\n        when the unit\'s next tape action is not a move, so the mandatory same-day\r\n        WATER can follow; otherwise the seed is kept. A no-op turn spent on a weed\r\n        is also converted to DIG (tetsutani weed_dig)."""\r\n        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])\r\n        pending = st["pending"]\r\n        tape = self.routes[route]\r\n        nxt = tape[step + 1] if step + 1 < len(tape) and isinstance(tape[step + 1], dict) else {}\r\n        ',
    b'next_units = [nxt.get("farmer") or ["PASS"]] + list(nxt.get("hands") or [])\r\n        for i in range(min(len(units), len(view.positions))):\r\n            pos = view.positions[i]\r\n            if not isinstance(pos, (list, tuple)):\r\n                continue\r\n            pos = (int(pos[0]), int(pos[1]))\r\n            tile = _tile_at(view.tiles, pos)\r\n            act = list(units[i])\r\n            queue = pending.get(i)\r\n            if queue and queue[0][0] != pos:\r\n                pending.pop(i, None)\r\n                queue = None\r\n            is_weed = isinstance(tile, dict) and _get(tile, "kind") == "WEED"\r\n            noop = _is_noop(act, tile, view.inv(i), view.seeds, pos, view.board)\r\n            next_op = next_units[i][0] if i < len(next_units) and next_units[i] else "PASS"\r\n            if act and act[0] in ("PLANT", "BUILD_COOP", "BUILD_PASTURE") and is_weed:\r\n                pending.setdefault(i, []).append((pos, act))\r\n                act = ["DIG"]\r\n            elif queue and noop:\r\n                _, repla',
    b'y = queue[0]\r\n                if replay[0] == "PLANT" and next_op in MOVES:\r\n                    pending.pop(i, None)          # WATER could never follow: keep the seed\r\n                else:\r\n                    queue.pop(0)\r\n                    if act and act[0] != "PASS" and act[0] not in MOVES:\r\n                        queue.append((pos, act))\r\n                    act = replay\r\n                    if not queue:\r\n                        pending.pop(i, None)\r\n            elif is_weed and noop:\r\n                act = ["DIG"]\r\n            units[i] = act\r\n        action["farmer"] = units[0]\r\n        action["hands"] = units[1:]\r\n\r\n    # ---- projected shed -------------------------------------------------------\r\n    def _projected_shed(self, action, view):\r\n        """Shed contents after this step\'s unit actions but before the market runs:\r\n        PICKUP removes, DROP/PLACE(non-animal) near the shed adds up to capacity\r\n        (fieldbook _projected_shed / tetsutani projected shed)."""\r\n        cap = self.cfg[',
    b'"shed_capacity"]\r\n        proj = {p: view.shed.get(p, 0) for p in PRODUCTS}\r\n        for k, v in view.shed.items():\r\n            proj.setdefault(k, v)\r\n        total = sum(proj.values())\r\n        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])\r\n        for i in range(min(len(units), len(view.positions))):\r\n            if not _shed_adjacent(view.positions[i], view.board):\r\n                continue\r\n            act = units[i]\r\n            op = act[0] if act else "PASS"\r\n            inv = view.inv(i)\r\n            if op == "PICKUP" and len(act) >= 2 and act[1] in proj:\r\n                qty = min(proj[act[1]], max(0, _int(act[2]) if len(act) >= 3 else 1))\r\n                proj[act[1]] -= qty\r\n                total -= qty\r\n            elif op == "DROP":\r\n                for item, held in inv.items():\r\n                    take = min(max(0, _int(held)), max(0, cap - total))\r\n                    if take > 0:\r\n                        proj[item] = proj.get(item, 0) + take\r\n                  ',
    b'      total += take\r\n            elif op == "PLACE" and len(act) >= 2 and act[1] not in ANIMAL_STRUCTURE:\r\n                item = act[1]\r\n                take = min(max(0, _int(act[2]) if len(act) >= 3 else 1),\r\n                           max(0, _int(_get(inv, item, 0))), max(0, cap - total))\r\n                if take > 0:\r\n                    proj[item] = proj.get(item, 0) + take\r\n                    total += take\r\n        return proj\r\n\r\n    # ---- layer: sell_lead / front_run suppression ------------------------------\r\n    @staticmethod\r\n    def _apply_suppression(action, sell_state, step):\r\n        """Remove from this step\'s SELLs the quantities already sold a step early."""\r\n        if sell_state.get("due_step") != step:\r\n            return\r\n        remaining = dict(sell_state.get("suppress", {}))\r\n        kept = []\r\n        for order in action.get("market") or []:\r\n            order = list(order)\r\n            if order and order[0] == "SELL" and len(order) >= 3 and remaining.get(order[1], 0) > 0:\r\n        ',
    b'        removed = min(max(0, _int(order[2])), remaining[order[1]])\r\n                order[2] = _int(order[2]) - removed\r\n                remaining[order[1]] -= removed\r\n                # A zero-quantity order keeps later market race slots intact.\r\n            kept.append(order)\r\n        action["market"] = kept\r\n\r\n    @staticmethod\r\n    def _add_sell(action, item, qty, max_orders, merge=True):\r\n        market = action.setdefault("market", [])\r\n        if merge:\r\n            for order in market:\r\n                if order and order[0] == "SELL" and order[1] == item:\r\n                    order[2] = _int(order[2]) + qty\r\n                    return True\r\n        if len(market) >= max_orders:\r\n            return False\r\n        market.append(["SELL", item, qty])\r\n        return True\r\n\r\n    def _sell_lead(self, action, view, projected, route, step, next_sup):\r\n        """fieldbook _lead_sale: when step % 4 != 0 (no town consumption between the\r\n        two steps) sell the lots the tape plans to SELL next step now, for',
    b' products\r\n        other than WHEAT/FERTILIZER we already hold, and suppress them next step.\r\n        Skipped at the last step, at shop-unlock boundaries and if a SELL for that\r\n        product is already queued this step."""\r\n        cfg = self.cfg\r\n        nxt = step + 1\r\n        unlock_period = 3 * cfg["turns_per_day"]\r\n        if nxt > LAST_ACT_STEP or nxt % unlock_period == 0 or step % 4 == 0:\r\n            return\r\n        tape = self.routes[route]\r\n        future = tape[nxt] if nxt < len(tape) and isinstance(tape[nxt], dict) else {}\r\n        planned = {}\r\n        for o in future.get("market") or []:\r\n            if o and o[0] == "SELL" and len(o) >= 3 and o[1] in PRODUCTS:\r\n                planned[o[1]] = planned.get(o[1], 0) + max(0, _int(o[2]))\r\n        already = {o[1] for o in action.get("market") or [] if o and o[0] == "SELL" and len(o) > 1}\r\n        for item in PRODUCTS:\r\n            if item in ("WHEAT", "FERTILIZER") or planned.get(item, 0) <= 0 or item in already:\r\n                continue\r\n      ',
    b'      qty = min(projected.get(item, 0), planned[item])\r\n            if qty <= 0 or view.prices.get(item, 0) < cfg["min_sell_price"]:\r\n                continue\r\n            if not self._add_sell(action, item, qty, cfg["max_orders"], merge=False):\r\n                break\r\n            projected[item] -= qty\r\n            next_sup["suppress"][item] = next_sup["suppress"].get(item, 0) + qty\r\n        if next_sup["suppress"]:\r\n            next_sup["due_step"] = nxt\r\n\r\n    def _front_run(self, action, view, projected, route, step, next_sup):\r\n        """Hook: if ``opponent_plan`` (their expected tape) schedules a SELL of\r\n        MILK/WOOL/STRAWBERRY/MELON next step, sell what we hold of it now (before\r\n        their supply depresses the price) and suppress our own SELL of that quantity\r\n        next step. Bounded by our own remaining planned sales so it never dumps."""\r\n        cfg = self.cfg\r\n        nxt = step + 1\r\n        plan = self.opponent_plan\r\n        if nxt > LAST_ACT_STEP or nxt >= len(plan) or not isinstanc',
    b'e(plan[nxt], dict):\r\n            return\r\n        already = {o[1] for o in action.get("market") or [] if o and o[0] == "SELL" and len(o) > 1}\r\n        for o in plan[nxt].get("market") or []:\r\n            if not (o and o[0] == "SELL" and len(o) >= 3 and o[1] in FRONT_RUN_ITEMS):\r\n                continue\r\n            item = o[1]\r\n            if item in already or view.prices.get(item, 0) < cfg["min_sell_price"]:\r\n                continue\r\n            own_next = sum(max(0, _int(x[2])) for x in self.routes[route][nxt].get("market", [])\r\n                           if len(x) >= 3 and x[0] == "SELL" and x[1] == item)\r\n            qty = min(projected.get(item, 0), max(0, _int(o[2])), own_next)\r\n            if qty <= 0:\r\n                continue\r\n            if not self._add_sell(action, item, qty, cfg["max_orders"], merge=False):\r\n                break\r\n            projected[item] -= qty\r\n            already.add(item)\r\n            next_sup["suppress"][item] = next_sup["suppress"].get(item, 0) + qty\r\n        if next_s',
    b'up["suppress"]:\r\n            next_sup["due_step"] = nxt\r\n\r\n    # ---- layer: budget_guard --------------------------------------------------\r\n    def _block_requirements(self, view, route, start, end):\r\n        """Planned purchase cost and item reserves for tape steps [start, end)\r\n        (six_day_budget_guard.hpp calculate_six_day_requirements)."""\r\n        tape = self.routes[route]\r\n        budget = 0.0\r\n        seed_bal, item_bal = {}, {}\r\n        seed_need, item_need = {}, {}\r\n        hires_by_day = {}\r\n        quadrants = view.quadrants\r\n        for t in range(start, min(end, len(tape))):\r\n            a = tape[t] if isinstance(tape[t], dict) else {}\r\n            for u in [a.get("farmer") or ["PASS"]] + list(a.get("hands") or []):\r\n                if not u:\r\n                    continue\r\n                op = u[0]\r\n                arg = u[1] if len(u) > 1 else None\r\n                qty = max(1, _int(u[2]) if len(u) > 2 else 1)\r\n                if op == "PLANT" and arg in SEED_PRICE:\r\n                    s',
    b'eed_bal[arg] = seed_bal.get(arg, 0) - 1\r\n                    seed_need[arg] = max(seed_need.get(arg, 0), -seed_bal[arg])\r\n                elif op == "FEED":\r\n                    item_bal["WHEAT"] = item_bal.get("WHEAT", 0) - 1\r\n                    item_need["WHEAT"] = max(item_need.get("WHEAT", 0), -item_bal["WHEAT"])\r\n                elif op == "FERTILIZE":\r\n                    item_bal["FERTILIZER"] = item_bal.get("FERTILIZER", 0) - 1\r\n                    item_need["FERTILIZER"] = max(item_need.get("FERTILIZER", 0), -item_bal["FERTILIZER"])\r\n                elif op == "PLACE" and arg is not None:\r\n                    item_bal[arg] = item_bal.get(arg, 0) - qty\r\n                    item_need[arg] = max(item_need.get(arg, 0), -item_bal[arg])\r\n            for o in a.get("market") or []:\r\n                if not o:\r\n                    continue\r\n                op = o[0]\r\n                item = o[1] if len(o) > 1 else None\r\n                qty = max(1, _int(o[2]) if len(o) > 2 else 1)\r\n                if op == "H',
    b'IRE":\r\n                    day = (t - start) // self.cfg["turns_per_day"]\r\n                    hires_by_day[day] = hires_by_day.get(day, 0) + 1\r\n                elif op == "BUY_LAND":\r\n                    extra = quadrants - 1\r\n                    if 0 <= extra < len(LAND_PRICES):\r\n                        budget += LAND_PRICES[extra]\r\n                        quadrants += 1\r\n                elif op == "BUY_SEED" and item in SEED_PRICE:\r\n                    budget += SEED_PRICE[item] * qty\r\n                    seed_bal[item] = seed_bal.get(item, 0) + qty\r\n                elif op == "BUY_PRODUCT" and item in ("WHEAT", "FERTILIZER"):\r\n                    budget += view.prices.get(item, 0) * qty\r\n                    item_bal[item] = item_bal.get(item, 0) + qty\r\n                elif op == "BUY_ANIMAL" and item in ANIMAL_COST:\r\n                    budget += ANIMAL_COST[item] * qty\r\n                    item_bal[item] = item_bal.get(item, 0) + qty\r\n        for day, n in hires_by_day.items():\r\n            first = view.',
    b'hires_today if day == 0 else 0\r\n            for k in range(n):\r\n                budget += _fib(first + k)\r\n        return budget, item_need\r\n\r\n    def _budget_guard(self, action, view, route, step):\r\n        """At every block boundary (step % 72 == 0) make sure cash + the value of\r\n        stock the block already plans to sell covers the block\'s purchases (hires,\r\n        land, seeds, animals, products). A shortfall is covered by extra SELLs of\r\n        unprotected shed stock, highest price first; SELLs are moved in front of\r\n        the buys so the money is there when they execute."""\r\n        cfg = self.cfg\r\n        block = cfg["block_turns"]\r\n        if block <= 0 or step % block != 0:\r\n            return\r\n        budget, item_need = self._block_requirements(view, route, step, step + block)\r\n        market = action.setdefault("market", [])\r\n        existing = {}\r\n        for o in market:\r\n            if o and o[0] == "SELL" and len(o) >= 3:\r\n                existing[o[1]] = existing.get(o[1], 0) + max(0, _',
    b'int(o[2]))\r\n        cash = view.money\r\n        for item in PRODUCTS:\r\n            planned = max(existing.get(item, 0), self.future_sells(route, item, step)\r\n                          - self.future_sells(route, item, step + block))\r\n            cash += min(view.shed.get(item, 0), planned) * view.prices.get(item, 0)\r\n        shortfall = budget - cash\r\n        if shortfall <= 0:\r\n            return\r\n        candidates = []\r\n        for item in PRODUCTS:\r\n            price = view.prices.get(item, 0)\r\n            if price < cfg["min_sell_price"]:\r\n                continue\r\n            protected = max(0, item_need.get(item, 0) - view.in_hands(item))\r\n            avail = view.shed.get(item, 0) - protected - existing.get(item, 0)\r\n            if avail > 0:\r\n                candidates.append((-price, item, avail, price))\r\n        candidates.sort()\r\n        added = False\r\n        for _, item, avail, price in candidates:\r\n            if shortfall <= 0:\r\n                break\r\n            qty = min(avail, -(-int(shortfal',
    b'l) // price))\r\n            if self._add_sell(action, item, qty, cfg["max_orders"]):\r\n                shortfall -= qty * price\r\n                added = True\r\n        if added:\r\n            sells = [o for o in market if o and o[0] == "SELL"]\r\n            others = [o for o in market if not (o and o[0] == "SELL")]\r\n            action["market"] = sells + others\r\n\r\n    # ---- layer: room_guard ----------------------------------------------------\r\n    def _room_guard(self, action, view, route, step):\r\n        """tetsutani room_guard: at hour 23 the end-of-day drop pushes every unit\'s\r\n        inventory into the shed and overflow is destroyed. Estimate the shed after\r\n        this step (stock + carried + harvest/collect - feed/fertilize/place + buys -\r\n        sells) and, if it exceeds capacity-1, add SELLs preferring products with no\r\n        future planned sale, then highest price."""\r\n        cfg = self.cfg\r\n        if step % cfg["turns_per_day"] != cfg["turns_per_day"] - 1:\r\n            return\r\n        cap = cfg[',
    b'"shed_capacity"]\r\n        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])\r\n        carried = sum(max(0, _int(n)) for inv in view.invs for n in inv.values())\r\n        produced = consumed = 0\r\n        for i in range(min(len(units), len(view.positions))):\r\n            tile = _tile_at(view.tiles, view.positions[i])\r\n            a = units[i]\r\n            if not a:\r\n                continue\r\n            op = a[0]\r\n            if op == "HARVEST" and isinstance(tile, dict):\r\n                produced += max(0, _int(_get(tile, "yield_units", 0)))\r\n            elif op == "COLLECT_FERTILIZER" and isinstance(tile, dict) and _get(tile, "fertilizer_available"):\r\n                produced += 1\r\n            elif op in ("FEED", "FERTILIZE"):\r\n                consumed += 1\r\n            elif op == "PLACE" and len(a) > 1 and a[1] in ANIMAL_STRUCTURE:\r\n                consumed += 1\r\n        market = action.setdefault("market", [])\r\n        planned_sells, planned_buys = {}, 0\r\n        for o in market:\r\n ',
    b'           if not o:\r\n                continue\r\n            if o[0] == "SELL" and len(o) >= 3:\r\n                planned_sells[o[1]] = planned_sells.get(o[1], 0) + max(0, _int(o[2]))\r\n            elif o[0] in ("BUY_PRODUCT", "BUY_ANIMAL") and len(o) >= 3:\r\n                planned_buys += max(0, _int(o[2]))\r\n        shed_total = sum(view.shed.values())\r\n        fillable = sum(min(view.shed.get(it, 0), n) for it, n in planned_sells.items())\r\n        needed = shed_total + carried + produced - consumed + planned_buys - fillable - (cap - 1)\r\n        if needed <= 0:\r\n            return\r\n        priority = sorted(PRODUCTS, key=lambda it: (self.future_sells(route, it, step + 1) > 0,\r\n                                                   -view.prices.get(it, 0), it))\r\n        for item in priority:\r\n            avail = max(0, view.shed.get(item, 0) - planned_sells.get(item, 0))\r\n            qty = min(needed, avail)\r\n            if qty <= 0 or view.prices.get(item, 0) < 1:\r\n                continue\r\n            if not self.',
    b'_add_sell(action, item, qty, cfg["max_orders"]):\r\n                continue\r\n            planned_sells[item] = planned_sells.get(item, 0) + qty\r\n            needed -= qty\r\n            if needed <= 0:\r\n                break\r\n\r\n    # ---- layer: clamp_sells ---------------------------------------------------\r\n    @staticmethod\r\n    def _clamp_sells(action, projected):\r\n        """Clamp against a sequential stock upper bound, retaining market slots.\r\n\r\n        Earlier BUY_PRODUCT orders can fund a wheat wash\'s sell leg. Their full\r\n        quantity is an upper bound; the engine enforces actual cash/capacity.\r\n        Removing empty orders would change the later lockstep market races.\r\n        """\r\n        avail = dict(projected)\r\n        kept = []\r\n        for o in action.get("market") or []:\r\n            if o and o[0] == "SELL" and len(o) >= 3:\r\n                have = avail.get(o[1], 0)\r\n                n = min(_int(o[2]), have)\r\n                n = max(0, n)\r\n                avail[o[1]] = have - n\r\n            ',
    b'    kept.append(["SELL", o[1], n])\r\n            else:\r\n                kept.append(o)\r\n                if o and o[0] in ("BUY_PRODUCT", "BUY_ANIMAL") and len(o) >= 3:\r\n                    avail[o[1]] = avail.get(o[1], 0) + max(0, _int(o[2]))\r\n        action["market"] = kept\r\n\r\n    # ---- layer: dead_stock ----------------------------------------------------\r\n    def _dead_stock(self, action, view, projected, route, step):\r\n        """tetsutani dead_stock: stock beyond everything the rest of the route still\r\n        plans to SELL is dead; sell it now when price > 1 (on day 29 everything not\r\n        already in this step\'s orders is dead). Highest value lots first."""\r\n        planned = {}\r\n        for o in action.get("market") or []:\r\n            if o and o[0] == "SELL" and len(o) >= 3:\r\n                planned[o[1]] = planned.get(o[1], 0) + _int(o[2])\r\n        day = step // self.cfg["turns_per_day"]\r\n        extra = []\r\n        for item in PRODUCTS:\r\n            have = projected.get(item, 0) - planned.get(ite',
    b'm, 0)\r\n            if have <= 0:\r\n                continue\r\n            surplus = have if day >= 29 else have - self.future_sells(route, item, step + 1)\r\n            if surplus > 0 and view.prices.get(item, 0) > 1:\r\n                extra.append(["SELL", item, surplus])\r\n        extra.sort(key=lambda o: -view.prices.get(o[1], 0) * o[2])\r\n        action["market"] = (action.get("market") or []) + extra\r\n\r\n    # ---- layer: terminal_liquidation -----------------------------------------\r\n    def _terminal_liquidation(self, action, projected, step):\r\n        """fieldbook _terminal_sale: on the final acting step (>= 718) replace the\r\n        market orders with a SELL of the whole projected shed."""\r\n        if step < LAST_ACT_STEP:\r\n            return\r\n        action["market"] = [["SELL", item, qty] for item, qty in projected.items()\r\n                            if qty > 0 and item in PRODUCTS][: self.cfg["max_orders"]]\r\n\r\n\r\n# --------------------------------------------------------------------------- factory\r\ndef m',
    b'ake_agent(routes, router=None, opponent_plan=None, **settings):\r\n    """Build a Kaggle ``agent(observation, configuration)`` closure that never raises:\r\n    a failure inside a layer falls back to the raw tape action, and a failure even\r\n    before that falls back to PASS (with hands padded when possible)."""\r\n    chassis = Chassis(routes, router, settings, opponent_plan)\r\n\r\n    def agent(observation, configuration=None):\r\n        try:\r\n            return chassis.act(observation, configuration)\r\n        except Exception:\r\n            chassis.diagnostics["entry_fallbacks"] += 1\r\n            try:\r\n                step = _step_of(observation)\r\n                player = _int(_get(observation, "player", 0))\r\n                tape = chassis.routes.get(chassis.players.get(player, {}).get("route"),\r\n                                          next(iter(chassis.routes.values())))\r\n                if 0 <= step < len(tape):\r\n                    return copy.deepcopy(tape[step])\r\n            except Exception:\r\n                ',
    b'pass\r\n            try:\r\n                farms = _get(observation, "farms", []) or []\r\n                hands = _get(farms[_int(_get(observation, "player", 0))], "hands", []) or []\r\n                return {"farmer": ["PASS"], "hands": [["PASS"] for _ in hands], "market": []}\r\n            except Exception:\r\n                return copy.deepcopy(PASS_ACTION)\r\n\r\n    agent.chassis = chassis\r\n    return agent\r\n\r\n\r\nimport base64\r\nimport json\r\nimport zlib\r\n# EXP239 native schedules: Yusuke Hayashi (yhay81), Shop Router 0913.\r\n# https://www.kaggle.com/code/yhay81/shop-router-0913\r\n# Ahmed Berat Ozer: V39 prefix/terminal, shared-action encoding and controllers.\r\n_R108_DATA=json.loads(zlib.decompress(base64.b85decode(\'c-ri}-EL&p(j53My5<F0e<Xd^M=BpR+!6(;<$}j(9DMM2FoS_Tz-QkXes^~_Syg-QjEsoPwX0hP<10~YlC}2Q>nAfZGU9*y@Gt-AzyCk~-+%pYKm42j_&<L5zy9T4|I2^=*Uw-6@Y}mT{`le3-4Flwzx>z#^UJ?|{_?;4%fJ4=|M|av{`x=u@V7tz!#{re{pF`WfBg8v-4CaqkMBPJ_hI|#F8QbJ{g;3G<M`pj?0cX7=iT%(e|`D=<Inkr&VQYJ+WyPG{QUm+;}7l^U;fOyU*G@o?#l=K_;UK;ZWF%!$Ir*(Z(sglG3q~',
    b'F{+y5c^W?q%@!$RV+uNV|@`v7L^ZJO>ujW5Jedfg{U4HO&D6@~8{5kenfBW<OhoAoO`A0tf`Q_1_4||=|*@rFuihRHi?|wWQ&lleR;#cwKoQ{8d{QAX@@5Cd${iHi<mp{Cm_qZ4SI39oc{O`XUKfL@3mdJ9x_y|5f^RFK-e=YgW;_av*JuHWGo>;Jyz^9#uc6#^m`1|s!uhT@P{oj5W$?OxZzkL16=gGF#s`ZGk>tXh}mp7Wv_4Q}wGgN-*akW_!8-M6^{yJ}X_J_>*zy4d?PqWWGpTps~-~Pbl^Ug;knE3PQG95u!P~PW;`HrtUPV@5f)iiGl)6Cv?obJU>uQBi7J+pa#{ps=tFGGdfO#DWFESYb;(b!cO9}r9|xIk_=q2z_W4M2TtVM4Ee+nG>sB@IpJ@}o+BnEbiL7p+)mKFL{xspA}Q{ejo0Z<ysk;kOz$_H|fq|7QG^c>i90_wApYKl1Y7!|}(D|M<7Z-#@<p@c#c;9&wky1b?s+@Par#`J3m+VDW9Z@7|J@$&Y^C?`fK}Tq(zQ%Qt+zN|%AElc8l%U^ct_Nt2HTkL;W^=iL~Si}V~d9x(r$Fq6G=<@#rCGfc35hsJw3X<paX%A>uW4kKrEvhN1gIr*%2`6_GNE<^8T<(n?P;r4auUygS>1>l6UP$D$)dZO@yekJjyVtFY$=d;*KU1RL?<Q<7{6ZZGy;vy5g$kK+V7Xb=#eD-X<-Jhk|1X0!4XszW+vfshk9F4H(VCBOjbQBEt{0D!2_xr!^rT=7>FRP%Ft60;08TLTdXTOIDn&RYsGNc?_-XKPt-2r**k-u+nQ|vHKe+i~lPCbX@`O(%`CiDyjaGRV~_6GrI{I)2J_e3Tjj#GvZWP5z01ilcOY|ELuj73;Y!TPEOoTt%<xRFEktN@l*KdX4It@SwwRf!i&_C@bCfFq-F^Ts)k1HR71?2_qSlImTAWL819=&oCoc|TR&Q+!5z&z8Z>c<oIKJ@~PfMNIQ5VD+x+S7fSt$88Ak7t;xDh;`!<^la9WDTIuCf$B',
    b'@p``Zv?YXw4E|D)K!0VASq77>jHmMCxPJimy?m)Hxu(-Yjl-x!yVYeWz%3cLQ@l<pz_zWr{_lMCv0<kK40K8V_k59)+*8MUJv*6oMFI%jpUg2-Bs6ODsz9A>qmtIO`SPz(u?#W~Jv9lhvkQFU4uCdp&g9_R%A-V(<s#UKC#X~b3K%jGiXAQa(I)ye^Fj6||3LnlB%*ElUOxs?l&d|cirEkSjU2>=F=RRSL5Zr$}Wdy}XrNtcpy$j3dHtPntF8mRpd;1Yb0f%7=EO7{2Mf!9&^!c!2KGXA-N*(aa#)9)|8RU;FS0_^*676(Gd%F5l3m)pjUz??r0Ue6-<+o<yE^%FmdIC6u@Dhq;fvjPZCm-C;4kCdD}E1`x%)w|0t$0-_*Q!|c`b-R&CM)1gR(04hC;!Ly<{#s_N382@9=F2JB{*(LrkB`5<JN@nW`|tk&tZkGWsLvaFFaS`98eD-{6%o(L(|h539P#Bg$uZ;l+3QtTuc1GW@u%;&5~jFM<bIS@as@Ab6`UpKwN4j7FLGh%b`!MBR;Yi*y)Dea0yo4b8?3bCdB!Mjt-7}8Ml1D~S;3FfL4qO$cD<+U-z17z4&nOVVdX^9jTa-lT$2V-xwOSDe>g>o?qDo{&V|}eobHS?9=dBXyyM=C5dwJh_;2btui@*<%jf?bj5z!ZNWc989>AoV2nPhq{GJRdPkb)Tnx7AkpHxHV^|y<e$93wHM^Pcq<qss4gAQ}Nn?0RTcTn>Ifdk=#uvg-QFd%k}3~(^Yn5KiSRFIBu*1fo{AVGpk<=iK;Pv^nCE@1!8ti+Nt_@>{A^H}H_4r2hvlSA}iZNq8z#GRkA#DRj_VaaAU0+raTDL)u&JX)!c4Vt>I1P-3=Ky~<`ce*(5%w*us;&ZT^kjIcVxy4`g@tjSYSkkB(jjlo92Z=G=^@7%(yrr$nx)kdyc~gv$L$=~1maL_0v~`xIoIa6f2@pPi^4}#mArOP4v)?xEn@pQS<-4Pm$8YuS@TSXC!~*?bvvQ8ut5s!5+1+EJHZl#=#3^tA&~',
    b'pm*EK=Wm>R{3#HjU{Ni=A}&-@my~)sf6w36YMN@#F+K@>|x<=QMbS!_{j$ZHPeBZ?FW8#;x&9N!YrZfyLl5Pl#Eb)!wD<P++EG)t@+`#7YF0#TD<J{I-d6CfJB<POcTPMdEVJvQnxY8jkgj!2Kb1A=n#6@1>7_gOpfsQ_thb=ShZ=^d3bYO0|>JC0>pHojIaA>h-)vonUEybph($l*CwgXudpe7s-3E1A?U&Xhhd7=+^@u+A$;=LE5vavhUtzBVsGM4hV)nRsb%ft~p1$f-MzZ5qw(kJ&WC^fpt{BWb-{FJ;3lV@GX857ZPJ2uID9C;pWM;O3YVKPNb}rQ3#D!t*;GcQ{X_P(v34R5-ouZb53UYE4#eFKVJ4oj><MU6CB;0ju}j7oUzg<E>REq=xfRG?LPhG{fB?|R<P0}m7K=@`ez<CX;JEc66TADXswVF>wvMu2hign7@8&<*I!0zgI0}%lIc_=t>EQYF}ps}a<6s(adZ{O?ZX-442zSc&{~+lYKmYtEC@iAI^MS5A`|mTR#{vMl5!k;giHjA(_pG%)~!?DQc0>#h4(3S@hQMDA#_FQSP86!Pkm2mx4(hSo(vhtQ_|2d>UJFxAXKSYTEc{#X^OjHTENb^j(CndPOXUo`u$bl&7?g62)tTr$;XZa{8lml@AA*DZ(ubV0EpvFQsW&}>dn@LjvFa|k?l9*ABTmKKu;P`kJ2r+n@>$XC@zkrcOn~;KLRPk41=`7XIt!2^cCV>40nm&x^&m$d4lSUrfsHuBkXMjJ)AlDGs#w}w6nPJjInGOZ*^Iv^h7a7Lt4941%!_=(~ifA<_ZZ#1=w7xQZ1#J+e)X}Knuu|j%F$tk-uVm=8Oj9n>eh!#8Hb4c-tH+(OjYRM4zW>VgZ!{0A#0gCh?A|Rwm1GMn8Bm*b+==$wjrQ%Z(LZ0YzTa+dauCm6*b_zR5r#IJw%?c}Z*y9|THHOyw24f{M)f3l&T+2CO{8@N!hQMOlz(0Fk3%B2q($vAw)<!K~o(p56HsvTl8qc<e}de8kDoO',
    b'(K3;TC*pitW}A+76`MjpUJI_6A?kp)XfWiW>u4+MYMUwXHctr^s;y^w>XQ1WZd9&8D+&#zHTjxlxFiWPh2a|#&>H!T%C1FzF-@O-}b_&@9!@!e;j2oZCiMXqIN4yX!i~Icqs1&{0+a&6M?hG2mKhY&K&^3qEZ6a>%0#rF5#9})_=h757`1qI!ODisH(n@{VYDvfi8s_AL*Y=svt)~HeE(=Mn2oofS`m|v-yGnaE??#aa6b!B?1}3%jnHz?SP2snEx>O1-!9#{9N8=W<=M&oHTcZ0n-UWzd-*8Pz^U}?q{pI9k-?hhyDJO?}a)YG}xm2$-Q%)CwuJKA2t<g1xjkss43~0^}R3Wr?-^%)*a!Iv0j4~`A{|oJ@T8*Pa~go#$<9T>beGzoO`eBkUzdGC2(X6AE``l^Jm=bdlQJ^8W)S9^L3gNl>rA&X1LtsH(9)*Z<f7PLE*?-^$-#f7pQK0Z*Dx&t$fm*e$Jan4;G_%b0rFbv;2AhUWU9_aMaogASmhySo-lg)aN#xKfdbob6!t9R(gg#nxIXNdOg0*+zQNhkZZ(ZUeZ?@#byKUmdz12Q@C5rL3YvU;l3KxnYj33h4<jN&{F-b@Nb|N`mQW&VOiEXU(}izC2$l@coD88*w40xAQf7X5Q4|541M5>%L2W@N0Qx8Q5B&iXi%jI-(jNaRWYv)Csl@pu?9Uo_`t=hHNv)eD}`)_$_XPSW*BWTBCLSxy(Uk{g%Z%(f;P5PX(sCA6lcMX<<h%-a^+PoGoG_iBX*hv@fE|zJ6rn)V$^&$L>v|DCj1OYzK&7ZZG|F#kufVQAMm7n9a&SZv6A|20w7OA89Nk8Fb}O#!F8;60O00|3Qt{_g`A5N%<W!IMg-{_x${2OnCc_!226(}8L%XrJA}oPPZGJ8XLg^$vZ=~DVNo}}Iap(xI_2el>WhnU+Lga&NFl4=)xik%xISQ!8q5P=U5x;-q#{<`KF64^VV=mF9pB??iuB7`-Pkpmd|jH%o$BtCREX|e>bJ|Eq^S@-u2W~nj^$;KmS~wl',
    b'%74w@K}mkjY@my21eIRSpTRhn$5a#8dZ0cFmD4~Q*$l`k7@KnRB={?^j2`O=(_RrjG&2YNAng<^RgIVuI90^0czJ;>8{<Pv=%A_=m1&4<1;rQ!RVM!@*-etvUoLG0m3+YKNuW|iOac_QDbPlL&`lo}Tl^=VpbFPBNH=KPR^+~%*73Wr7nKZNi6wY*)62!g4g+in;zrm+-+8WeLeD5#=rW4FoomrLyNnCa%*(pw*ThgtDwVi~ll(kcFWgiE1_?I{l2cWxQsM!Keen<hBWpQSgGsR$T}FW{c4Do*-ax_AlLAxv<7<rF_Rmy_Chd7ytxpUz;TetcRC1!F0tl|%_!)7Gxqo|YPuNS2m4nQDq#=tg+dcdpkr(9~()*ysBt6vGxL0E%beIbLoDrpXht+aOt??1-&H28-&=9OoOJVd_I&9kCSRFK|0FjM<S1ypNHaRNOK;M9o57(kbN(dG$2U#7t8Mm?~1;o@7$E@Te5mi9I!vu3-tM1wMVF_sj>?nPp`!d`Vd}hmQTxuG{$}!)&kQw5NYq&rqlT16N33r0k9B2i<Bz*>LH`M3YH7O=qqOXiP-H}Q1)^$m&WvEL0d>7DqLFTz!9-~*-rpnIt8tiUoPioqnIhR<_R0-r{b=orNcM<}FZ&L2Dix-RIWhKR9-<K)-njSSJ?1I&tkkdd5N@I#1&9q#+!ejAGYp{>e##q-UFHZ285osk`^!2X*K&5A;wCtVsdRpG`h2T98fWac>h~d`u8`Uf2Dh9`H+C{n%WfA!EuwNJ^mXB$|d8NT)g8Rnmv4V?{JwMeeAp#-i{FzoXw(`DID)cADiHZf|O0q*Vc2DR+iXagKY&30oTd_FFgI2IaLk=PH02#D0rbv{V0T3H$;Zdg6rq9w<+0Saze5)46S6JxtAy2Vw>s7G!5rAPpY+ge&uIP0v?jX~>*!<XRD`WS)!w)J@)fW%6?(R_Bd2d6ll8~~%l2yyNL<vo}Yh@Jla%4C{D&GLbYyctDE#g$HYkR74hUFr^Loc)C?FG}&a%5-N9?E7q(<KP',
    b'VV2O#30mzWy`f{-q#RAj!%hn$H(Ng9`VM@{Z<C1@T%Z6dIagH6a#aE%(v9ngeBbT+M^`VU8?E3aI>&5ic#73V!0wV9z^|`k7U!tMgU6Q+rgm<Tkt;fIr{VneN&O}{80L&<aNnCH~0rJdt7iuQY!K%8a4w$RY&3l=Ho}X8vw1i)%#-S@v#Oa(&Y1nDvR<21M3n_zYO3USP6vQwp=EhO>g5q-FEF1`dY;)qMcA+eqsO+^bS0~TZn(_s&+{4R_1twSPJ`v(E+^-;=$XXmTy_j{vQAFIxWvS)?Ud36W5Jxc2cNE3NkMJ4aM>l+kOL;yQz(I1Xhr!Q)*Mdq-Q{8+Gfc!+@LtcbU7A<KxLk3{+B{8ID@}!kH_VZJa<I(0DUV-kgNeR*>PiQ;4X7Y*{l@_?AZ&P#1A;nJjNj&a-eUzWx{~hcfOWdasm(b135`U?r)@vrTC1~c;<GP89qeqy8u3jFWKLp_Zv3JoWuW(*f8F+xvli+K&z;=4@s`|bxzz~Ryhk{duf+sT^^pAe`E7LK+F5rWcHAf9LYoZ3QZq7x~1}Mb7`#MU<MO&lHxu2=zoN1LuxuxuiV_7epf@`kfpa7-&(dr@g@DtGp*5I;n*;Gy#u*-mj0TxI<peO*YnSx~wQR=HU126>zCISqG9rBf`V!p6K@H>HR>qNyav7H@JgD!RxK9WX)5{$xH${Q&%Rl+)%*a+<CRqv1{8*L9ZR)7O@FDMNHlF)@8FhW)D5xp*P=dyLSLWFvf{p~m=wTjM;(m^w3<+Vt>cI~4EEHjcqzt$=qc5R1Rag<QdI(D&K)XOK~^G-Yhr5q9!5#9Nl^)yUZ5Br_D*2kelAp--iKq@XN#-Re2F}FDOqBs^)SfYQydIFH5O$&N&Pjx6YRI+>?E$V|#-cZ|@dvxBxr#yV}ithP<iKD6Tc*y;B1@&|}2=fAFP58W_iQ?COY}W;1P^|!+wJsmyj2yDVnb<L(%iF$S8aZHrc~ph3k)nIY0UpO{{EXF_7z%71Wwvos_{CIHV}L?zY6vt7-O!Mb=)Z',
    b'zcS-bop@HKWeU1eXn864TTjSP@z6{kb;V%myvml_TF`y>bN431SZ;2Fn$F7+i|NsWs&m^ymny<~0~tvr_SKXAo$s?2@?tp^3g%+hJY%*rR)esqlnewh2R=1c2B&#|zYf?}iBARY@4Q!#)+zhd%mkE6V8&0&rC!7OvLJHW}Ps;tQvrYY8hX%YqLc_*o`mJ~i1*>OkFZ>G`444k_QU5SBZJQcIiMYIk(b4d+g-HJm=-Z$BmJ-q{FDo?R0x=hGXy2hZ>2m#-eSr!s25thbx+hc+y<FrRMRvjv|Oi>fz3f;7|yi{zc2;mVtDup_zD1D5Om4UK<_Z>`tgWv?we<=lpx$JL>!x;sr<$Vqj0z=aegG{WHOGF!HgWB}{UE&N!^DAP(CKX<lQXarDOs+@#M@fP?|H>7m55rsbixh|&-xb1$n+Rm-^8!O=25xpO_!f0$$>TLv9;J&|-teqknU%|=|9Ocb!rm8+F9>lH0}0`)?x>gy4ORRlE!uiWg02s_kG}R4m{-xSaH~Qz&1XzWH-}cisVcl&m;zBnAt}-lTdaXn%`HFKVYN>l*A*?yIPIhG!l_rru%4rA*=FWgBBg`e6{3XJq$ty(I76rk5&fwwm?qIq#<ixvI)EGGk;eI0O)d-Fl5$FuHhTu(H{>kEKIAg+KGclc@z#}jz$0U;pOvvLk#Typx6A^K-D$MW2w2x`*=TAAuscFH2mv+th%^*4(L;i^xu~gbF$G{~nfZ2?kW?yy7rG4svw*;vf%KgiV&usTACkx1gwm|x7A&0XASoy7Q;h`TSero2IudcXk1V52MNX<FN1F3&pvQv^o7{)@Q*L6M<`2B)!cIKDdoCU+J=ZE1rZTv8Lo$|<((t;&?g7a`<Obgrjgg)U22%oJ47|=j@~Vsc$r4%t^S{#)AiQBIhHZx2ho$cXg#=rfd2IFx=onK2L`r<<g*mFHrB0$aw|1L9^)?<6!&Y&yS@&aE&2ZNcYk?N`63nA)+5gcR2y2a&>KN%WHN`U+PekT+;2NuydYejlU=!&ND@4i!F$',
    b'RrWPP-QC(aS=PR`f(#+&q;;vFkD(g5@PF_n_OuoRMzIwk4kJA08r61U8WtNO6~K_w2l@xJQ!UB9X74nt(^c^@nV%rc!VLP$t+zfi#^V>vI^zF=3o-pQDh85IWxJYp8J#rkgmVmz$bQ3bTk}w9&afL9KY?S$BYI3!!5}6Nni?NM2g5LJ5QQc~Wdc1Nl3{7?L&n4XHWg01}YS<~eY5rg9O6r$V!d<Yz@X24v(1(RYKl&NM-pMWU(M(S~C~0_J4*4>!;5sl^^lCl9<cgt;kFDb0=hP1RU3AREe?QJzxdKuRE`TH<$VzLT6YO~H3feG@|&6*7d4rfsVBLyNx+mnOv~hBaHs9tG^oqm(y=F>a!YCCV?HG0M_+-bk3W)q-O*laS6O6Sryf)=96MSw8b1=kqJvM3Xw%DzUs^-$*SoN?1RQ@v5vUc{QDf=`pMRmjDMrg(f-LiOZDOOr<uqV8ft(546lXdnOaK?pdmUl7#?kqU+e3u-0@&+^3fek^e7r*gsWzTeAg3^w|A0E8D6D9l>h-&nPTXs=;XL^RI5+z-G}2l>&?Os<#tHcEJ@iK&LRTHo{5J88er3!VO>o=8<Ux>3~WXXCfrM^8<?A)!SfG`5mI~*6nJ_`w!LmMs}b#`-Uh4F9R7BEruWGlCC5{PgO+kyo20tZmDxJ@Ld__(4xn6^8>TprGP@Og}{6fN8lLDtT!XNi)_AK{|oo+;PMX!U7;o7@i^qA8qX_w%g~dum&e33L9}+~IvRqzTpE?K0U>R%;(Ro<k!k=;MU<o2G4Ca=wEA0@<>)r0!wF{w215$<?x)cl1qE3Bx(eA=sQMTfKrJ74*ZO+7680;Jfb+25y3tt?-YVp4j;O;K=->>TUBJ`_I$rWR_yaIla`ceH^t9IUuk$RZCrE``w4#Rt+q)P84r2T04!`K&sguW7@ZS`jM=*}d&wLORkixKYAkEnZ1sP^Wf8+~^<Zzu!Nm8nF2e-;Oe=2$p89OvC@}9^qLr#dKD-obw-=xBC4dd=2sgUKW$~B_t<Dk8NbEYhR`',
    b'mfKWuT)nT#5Hmuo<?F<O*kHJuVhq|r)sW1hT?ahTIxKj><M_|s@e$CKt8d2L9}MgfQ4uf`wEyKU6CvX5`3%VK*-ceN+PsXS&>DIdTYi@)q}+%SZpQx-=t*3E<C^l@(C?(o<Aks1l!}*?FkX>Ck3>DUJ*1rBV1VW7i>$9ZDCBQIFi|2Vv5h8f2dtcJ(4xUAxfRXBv1q7)pCuW&g;NkBICu5dMlnn#3e4$hT*i$WjJ^FH9{D(od)%+*#H)_nOH|&Gvr+L{7|F~R%)f^IGkZSR!F?!B4-MoOfPRW^rN&hd}$kA;x<FZat9Dk8G2}?f*%~l6O(!%b$vl_gQVT87F_^cHx-H=Q_DyPgR0mWV?c(a`%e{$xu<4hX{mRjA&6ke+eI2!X*G2@joUttLU22dhTU$bd~9i5(}<MPo#kAgljrP=8El4#p)n|MlGB69r#Dk}Db<B~ZFA2-&y7W}JFq1M*KZ)LF}8rY$I`qrE5ht}*3U!NktX!kG=w{X9HppE=TG}@?#ZGAM_rxHU6O3`)&3PFc;Zi{Fb5ry91Q~#%CWF4ED{xv0`-kcERox02Bnc-ZFJ|96w|pwhl&9VImDItAfzG$LM1>5rNo-1R>jGkCtDCtnj(|_ocS6UsCP2u3_Rb01a+#ZZ#c5vHo4!Jaq+xuuP<(igF?2r%FjPl;6Sa;RzCjGFo;v{!q1Pb#ZP7SPIW6N`gTlbgjE~PEdx$9`?TMG87J4W$?Hr+o(2Kbg{+BH1KYZCoJd|GU5>n-yh~C>#%MR|l}quKyM8`I%B(SyGi@{NRlTWLPL@l(<u;luJ+0-$IrG164?){~JW3P9prTHvjAZyFR3Z9NJ#ylRkPMZbgho<W=*!K|#$^U<9ye46ofaS&bHH(;liHY~CF6WXod1irnEB4>8FyhW1z^Dlzn#gkF%;5sNt>kE+5JUuvxJW*(nU=*RxIQW^l^SSw<cZ?7K@M^*(UJTm9LVp6PUZJQF`lBE*zh7-qZuMIK!0qDl8ko)5Skxo0+g4<V+ysl#>L6sBGH4OqTSn',
    b'(l(9XhSXgG(U<|YwAvq;N_;j@UtXhT55ptV2^5dtz93&~PtV-z>h;y%x6?&cxTiH8;+C*#z6kg)^Qm6`-?B6kzi{PMf>^pIRHyKoOi~jC;^d1On8T76)*W?nTYUOe5EIaJ^5NPCXF3u<+R`_gmRul8%FKP1Yo|s$KwIZ9qq7MNn<u!1EL>X_xN?zNqs4Ygq#1JhOrQVi?=R>8Jnzo8ACQy)Zl9)>EzFkn8%akm(y=+k?#Y#_5Cm48Bs$(w)DmXCHj-Wgyud6flS!@_1LvqQq4BN<;5$^kN>V-6UkTPjWwDl2D<19(s2}EO%{OvKMSro88-dEUxJBu;!>bDTiZFYM?gQBOOi3uTPV8)W6$v2lqy%Dt1+e@U!4ESw`(_22$_iiU7fLN)^6!iZ$fiifEom&9Jf{;DFc87v?k{*vcn`?ooK1lkE&v2_f9Gy1n-0gt@Z*Bq1xEkl0BZK=wJkxP0r;~J8^wY^RAxOqBF%!2(S<>GO(qurO|p*aM5hH)Q~=UyAbW*mw|Bb<UF7q#l-IgS31YJzw3;i>&@zbT&glLGES-u*=6Ts{Y6WS@R82O>TL4vnSOr*>F8f$-YK}P+u_vL8PddNdRwf)d#gIN}84@qSfaiXto=SEA^Imb21}+7J9wmv)#2bh(Wz;_bm#VnK>I;R=brB>CDfV>zbMZfr3lKE_Et|WqwJ-}rUYVo;>_S!?p3)_6-UUwcA<L++?MnQ)xa82Yl{WRAR9K^JGBnX<I~_b-<EbiHq-M>~9FzTH9PcoqEBpFb=d4N0d<^H~)?M))z>B;ztH)2IFWM2rozlCgqexJ4BWWXu&!@s{`Al1&WNM>ElYzKG(&41{5Fzp7ZqBf=&)>fedUVbr1cAnAds&&mlR{8RFwGKC*)b{)xqylFj}7?T<t-NuPQjQZCkJUc9UX&rIAb<GWJ1Q2B{R3`hP6$BYo+yAuK{NC3H*v6AX!kCGFzU-kaS=SpOT~k80YfJsSg;4T+%cwuaT1!Z(J)GwJ`?(DB&u3+>(*J-SkW8sAF{M^M&j',
    b'l&<`G*>l7hu#$v{eXoPCj)~!bVfCS$Gp&fC$Hu^%IM=20Yb+^?HaPqK*O(;-4mevC^&Shm}+pOIOt_spBP*q#pC8kz#i<l%ehmPe=GM32%5rHNM<w2-SMp|S{CGS{}&Q@?VYSfpBZRIH}d$Q85JPTUL&_iJcst1$>S|!OfBBV{Hte~Yv@fZLZyNg@}COlW-68%Re{qOXIFze=%b$k%%;{>NbL6SR}ys^zs(bd0G*Pqu6(&T-iUJr7{Io)6E_rXEaX%<@*)+j2r+$EN@n#1L?V%F_!@_lYV5)YF|nd^HXL8`XOWV+4~Tab-S((R}DZqSg*#bx%cn_s4FLVL}h=~H!ht)CND%051f;me)g#sqYYtcy9aUaR*hZ$a^Dn-U=rGKk+v63vj@uc}%f6GU48r{W*O=oITnua$g3Q2;ZchcjR3ith63pY(<FUF!oRf?8s(GfuQH_jarkUGo_0^>2zrEKggHDL-dz2BPYJ*WH7FJ`|5!tdAAlpt*McDtx<AH)7mirTxjTwX5}?*myy8M0$964nUntjA)6lEi?(IJ(4@kv{s-O(5}P!SG_BZb88AE2QK1A%$7<WOpNv8&Qv_$v_z5fYANTA8zF(VI|9H-tpx!`0h*0@X=$i34(7s%9#o-m-gARh9N!~jAxR4s+Ro?uojp9!#-L0`==CRe*ippcXr7Ele-J%^4xvyyb|^i%yX84@ke;8XQ*$|t@9*yi8m7H|Fs<QIuNZ$BcX)sEZ?b1x=DUZACC_(p<RK;FG>bnJOp)2?{tAa|?kTa2)Iw9-<Y8<omI_YsQDFx(<M_oXte=+HUNI-XFcFIYCV0fe4n2X#{(t-P{fD3a@#D+eHRZWGONqsJ5t4cGO0xz|;A$n1P$$-RH(K3^Uf3XL%Q68)-mYZ|IGV{{Cng;LveV+<`b1vd2u*LHLO6v8uw1s!tb3MBG)b=sm|1((Ua79tJY1kjw2hN8m5nfY29iCsRtJhXV|ID|q!}ZaOBTtZT%|kkWMm!SzNm@&usWyo{2f$)qKk47xLv@m7`O',
    b'4#Cnfw#>uV|AV=H9CC-DMP=I%``&kXEYDYND0U;Q~4S;25vd)Q>jENK(qvL)?D;cJeOI%%g4$%RRyNqzVrfM)Z`4ZmSFpE{wh^aORqsi<nb_*C~f3f;mtT3-Mem>~*ZpAMzQ;0mPf=YyN3-xk1~>*A~YT8ck^CfHEmDV6wOCZw^-ZcFQJ(}lUqbX?;XvtLZ+puGR)@X^zf`SARFsLFYrhjK+WL$iMT;_}1rPYrHD0w7)2R7)@Tjf!))IWTmx+8vLD7}60{xkSZVikULgq)w;NsMRrre5jU_C4ap@vI=jC#SCt2ampG1Y8*FSAz~~7K{}qF#?{<k(iO}625^tcp$V`AN8u-ujo8c#UC*KwepC2SRib(My0*}0VG6z|7Qm!G)L&@&&05Rpj;jWhY+yk!VTXo1ssg{dP-Ut~(=y|B)qM{iDaDsoz&F-`W7xpL&AF=u{bJ1#ry*G;(nZxr&?K=0ySD|~IEHnzZFE_Xz~_!prGgwJ)yI1<l!M;};PcmT=7I3roOgM7aJsPe;nnPQj5abCqWOT$1#KUWxxk`DwUL<46Pb@Mzr6eS5`K8P_Me|-5$^HB%daF=5cTIbBunHVrTmyyE|CNpo8M)&IiC3&GrXL*e}XAyD);|;e(&###OZjf%UgI^ojDJP;0I*NjjF>x+mwT3Ac|}PoU=Z`3xw(acQZXNNxuFv>X&EN8Tb<^P|bkA`4^0r>CFFkICFY$AUDlo=fS+Zw4C($cgN;DTlMH+NvX5+oCg0jmJ0A62lkDQ<{yRb0OPFNmFP2@Y**}#uC6y~F%28xW@YKpWP!Tsw~Nf3#pl)#fI|7ayNUp+blKmY#i-2dz?uihfgv-dCTzW4+@2&oL&S|fTT7Wrr_n^)gn9*y@?PfW{z3A|Og$`SCV9_t?s;`&`kuc0%ZGQr{&Xp#y!-Y2FYmryGcTvf&*0jMV=sDs*<b!%f>MyXA@d#qPovL}SB~lDzy0ygoM;&D_V{b4F21SpQ1cK1%J4DjUb)=}jOHAmzrCc~Qf&III5CUwG;Z=RG(',
    b'Z4GJ(0_f&Pf0%s`8Xi;E`+<9_O{lq?%Odb)ko}JfT>JDEhngIU6$P&H0qWVd0wBqZX|ybOGoa5Er~JXZ0UHACJFzUPPAc1=Y}~{=DYl5;;c0OOSRil$sW-;2_maT*cAEibWb~iM{%k`tan+s>vrLe}orfSWIG4#p2_t29V1#f8B2dzaCp}wVGAqxdUNMfQM2ky5g9(_Gv^dM<eO*I>~Z{cnpWG!%d_JX8Qr1^{ILcYlU0LFciB*DHDjP-Xw8_(If7eNP-QNdxw;CAc8%D<M&)Namhu(7W0+1<c`l^x8uIX^vMzY>x*NmNOp$OdTm~K37l&}<crrie{UljGbWjFsibQu-yhv&rTozxK}qfV;b)<7dPwpcvzG!_%S-D(F2WP4E__{?3C2UG<%8BMNx9#~@*#QzjThusP4GLcgU&b5Gqg0Z9l^b=?xtO}K4JOhQIBEocBt9k5ogNct;EZ)8!Iak8qbvRv53=Iv2~Lox(A66NIVzR6IGRI=VT;il4nsg3+s4Wvf7wOr%Uq;Cq9*9gzs%zFp{OQy)}1|mR_@fb+v(xWPy9L(zJt(p{wcmP8hqrfRHA@T(i6r#HNaAV5$rGCRv;^nmxHH(!4{)9jddEsiK1a7?sJs*}B#sIu)-eVS~sn_LxSkj6mL>xwsSo>mUmSMYmDom=v9%EKE%|dUN5s<O~I6#4#&@%x+QAiZI#BjOhk|L3PF)*Hlb+PSz_`<$alvfa8n(B{4tL<Gj&5q}CWXjcU{8r*2rEG?eJK_FT#QyB>L>0qt&(;rc_|jWi=aY|=W*w-c>{SuwHflPYyyS5vJBOInwJhs!Z_Aa=$kKRQ+~fvVZ%P>68-Dub}S-fjOR6YJ;}JZ1L!4Urfo1V!hAr;sJ(b+WFg3UnwO!#WU=*)e#c#QVKt8djSapSaqA+zrjPVkv-;RC1%9oKG}iJw8eA_HmO0Z?Ulop2#j!pggx8h$k$Ln6+1yWszlw>x$Esu*S$2z5s*2CumDIU4;H<m~;T7aKwTlBaN!8t)~Fb*SWut7LoXx%',
    b'RY^_#WtWSe`Vv`ZADqUz8}0b*eba*N}yNX&T^LF(tmWIZmhV@SSax(&E}}m$Z#6RhpD^rrhk0jm^C=cy|L`ChF*<a@-J6NZ4hzE7fc9=^AJ~{1=of%khVfh^<j|8?0&`QyQkyIp{x59i1%Ib`mV|pkF*QuA+w7srEc0j0mIn^5$A%%JYnT3C0fRM1vcTfTV`j3rc`j8Ayz}^lt|bWFSMn&?2~{>-nz68VDd4!;r#O%6XWvtAWaH<DpjLGzkt4X{R9LxS`B({e{Qq$Tx-J+xbz_U4T2^rM9p{*LO^*K;eu0gFn_`1YG<9F#={;~cvgqyC3(`^&r4>5aT>ym4o<^>YbWh3LJuAr578!(-7AY(E_*m~BbLn3gzs@;O^Mo*M57AiFN+hGy(!8$P1^~PlrV9$YDQaeQLxeS*v#HV?G{`{FeGGq(2E>}>M1!(J(^*|w=ws8B=N?V4Z3~u1%+8d_T@&qlUvUN5{Q^*yN0+FV4P3W8qDq}e@z6m*-pljo!L`2S^1^nIFOU7wIn9=`EdI^wMyu-Knm6T>JD&lbbCVU+YB@j^Pt$lEY?7TR2$K&tVIcDSG+$-2c#$sh*v?NCo32v|KyMYplME{LGQ^5bj(eqbWNxb;9Nw7+EuB#^ZWUrIJ>fWOVk<`zo&Q+<f^CsVnbb}h0<m6SlmgAd2+Dr;jh1{X8y>|$w9`onhC+)QM*{JS8lj^Y&CUO))jzbYa2u^QE8o|Bdf|LBDi(Mh?$A#Co|;rP*>b#)a96<Q`#jyzrM;DIcx9XiuGVzks{~%eXcTYFvydUJgC8vU#0V3Q8^9=QXy|(eZ6o_>df=O9n8)v)=^;Rsdf)Q37Ap{fIL#b_?PzyT91RwLZ~C*083y@YWwCIF6D%qMCr8RuelYrwWQIU)BtUX+XCsUVOK(|eXZvK1CT$qjM(ioEO_9pZ7^bXH>q7sTn>zqOoXrylxi8NxsaC}CwTD|rG;Th#*aH(ihW%zb+1SV)IL7_LE77gL8o1%BTJD=p~pMtIk*G0(Pvu}8qLPzw__-o',
    b'l%5jvr;fVGVaDs9+FGEyw58L>H6WNPI8}p485gTinNdtZI})m`F11l|*Ikt?)Aw9xtt4$D)D)1S5ji)y*ONpWAcPxu+Fia5AKa99<Uw#^&h53MtPw#nc4J{Tu{^PiiYm<$b`>%Rni|}v!KN)tbHGa&v-kkQ?sd)RIoxJfvN5%lCcMQ9S3w3i^v(5kn{?04Nbw6a=nUm?TS_zD_Cq&IEQaf*J|fzI7A)<_K&2YLO`|Ey!3eNKsz>$qkBM{?n8j9@UYkj8nNhQ1&}(&c-2%iWsL8}w;AYI-Z@=#7vPU?6PW&(oVGNmi&PXady4>0N7wW2#5@+O5*eT39f#Mg244oxYS;RV8?oQ`iGE=YkhAlqAwHPRcqQsdgq`5sH5IJ~eMl6<Jy#@@HL7=!9DL+KCag0XJCftG00vONY28H>IY#09+42nk22L_Im(kp5ea)gF{2TLWg`4|9?rMmWX-c!~D4HVuc$h-0KMh9TQ1fh>uHhq^m*PQ>B2BgVUO$G#VtwN^_O`6I09ZgKIoc-}w4UIC>%~k?B6b$bL0U05Gpq>`FoxLLNfz|Q7L=?MFAjK$8+$(;3O}3J1aHROy5lh6=H9N3EmDN#MbVU!zUys67o!6Vu9$b<7LNwmN{Ls33m1HuCUenCWw+U+|vT8*Zz<-#A<@NILmseS&AdR7sl0nU3<048!*rJS-0u-zxZ9k2zL&8v^y-!yyanuVpR|y?p!bHqufc7%w*k4U(MBQdSI|pRoYjYgHhjEqJ>gXLDz^$o&wdLM@hgy2jS>Y@RPObNxupUc2IyFCEK?e!CIC=KFuYI#SF<Guu^jrdOC^nYOmOphP)=DGR_<*1QV8#&rxfp$w#x%X>#kdqvM_&IaP~TTWnVjF#=P1M^ti)29TNZtSTOEh*#0kZXuC&x{mo$*gu|?7+Rx9xfqEjLUPU-I_hj39_-LQK&%Q}V&Zi_c?be9IyG;kOuB{n$49LS+7@gNsWo`K{K7C1GX&!67^JwIEWg-p0n<%pb7e9<<WJ_upQ_E$II;}P`u*+Gx',
    b'Rvg&AB8LpW%hz9a@;BL(IhvDkKxVC5Vw+}m0Okv;TAJO#pWNZK)c(w8<8c#=ZS~n!g!KtC8&{?s(PU7ed7fui`sSw6v`ts?jSpl8p{k<g2Agc5o6DB3PGVS<!N}fR3oY)536m`CBhYn}VX|)?cwxCQ6cJeLezguZRh=@bU!C0Q|4k_%~91B^`9wP%q7)B$N8ckVEjvp)pbY;+Y-v&+|Dw=xN8ntMnArulWwDtOja;3Ri7R`%yj-NM3Q5mysu>bdz$&p%Oq3RsvD(yW);`qnMN=zhAOzO^EnO));Q%D<Cu`#B;Ab``66`xznrjo>~X^?ib6JUBt#u$m9bfx>gV=71cC3tk?m&8p2n444onKo^K<2q0HF)}&{HDUT^6}bvhCY-`dsq$vKBzrXrmS#IR-ld@fDKutBeMD;gex~KxULjerBnzmZbA?&aI=3QgUVgX`%LdOg-6L>oEQe0YJ+X&4QtMA-EMB`5?f9q=5u}45A)74Fo~Lxv6XcO3@3f|N*Tf=?=2syy)AX_`XTI|S$l=q98$)bQ1n4N|nu_quqO@~=f1mNi=l?WQskx7LY20X>6d#pErysBpFcR(l>L_a6Y+A$?C!o~1u%NhFQ52Yphq&0)4JU=Qm5|P1q(i0jRkN-;w`u1aJiAYi!RLU@vmx%XM!(fJnF6T%J3&vL%fh?g`+b}JCkz0iD3EV2n`459<WZkO5|uEi97`yl_7>ME>Ake0dLl$5-u~ym{QFD%_Q#iB-hF)03*s*5F3-RF^%X3C^0hWM|K=r+rAu>6t%x!SFGWZ7;x0dE`iJEYI?Df>92?)*_seCo!JiYjN`4U7M2F3Km^X-qnZg6v%&y6Z&!>xFDZ$&WpaLZE8Je_I$oG~V*Tl%FDx-58v%$ciuP@h6r`giK>6SDUa7`@BNq$4qw2n*xDysOcM{|%xpo5MMA`vgZ6G~R-iHMF6d5v$gu^Q`Et*8z~%_U<pCh<vm=!q`J`65C|sny^#st`l(4ZHKoZ<DKZc~~kk%tO%`j{zUm#Yjd)XQ<#DWBo',
    b'#ijI@ig5dN_y1XgL9=UR3W#1_7a{_2NkHzSYJqqK?z)9TyCslJgN!izezIb0pSbpOk{ZuB-`6@R8|%%9I_c_gic@fvOdQR%Ial7Uya3|NILyWKqPuv}<rO{FK+QO*uaw1*8$9gydewg~`d?8<iqXgy4a2DWv~5Gg+4-4OAUXD8Lurxiq39Zxg5(lRG^cIRyj?1y)1DqN>&-Tn#(m6UuABdwM$%ECZbvqModojNE~#+d-$w^Vhiy_>L6k*QLJ5wYeRKdtFKMg(s>)n#xr8S)lZcV=YLFb-O+rad;=jGa_q#4t5bv)QLZV=4?Rr(-b&M>jn+o7+kqe}d@WkZ!2AwUGh~r@a)({D~4yw?hTc+5E;Cx}+<~KU@@vkWmB`p*xgV$H!_9?4S;gLc_QyPFT_@aKz%B?NI;miDF%-!MoiAxn748HRUQex@8xE$dp?wcaX$h>VtIem9&;dwFnFCdElm3On1Q1A^i50#;@Q*D$C@1kq>I$po3T~X$XBK)dou}ka0@OD}~?wKCO?4XUoKPlQ9yK?;k2>50MpjTSW67r8TJ4xeplYNOg25DvMpWdy;|1scpj53$C2e5V?4ax5F9|NWc?rYThn=k!?Zc`MP;G1@75`v~nk&^MHTUk-}P2;OdPjbaR4BL|%mq)S&{v<bPoWX4^WFSlm1uhf~y+s%B6nAAGPoK89EsoKL-4D#luLh<V>z(iv~Jm1mbisS1~a3<b!7&>n<l7ZHh0@xr8+A(kY##3f)&!N=!_$GSqjst&<bDzMgrCEF{xEbU||nN@(W@_?~a%aa>Hc^|N3Ki@K@4C3<Ok<0(^snyZ0GJbAqLlu<ydntjW<OGU=I@|z~{9}tZ8Sqj(LWR6tEP^Y!D&KJVsBeH~IXAV3aJVj_;g(z%lK7}j$P8*^&f=_m2J)=3UBH}4|EYTvX*ncya^=u$Q=5;uI-OK2`Ke**QODGsrgk<#iem&aYI*yBdUh^ePHPU4f^bi;8IEZ%JA?#l!5GY=$?OCx->haNYs-?k2MXxbJ1o9FYCzMWH#',
    b'b0p&Q=Bsp7&gzavM3*cQ3i6B9EdehJhM|&_>y*tF7cIY4JzcJ?CLq5%y8Tnf^ONhKw)l3ojGDd@>+ULw35Tn{`|8Me8(J5`I)jU_>cnAeLer+_Y}GIFAu<j6s-ln>bjyUGclkNPcY_i&AzYH6|<ce-OMkWrw04P)CF%+SgIJEd#>wu>Qu4k6d-mB;vJCo=7O92ss0Qjkm*4@yXSO4rki$K@r*t#v0F63V7sjR;EBm(O|28Oq?y04^WCV1_a7X4M}LbS#6pWK39NF(Y6t(Bo+nnY&rdm*ir9@QK%6?Oc_IzDcmQub-DRi;x7su3CF<H;TqV0Vr|-07Jj-}DqnQAX~W~_;w8=Eo|nelqPMiBX`#?AUgYX>E5M%AfGp&|gSyZ#a4Q4St(uXPhjdsaxM8Qtwd3;le6Lx4i=MfeY(n8{8}uUu3*C^h@GCu>-)=L|zET=xMr)P{tP5(bbaKa5s*25l$bRAt>;U@RK6daD&Pgy>53=;7%QWh2+3GcqTD<bC*KJrOjJ1O1FVV;ZtK9BWrw)YO{z;{nNqXD3qyg&*m0MLdxIKq_bPKe#^a&3U*=?zrkGHRDJeEA=Mn6-zC2;^Y_6Pp3xd{|A%LDh{M{x?Ah!SndA>Ai;$V0?B))Os|B$@zQzhMYk%LhQr`h50nnf&rl4X(82xa64+BLJlQ<x*--1Jv`SPn;K>bYj%ihxGI(f}L*3jPMn3zWQ}Wj8(A?>Tk+*C`FF&K0$;*yTXh|fuLJMHkzSI7Ef@@go8@|(CtuwLY8aHkcGG#R&ugzK?@m+Qgw4Ny<I`IMhP`f@eiEXQmmWX9^X#ci6tQ5lH!b<P9Kd#%so6nAZ6-y;8oReIi$GT8H9S->>46tnGLK~RXT9Jo&P#ozVY25^Q<LbhUGhBIS8(BXmO}Sog8cq>CGMUsg8yNbLA?~p$n`7$bTzPC|i^RQ&>)#2+WhmCue)-TR`C`>RJ(o5(JN<*hKRfb9N1WB7xmme&GdIYJcYlvL6u?W|9X~#PE&wr9q>|8q=*M|0F76CexDLB',
    b'XJ3OYprA?P!_o|Cr-6xmOJ-pGx|4k?_6|ZLXef>DVBoPCM6&me@vlOhhc1LT69l3KXcV7%G5|f9ZxeU9$u7)6BT2!rbV7Sy1>(Or8qAG7N!@|@*30}6ZqOGxLEY2q9rMU5f=INJW7BvYQmo05{j2)0GiK%*W7jf5<xfxzAZ=tE|eB2o=MXI-VKUXdk;a^4JBWfUP{(j*69~$YV?V;(^uY}yYD?0YT$1#!DU`23oIsXmd!1RoY{Mp6=b@CY+!{PLzH8|iSN3os!S#D_#^LL7L3iKv%f6J`>j{K5lU3|le?HFl}?>dT6p_2$tKQLOUw&h40sxzI!{wK%r@6FLx;fRI7K!d+az>PU(CX;FVr?JT`%cnsMGg}DqSEE*=yT_LB-hv7xuyoGH@s-@BZXahBRTa`c$}aFe`w;hzC84C#bNv<La>w5udf|A##VC;yG>$S19JD{JJt3hcUlSh9in|R9M8!X@#MVsW6ApqWzs<BE^NRwfP?CLuKDNSY5n}^{m&=!>#B@lPCj$H%wuiBSxO)dhkptJ!6g$ZA$I<RM5*((h#h!HTBi0b5G=jGYG5CS#n$qI&lf<$=9Y)N%LDz<B7W*s<=_5Q(TVybnHdz;fCfm^(uen%@ryCuPH9Un#7BW)uOEI^eIuLUXg4vMPXmE+T=7a&e(aOY@DZq_{lQhlRDcm|B17LNCfB4s4sf@rl{L`LBE(~xG=A+KZhjH#0}#46~K5s69ul18^~P553cW+mVZG-aQq=f4U7m5zAR3YLAwa(0CAabX$aMOV-!2GUEM%Hl`ck>PVBwe&R4!^ZWvLU4*9Tnb(}vc;ux(~kfGlAi<7zwpy@M;XXBGSfJs7}B}#wC<Y)w)b?3Q37)~E)hW;&U5GSwe{^W?>NJmY?VtrJXW!xAoQm@bNT4j^bO9UJVqU@UL=B*r}7_)?qSaHQvf*Xf4yHB=C*Y+YWd&x6`<B-f7&AEeu)?Zx>s6d4lIz^_DVma)&X%pRHWwH~g+VGU$HwdhB*B#bCr%V|dPqzugU8do=XF2G(',
    b'osT}x>1gmVP3h@nJiFc#Sl+f>pEdr9Qe33oVMv<EMInx<CeWdpYT%o8GdVBhpbc>}?6`~SsUT1^wUq#PRk<>{1%9%gYmNzWNki1a!dT=}Z7M>QG*)1+KNz0@v8*Dh#uTY2#KoT4Ry@6?i4{{&Q*)QMh!8ZSzA9V+Cm+A^(_KMd@@Se99=@};)%r3E`g9r6MnY4Q2+QZvq{$0@OnA*i+Ryd*yY9(KKbz5~Rik!JHmX?WF(IW=H>ORjgL${#6PO?WDrPXqS-PFk$-c#@Ri+m_bm#>xz65EAa{%jMlbsTj<5PalXZ$l=xE@3~?6itO8C@q-)w<N#mp*{!|AcQ^St?;ii&sir;T8FNmTf%WZ6j&eJpS>q#PTX`-KOk6!k>dw92F6C)I1sKxK`Bn0IcioqP~o|I_9VyFXv&c+rgKiTM#T<VlX?EdM|(BkbaYR#IY;tEA*A2Bs<XYy8O<*fsIYa-4jk)OaUj{a0^;`-ZPS9Mg*mgGQW0rswq9oskv^k@Gm@htdr;d>mhUzy9O0V*C5vKQ)p7%iP$2md(^plhEYdl{Xd@8N_PTEBm$v^848j+iPgdNN#w}alR5%dBQTsk>LxC}vwQ&~&)=$8`77kZ>*y854PRRRSr^D;flca`Kyl63;ucLNzUOcrO{J2H3T({xKH+YRE%v6(`XWhusQC7nSXRCvs11!wgDl$P@*_&@B@n#(X$tj>GlTk;%nEIDgCPbNP)d>Ym6(zM^b4Yk5N2ciJiEkhY(S#90}gMS9;S*!glXO+YP0Sm<%nSEme=ZQ;20AWc1Feq%arZ;0xE}&0e+{S@+h7YuS%Wx>`l9p;U&Kg`E|Fx8Z=5-iJGQ?)(t#w<h#F*iR;(zR`fy{CTG3q4usWv4qL~>GGLRve2$z`??C`cB1E1hO^-K)hR|X^Y7aKz>dbolbVuYAnz>xgdkTF9wr$tk%_Hmdj^S`_g!6-ny_D3<4~}&CU9E|@Mwe#{Nhuyl)Snxyq~v+Fm3Ac&onA-vw?E&1_~{=%zP!1_#&-+#qCD0TUGgi',
    b'wGS(W`Ft6cM?b1_Evh6};B%7-Wlb9AG0Vh04!U)abwvNB#0fYF{*KuWyn2Uci==DWd`t?;lC2<VR9b38td1^oKZ06ns{9_uFaoJIn6`_`--iDMz=Y~)XCkCY|uC;DhL_D&RCJFx6H9Korh%X7wE^2e2i^X%tkmVYxO6dw4*4(57-Iq4dSPm*oZ|zF0ntjLcUjp%*;M%Ef8q!#}q=<BVAp~ARHCzNgOtMbx#LYaF1gF{qRBA7kn^&ieTY`EiUnv+B`+F5ha7saU@43RgC75@Z3C1LOTvBn(X$&}A&npdYSw1kRjH4x5@_lg-VjS`o?!FCC``C~Ho43_`Qsz<jeNBG}QUE7p6c??T{c7^SWPuMq5ggU46QJo!cllxXr+gB9?MYatl(dFx+s9qPo$1BPJmuEtabvY7gDNj<tl5oL-S(=xfewwI`xaqzAstr}E)=1L7^tEO$vOjdLgq|s_=!~bRLErL6-X<9B5wk5QyLBRSD=2iaF}3lvoh)LGkC1zA?-%u<<|~=)1ye>J_j$1qlp6#-8ygE4%YZ8WNXhL*bs}{wAn(0a7yPRX~AM4CaXG<h2Z%Onz%m2b3}AaW|_mQk85>^FwZ9X*o*T4Ps!znoYN#WK~Ry%TRa}iRq3}H`1#j|3k8n1iGnq$^Bj8PWg?ZQ93u$Z{F&3Rhg4qKBayf@M&upwMf%i->QlddXV;qt1*=@1Gfqm}V$njKQ-|n0JUqmS*y{&Ujep<`vf|t2#}W0-NMcq~CC-;3%zn9<<?<fzMJmAX3!4pGu9q)xjBu%#<%>->`_1y*+0!Hh4MZ}SefQ(bFYi9SEWf)Te@x%~@x#ln%Wr-?hZjTeo9{qd30R|)VzeX%Y|~QXIoWm->UfzTtDYbx>BGHtkxB>NoW~ZcXO|V2QsT<IOiC%cEOf?xKDj-Y1iCwiK|LW0^Q&CFk#^Dv!qoQLb<jB`X0!qq`!tZ%9ScIx&traFQK3A(>KJVETZUFab^rOcpn9lmrl-m=RMHjw=rB~n0%Wc$Yw-J@OH4@qRlympElC',
    b'7=Njbmqnt$DJ+de#L#2fe>M^h5k;JN6L7Kf8hMvTSjqUsqf@WRLdx^He(LYulfI4ne&RL}SByzF(bxFj@G@-;`AMsqRgWSxiL`FOk3Wpc=wyV+IH$ryyl14N1I5_U$Z<CR;+qA;0XskQ|zI}IG|!e@A7@v{IVV!NJVhbVIRe2cWAQ`#GlAI>197)OOO(_x`|nxa94GqQ)QyJuhOWkwIS$*LFp?7nUT=jW!esKy2ktE}TEJvB&5ixF+|(sUYxZ<~ABW8kvVIFSH4ss1HuiBa2zB+90FUBU8Lvc$Fh@-ok&MN>JTBxzu=JR^=Idg!WM9S0rkg|OohfNzWS1T)5EW4dmGu^pUw$Jmi}P%!V~lDYwfT;}V5yB~;3JUDoL8A-p9o?q-lxJo(v2BD1CZhxfE+T{AyIy9*pYAz?BxhxytqgULxQ}W^>Em_QU{ElzR1U7?%g=drrn`I=KN2*h|;g?fI!Cwy~S}cbby&$&dXZ!`ISVlZnn!YQmN4-9v*;Q#I?A3tSlAq_{TzrWwo``j(33h!`v?o3R?-sEf$yAJ#-v4^|k;Bns;X?Xp%T;1Yu3`AZr%DaS*zvpc7@+cnW-<^xSSN+wI?gxCEs&&LDc!Jfy+Wk|MK~o919~FPA+N1`>mbVY(D|@&tGr0Ty$Wi2+0J1+6e(f7)!{co8fhpc)!nqv2MIKYoE1~k%SEWM#2e8Rg>m675(G9Cv12Cfn-0=)ifEg|`7V#(3iIq;V)G6;E5Gpi3ra<{t-5U#%4EEPMiW{%<L26rE9-V>Ty?8U&(^Q(k~@IEdDhU;ErvZ8ie3!T0f}(V`u49jM^Xai;N*H0;!W&Vc{JnOUs0M#ssyV#?9q`fj95zf?PVlT`7s0z4Wmp~(dNn2P9&via$%~s!{s3Hg)e(l!BATQoK}xF&(fWVCE{aDPS$3qt;(dJ9{1o%R#jTo+*YylJ0vv7%=UC&lZl91vLMpI={CAV$(u`-f=$fGR}KbE^rwz4uL1p_7$?aD%g%feChBFpk3iMkV2Dx73{*<PB;}=K__',
    b'r5uFaDAF&0f!hnQB;uJQHO$ft;r7rfOP@G6y_FK^QUN@TSfrs3*<Gx%|In8Wg{1UPvVMq1v0a_?A;JrgP2e3s0tay)6O4MC0Pxf~T+csZS;Lm%_ZZ6Bz9#bzV$(5MI?W9l^w9HV3JmWEGLonk@s=q=!TRYzfrn*$II$>jg6N5gOaX7zj%BpLuD)wR`FX!Xg|;U!h1IZ)~85j}$+VF;|1V;>(@0pr51xhBYUba%~@w1Kk*v;%h(?3h>t9TcKK;Jbn;`Ul{w2lhX@ML%3W@OcNmWgrZjz$0|g6O~;Mcb18#5G<Ier<elKFz~q4BUlH0H4!!(J55{amUzxjNcx+Ni3M4eo4VUIYIn@Ug(KuQj=fZ6_IB|$cO0ceLRzg*3vs^{!7&U^8nt=x6`9<R$s*6yI|I3F1o!+iJxoL(>e?jw6i<ALRbt#>D05C@R{6@{1#a(cq;OF8(BFg@m*->Q&B7sQOhjGejz|aNOG$t6y7ArEzN5=vlu50-k9*gIREv00Nt;iyXha6DZV)HnNV&e8yH=F@hP6FXbZwrLH-OBr95oa>VkABvu&3tLrHxZsx8KJqg3zS%$i|tpY8svE)RpWMvc{^5^AP0rRRIMgwk1cx?mYSW8PR9V8lU9oZp_@f8HSa0gP?3xk$0X7NhqrBk8^E32)lIHgTm-_tvB5>EP475Voq#P)W<p-SQ@C3eQ;U%ES_`u|Zw@?4zf*#_H-bwv#<Z5n_%$FMELtFO7OM6kwb;F6%c%+RGD<kH8aKGbr@ZKH^kaXvDA*fF^{~wcO@v{3rZx-%Ld^;=&SB3{Uo@;2u2e{7Q|}@(si0&omARgh%Q}iLbeY7-rs5}+pvH@al&FIaf1f~9w`lC&yoQE_=rHOuR`teR{czY#Lo2fd=fx*bYty{0C_tHcA0=zPX)*|06IU@DytJC&ml+icHb^jVgBk~9GR0|dRKNOMyI%+#-Y2CBYGLH=rcL5+3H1{lA4GKDC#Bi14|~>7yia@`m>jz>ve1tc^n#0Ii(HEI_FfQZmCz@@&h$TnO',
    b'+?;$S2d0(3T^QX?8>+MlT+{qj?&$R)XK?eP-6m~QMt9;6!?%FrI}cFqC`Oyu5qBK6yjqd0;6K0wSI2mWPzfuMINW-yh{w>War39oR7F4lKg_pKs#sC<dbRp=)=ZgDfmKhtcI&)i8t{Onnp6SbL!o%?|*ssbw~evJpMK*;bf{xrH-_3<Q(nylbKOm=`0M}hEDK9ae`S4Yd94v#-DEho2@BFS~XQP7S!<9IV+4^UON&6Z1BU));(|bz>x9hgRorDn#)7(1twOQ1`H@kg1XL3(jMa|zk2SXzAVC1W_;3If{b;qDil-^qc)x<tAsH*a>uA{4;rT{a~@Ra-oH>H^VsQ$9C+!`UUELCG6`U9c&@W!9CF_8FevVv4L6_RI<s7z?@Y>v<R3%`OW%HlmO`eXk)jZt!;v9{G#~{f(l|>^r4vs<q_Fl_IdpWZAXsc`Y#ju#ff;mr|Gk|;Ib+fKrC^GjdAydi;=u6Yl*G}YSFj%$A63#Nst+fpsZ9TyN}d`C+cr&HU!7wP$)ty$GC|_$FVH^%bGGH-g0tM$KeySXH?+q@<?{M51u?WbjJPd`^jv0#J?z1$t6G#FaJc{N&-WjG`p4&O@$pZhS0kQlIbp`7v^Nhg9!AB{M>K!B0#9jCnM#&s@r2_&F#Y(hvPgVPU?2|acK!t#m#7O4>`Ula%&RU>5+4$Dz~tQm#<S`A+_T$>8IVrMAzwUdAaNr3UZ`y&x8h>*>akAryd#_<u+|X&V$vuX#IUky;_Zy)+xfOePUU{WU9y8KXe7azAnFK*%4Li|_6g`L;rz+((Ul-Ek{tu@C@L+g%|Vs|i#oOQMVF~t&hoA%aK+BYXCXfwL5UnVr2bPORvH!CXU2Ib#*d<y)**BK@i7cMYUu#Hlj-=9(yf#ukxjJm;|gdu)w1ma5<YH1SBKFKP&1ODmU{%ZarQF<CWh9SDHcsDLvm&s%S^^Fs_)06NkW9Ak22b_bA-|W!aafm+rlyr6(j|{2a2@8NG(^n^_DI5n6TU!9OuK5*gvyID|BI`{no2&qNCYp',
    b'SQp$gmX%#K0`UwHg=9n@PA~0%pcV;<9+>>baU`ILAQVEAO^I2%7r~PL)|8$M`nE&}m4>CRsZuU`((#Ic;<R+psEk**P!yo9{bSdZI%2mIltLv0`3r0~y>C?VzW=t8*EyFnv&m%~2+Qru)|Ilg9*DmhJD4)P)zm*>Y2gHggxSibLIq4mq;J5q)#^yRU{>kZ<St)Qv(oLiI8zmk2Z3UO1#L&ceV5U(e28QXZj4S4ss$V)Kg_=+fl5IjjW{q8F_n2U%z(pRyz?awfg~wh|5&5ykA|TiU~~9jM8yA?J&UO^;;F1dQ3z@JHr@HWil-|#){aXBbcCEy--glV;K*Bzs)f;mxfG8?Kf)9kjC}gH!>c3&?{(T0nk{pqbXiqV4>n`Q&t}SK#Vg9zD9vWr92}#D@_ufzTY2S<bAPn*6c^nxg~bqUJ~mUc<mqgNqQD5_3gik*eW^4+++)uu8s9x#V(V@3yWv@Ewq#GKXuuTr;@Z-*^*#+(S$m4rz22G-(9?a(qOaMxM|>$oJ32AdQ3Z}0^mUNu04;c+0!MGQhLt$opPV_)Ln=xO_s>;sah_l-)H6(uwS|IO#&}sQm-AUH`M`O1?DyTz9Zd>XS%RTJk*nemI*ONpMd?sRT8rN>NE5QI;V2gu#iWQ*VTT>TO7W^fZ8?<Hd#;>80tPFNrK3tfzlsm!&On5i9DTk)xxWrU^2MC_<e{7bJvVM8vTi?x1C%=%%a_Yv>oy#y6$;r`Q4&<56xGt8w)|j=nxe|BYMVkqFRu-XiY$+wy2IsPH-_1_EL%b9n|GDppkZFoOwq%epU(NE)JZgYx*Pmg4H7s6QSy<R(}A4{gg6{fQW_cr+#>XdwMNiU4xoJ6MRP!bh-i&cJDXqSGCuGApGcpT-(Jot50v_#xyZ|)F7a)p$^a}x%qSy1knnb~Qs|CJJNqgroZ*Rxpl|2<WNdBv1gD&`rlbmkWBGS@LN~o+K`TwvGe`lHPNVGXkE3<TCWRO=m;@_#?Zp(OP-R3bNa;o^&K(_X1l<B4yBG_O&3rziJcHu',
    b'FM=4&UxhsMqLeU^p{d*}1<w+;g%9vvAy3B@1b~!QQ#xL4xq0<MCPxwrm?Jatp^{VZrFZAlD7HSCn7jHfLc2pe8mq1U9+G7J26c-9KPNJ+v+VWj(&n=A<y3LoCEZq8Uvp0~zH#uOKc=JX_%3tvSbQpv5`Yam~(e1y`XeBZ=x&W7c5hnH89oXNE%u50fcbgi>he#(smL`z;$&W9mkshDsuxc7^oPJ%eqcO1`-)#%Y)tq<1ms#z)NhVJ#%LS)nE(3!jP<H|ywIp6?CThgz(UK|i(*v5KN)|tp84ZDFmNauTK<qm@en+^n+0#PZW3p8(aewPlSn1`O4Y?|t0cP1T{MrtdBv=<Dn9!_Ytk@BBOg0>UhX7w3O5kIF2+AZ39aKSU8?gxko1C~T`AD5N2LosD=LnqQ#?C`;>Zo&m2@?=Wx6YJ&i@dOv{t<n(kxyg~ccYK&DvZMtzb1@%W*F@h&Cx!4&B-ee`<FszryR(1%V3iY6aL_VoyS<2OI00Iq7;5ybR>B7vZ-)8oHmtFd0M>mA5*faLr2@_!;}q?dDf-OhSGLtB{?AdJTHtb`L+etDp3g&?!X1N0N?7ie;3kxKxGy5A5ogps4B-17c`1fA{Zvm3BEf#c}fLRCRM=JOh8YPx3z3PX`zjIX3jDaU)9Izmn>@2GqriJ9smqtTO#Rfj<nzjnN?K7wKIZZdDPEIKvNUs`CdkfY_i;eT8v02J@gLggwh<#rW4onc#e72*AZSWE=abq&GvjkjrP+X)?%k5Zs@5YUte7>hM4kBY6D9Zx={UM+S^T9YeBY*mK!?@Pjg6FYD?2s&|cvb9_7cgzvs^dtQ>+@KBQ7sm`akKI~IUixqa(vqWB7|fCRgyd`p8{{|(hzdA|SG5o`S}qNv1~z>!3~jqoWulc?^g7qfQdP&q<6ric1h78w1udY$W(+xx1VYbpwUGg;2>C$hQ7H2yjwo7~mJtf7ocX&PmPZ>OY5=Hz<_X%^?^d+KPWG4aqcAJ@yI<s83McvEdm)^1~K4BuWdldeWrC)KZtfm(',
    b'1uL^DpMOAhUcXto%ZJ)+2MbVS-L%hDHR|4+A{&~m$?B07B+UJr+31d(#MPg-m%ks9|kaw(zIN!MU0Eh@P!49un_DSVSversKx)4!y`SN!jIEa-+&ML-rW+6X7q?xU^}ML5_ph65x}I51{r&+7b^e^$#)f(?!7kn8qwW^;!s9Kza-jU9a!8mep2$L;u*H52w2*EU+o4Zw^PX4U3C4oH9E5#3Ob2+tT+;KoK9!C?UUa6@$*KPLE1^b6o-02+zL79=z9FmAd?ADT5`AfI?e;AbW9A^SyQrx@d6Iv=g0M{m_bQS5|Ot7y2w#0@%HW}A}FU@Y(Ex*iFvo1550A3V}0|2DjSa~oJvPW{pALYK~-d9~h7d(j3ZF!lAviS4BVmPq;6CuzX|I)IAA(lXvZ?RtIA$-<#yto_As5gdooUA(+}*^YvUTO>58mmT+`;FkF1Ro1k?i~s4N6A_cR+ZwBub`OJZy>6}_UtWGIvH^;xc#IY9TQW(rAOzLYw(IzN><3E6X+*tuYshOLOhL}ZVN_X6@qNNvI;Ib1I%tRbN8WJOXHjPTvq`x|bWyRV7-~f!j;B6*ja5A8$dCP&HO8i|wZ;}A6<73^^A}0vcBt$k>KtRTq@g1dddE(GR_{%M$B3IpuEEFyrFHI>I*2p$xP52^i*w~V(o$oi@9rF1_){KmuHo0fQ_}U2t$)uf!!&gp6(z~LV#&QzwNU7tXEv)EnDCIj)3`J(H^tlWM^wF?8Q0TNE!rVg-Mu3CDoR!ArlhJ4nov`JD{(c&o#0t!*``6J&|Y^{y-T&r;s}`nFD=o*YWdj8>G}zpBW8UytH|~-4HMBmWBZ<^?JTaaXlqHq;zo`lINqtrdq{eg`_~#c{sGK@qA8yh76GEw7Ep{%Bl8S#_e8!$NMtL3X-a{JEFHk^X|~s=z)80&(_-)x20J$DA^+LKci;nIt^(tHaPX!=V*x%XGq(95><MI$Vl{q4Qb&oP3MXQYT5Jofg}g#d;Dl}4yT$VpVcJ*tX_qgo*2WrvCMtw5|FEr1N9*eN%0',
    b'K`S^d+IKwCX&=Qd3fFoC{rFPi9ix)y8)g5RUEecTG*CO}`64Wm<%+&2k$02)!21Fs1Zj4Uxep{l9=gWu)N>iG9=VRie>8iKR}`KYES+l7d4n>ozfb;!7)z375rPeHjL>SDRU{p^;9lSCgsys)XkjaJ$)GXEo+loM<&#236z79ybkTQs7NzDc0+Rh7c1{C#Tk}<kiKNw_TGvb&Wo!j{i~KfMwW3)fW+3+WA=M)#Rpj00kHopVI(K<Ug#Oa&Es}rpfrUP7g|&=Zy!A;0p92qQX!bz6ex##t)Q+2FVMWgLU<auLN`BaD~HI7d%zw>ZG#ZMu80vOx3j;Ex1&Iql=}I2%`5o8B2iHby)c+Z6dB<DwK)Bd6R$|@1^amUlcP*{MK7-hqOlfzOAI|$Ks6nfz}F{>4K`Q43m5-)OaTNUB(5CBF|Q-r4YvAtWMSkNDN0-ii>pD_1MN_!J*5;!(}4Do9>I5TFU;@I!h61r;xowsKojzWSnhZosM$*)eh+>Rr6>BDI6Jfqt0ycUDqsNx$*-$(v_B_8H?ZlF1f~C*VX5>YJ=$Xu)2fFO+3~-YLn)>#kPO!bnoLx^2mBJPaFehN=3nz+)wsE&4`%?acV+E6MI{r<iH9YV6Q>IcKLXAZ!#Vx$}N%u5H3#<(s!U0fu@IB|Lrg9YoQ=7!El3`y+>Sh^Sa_bS(!|PBCY}*wy`9V)U_zki&TgG3T^V{ntaIi0+jGj?R`_KS$wHVNSPn@tqhGUHn>>R(jOY-?H8g*99Wg+CJRvZr>bgyrT{Wdv-FlHdZAVpZ@46t@<=K`7?&nS3sUvx^3Guk6={T~n)ZC}+-}8Iny5QTdIf;<bd5YWKP{QlB(yWmD>VQk<V&_)3OOZ-c|oIjSc+<|aC6)qgPylARsAPZc5wqK@_7sVvIF@tH62s#Pps%?X6}U)p4I=T)ZOyr*17oZ=D9co4EBW%VZlL#WJ*IIZbQ@agMKhi#ST9HRz#t~$Ew2d^33Zap6<RAI8;v&XhQB?e5Y{VLjWYoRcIRNPMj(gdFxs<;',
    b'1>e~sWE%BP4$5`P5avEbJrBRtw#{1N&FE=R0|dY2qjR^hVC9r^OmUp&MfR@Yb&oYvE6_kfTWmGyc|Uv0w0j<A}m<2><>7z@ABYA9YTYG#@AH6v<b8as)4b{_d?6;q<~G8yI_Y^CDT-EnQ1^$k}ADOn^|XJX;0>Hx)#*>hHYP{L(XB|HrKOK6P+RBZcYG}4A`c9igvP&$b@ktOGZO2wttr@6U9`GXadkCIN#;m&&Me5foX&hML@%<1HN~aQ`b^lxGS;fSmDNL>PuUkNk)>84RBtYc-54YGNNc|tk9aC-W_Azfw8g~tz38=k5N@ep|=M;$~v}Z7M@Y3iB}vlFc`9Jfgx|gn){o>AS80M+iiKiM41bQo7}Yb{VCJOo6#0>yEXse(FoJWp$QBxgInS%8h2n4{spj~$gdZQ+Cp_Vmg-V`K0XUMC|exuj;55)8&k;2=%jMaD=vHyuBWLmBX^OwYb`GN<cn#p;>ujrw7W?hu2d;l`-M}2u0^>m-?>6(L>UolyMSbH1c-82Oqr|>hX_Qc7p%x>nf6u0aB?&4l<#_8aF(~;&RWC_l<=`G&{ciJUJ}FJD*jqq$8E8?R3U^<<gc?^EW@F1(bdlw7UcY82OwR+Nvc-0D@|}5cllCAdFF+(0_IH!uyw4IJMs8*3%%N@!8~n*7Gji?{oA+eL+9<n5&;dCsk$;h6}E>u-NICIj)w?D;)3e(=(rc-seJ@0R*n5l+hwbvO1C9JLUt*o&l76x8+x{5$I5f@9;c`@g{k;gQh?vntp5s8?EflF6O6ST+bBYp36XhE^y||W5@0Ofc0sfVBSKqD6O4N_GA6ao_YB2Q{IPj_%Xq-!^pt=N%<R+Z=x;Q_Q8mnJVBf_=e58X=yYEWH8KyFKJ!FkZj}TTCpijRRAv%pF)-X+msJTIA^fK2$uuQ>OsMDR6ywy5xRfV>nN<aiOL|mr&6FSCxg*4QgOQoX*2MwNcaxc=A<Ya?wnG+Q7pp$_ant)^TSOzIdVJ@5-e0Hc+UZKUSkTIn?%IvKqv9PB1IwSmW',
    b't!1XhMGnw4&*@W+5M^|m*o0qW(DXI+m^R7Dv02NN(j|!bu=!N;Ha7bQflNTNt*QseFB8G(GaqP$5A$Mp7c|G)!e{rSc%nE2#o|GG1CviL@FWoKbC367ISG!t&{AW;Yx}-0bQu-t$f5u{(xNnw1pc`tBv%!KC^9eDqNwW0y(JCze-)sj(?Eits}#y$C$b_<YzogC<#SHN<vlokPg(8cIcw;JQi5P2m$?VM653#xVVbI~ukmE!3C+oF_4J_N6T>fn6kt_G#)PG5BTh#<d>-nlJ}FD;e{?a}4zoP$#?K<vMAnj4n}amD*BCEKnDvu&B#(Vw235*u#n}YPEYz#06k~~B&j}|PjfYeZdpntz?J4VxR=6A58k@pO=LXVLiWxi!hYt%}!{O71IUi8VOb!cm8sJa$Vx?#$$`GFvm{?UETvXoq8sO=5ZlJKHh&{!v46zUtRxU{b@MZXE^urpy3;EcrZm~+&zVkqLAq8B1=c%vy#o!}0d}I+<&kk!U==FYprmNUWIqczk=CBK$1V52wm3d3oaL@4z%=`xl@22zoZUx>d*$1(O2mx2}8EdpLon=47dgE3aPz2oQfSPD-Q-UJtk*-#7q@euZKnEGIdBvM46m|1EShE5VA5ZaEJYeY;tx~GO>mzO-uKPWxWz=EDaDsp@W>hRJR!f<jCP4Fexsb>EODU1krUEed!lQ`#HI#%JGt0$E#KshOhxnrF?MWI!Rv0UIYl}>})wTtg47Ain<qcb%aO!Adbuwki3ZyW6IvRVwu#?!Nk*F@^!M-fee52MN#HsMQla!rfM`mUEzT>E2&ma>|yrbfW-99W*47J5^S7hvgvBm;!+%nobeweO@Foq{QP;^d)8y0GTy0lXhi+K7}=<5g1w{i9vRdz7F8~jJwd}>G)Kqo)UCW${vOxbUVMQm=4MKBv!m{A`pU_|~G+N}T)kcgU=bEINFq|}fx4~cLit@#r78$JVtk+K$9`8=$wT`_j20}PX=G;xn^w;mYl>$8^%U3u<q8|~2xTW~35^H9Y5Io%MgLtIs',
    b'X_<$+kuzGdZ)pwOb$}wvsu-IU}(Q3Lc$Oo`;eF-*{QdwwEX6uG_@-5Gl8mdAzBQMIziXC7h`mQan!a<foU21$SC>6<?a#YMsOF6i5nk7e^7?8#qwEzw0IqR?#r%Z-!A@V|A-9{R!VO6P9v!wb$uM;YM6t_4cow^+8g07_1#grYO>gvtmowd#z06}}}SLn0p!<z^M`q+2I^-h&PE%tfbe;I%KuIupOcHA7z<B}P@^Cqs^J2Fk?3)Dwa@2?QjJiAcS3C%|1*1LQV1pzJYow|!k4OvA)cXwm#3UG#fxb(izQT;r(et7z{Ce)$3`fsGi$k_(GsXiwj+zRA)90GWXBz30ZlExjS)K&D-ddf1)1ZtdsN!%ZWFN3dmSOHfm3J9eR<*GwtWzZtj;y8)J`OoPTO{;32RhoQZ7Ay8!e91Oq`o>bHp|DIxJAUn5SJ_Wl^1ulGz2)N5HO*FFT7f<t>l(HU+|RW+n4$+b_EKxs6_zxnAZJh>ujQ9c$MN<t+6-Z>-K5xvpY!a1;;IIQKWgAuLLC?GQo8g#dkaHgppf0r9yHd=gbK!fw;|}`6m1WrsjyC4=xh1LO&QTiXFLGP%U?AEAy&U;l0=HO3UONAb+ui#<#AY`k@k?m<o8gxaX>#Voq(?orc+y2Ac0rls=}~z)1F_+aP$&32456pD7KvAdgecs%A3zg^!h2eeQS@ZEqk*d<*E#(7`_y5GyYuZdVn^jUYkc7W0>vJE$vsDfe(iW)I2<l5ltqz8p2^bD(@+eVD*w>o0;@)m~AK+sNkZ##r;ZihZ@LPr&{z1<|40v7l2EVXHCQvx_LoJ9nU7k{VU)}`-bFf1G=Vl5|jsrOd-+f^DP^Q4s0H=5^biBWyjtWxGtoTYE61wYmJl!&to}gqcPL1gd{}W(-(AP)kfB5)u5MKft2DK8!j5{MF6_OgYj-aA{c)yUY<<KmeQaZg^qew9*y#eThU$V(DH~T+FJld!SRFY<yzw5a2|R{hD1HZWgB@z-WG(nh(96bQLrO{v?wrM{McfqzSbg>UjE',
    b'_LPhuug07D4&_O&FhKgp;s1>r8+k`RKM7vEf{VSw}CC?Q^R#ZksUtcA}%;_&VC8m759(@pi@*l`{MQ+<hD8Bj5?`tuE=D%4JxPnn3!VpQ4bP4JE9tg-=<-0S3eG;llLoV;G@$FjEC@YY0kP09!5>)YKPN(~Vp*N&)|TdU@<xv%+meyLKUq5LRbV@?36MZ0<-EA19?h#np6Pp`WMaA6VATEC8{o8^|oSOJDi#t2e}W*VG!387f_wv5Y1#O2L^n#<}o{NG7%39;<K0uzYdIxfS^>_*jd{FW$Ia_2y$G4u5HqofTF3V2-v!hT8S#)?6mYe^Db?B?DiXDnBz;!#nxdHBXirpoGL?N!a_a!mL_G`Mt(?`2hyC#XoM)9Z<^Xv+7R;B9%nY72Z)7rjyfuKzSk=rr-6m^6)~epv%Mb+d<Qk|n~oN%`c)ACp4u`>do3xc3hmr=)u=*kJc*>^3KQ<;kyUC%t&OfZznAaLNhY%B^(9^i*L5$lIP8oR<64oxBvjaMAHwlCL^ZoR@$D#pXGq7@POsu2y^IJ=x%`GUY(<QzN?C59W4n^;9thOi+O(6TU(%0u(5P(!CHapSy27<G)i*Y~=1EB`W9;l0|dmXC&B1_nJJ=v`Pu1$8zd(%^1r!<&4a0S#!vmv`Mu0Xlo6an^ZJiv1Hd(r)=*~x`uL~4Q$_R^?-$TBjH55BFx%^7^aglE$bIKhulpbj<aW)dzLN?(>_h~E1Y-plD<+tt;P8Q*XrB1LM5DSMur#mK{VW7oi*NINfpal0L;VUu3Cd^pw^=u!<gtNU2aX8hg6}XYdV^nzLu~N&p4sgIxoAJ&PyQDm$zzF(7(62RpEHDZ&3>G`nX5jKo`pQLytmPvP>t86n0Am=`MrUGOJH($akcq-Zv1wNy1D8QB)}Qd6s^EL5J!2WPJMpl|-w|$gSm<{1C{Y8MB0#?0Eu7qZYPq8WBT1rnVqy3pxZizj`NvIl)|azCe0{(C5eJn&9UTsjp-DO1N&o(Jxei$;g03kzu3{0<=rwhnCkETKWa*T>',
    b'EJXEyZ-90k(Nnc5{taaWbJI%3ENpe6|sssb%CP(<xE3>WrR_=f4>fu@--Fn2b_ex<)57SjV_jF7zo-uI|<$W~9m2I`d`$Fs088qAmJJGo)e(i=MoWDUi{#0_o+E4Egp)bthOCGUMoo7qmj`pz8YK7_HoJzN*AyvY@uc)ttlCa@U~eqpYrz7i7$`%gv`gZORsfq*=7MdjTQZ>r$@Ex4mVck2lw7^IL39!@H8p`8F$_@7F;FSX+@-^<zG~CSY(rnf=jH&P~^DTvnM#b50?C^`%PaXr*|o)m-kGA-gJUYseyVWfN0%O=x1&6juEavo>S~Ia6VQ;%MFeSU-RBBT7-q9LI&!tDx5~7p`V+rRBp#Ln=l<Dl7IE=*#KMdS=w9O+6;_f`k~0h(=!N%PFx?(-R1zIlslC&CHjdD_A{Rui$oGgrSpM=e3|<T-s|a7rI-b7z2Atj5*&CRq;ip)zv&g318!U^A@YZ>~RXi+X@Db7L>XZBXk<2aC9QS-J0#HG%boGZ;@Ipk4lHQnWa#l696^Ms)W6}xy={MPnSu-R#C=y$uj-|IhSWC)B-7(O?>)DtUSRvqpYCPqH&%N``4@GfLlqz58)yMk+Js1wsfi4q$_tz=Kb<-?|yy%%e$}J;pgM=H$)XVrtQ6Udr@5*+rx0dBkR!%$`W`(ZUbTE@?^wt%X~(f>qxSjF?;dEhAh&r3t!XN&v(USAtoEZa;cf0G5)em<>-p`Tr}QWch=RSWh=(Ec<uDnpN_=vhkYxi5sOt}1bKQYJ*8lx07xlVj6|qIGo%9(*P}dRS+0HgdzZtVWFmStks>-fE#va!5-0^hM_-(sm3#!Rg;wiU+Sv%q4Q9x3Ip!8|i*0nNiG5uGO+gQkXcm}F{KQ;D6Qyy1t7j4AQcHM-R#*aL^Uv)}qgxGS@V91la&76Hut|9g2oAZfGGdNYpj4Ng;8;s=zDbrWuo6KQY|eO$H>|Tvr|m9rl31>zZMDY>BNg+K`(>~Dn}7QK<u^H2hi=?=^4=EG{DYV0erxfno$E%;WvA^xd',
    b'^%(+!-U`kOX6h*F|Lii9Zu`Skt~t7>Y}`3FhL+*;pSaUG^D5k0{0tZ9$g02Ps(Ku>9C)1pvgu~Gf2zUm-aR$szviR87|Lt=GfC1GZcc|623by-;2$!^eYLjb%J*nN@5Lu#1w@kBddDbsbG#)5_LJ&7T=@R%DJmF*yGMTCAk7+_xcj~1c_la2%10u!Iw1Oq5SR=5n`2bg2?-#1AGqZNG!*##=w|%9Ril~1-*Bu#wQMas5Z`1#VVQgVzD18#eK$N$Zn*)*9%akJ>^uW<m^{=Dc;~kyKhAUQZd2M=>wAK_PE*ki4O}{ZliXTRh!{mM&<tH_~GT(Wz7vd0Us;#!+^iZg2wBRprNt%Uck;~qh*($jE>@5hSykp(9+|ekLQG|7%ju0!}_-6TmSawX2Mtj>=-!J(6CR8!O!RD<Y?mAd+z%1hW_~?cV0lyWX#|<(Jz2Q+-nlW@3MU;UY_c2f4=|l(?32(A|L-;#?2B#$v85FeW@4iVQ_8Ih{iL8X8qtrh;q)F`0^4n|HIU7jsgNi)gnch6w%O|{FnxTrKMF~;F2uJGDrHT<g@2>mg`=o4v6u`<0IHYu?y-<Y21{$4lOZZtqh*Lblfc1*=qT8p)yKZ*5W^*g15vQeI~x^qe@!#Q@7-$`m9{JFk>!Qvv>ziOk88hNiJ4PXoXi!UpLi{FK_v@Sua|%o}?-w`)@>Qq!c>SFLN6IZMv15LZHF&4u0Thc7#P(jCzKUx=0xwG%y?HC)UbH4aau6ZV)Z)n5G`3t<HwR_$sN!O$X18!?JWmOJ*H*Sjd)$Q>-Cn`K0DgI}GSQqGDBzF;L*i9CeUF!KZbqoffhfLrqETBj^Kv-OI#F*AhG0FNSo`N<2Dli^L#<NoF1cfPqC|l|RiFoQa9}ksTBL<GsQ0`pTGcdF<BTI0z09K|K}KBlrN_Fknj)DlXtHrKKlyJG${@5_}axWvWhi%&C>kE+Xf(2m)IisKv4~oeTBbSa)hkeJ<uZ9rr@2*tjA^3bn-n;(u<oiuESuKUN_|jW=(Bw!WOi)3F|o',
    b'=|}uX`7>4ZD!`<S#<$=+w?EaUCV)R)@NHn%FC}(8+}Bu9Of%Mv24)1Khx;_(&a${me)swA_^Dg@eyH4}q$(CurgpQUInYq&puJLzEV$;7m+wYMDzk7pjt`V}6qc91yj5B};!VZ0avCHy*1;O?)Fsvp*Kh>H9f;g0gBWKGTd5HgVjzAY)$(bW0~AM<c-qa2q#TI;^mq%jV^JE*1h(e@scH`OE^a!!E5?;e%Z{bokV|_ci{s?@)VijRu-CdQY#$>mRzswE4n60-NUUNfYuZLtdU>i2m^Zq?x9WGCcHXUf!?Mo56Kms!WrEUcWxFFvwUda_zq%2VL{nE4F2hV%E3eW?jG0zR;%OacuKUXJaE!^JvdH}weMXcb!V+wnhMaXP;mA;C@lfuF%3AaoCQ$ovnTQgBLrSw@^}+SB-$3N>bm|1nN%#u+McH-=sj?K7*Z@*yC%Dy32@V24<<OJg85s69-+L^qEhwI_vW5!LJ<Y(bYDi3ovduz^x~?Fjkr+6#Unw{y#>s=Iu65TH6DAwXk{)Xc>VOaekU3Q?s}mnS^y9-Ez>|(pzH<ZJv8;I@lp-KEP<J;BuPJKEC6fa8oi^M?VbzA}A9<;GZ|BA6l*%sYM){o7LMm|<l9hRlD-g43U1k(s*O~{=7nfEDS2JmoY(P@<t&ha=qz6h|_D2d5;%(W6P-%WQgx+sDgJl&vnl^Cyp`YP1hGFCZ@%FvgDRku_S^@+f(@<oQF3k|UeQIKGy%z!Q!UB^NU|WfD1i)}47v@D#EnQ6l;M0#7ABZhh;9~Bib%2h$DSYYFp?nTMY1}YL=_DL=S)OiCYOQ3*Hlrem#5TCMf*T2jG?B5hv(?u(f7J{NGjmH%aonYNdn%72by`6{)zW|iyY5)2kWrU;cUuv<C$ke|QcP(zW1W>sodt47u{-61KHMAy0<h{>Rf)1wfuX`W4C!|CV4TPu9F~$v<s2+r=wu#hm9xsY?#g2MYt5hG$)YH_%m^7Q2FTSm_5HS$W+)nFTsh>DC|gNoHJumC=dM<&COSB|^KfI',
    b'bcV+(`%Xp!nsn|OJ`ig1+f4sGz+CPaR$+-N;UtO;wdBQtWW%Qp*aG)Y<LE8`hN^RPl`EP2pJY=-0ht%nokT|z1lOwpPilfH5MmvP0WmnnGK1k}MFcc$WiQ5<lvy-+(bQK)spdmu`o@PmDjmJR{S0#Dv8AC1JVyoYnLB=IUNkRSug}1zlYOk&Hb5yj}&8A~Wf^gZ5YgmvJ>kEo*&3L4q=}RHK1s?_&(IPiDX~QecZF~9K221iFTFZHpSCs{<VY3>JYn(>2fXF-Rxw<L^>FfihK+C3(j(r<hfc{MtlU>R;2&%dAZR(B7OPkrs81X&E2n)roBspcw!3()E5fm4^1pj*O@YLfnC-e~jM5lI;==kp~h_>^%H#XKHKo&=*_W;XH$c3zHT9$!Qp=SYXZXy;ggt1N8lqEUcUm)N(23Q&jMttKh2@_3{-g8!6IVWPbKQqu$`3A(4jpoH%;76&P5mcDFQ`ZqrTkFR{R>8NTB~IVlW?xWpd%C%WZQS!1p<=>r13iTcU^o??denu}IU;=si+(Zejji<s701a#96og`mr>)vHbBT^<+c&5Ka>V<N=AST58rKF6HMl;0<IR8`;v5`<h#xyxq|mD^DSS)frLf{@sz%tbzoe6^y8=7Jc*(T8=1-|liKJoFi_mct);?4jV>%19RBqF?*%c{cB@Wj)j=&kHMCmx==U?{PL2w(qF`*#+KP`M#;#86&7`$_nh#%sd1z{O!*o{_fd5sfM`TLaRGqKS@r721af??HYnSej2#P%zc3_J(#~#-apTD1D<6e}@Ovgr3_;|v$uMV8~Q8?~OY1CV9*k|6<%%mI>gfn+yYct?L)yMLeA~uMLHyyUz_Z_yN2M#hA6a0mB+DdSE18hT(V!jDkuF*!%ITr}QKDxehm8j0;Aba3$!D`Vlz&q2hWPsw^Yc8IHFHK@J=esoc>x|ZG)AMR1Uh%WP`0>6EVqqTRa^wZrmZ^MN#~_PvUs!|$Wg`7_cSL)qxAM?tARd}tnUGFZ0)aQzAv!_WwgQnqlU6851eh*%@&M',
    b'{wFoB7@tLRcdXf$@3F)ngNZS4(}5o*j`G`pkWaaYug&-mO51AqPw7&wgiQR}SJsJzi`2dSf}F9*e2?zpZoMrRAgMRryUz-hodM^P}?^kZs;ocImM9>-MkXYw>Id7SIv0P=~V0nZ@*eKMWQpf)}WMUwF44MtAu>KsKy@K~YI*-jZq0ft=bd~`Y_y9{C-azpL+@RrBhzY|^3#39Zu)$AMAFD6*7p~YsrcA6Ll+ueIdhREs^6MZktUY?HdQ%mcWz2JhnYZU~|@iEd%D_{*$CK%Ro?QU7dx5?lQ3T@vR-)Wci`03~g0xb~xCK7H2%9TbuT0l<%W2uxVn`9qu9tp!>eP2M$1x*BYej7$_8`({`DDrmPNzz8KSr}kGK0J>rd<S-q!C0R)yhV1%bn5b*YR<Z-dRD?5AN-OjKMrQ*6R@d1_NLIZ1m+rCksR0%E2;tV`pLz~W~Sl>s3jfiOi{@1#8QZf3R~BuR)X^x<z38;sW(>Y^<kAWPg$j_ft_vE$m$^H4%34m{n@L?h09kxUn6Qgi!O2v`ZghX{QivcqsOcPJJ6w=C94C8B~b@wF5KalJsiFHw|n~W(_pMl+LO5ENCDq}=!e_1wB(VpPIXNlV?m7PJ#Y_Bzj<XWj`?n|h?FucSdW8b5Buq?u&m~SJkDKLXl^m!RB+sYvGP`7uP~8Nmpc}~M-vyqVLd<2jGiY$*tKCe?`69hGRSh<!)R)I`#uK}K2o0tt~x@?KZZ-_>$E-v+~r~DfzSx-YSlj5l(?+L)cs7F>^<;!^&~JXH*APG#4f8=CpPr}@OuILD!9s;mlseaF@#ghPqY+lJ~~E#W9gr`4J{ht<cn!m;G4T3bu~n;@MSW(OCUd};2N09@H}}dL1o#ha;Nh3lh<TLb*4PSZGQ?|FVk#m{kjXix##Srh#O@4a+<kLr`|-sZ1+LNfg(GUC6AO7&<m`S#9B1CcGgBorry_eu<BK2bB#>29E;Pb)a^!gx-e}UaBNf<cJj=7aH#AUk69smRiVMD(?<p#fK^?I{w)=urKID9vG',
    b'$I(;&~AW?anf(Z~QAFfmpJw`ln&CF-i8wZCeochI?Mp%F$Re3<JjbnnhFO82ttK?(tKLj1o^cp%O{1P<H-NEWOe`L;1tB6|kv9<UUo+q|HyeI9yeLWDR=IqxEJY3Z2D#)2~I;N3Y^_J>ywd2qt8<Z(S#X;TaDKUV?kaq|iB^=7xe((E-~cpcrzSr4R;6N8{x+s54pV&s8DBkyusmFuBy3<*YdlX;9I6s$3)}13t)46i=C}d{SzS>FQARv>-6SaC$J!ge+KbO#uT`MFVCLE*Yy*{7!DBO~pF^<$MQKa`N2keZigihZh(Y05GL8DQLq1Cog+z!HoIdE~xRZ7#hlDYEhzm-!Do>#RcGa54vv!0<&zT8dXRzlgLY%Sv~Edzp8@4kBEi+Wzx$I8<ctCgap)XtcjJiLm-PjVIH(OCB#xb?OsTzfn+Ll4^63&m6vK#uIol3h=ztHO`r|Q93@ZZ+ty@(3T=W}OPtFSk1+fn_yvHU#cfi{7^F@iHcLSTgCa{I7?SB82iw}}qPc=izf<mly>h4}i!jgpb5U&$QJbwW9v(ww`b$JdUOUa|xuod94yKAo0k74er=<ZNg96u3)Z%g}R0a$@0jGiv3A&x?RTM#oPS2V~2Q<@m6QwpjXFN{?VXC32QrGayDtx%ZFU0odS|>%(a_vsL)&)yAmKhk<@!b}|hIjm+$lGgzQeh>K$bZo)=7;~R3T0B&n1$EA(-vlhHk=8^ej#{7Ml7sE#dvCsKSuiOhZu{zts7^JGolxCoK0M;t<pg2?Jc@cUaDaY)f$R~pg0Igc)-Z+bE~hqJwhzt#sXA1dv&ha#;nWD0#86JM-x;KCa|jzoLgizZ!$g|Qr#|8{WOUXEV!dEZuIIw0R6pSb}oO4B~-46(xcN%D%O*h?*PE6rNpOT0Jb(@9!=Yjbic3FHM&?2&qe}DOK6pnC|<>q;-!e%4|}V!0c&lEI=Led!adr$oM<My=!yZgI0~$m6(dpBic4IOTAj@(w|c@E0yR2dh>}feB_#_=8ISsO!76&@{91MH8',
    b'gF`%&$RjEcVjU_n5-}wf0P)#`@=rO_8AFReg9GtD|D#_VV+IGWb3d3Z*iM;zcC@5w=;Y*RAq`+ZAQ8|<#inja1o7FbCQa2BzT?q3j-&9^4N@!SaBkHuGI>X(a3n%Ya#X2%8U%=PPw7Px<b`)GXqs&ZfQ2lu~g<@q~+vIaR}S5W~Eb7!i?SG%ml&VOvAdK$ui6uz>%|b+|mbOXH=YQwQ6eAJIfQ2^PCAv(a^1=)~ZOo5+~Po+et<AuC(8@>bg+|OFKH7au0shJ)>=<(k+Zfu;daqs8|SsG#Rc(^8{gt2WR-!-iB#~G^FsKJHQPq=bPHWk^feZPwXsmxMFw8y&>}$ONCb4qLjPRZFM=4qp4%W$-eH&<++;~CT+j5VNzW8yza&Osqzxo8d;h7NaCvELAXN_%d0aUF8^3?C*-;ooUg%dUyI(B`^8yqPH8ypcT)LWYn$$XLF<fvheIq+f6ZJJ5juGL)_tD<VKza~lE<kfDt-T{VCkB7u(7^3W>iurb!>E648ja>jiwdzYG;wx#&MqMP1Prc($lORkbv0u(a%Dl;?NlpE8&R)-2|SzV@ZSX&D^>WSyE~<BWW^&vqrrP@Y@Be4n*69qdrq3EL}zsCn*{Com2r@WO)O35uRMm(c}$`lM?rN*Hs#0DPb2T7l*>#0(uVUqkeq^ojn|C;B8e=hi<m>EEnRFuzvX)B_C=7Je`h91~$mNN=bt9WUS)?!G$GFMq)cTxddu=A+~7r6rp7RvoV8}vW^q%?8zKsu;)*OWlva$CcP5Z$SJ;8LhAYX0Rlht6@9v46G{;n+qEg)VdP0Ky+#-N+PMrS$~~CrY1wfd1^bC$+3tniRj(k5I#~u&SS$rYT!$TFrAj9+`hS>~*3*B;*wK{yC6@Wo5&O=&^4zzVNbH%NY9Log6EjxMjR$6D&xov7X6U6BpLNg6lZyihX*~%%H>?%G+Ya$xf%nJi0FAb%6SQ1Oo0{B)F+#d47@bpkgJ-(oX1~kJYn3`W3mx2dDrI?=iU)N_2As1nRoo`cY^|10U5Q%5',
    b'&B!wvv(p~G+;p?L(*8zuCEq8B$=+~*iR#arMl%N09dX%n4vTwPpN6etz1;CM15~71x*rD`2v}Y^!kfjePE)l<d?Ds|5}o9gnyY=mhVyG0ksM+aOmO$nl~E`Yc%(~uoe;4DRlcQiRZp_eRcN@Ey4}2&7@Q|bfFX9ElP@0h`Nm*?%hJ%<D+@m$sO394g>uU4u%MrsiK=rB5?CJ<pIX=M%T|1oDk4_#(4IsU1FNm5{#}O2%lGA<2f~(#z!E8*T66)n;S(&r!R9UP&t?T~;JIShC;99tWv{GU?0n$gu}m5eR%o~AoNMt~mhdmsjI-##pyom_z_xT7HYp*@_+u_mc<<C%pnDx_0(6>c$gi1}NY6&r2|@>uP$+{NEAB6~piRHOGiRI!>`l#Ze$+SmO|pvKgBG{IEhEH_h*mP1l2FOy+xZozI8%*m!fRo3ttS=~hJ>t{5Z0OtGd`E$TDigr><wiiWkBMfwsK-YcZ8qV>Fzv64x*pa*H`w%{UaMqLZn$Y5RC-f(~!LMI>zhjoNM5QuaO+_06Q~V9bh53!X^oQjkqLA^qY~SLG8CGPzgGQCs7jLwkm+B5Sl3%<m<W!IYp3tB!XjXwV9&bf&_Zr_k~K0ICJ1qfBRM_(Xn02Dq)t_8Egc~*var>_moVT=ethmL+VyCs;%`LHC^H~IQ)P+`zLwa=K=iwa(1}|-+#E3I(Kf~y03U4#r(}{HE%x&wPIt<<D>*rswmZ3>EC9h<|t?RK&6GDJ5oNY0Kc>m)4wT5$D@`^rtH0yWZ8HaG@NVF4%=R6l~ZwI&SMJ*SCC*T<0L}q7#@LL%h-}lYC0jGydAb(sj>5uaizNRhtyXsoo(J$c=XEzU@|g<s3+M7LkZZ0ktec~!D+Twso(-Wp}df}=jsYHwL(7=UE6>wWU9>c%&Sf&)<RoUnKZ4n$ub2jewv+MZcGW*bnZ5>uZj(z(bohP2`)T}MpYs*$)F}jHkS@@mRK*bIVu86rAl?<?Q;n1cO^T6oO!O!lamy?UHRUYWzzs|Ob_Y81?2a6K4e9',
    b'O$K?Bt|5uK%%H08eP&VN?;X$89BATb*H!TaXZzf@JPJo}2?{9C;LG*iwOhtM#mjnVr-6Y!Qi?a@AT6tnNCM|HlAG61JV1jh1FJ^t5R?C`a-6Av5<vmg#E!C&zB1<kAjTPJK118M`HGM_jT}%0}ghXk8#!ro>*+%qwIXmd4ewSR>KOPJu#c20eMqFGcaDFf>iiQOkhelqED5AnI&q}D-_fn#n+VxORc4HN#1QwE>Tyk)pM_cwpSB)EHFF7T2acK~S<#QWoxTxlt)5FHR=xtS^x2Nx^$>QTi9pS4nUj2o}){C#GX^H~*nbI!gsn3I5OQ^1IrWMP)w#VEQ;?Ub<YOhtmB$_XWt!bS8x(gmtv4iXPkAMIB8$<3KxSz9~-+q7j>CYcOs^%l`@;qknJZO@j6yula$$7Bws#p(@PK~9``K3AAXwfBi6yp9u@e2~gU)t4SBVLWH#haL!6%2g+mOJJK4?OxQ<OF^Y2!lhGc6jltLEqim{`3oaC;IA)G~?*Y>ru-btqtDfh%Pv5?9RxD_<a^|F@MoJ2v-LQBS-(&BYq<+E-FabGed-`-~SfA?;FCXUHs^>=^I_u%-1nh0EsWeduDzeZwe&jzC&cr)Y)|CDGzQtJHMn=d0Z4vGb;#iM4GzL+mu1oDyPT0kF`Dau9dwzWcVt;g$;;9k(?leK>Xz5fL&DE50s*@#olOu)JTzsZKt1r-Cxx9dh&De{piNAM|Ya*@d!-eJYtI>rAf~2XoB>RR(|XdMknDk1ew@&sqjSn_#lP;+XgZ5g(N}DB1t#2Gw{{3W#7M@u~-2^Ti(1o{Y}_DYRb%fo$uXPXmZ!D1m%=jMzBq1E1j#gBNp~7m)%MDha%ESMOm7{sJ^$1(=1nOjVARpiNpHvYoy-mgOHXcY44-%+G!Rp3|aO2<g#+=+1!r7rlC4&B~*YjUci+*YMrn6J2doR?wFtT*K_sDaq`LGgHFmKhXf(>-;-ma98UBli1s<KczvB;{^#AV?|*ss<=6lI^76-V@>BRmcqj<F)HVtT*JDS;>mc',
    b'=Ef6t%dzF<Z3_K|nOBQ5$aq0B9RG0mIlZ-2i3@Y6qjd<ov-aXc4qrPts2{QZxA{yz6sSlsbymbZpi(%WCY&LSnWT)q{@IDOJm%?CsqPB+C}7`*W+m+(Q+{Wq?B-je87Q#W*ACN6!{<hE&U$!cMmth|g+F9+3+FQ<7l_xgnQ6)sinXsVe8Eq4-KH{sBNhnWn!K&-4I>etu6Tx=t<8lUCzUjF#`c>HaWaGMthQNk7@=U`|o)rxlMd_%9H*vHzFB3zjMP>Fg4juY26_;OiLU&P*MK!WtTb_`16VzsXM2AL7f)r}qOZzJ`x-lt4k&k>%#gAm_g7Lx8{XFDq^<MFWM1Po}TbF#}s%-bIdXFUQQ;x&3-l3Hr}0T+)nW1OUohPUh|dnC@a@9v5Qyd5e*aancN`OR$JI(!7yo=uBHL_9kqNC+sO0wb3*FsZ`k(Bxe7EzZ%`)p915#q4I91%OKE?Dx0&jj9~GfOb4=irRbU-#&hL`E@~YjD0L6sCuCfQ2-E;*)jveaF}j=A<+jJJXX@H7!~qn(w-$y9&vkjdpVDMD0DRxz%gZj%>`>wl)#^C?3vFe>*R;rKK<FRaK;)TjV=C4=K|aGNVFBeGxuF##;V1+j<;1ZNiINJ^221Y&vL%aY8Jb;YvO2@SBYV5q#o@7^#Jq)uekgGZ4jy$V3|%yM3aO9iNem-K)b?~sgLu=o9pPF%2X=Epy#A2+Oc1rMlfP^zraoaw%=%4VkK!&V#^USit&12Z-Dd5+F+H^re2r_@OU>`p%IJ#krF_#q25IX@NLUc7LJphoR{;z<J||4vqc!GR2q$41y0X77n1sI`}&JeZ<Us?_a(m=shzeFuBn_REZ4>(^B65qc4dnfvHtR<`*BNUC-f|VpQ!F>H6twAgOU^h1Zjox!2p}Ul~O8c2+L)bFhv2ii68J+cGsda@7skQZmX`+<qMJgEN)R^uNe;bm}3xbU!3ENnENGf&Gtfo`KBB)HW5ah5D5CY6(O&=KuWUK`66BB6_f-URuDQLpl;EeU4NN21eF$vD2',
    b'3N*RSAI?^0S7~-t?fKBeDkzeWh`rFCbucvC-;!0F>d?0*W!2IDMjzS(Q7Z5u(G$jVZKGdbF*e2ooiVxnX$*sk~3;l&TP3LU5tcf$kl@amDSXjTup$&}fQP+g5!X8|QX96z6h&HMi%P3j)-2pPs8%{=<liIbeKK5$TdE`{LeZLK(YSUla}5QTsg7$y}D)+zph}r<ipeqX~*z6SgT0v=`mCw>(Es@qlG|LM|0~3?B>vol9xLk^rf|5so1P#U?0_kY~!&N@&tf#QHqAo?(*oFH~&1Qu~ffzViN4t9!(>G(G6c3ZY{r^ZsB*)tRwPGGM}?0imaKqID3x&T|nDYsJ%mKNvhgz*{Cnv?O8!=%WI$nC_YC#Lc@@`bH|eWW)|E6*G@HwhCRMO%`r&>>=-<+X~QCqwAaDn2Wa3W4*Vs5O!uFW>9^?{2TjNitI9i8K<UkQyncUVy7*INZ6=g3n=Mw`y&yDV>y=D`GP^{!Vtb5f-zX_=5j^Ddl%&reiF6(T#F7EuEmrf%9$aBRrz@Ll_XjNYRTC_Y)!o$cOl;9K^-mzpoN_3r45zq@W%n|URI-zEbTfC%AzQ;Ecki~0yUT<G=m#pe`>14CS_N<qf8S&6bix`GT0S$2Uj<_xk|u5%uSd7n8FSKL2{XbLb;(5J(CK^sqQUG;e-(@o5<%6J%c_x3|9T}ThvV!uZB&L=?@x<Bb*8%{ajeexX84Y*!27R;+2spV41wpNLFd}$iKCi`ZP+YPd=+5&C9<^s-n;!e_uJ|8TbD?ko`<{j8v)BiD3cD&oAOby#l|Q_yG0Yz}^~Ld6y?KOv+x`k)WU^USfn8iO#K6z+Xv}2JsipYmrb7z^8lcsJ7sLa@eqj4<?j1shaQ{C)2CCK)QZwCnyeDM2M_1-6vOcslgS5feUMKj;7G}(_D3Y2MgyAh{smAA4tG@SR)Ei^5CdRbGj7-(Y%UDGZf^Dba^lC)aG#_0lh2`LvExBI3D3SCFhTUg;{N6i7Yv{=U@@|tqzY9LHa>KA=$(fM?Sec5gFG}$I3!Ko',
    b'^5`iOB2+$S)YB1s{>DB{-o92S*l(_Z(6(9n-{fu&Ed2ef)>`%LqEyhc9u0QFiyQp{N2R>8c48>#KP6OyDB{Vl=kKsJh@g{OeaDW$PDlv{$fTdt!_S1w&u=LU<EM=XH$+aa72t{LiXsDr$+ie=L`RWwn!!z=|y+<>JF#^MC&MHJ9oMy0Q@9%CTz{d^&LZ_WbwCYjT8QCpw0&F1CX?OKG4+LR>fB^-d4>7SzN&AQ>4TypXzZ9Ha@_v@#|s&H<HEKXA{S(r^g~>=NK{lA*b)-4X6h3X>~twPQc^EaKFRUHy%s<K+zA_R?JMZIUH}5o>!w{yGyZD8grUJa%6ZQ_+3EC!L$W($4e|~yV<MI=jy3AntJEYB`bMq^hoNH<S*2>P30W$9eB~BmHJ(D{&_OFUNR3lTJ~e|;N9p*Pm%(d_#(i<wTfAdtVtVuWc<8{yMeMr?FCEtFLfNUEzn~uXy!GEB^yayY;QK(QGjauC5h6yHYTf<GI|p7sqoHX<+M8(9`-AwCi4Pqx#YanDh<}J?2-$w*nc;DnkPejZoKmwAymg)n+WnpjR-m;i_l?Xvo*1ISv@w2KiWwq?=YS(cfT=5iPu${zB#qGqR-2<#^|$(J_U6Gt7o)3DvGSO?WYn7h;HDx0M9=q<1`Q7h^SWw0wj`K#EVqDuK25(uPhsNwgZdqI_H<3A;KqwPy_9s>a?_H)VYFvwP?)4jfSUMUxTUICocM;Nig@-j5ff;caS$RBp33)k;g!F0f&=v1_vaOQA-|<E@Ep(N)e>ZzPS|Oqoxp$=@^_vN|bC$9m@+n>lio`?wV#Puc~Q%-T>$S;&no+8A41l;<!9BgEmo8($G%YP;CvxqfE4COiWb4mbEq=x<Y*5RdiNvGq+^LrDY~1MZt;!ih0ns5R4F(RVX78b-hV>YfmQIk;#SU&w3~sGD2e*NskwvwKD94OXyI9G-4$ll_M66JvB0Tye$b>Riq0+S$Nrqju1Vu%stIxq__#N&rEPFrFlEClWVa1H+G{e$#{^ON4~IR4Bp22G!mDp',
    b'ELF3w2<KV_R~~)VVR~>a)k2^&=rguPh*TRwM)<S6XCYXYtX{e4pGzf7s&`a4n&NTIj@5SRngHEL1Ek}*ki`>T`p1lA=59ZpW@JVlRMUnR7AJw_Zdy;d_^}JETuyOQ`$baQ9j15sEFxu^TZBDH@*``B%Ee7f^;AbhDW~I~b3X#u;ce&L9ds8bMAkZ*q_n>>N4&5Y?wShtVmD*>&(!+?OGtu<y@#tes}dKO%V8iW5qvo;f0Z4{E<Px>IK31jdZ=rz=bx+r^2y6^_P>f$5cuJsDghsyykqdK6;22ri0TT5QYGVN+i~Io$K^aAh~mkc2N@>Ua4eJXcdHA>LXR=Dr=^0~{8vq`m&Nx<wZI`ROH0|bsBHp3OkE4Vvc2ZQGBab4#!1RAk!D$pEZ_T*NuAj7T=XzSdoO=M0uK`QIJ0|o!N2TFo&ii&1I68(+1x^veUG^;q0!ylIizw`<IFiwk0c_;wQu*#!lkpRRW2pdjGI>#0AJm-FE#?bHe!*d`DCH?ISD>|BOkmG{jj>53OLegB5#CEBlRIul}WN;n4HvTFuz{}^WFK5Sm;DqaUPh9e#9Y^K}#KF8Njymwu;zgNUf(kNFq-t4JZ~V9wW`;;l9>xASw?^43YNe+2G}!v_B@7X5^i)Bzgfy8OioCs90s+QDQpHq1mw!w<I6D*ldB0$oqY0E~>=_T;=e5r%fUyCwH!c$e4aY{X)S#B}=6qB8&tdt0xlGAipINEb8PSkVlx15yrc){MLtxCUYujSLBTMU?A^X(oXTO3mXZdjm)p4wA*T(4GA44^co0F<Q&u5)ui4wj%Mv)6(B3S38*|QrPl<B+gNNASe7ZA!*HR8o=fMUXM|xlJvl1H*dQ=x4SCoEk!q3>Kq^QEPk`{6CaMB-ylGmM*S|HL#X(UNI0xd42=pYZD4U_ufa8pn&23C==eK5^-Rf4oCPQ}~c0`HSzCCU#;lOEKC%>n`L%q-$K~Ns3thCIkxcm|)`O4g(N`e3bh4tNtwkR~rO_XX&h$)4$;;-5bT<^FMwi8L{f`DS',
    b'~tb^a3MR@YlPQ}3Vy=g7AyU|GpV1S3of^rg0r|g9lOLnzNfuO?CRvMbKipTMH?H!RcD8@z#4p6;%Rhc?3w_}bbbc$tnq~vC&C{=w=sQ8IzR@94y3)@wvn;9H<W?&x7<<joG>5ZeVxWM@T4vI5f4m$`%*(u2=c~0dEoLUpe+2<5*qE66g%TxU;UBQrao>EUAuh_9PEOGyeQha0~U9)!I4iU5%av@w1+cWdW%9V;Eq2q^_U(XhKX$9=-YJ1GCx9ymPM1Reex@OF7J?{n&Cy4$NhZW#MK9k7v((UR!)y4R@G%wseyQdL35U<aHw<MSyyukP%jy-7oV(I-vlAcQ~INxC+W(hy~RA#kgz32{9l!u^}4vV>j{LUff7RDI3EyQT|G(PzNe93;2igny}{@wixI+m4xhSo8!DEd^E&?$%3NTUaMVHLAo5SW7VgJkcQ&(AwUYqdPJLWRt%<MCB$>W|&?YU>nSPh&NK5K!d6K$QZ675S$!&D515l1hbd&B%muy@X0f+!IK52p(St$0?jy0Wx=Xk0Hv|dmNrL5djAcZvu{|2DLG+Tn-0|B<G<$apY!=s&OO4aaT==&E%BUm(3~dxn6g@hzrbWHK(BQ#y_o|Zk|3d;dN6TIde6=Qw>V;@$Zx5`^)zg`J&ndVKvRtn)qvDNKPf$>Q2sxJzu&_PE?#6DWhd9fyxNCl>(Pz;#(4R&CuC!UhEy^fkOSw%2uK6QmO=)j?};Fl$r0Tq?*;g@b=H;-Noeo0K&mMxI1%9X&C&DvoWNGWapV<f7cZYVR0Fy)e=P3*lv%dy0=~97*g6yhn5JcGw6dhgOz%fN$@+pt{!iX57&Q*dNtpbE^I~m{KC>~dIMeik-v@AGwDA{rPdPup%RmS_8!w$s7rc!NDDG~NXx%it6;>#<+0FC9C0cj>VMRp7+!nI<0`&I6e`YHO%n(j94eLj`{}vZ#^%Vp)(woa(g0OEd)7OYxtanDe2VoaE{4gI^CEU;khA?%@av9fMadzBMv0Otwk@onxDkBiG29~@vxzh_8(l',
    b'HtYE@RkU{R&<?wY%nTd9}hKTm+S0W>`g<UF;;7^l0^Hah{}h0<OD7e7d4oL^mhGdnv?Y1=o!<4m^YsS#J)SG>dFIYePZ8(x-7xfo#VM!-doqf%o?NoU0B+Y)JfG~78nvYd!bw3&O=p+b^w4yKkCb4Dk`L~!I;V!rWj3i_Zv(~Vuk;fX<SNw7kUEA^h~1TBx8;$FiB#g@l5xbd!HJ(;J3*<`;<GhW{#hd~aWpE>;t6(7*)quqnjP=zbNi*cr##mSTSUURF(Hzmxm1!Q&cXCaH#$Z+nNr2H8eC`-fbRmvV~JOM?}^_{(jx<dVZ4_bs-8}VG<Yw^eK9Y1+=i8*v<x)O5H2_qzl5wa~)<n?s%+uVpueEYZ1G?=psO(5b#d`zaW>_i*)kn6=1sC*LYCBe4S<ScB)byq44rgLGZ+pb7OZ_5;<z)Lmz%WC;QV+@~2>uDxL1ZJFocO$O=ja-DPj9Nll%<>k*+H9M*EZEwo+*<rZvav#f4`RYEr<ZLpK))gwp0$2wquTVB0yk-o41#**Ek#M<8W4^xn=HY0EKF^y(~XG#b3>peL}Ingu&%}|11-C}Qk-b>sqp@2Qc3h$sqkT`JL$2i%Ew!j3Tm#ha_!WVoJKvBkD$a0+1=SDMW)H>rY4qmqTQk?<-4jp>($&VyVC^D8*M}!9&}cJO`@?!#G+)K{9BD$!DESWMzJ4ocgqbMk=`{C2T~+!dEg|nHjDzq^SB(AY|(ES%*C*X3u=dSg30Kw=z+$UUDB1|cwkgnMS3oCzCzZpY1TW<8#;Em;C2OF!Fbw~XnI-FrG&MRBxwv78~ycnrpt3g#qaxS4GPm>tJAN@$Cqq|$(}cjn?hAI6<pd|iFE$@V(+zxVJzDS%{}eUu(araB}LL(6I^dI=i%V!F*_rtsgA?!i+5bSm+S^?>Ss}JD|s?x?`*{=+fL|+S@HB}G)kQ%L3zDT^2~D1i`wca!sj`;MobKn#SgrI;s&C)ok=K`gds*ktGLbShPtO6IK!Ii&1QruT4D`{=-lT^BCrv}@+c*S;T',
    b'_{)Bb7j-^|w^4ugQl=BV=0NgBK9A$*U8Xbiw2)tl52Xc^z1Vv`v)0cXfGBw9^GbO2Q}-pIklrnK(M$B5U^_mo(a%V8+w7l``ktV;w;<7{g1Jw@uMkBcUOKgei_U?e^8TEzv+&S&8lnE)V5YowjM1+v2g{UQ19dItcx_v?Iu8=NxJ!atm2i)T*NHinFnIp9<$xK~s?^^?-duO<0N%Rkr5tjWu@xxT??*Cj(kK$RQ2vnU(?!B`fDM39FQsM`*Ey!s)Ied|@hcTl?L8DR{S7a!<Z(VnZ1!?`|L|-`;Tfy*%Na_rh#>V%inZYLBd7v5tOTEHF7HqnUgojN`XMJUxyXMIPdgsU^ONb&g0;NX#$qMYPpCD%#~TVtO|C+%dE6QMcKJN8|w2i?_D#^L@7$37k2iA{md}#5Ml5uZ_cfWRyarvA_^(ycUYN85neq3s9)jq?=?dk`Gj6JeK*cM@=}4+$QM~kAYKcBo)tX9Xt89!1%GAdfH@^>JpDyiimTI@h=IQyhiW21nIW$6e$DG2lh{6(nSxogR=(}pUAJ9SA&0z0ox^!Uq*(HH_-!}1bMB3x9=9^bINmEDn%4NyD=tA_g#K<l6qTVnop*Ac;;?EC{Hx|mF75?$4|Rj8z!hGkzP8aSfuHyml)YW%xo*J0|+r{2bK&Th^Iqo@A>>W4iS|(9Ge{a+*c_Yyf9fwNAUJX=aS_Jy-2+GEk!p1Xi8JpDIE-BJ*HZMXv_2|T%x(-+Pi^=ivXiz!%)hxtEUA>kMzj&GfWFL6VnAzB}{S+r=0U?W~aj-4Dl3NG8}j7gIT-MTZ0(x(GWW+fLLB*`nHr7<m%eah>+(nUK1-=@WyN<wMwfhy7h>HChQ<}6amf<6{9Gi4JpvxPT9|6t?N^;+&pvr&FLO?qo!>L%i8;ht{>P$`oS93&`wW7#q`AxHN~%6-JB+fGYe}&i}tXlak=+hHGjEHQ@aGG5i|h`sz)tDF+V-i>3bvUBX|xYB2bQ|iI!|CJ-YF5H#QI$cme;<e5|uISuyyr1Qjno1S-eS<',
    b'htN!9pYOa^b|{6C-=o*uY&?FIP0(QusWj`qyW~iFg%NTmXUbTpx}S8_cpzeEIGE|e|hUzgwgzSSzRS`7f@BV(bWYl9xi(I)&mS?R`chFhpq@kxY0<OMJO|r*MaU^O36cVcbbv3L-L%+9tt-N1#I9vc(CI*a?XgQN!R<;GR<<6U9M`NTL`JF7tA`bBgDCZdDiMKa%>q#De5Z2?M)Y9u_XJrU^a|7@W~_aR!pg}-702lbzOqBYs)Ck;Pc&GnTw3+JT=&QP*uYTyn@cT+oZIOvDojdrGq=~ZDWl7h+_|im4kg*dY)0QMVH6nAU#*-R-%Sup9hN?+{4kvMi<WPYEd?Y>$L)28?T1fquJUYTb|c$7vmW1%d!=a%Xr&BUu93*hgF1YYLJFEZ=q1}yW}IIGkbI)n2MS+{$~B&q8E&ip!u)Af7@tmdYAu=u@j$SK&x|5R{~EisVxKTeO(Ht?oQE8Kc=v<rI~uw96bH+heaqNtO{}SMNz6!giw(|40KB>N}aWyp6V#MO{_C3Ope|t6Is-UojgRE6ujAq>U8nswz)D=CiwJAJUWrqwkK^e$1Fe?3PdM&#@o^ZY3YWlkOo)}qnoobdQ_j+jA3j0l1f|E#$v;$-Eo}2!0JMEGxM=6->zt3yxWMsTSHT%V=RtleX*EXk6AiZ{<W_&m0xJY+C9nvZOBiECdWKtTVFH~6Gswg>nLQh4!#A+6Z*?afdhP>URm31ROaY4!9kZPwWIOP?47RD{kCmsOQq-ug>a+W(~%ugJ!q8ME>KdSK<7@Bqrji8jjI*Aqf-8c;J@)-ziDvIWy^nD;P!{uM1|h!cZD0ACXwo6I48IJ{?sKptmw3qdlu;6?XoGa?iWyH=0snGVG2lxqo=bM>|x;2<n;8~UOt!20~n@Obg7&it+8{90Izv7yS1Eq(yw^9H`HdYo}!0%sMZMHk%+aTH)eRWt33KPQZ&MXO!U?Bwk>6udYAycL`FkynqW9(T`1qO@Sc*=%qBZ&Z)bYWS8ZLh7G+!HI~Vp*N2fH_3$<2WZUOcr',
    b'`cW-VR6lh(v~wy17Np;-*{DggwU4#u+%Wv%pe7-C)5s)ux5yARD7LBv2(xX=Y~TtJIMvjI%x5rET`gTypY5PVIv#}&=juN5W<V<L|5JqOg5NZ7j63>_4#GXRpU>_Ro=9MHo0b&-%d0Uz1G@-PmbQ2by(9GUP|dr{vZxVXQj~edB0DiKJ5@{5-eH}pU6-|1AH*6PF<7@1ys|}A30e~vx7%`R5zyIa)u#@t@3_NsZ}n6%K55d}?h`>3!soz<!jS_eXaN0F?AN*-m7|{}7xNa=(rf7r83Rw%M<H55=%wVdl5sc`cMq*G(*lj)bb+MYZX}zPsQf@1b9y?r@l`mgt{$JOf^gl$yvUUHQr#j^4z<AOwslBWpIf=ET$j5htQbzTKUJP$!=C2&oe8N}E<$-s<?&R^pc4&t_mnM6Y+-xxCHQbH!5wV22n`}L2)Aot?&lYbRTY#5ux1xl_6axDdF!ENt>%C8`Z)4<x97_fh~p9hN0I2>Q6w<u)#N&ABqCOBKgyeVizr^(pF(J(b$PrMyawvDQfvMIbkYpXgTiLzG^_=o#6qW`Xta9bRAn9b^h7;F6k>zbU03P}dfZ>rS3rm<3N2cVSn+bL(uJc=_U$Mw1H>{l-<sC|=Zw{q)x`3mJi^`?TN^8{;X;kokFVLI=zZvLqqd}{RZlF;GM!r}S{ft_>yobR^Ip|^wbkb=#aWXcQO!)O$Q{F+?wsw?V3KmDc<G`ynrFF;!c1*8QKoXO9NW&$gQ-$wzTwfWV2_=|gTME4ll6EDDdFMHDb#>%z;The4D0jCW($Jj>EW4x_OBXfm#dkwK2cYoD5qH4Z)mck2x{dQ%Hj31Bvk1<Z9URjf1~M(HaN`HfG*;Kp7p-{?YDni3gfnjV=WWj7&W>R-(qX3_gS!A-`&CVXfhtB$hMm2Y|0a#C7gV6lY$PTa++%#P>DuYLt|Z;=Gmh&#t9=jfUBHq0t=_Qbmy;HI0l=5BF1OTtptJ5@(|m2C@U$XvhS5+iHk!_?zYYPB0E*bRdHn$Ty5nOt8E`)+OI2',
    b'~XS^+As}uuk7jrkaCygHeP3hB$#nP>MP&`J>sofIp7Tc~iqIdg80@<w7&q1}Of(U_CnCOZ^Ge#7R$ZHoU)9|Xye&t#_P}LI@Uqw{d$HwBQ-P+&KALL4!2Tt+x_>J|x=@BvSpRI)b@dV#S2fNB4VaKt4VeFzw&?{E^v*t=W!YA?<L079i=HEsfx+VpEll+?E(9|gotvt7xk_M*swYsL82@D-m<#Wqac{w6pL~%>-Lp#N&A}o#CBK6kLS5fiq6QlTHqTSEXrFgHOUOnfg2KyMW+{}7r95%|*iRMbn=@`3*yOHlO1-79^!x*Lx=#yNDw+MU>h4%&)w+0QZ8q+{m^e!Ed*N{<55WTHi?I5*MV6%hTXVVJ`JLD$mFS`P04X4#QfA`(1K@%vlC!a6YE?8A2y!#}RO87n2**N-bvP#LsYS*hN1*<!^XNukML8Z?(2*A2DH4XSV%u23t$f$%XwWE^ixHVCbnsu69p)G(y5oQ>(jZghne0Xx{WnGBS+v&M%oyfKG#p><7+89eiYihu2JegJ2NUbf}zIS6STRM|oBJ0!_YqK&@O2;eeQ%`@}7q2m9Z%gxdWWq|}qu#OAOV^@QiaW%Pd-We$y3s2H^G!T(xyoA+5XjUI(e8qoT*AGw^0X8&etcq^VUu2!)}#~d@YRHZovE8B3U>aUdgV(SUg^SFW2~vS*5I9nuPXokLdxU1&Un(b$$NV~UP>jo?0N3fP1aSCrw3G$UsFZ+8>$7T2UuucPL1_DsjjZSj+E)Ip;!8X@|WLR<T4~_`F*t&zm<UEo?AXDmKf6E+w!2ZnI@FVe19MzzPqx0GEaGr3-JxVz9ihQApMs5^)_BxOYNH~rZsjD{Yt87^o?Z4eg$!{@2gDpee|bl<c2lHsfuLN>#7}nPfa6TzvxxveomeAN3SLe@yfjYef<32fX}|#vKyi{`VJ)WPB-%M9r)H;+4dgE5WfL&`Ez;8zXOAJD_?h&!8@g=Q+VG{b%%$Wn!%IrPc7Wd@%eqIg3~vm3by(2e?<!6Z^;w<YRtHyQ^m',
    b'Jr#oa&bFF=a>{M_oh@a0Cx2fiO!ZjBK1*CEl3v&{O>TGMR#nJl^wu~YwaCf&X=!Ve(Pee5Xly_j@6c&2N-8?VQd+p<5h(#`gr8E`FCt{QvU3$ovaD=!KYUK+j$&+H4Z$|}?cW}^FM3|3W>=*AiNu^Ye8T7!zia79476?Cs57Ngr=8;Q~;Ifq?EaJvWlG-<8rG?)6cnkAcMMLnzsfO}=i&V~it@~4`UXq1ZvL8cN4jbV5Dec!*=_0gJ1wYSCX*YL{LJ&Q$WVtWb3<wpdKe=5K7v#QG`fp6T0I4NQiudjbHi>XjqB*{)v`oI*Ya*%jYvg=l<wOv<g1L=}^i*9FF@Ibmdhj*<tFCnp$OOtOx^}qGx%p7L_iPYTA%8$I{pT)*q_d^CR_lt6HKW7%=Rj9Y~lc~3Z(#DoZuSK-=V4IG&mX$QrB=Fbi0G0HlHbOYtXJiWSTVBt(*h7E%$6x>cC*0bfe|z7Z=)=^7N?9YIaPYto$X;LPb7S5twH6kfY8Rn3c39j*6ql{dGPS>ldP}Rt=JH;w-Wun7_+==Y$l?mMzcR%z+yG@xOWdAEZiYsu5a!_WV5hpw?Nn^IfBwhY+kdLcxMoK}jK02r8-L8KIh;t1YDZ~>zAV3RcSh~)M!eL&{p+uP`^*3E-~8`?6*G3_EnQbp`LWB=*>OC0cc1+RUB#~z_oMuEy_Gvu4lmY8m-n?;0*&l-=d4}M{<D847I>h6Tf6IxKfW)ihlW}j*imDv|0{<0zB1P+U!j?O^o&J5;~hOlAW<$lB?i?-4<3<s3*^%GF1C6SbEveo=Xsh3`5`HHEKs~hT6xTAOP9jv-Qo$4-KDBeBw=9NSyL~=>3-fk$WIcD${#;rfB%=to_{#~siyH85uYSkM%*-e8zM)!Zd#AK2|w-D%;i@d95&-;J%njzi<<;FQE05$!Jth7l1M1NO|$Pq%lT}l-g?;S%)yJpgO&?9Q=?|BSHmP-va4*p70)BrQJA7UY;)?^yffY)TdwRUPdBMyPCv*6Rp-E}Q?`$7-+c5{HMp3d&B',
    b'{XkUF2<!Zov%Hba2Yhr`2PtCpvGI-oR)g8_L+BPCBdHvD=OAK@LhOG0s!((0gj=bel+x^6bt$w0FzO=CqD<QLbvQWZu{wfAONxe77>as=sw>qS@F1<!dr_!ZWD%tq>mXjR{q6Ypu%#;bjw0sJCFKtZEHAA;vBjD+t+r?6H2}DslU%QGX!X-I!8HK1CawwL6B4HE)X~HIO*d?`>?~#s0i~EJGlpe>Z548k%#f&6y%eY+H9d6~@5HMgLYem2qV&D}PMMFg}@cE=W#E^8WGqhOh<hlJff#&Dhp>;>tNXNs`S8={r-C55{`c4<|&965w;zO3wnfr#mf`g|`O;2RwS`daLkzX!8FN{`;7qqeXON^flWpI=gyKPhqvtaabyk;<dJfo10@Yi+jvACjS<AX4B(DWny|X#cLe#<qv~TeC~&#jkaRV7%FTYQiHe5*f;L>W>S>)q_SrF))xU$mea8EKBJB6(J@wI7dCNXUExSq7t9SdHppR@JiU(MDCfYsPCir1wH@qS9hW(u&utvdHN?20zmv!bfFb#WxLjy`sXzQ$ni-!>0!X#-y=dImnsy`O5kKmsSZ!DaHKZn><8(un+!u9Sm+&ij^0UV!Vw^g#+Zt9BmiO`cr@sQ_i@UnbTDhqHMWemT6Zg{}Khz;{!=?0;-iUsF=+OfJY+>Nx2@)iOr4MwkIiWtea^)B~KA<C7Wv#8YGkCKT7GsKAxK1^;&O}{PF=01hkdKx{gcyOT`k2>|hWicm-@pC+PyhW}@oi&c{{8K5zx|{5?gtsWg5|7h$?UN%)8kv+IIWnANcrIv)$+`?qaU+jt(K=T4>grpwK(ya3+S69FUC`@e)6uCPL1k}nP7zoLz|9bYm^b)8Afc+6hiT|{KDuxyJ>#1`Gs5B8{Q3iyl5B;_5g%Y;zGCXCbbBKsG&N|)7KqgNrPAuQ87iuP-Dl`*H=)bNO*$$u7>;uR2jNE4&rnPL0v0Vd%?q}t>Nl8?4*0T`zN>bTkOu6-XeXu$y+Pa77%O^@IIqNUkQE|;X%W!Z)|9&s',
    b'r|z&b(B4H4W*c<oLfT?OkFS)-i%V|TNSo+7JCydU?9X>-8NwyRgwG)RY5GE)$B&DG`}}B*}R)P<62)+cftOvifKJ;5PpVibAc&)x{tm^>6N=4HOx(G@wfCaS=DRu`S<?1v661+>&qhguL{>UmtI;beU-XinYXEZ%VFM>r*#{|7r%NQs|YhW%sTHqq^yfS0O&3;J0FP`c^nD%7&Gv>k*~DLsVPZ{A(Nq=j!h2k=Q`sO=L&UPL^?cU25N3#Fz&c~<DJk<Sz56I`~AJ{9_S-FsNk3Ktkj^YrUMZkn$f}jTVYfPP5oAD8HJ2?d8rNC?7lD-Rh)6B78l(%sLBj+v9+~Zd5QpAk8Vv3?yovz+kfplh?|(Ci{<c`u%p8oR(gsO-uE`rzWcH<O`E!>)?En$XDHSgPi-g>mh)_D6S`&MPBv!QT}>;sXN~sOhljR2*Yj?`(%K2n8nS30Idie;!4&$b95kAdcW-^hi`-pPmyU>7>0Y$cL_HvQC#R6ccYrRyV#W0zDa_t(Q1V;I`Fl{c$Ap&4YL(epZOa&jyFVj6&mkXP_Jyg(Yb(EYX)=Q+#4Er1b7hEKm8g10Q;UYhdnsO49F=OAPczEcB8+XBuG}~lKff|`6PxmD>0(!yGOR75(ocqG=A&n)sGhe`x{ggFHq><D@`Eji%HTT)C;IE`MB33gO!d2m?b^LupHiGvdQyUAjK8@7Qf*a&v+-+t&Le!V52}Y+DDbIsPSBduym|>AskHJ_b!@^KO0)|qF}JbZyX`fi4`9QPfwkHkj4T!KGEP`~D%j_vZ6(7)_(y@BTE8`l?4cVfO++8+T>~vDT|3suw)4_k=4hM9m)j3W;qB9+gQ@L^r~-XV++{UYmYlJku5G73pJJ2fLEfwox?L^do#%jMHN^|3JYKv3NB-Tj)$9XeE)M2ldXAuU98!-h(I6nPJ~lJZ+};>}jXvfQd%_`rAerP+o<L8uT_Os+ti8aCN!RBEMym&<Yd+Fs6>@->&)2KLm}-YW+Z>}QTW_OmZl*l+E0q~NHZEg>KRMFE',
    b'9{X+{PZ~HBT`3GZXROw>n@^6FRTVzD+ch3jS6oY-9(^2hQCUfPo1q%%uY4(skk*6Y$eaU}mnW9Ql9$;9rwPV79WKE{ud)QolZe%IZ8RZ<fXZj;ZnPr|){?TINZSMZokEUCu_Jii(D-ZD?EBkM!|qFUqp^~nP0mstmg0KGO^a#|<KGC%D7(M5oXP`EAEvtJ>upyB^3tc^7O+s0R_WEojH}Xt5z~ZdLiF89hhA4@#bD=<deesc;SYiw`8DC<KEfIO++42r`|8`NRoKaln@@YbkS2<Hpt9UUP(7=RrQKPWG2Rm{MYwK3^UL9A8D|_Q&eMb5tP-O1R`#MrQg7vNJ^XorURDD-xO*?9*7#YI`D&ovXdF#;T3OUx;-V*$Y1{HU&QejpTxEbpIU6a_!Wiy(Mq>+g<11E2WIPxe4T3J7Q4{li0Bvz5wM=ZM6Lo1LNpDXtV=BYplcdeBq!r?teQ%5Vw3xmc=go|PmB4^Q%jAr6z_Lbt;sa+^S6p)!(x+r>;5k)zVV|A2^KxZgU%nql2sd)XY*8qV<xwruMb@d#o=9weAh*w=+)J?$W;N&ZhjE(9RH^^O8CQZfHelD>bI+Kel2mP&T=>lGkoh~yPe~e;Z#DF5b*@&co6bET8_j77g|oGwTN)S?in=joocbO!lSIWt4r4&HGq5q9UH)hZ33O5MG*s4tfK@s_w0i;*Y|7fvdr+eKvX#%o1}@8T@DhCiF){2)L+!qZVC$AEeA17#c`7?_j!>@6UYHLPj@j02>o$tc*7mUgYhkSXp_=XoNGhJ3Zv|eQ$6#v#e}P{gV$ukrC}l_6SlOKsYJqw&%=JY!YWU`JC*OR~1zFa>5O|x=MAx*PNq1*|<vt>rsFBf^(<IUL{uby8Gx3EwQCAEwdSF5-FkD~;jo#J@lbcMF{r2_0V&HE%X%H>7RF%BEhu9tD0yga(qO@UoOWN6@H#F;FZJJ<q-cfQWc*Gc15x%AH72gDHLR0QOP|x;JNFw-n6xain5|s(FT2U^BNVx0BTmI%r)S{lVm62kNh0Za',
    b'jg>o-AvTt|L+n_)T1V{d^u-iC%geQJ@b9!Xn+ge($4aUJkSd_ruDfgNS)tu|<XUJNb5_=PvQK|{%vTKICZz{WeUfR41tBcs1!8N;Cg69w43wzP7Q@z=iT&lmbj=<t2Doss6fq~EJ5z@#sgKhhf?^Mr6?{5*&N<-yqmjc}C){#pvAD$R;m_MBFto>yHOFG92;#Px=nS)mgLyt->TD#bb=x&+9TFp&@nXyCgGeTi(pL#`7-4mK)eoY-F{Tuz%%jsLnCe%5Q-6-@PX`Xd?!Gbz!<+n-^3<ulM+CpQRa4E;<867?wgNn`unzVy+<T=bky&}qPtHIRaohM4cl#Y3KpF!e$^14CxAISHw@UqYT^pC&({ZDxMfB)^bf4<#Bm>`_>UT3f^C2O|7QOA8g)i*3e=l<?)hx9v}6RkG1OK#9Y+*-@-^qa0)A@Ve~{J7CdVi69Ov}|4P@ZbLR*T4PcfByWNt*Bx%9aPIJ%De1)H5UVau1)RCRZ~Q6LFay;(}@dFSq<@Rz^u0Fb!xl)YA4(0<etZgmI1V{REEF+TIU)#mk(NNocN+gGnhkLge{B?#ps>98)eqd&8b<!7qlo|Z)?*~b8~uA={D59+%^-^!f{snTI;o}bhS0nf+_yg0ZL-+OF``a(R<YblqsZPWn~Uh_t^dEN4YQw%vA((({-CFkfe$7+<a8uh(lxFeR)$AO^)K$%~z#;!2Bg;I&Jn+TW^O3BMbzE6!os9;`Ytq82bZO%y3gVTQTk;92k*ZQO*wK829ZH)RmgrTg@$132&vxS|OFgpOS}sKD+sJZn3m@W@8E1Pu<rRUp}njB6r7z_welLCumZ>H+5Uqw_vX-R_*L#FIk0ZIx^nxjR`3XsG+6o45M4Nb|Rhvp+#iD%Fu7O@jS2}cXlgJ$@fah*{?)~_hP6usX~l>e^-grz@VIPcaOVQs&<Fboo9<twB8g0y6EcF#`fJ4->O4|W}cPAr}DJBci2mAT0WO@9{P0LC9rC5G)s`6cv~Trv0ig2vhoGm9Ws|EyH<XovC?bLaPm',
    b'_Xg(|I5ALv(lEeq<qH)ZMEG>srPebLx!5>`<u0*eor<%iHPuhga#N~=ktnRdiSgj+!4k5y|5xr86Flqd1LyjCd<jtf+Gpgk8`MC)t4K-C=Dm$YxQUfHQPf6Y;Q^=9dt4VC(p<MmU%bsTsPtGCs%Wc3M7oh%tWGbLd@4RiG92^1d<0>>ULtrgokbs#KYwminWg0y!3Tw7<dqTlvb??ZVva4a7;(<o}arjWP#1frt}Xdpf{qSdfWibwwu*ZJM~^`_L}=J;ABFf}Q=sN_>a2X9;Y)FI^7QL~%%_KB~k-j%IT%30qOiB&3^DzF*o?WO7VhJWvFDX-oLRltlo)rNAKZmXy$hdzr`BRnPPiSjv}E4hhmJ5DfR&am`SXEbzd*~6n{EV!n7Wh@D{p7!0RnO%l^4828Si2Plq?HV_kH`3Eq9<*5w8m8Q=dY{b>)X<`0?IHCYzy15aKht%``@hVYAI&EkT7~NMHu`(Fmj`RwaW{JduW#1!sC}r_=s2-dWA+_nLFzMmCa?&$pQp_?i!bIqT8H<2R=?Wgdt`{+2%pb)Mm<xm*Eg;tpgT;=XuqGD$mwnfKmW__T-$H9;R!RmEsqiIVX-<)m6XTi<mk-YBckPcxN0JQHd$cpue@MsAG#VM(cncQ+fk+T<C+5%b)=Gs)vRFOpAzeVH<5ayjbgZ=rn!1|S*{MkXKxt&=FGc$AJ3Se_bMH2?3qEF^Eu*s<$jMB4TCg7eGODTA9qukLrn$LkzzX?t1yl?bs-o8CQ3uW{lI;I`l&{x?({>K%Q{B!(Lp(A{@ila`{I6deoh?#_lK+6vcafSuEnLIjWE{7_V3@v^r)U<jeNLq5Z+ui4f&TD@tdLdct>%7b4MRMskfW#Y|+N}>k(6a>3P_i(s~FJjC^$WO}C%ybH}+s{dQh%%WzbAvi78>4Ok0_$wbG^O0uAvUWbl%^m{{uu-orVwxfrp`oqzp(wQ99o@~2Gi}cwz4|m9kz8kWI&hUUblkMdfPZx!Dl*Jm^c&?a$MyogkoA*w2b@B)@r4$eMQ~',
    b'ULjvP(MYeCD1jVQNdh4XSMIj;+~i^EXsbFgxLP?_OU?Q8BwhnMHZaZVvq(3Ej`GLdNj|bzIBWCaj&hhZx8`TDZcwva9Mia41SglE049klks}+QVS?Z}zq%n8b2rmeIn@5jZB75h3s-En?BNqg=ent)WWgO!e7aI31Fx=EL{7axD-}F13)ws}?`Fx#Sj85JoYay$SY3l4=EEj~~~xw+o+UgS5f3W{Ck*e^*37s=K4n+ec+@YWd#7o3e+r#UWoYsa)JD(&Zzcsik*^mR5ioNU7QsW+LGA(_MCo(`}XJEtjOoiFy0pqeo_wpVJ}<)%#523Wq96th%$Ow0c;{wUw4#m!ig%?y-7Js+3$ec4?*lR$z%WmD)L4JLyo6GSMO$h)VmaaX}~2?eN~#7HR7nG8Im%15HhZ1l1G8gz4|-gUmK0x6fdx9PAg<vE44GStHrn_D-$Q(#B8AcpfNkTB2sv7dV*X9TmTv{gai|NnVe2R>y;TtWnoSus5qB!B_jq+}1-EPL$#gYjIyiaFpKKbybGd6%{c$ni?o<!tF{{p-gv#D*WS>@clus{v(W~F<%TtDI=r$V@9WZ`+9Db-|k}JO364<`g?1fb9=^?^;#R;Nv-M>?=AYLYq*5Qv2#mtMit!cO$m(l)(E?LEwhYIB-Frg188=)f6Y@wjq<dvdhL7BDLRX&yumaIS+TA_uL0z@FU)eyfqhp_RcqQ4ZAgLDr5x#j?5qcRJ0#u~WgVm%WFKRjxRNpkrWz|MA?ok;`Nk}(xl7!KN(-*Z`;qO4AWnIhRYsI*ZaAo{Mvcv7mZXi!gi=;HG#b@Mm$IY%;%NZr*Y?a}KVy2=NbRlas5NY30u8szCT0Mq@2OTqiNYe~%$I_;8ldC-LA;_Bet82n_d_N4n01(PTtS(RjlF2Iw_T$Ik9%w7_qd;q57icj@o@Y5mzJ+h-N?ps{NPQ}ntkUK9)I;3=xZgg8W-)O4flHMoXuBR;tcj{H;KP$NK{8V8hO)f@jeLUsqdXf1Eq>3+GhcqJD9p<Y{x5+tIb@s)',
    b'T^v5c96<eP6(O~i{GGT<|biMDZiy?`n|2PSc^2y$zCJ3bdeHg1|xzG^wPhm*rP?Tq-SjY>DtCV+t^FoQ@b}x7fTl&!<yWj#m8e*AVb;A0eN5-KXN)<j8pQv-BwzoR(y&CVw*mtEtuv&<H1R(W+*JEj8OCFu~4oG3U<<h_0SzPYde_)8AfTaQJ~e+IW^`geMhJdq&E!AnQzqXk<c5#LN#s!9J3~)?*CX5MXyCa)lxQS5CN$6GBl-UHX*&+|5V(A*-~f<3$!gHWtu<Mz~Nv8F)5N-VZP7JD7&+k7Fdin)hPm9j<vQE-6tJ>Zy-N%rr!<7E;N4Nisd$tx%r(a-gZ!)TFK}Ff<(?5ayosBjNj$S*5hlnCaZc@KgkRqS%Te=4WN{Jsekv%JVIpZu}Zx#d=(Z|qSz`N6q$vH_4u?7Q+yJOyC9RmQH#>LNzR+ub%S=3^%{`3L46Q8Hak+cb?|7a&%a*gsY(dCspg<$;5y`g_t(NoSoaU}z(J~P;Z_`!Q@CreQ-iB>*{PZ;Lsmb7Ij!0<#1oN)>792`-@HW@(Bha4>>~Pi%_E$aVJa`8eb}@BF)V(_viq_zO9{<OKGJ#NlDcU4%DdasrsD>&PVV~QpzWaD1IXBSVKKb5b_rNCfbN0bQ~atXdfrHE*A8g!C^F^4JCELv_Ip~*43+*M?eMnLa*ONuZv|?N_v$afQ3|IpujA)-giz8AqxYcpqo!_-Sf{eRqKp%33#6fNnmX)p27g+M_m3Z)JGXGT_#?L$IsZG7k2c2lga8(m$BFX`Rp<F#GN(S2z?u1(+=Xp)vfJ<B@`O!;MA3=cD8-Wk)C4<<OZDpz{MxyN?CAyLrkt~m1*v^y=BLJMr=iY>1vZ4c@0V&4Cr~94x@qhx4#O25XiH9;X4C4;Mdo-+um$FB_m1l9raQn}AJ-ipXf9F{7+a9F)<4uby``++X0h*~SJm7M0%kI1baC6ip@JB8QYG4d%i=3&8wNAE5}V>p-vEIHPgor~O|;teWKC%)ZEmTk%R4@47(FP`U0W;Or!C2A',
    b'bEd%QZLh`R8}6&29z=|;ZR<6A5V+l`=eDR==ZQc`_WpIA+HW>CN$Kq+69DW5U#u-s6&rDAY#u+`m>p9-f3e3FWxmOVxBHawJNqcohn0E|#~0As8;vdI-#vL_LPHc$ak&NK)Ejl-6-`6D_<LGIfl1R5KV}ZUw+u&Ev*St&aA+jHUCq=zK6$Oj5An&B_uJiJXZPh8Z}|PKeWWmU-?=G=Hk<3+JGva){nB6$)s(Wq%DAabK?GVxlug&UEZkbDHi}=g<(eHwYOAqwyA3m43dO;L9k^Dd#UN!3JchY6{NC+;@q)+p=I=d(rByw}J8Xgq-uUr{6PkLwXHlxOp@KHVv>|c<y*RBk&y!Kn3N1zxctD@{Q5=O@a1?9gOrr|28&a%~#@T_WF^eeuXiL?`njNd33tQV}Jilr6#9S{@`h~--ShXCui%91^`c8Ebm<9@_GWzw-cr@S}Rxa!`n^kv8^Zh#gv0g;>-nR4PK?cXxZE(FSEoUq@!{w@m=CXYz_zZ>fdfpPYLi{wZEcswtcst*=X@wxg@-wQu){u?WL*u>vAOWz}&e9fwZZr~$wicrX;g4)ug}@kps74i2U6giheW;vSt&*(yGv$#Uh*NsttrwchW27l$Wfel3&0-wNJUGvwL$$jTL+D)35#5{*Y2=~y2FW_OpzRpb(iUloYCS}#{oYWUZ<qr&9iVnv4f7H!PB7RYXSGLI5Y3xTe%?Q;bMXV0|J8R-(E3dvfHK_@coku^#-7qOhw%-Ud~;I&{ZW<KW_k6%Sk>dO%N_UE@a23vE@tSAo#YD6rk=>F33IS3=0^VR4su#5?J#R(0BU*wlTn(8>=7}yxuc28h?MqcQzEXF3Yxm8px@sdqMi48uZBY!ZVlo$vO`V1909t8U2Zic2@D`Z)mfyrkYlm!*KR$|?CUg%c20R|r3rl%Iqmbk#5)ZS@ZJoz*?bljwU)yM(>q1gkr3AQ_x@k|%2N47WtgM)S!~=VG$Cb6eHmZ4qwSI7y>&FR#_8Up<6!+|DZ8WT8Q4HeUrEhqL2?^c`Y&Q',
    b'Evz1tzbJ3{I!jhr^NzC5jc4!(zEgIXJ>QNHYXJF{&X|qBW>TO<cPlsg!_z(TpZ<@op?Dvlg+|~}8vejGst_G~n1|51TPqgc<b8~5?l5ygs=W^S7B_ou+_19P*cv04@HTu-_5WyC#hjCQJUV-!=u%;fsT{RC`Ini6g#>%=gn;4(=P(~J@xBT7vkRBNqO@=UJyS<QUH;I!|2U6*%l@gM^m9Bd*%#$4P0zu_hr;ud$1;w=5x8e`g-pBNuufCX{I!*CWsnHS!`w?wRWYpPgAv6_G3$jqFuvd4NQp<pW1E3gDzODjBJ-tf~r*ulbb<1G|wcW&O614&HccY!D5$GBlrNN5ol-Z5#6J6HjR37Gss5stIo}hu+TS4sfP1?+CxUV5s^X3LnK1{Cjj01nmIWs;ZI<z&$+55(^nS0+>d!hkbx#P}Cb5;a}YDtx;trx-VqOR<%l7GO1xl4}{Pn6xFW#?lQoeVd5RD2>LGU?ZL+UY&kn4@~usE?WZciYLV%p3d28qJK{Y5ate9&*N-MnUO`OEw9ocdZT{IE{pThG>*CD{z3@J!7Z+<p+Y8T5HiLS~&6KLQTTshMSBG+*-#x_2!l9%B9xA1NW456~w%)Ev`J6O<V9xXwuym8ef~`W6cAwYlnwnZ+EoSN@>e{rLAyf1KFCGb*pzNn_LwS9A6*Bk+8R5WtaJ{_OA@Dl+WJOB+!y<<gW4K4p(kMRgK7&<sf#QFm4xF`u!WgV(v{Vj=^s%OGOXWTEofc$sK2PV9z5u3{fx*R>ztDdTHMGjLlEQ$E?Pp_XuqC?xKa&xAGi#Z0c$S+o?jTMPvu&GbJ0I+Tj(iq$)H0twpCcN))SotBHuDNu8?!KvAe@&7<6#AxT_4Cbo=1#zmycDF6-nxoeS&w*T~GBz^i&JybKDZ4VChR|KLm2RZZvwuyk>Zkl_4d+e~O@eUerl9C#hXF&C$PDK{GH>B{tPV#AyF+T)ySsXqpp?tWtGBU*5*)`?TU%fSt1v<xw+B=Q{S;~<;27X!;>(1dNzFE!52@}OKl|x',
    b'PXR`rDGJkdjld>PQaL%LTNv2qq|_4}>N7MHyF;u;77O^C$p+lnrDFY?=`4O_tC%@W2imB0eb2G)DBL8u^(V7Dwq->*I87(Y3CKUXRm#?q*c4@NR7%OOVQ0<F(0-l<{ajUJwNYyYZ?2dr!a3r!y0bb?Nmw-wI-OqHI`wx9~&_wf@p<+kkbk3aq6uYdm&KK|$5-c`8p4l^k}_3yv^_RqJ_7P_8fw~y5Hij^_y`xCa$=@f@N6?*RvgMHp)tfBtj{`J?t{pEl7_x<<3K1{XyPJ?b7H8EHz+^ApF=T+@<n!TaNFoVwa1ZNG0+Aq{+zC`+}Db&kyYEnE3)4&A>*&f(2xuFfN?1TB|f4sf@rxByC?M%D4@zfhGqlrZ(rW(xa*L{2VruS=ndvi^Z-R@m?pDRWX8h=Q?1AHTNfBV~S|B%S2c2pHtshtsbqzA<Z%wM85q=v%n$FHMWAMRK+_ZaHLC?iUlm0dSb*9=xjxqeqo$-kS-{E5_P`soO+&#*n-pMU#!0Hu6rw#9Ub)!#iyR63?OqFbxst|R$wU*w`Nm7iH{I+4Sm_#RR3eUESr`mgrvd%BO8en+|4(3EToRtM@vb!yFvKfSrNu;fV$x36m~Xq(rFub)Y(uLWQ5|5V67o6XiQom<QB+uEATR@IVc>q9Z-Tt~Kwg?L*j4N#UUp13Tw)RF1xV$&X~em32Cw#Mz(H|SAL{_GycB7C&e9CyFw^PSnGVCLF13XcsqL7Z&UEML_g<T0}uqOp!1b&(6kU7_i9f}5^aEVe{(clYb#K4xQE<(GOKb?vdQPqELhY@3sxCE#HKvBoY8ls2NWD#a9nZUb|+rQ0qZtKi*D#7H%(8>oy3#{BEvuZH(r@Bx*ha&S&HEGcS<qc#{UPyp2CRs#Is0jH+5_qO=Cwh`&tYExQ;dcB`D24Alcd<A)G0A}$<!&=Ggdijn#p?lNUsS3z#q^C_Y^%3z*lZW`3B(z_m0~7qIX%)Zdv;N+~R%|J>nZ)?b444nIE?CRKVur5gU?Cs!Eb*GI=+x8q?A~UXeJ',
    b'fyuNr$F+1P#0h!Z0<A_F7(XaHZT*N14Bz*0VgoS>!pkc1juXZ6*Wv@vRr$*iP+EN{7gU%nsEJvhZ*O69=(mS?wkiNBdGG7TFnRv~lX%3MzN#Cw8MXzgW!KrBJiODm%4d>qU7w(p9W1>w4So!%PKc)Uw1?t~XVaw=xREQ=l%U9S%PWj+L{MI$cnjyK%nMdz>8F{c1ciaC&Y0Ksyz+^qfcg{K`D1DiWv<Ks-;y)JjdlM0qioBG%8`v7=m3#?U6Tk)kJPxmeBEHlbWl5$6%+%7%t(1A{ToOS`7`PEBB~E2@z<%v+H$4c%fZ*&#hdvaIqu#)fnr*-{?;V61^xvr)qvZl}YsP#48%YRGN-+HCAGsV)F(sa7rxQe3P}BScO$#+zaYyQp3*GUa4zwL{wdh$f|Kx71ag#l+&RS8_cg6)LB7FUippG$!>aC%;Xg#^C02zvaqH*?af<cX54AYfLpx*=E?w(nNeW=FQp%QVk|Hjf_D*p+pHYzL0#aOl2&J952u`+;7=S?e$6YPC)94k{$)N*_8ef`Y4D|TKG=w?<?EVs<0_Er)Iz3<0QQib+P-1*wPMMXCG<Wl(uks`NtIB;v7yI&q?L7-|=AFflpJL@`dI?nhqppYv{)vHK1kMdv#N+N)5{1BVLYmgP%o`mk4!Fi3aFud3$qOM|WSNp?n@2ZHpfwl+?RE!S4+&wvIz9xvqh>xB()3AhNi@io!!1o)sTxFqlN%jcLW$n=NBcydP0#j|}zNHju7a=$7I)tYcftP;2*E4Yz`-q+usd_0aKo1|9Fi0GC-@u2G08?$0{mKLRb-?cuguyoqr|K3bQbw~maqI9?hoExg9_Ve%_2_G!bGQ`ZibMYZF3WYKE1N}Z0ehAdjNQQcd%DJ)HBsO;zV8oP7l<W(n*&y9NU@wxx-?4DN&^>qaH`I7dgqf*7a#Mw3WtClLKlF)dq8G6ocaL!ooA<i4tTzj&iN7YL4rKUJ%V|?;vvi7kwSk(itMx#x=nQKM1Dv2H(2|ZU-F9v<?>P|)?Fubl6PEEo?O',
    b'`OG6WW#H1**E@t#^?ckHmWzonq)@9(N2}19hWm1dnVP*(KfpV{$9tw-rvHqnTk(S_l=u{+b2d1Vc%cY)}kNNi8uRn9D?dy#@n3PmYdT&wrbh8S&BAD=$o<nvFYaDjKOB5j&9W~AM8hJZ%uQKXTjQ;->lNxxbjevVQLV>)r76)mZtpKf;!5B#=EEg{>UtKEd*Co`;1bCZGt$UW$JtOHW=7EZCxEL>M&Mq+Vg3+K!XC0^&-uKn<sSlEHoyl+GsgNcWJzL==EBKu(<_fGBal-$5B%ZSGcFQ=w-mAU33(co)D|?p|$hQfKT<_-(rRJHLdA&)o8cq18`6b3$~63*8`fBi^OV5r1o~y3Qg@Z_!gFG6*MeD&<2>?KNSQ4<zRYR0%es_X0y;maS)#>s7Ca3(oAp{CY|(-{P%1C-SmmI&NX#xN3E?XZcJ<5wrjq|=Bhn(nUtNQ<6;FS?#$NJ**cYehZ4)Zo02^@`IqVgm(FDOR~jOO--uEP4o0BfAKTlW?!n=KIAv*5)H4wX7Y!;$*GhcolZLo3_40e`KwgEO(CzKBp$`KF{|IL~+%;5VWNl#2f|}GbKY4>A+Ng&11|suzU(w!JriS5OBwLZTz`i<~9izh?R!@xXY1HgKM)&jtkj*HrD2G*}$F8E!@n{R4vP@c}Ejzd@DxFKYr9JYJ7&mn1sqNgofvDU#4^51O6-jp8TaoE;cPr=%E6|)8sYp64yatVxes)xH0j(9ZV;g+FSo@q?=~Rj%oa*+C%y52CMK;YrShef=h*@WSyCvORS|Cp2=-JSPW#GUXR2z8DS~I0_)^SF*Evxmm{+3L(#mO$-)2IwgDc`bS@B5UzQ_(UCmzC*QgNTZL5N>$idJ)gAhr8KVgp#8ZCN{i?GPP4rSl*2lZXR$W3*0%Q)fx!vRo;6A(F<m`cjoTOn3|Y)G?}Q_9eR`W*8Sd$h1EbF2Xi+yek(b@*(6~Rv{;>ik?u*gPG0M=*xt?3H8NN0)Tniqx=p=RZY#GgT>;Fh=ggs@o%HtM0zEYz&zD;Dss+v_',
    b'&Wl)PS81VkuwygMcxOtZRq5l^PmMi(uD{co3aqL{la69>*m64CjD)kqP}D5hc!SN}_$|WqW3F6#XgHb%Q32wOf!kMzVPd74AkSO$?gy)VurKX!5RU+gV;#!sQw6~(r~*~)d&)SAnvxnc9C~^Zg^xH8K=~cIP?JH~s>nayLN@jDA}cjadr7{B)p9M6JdBm9NpKV0rhA{U7*V~-?&R2~UE{uHO^so&g<qvo;jPbCI-<6b@MCm*L~q9oL|P0`@|P!&LMaRh<?W~s7wOfH@TUe=*chEVh<b#Uy<SaWLroUO_uOSS_Z4Mrywx1R4WkiXIRa^|M1Ao69)_$`x304X>W;F|@#5D9DgX|yVM=wnsjI=@&<@B1bJze>X4Vu-M`H@H+C)_C67}Wk>YFvP=+E?RV!?PNkn;pDXA2#BRT!(Sz8K$m0!(RD_<}3sa$E5pR;vf_k)l&eM`F1mU9i~L?)E9rXDS#2ilE*YLU=Yz{ZOa_sN}}_7+mwQPP3L7=yj>%n{_UuI2czF4PT0bMJoL;l9}rC%nd{>_FT9!-EA?r6+69~V*32Zo^<1YNt@F|F=8bg`>FmmaJJf9@a9!ZP4vl=S}uFF;)<ze>yPq7-N6(-FtCUk7mFQ5`;J6jE@+SOhi77SrlA@Su%6YZm7LnVvN_)v>q`yvx|`D3g_><~lgi?%f~{`WoGOnIVOdb~QrEY*GZcnsO!UYT#YD|wqN()Y-uxxQ>mlCL0VsF;OVgw;<`=cqQ!c*91|Ie9^n2zPO{Jnm%EUw0!<-!!q8i%v*Rau4X7_ut`&k~l0QNHNOsnUzeC6l@K3yug&ckZ$%|r^>7ZVA6WT>=W2PkXK_LDX8;VN;`LdxFiszm`!wHU!lBr#Wftu{~+>20sD?ie5K=r0JLI9f^}AqrnhdBZ76MR*_s4dXt*tn~24l~wC~eUDMa^n#pDTf1Tzr_{Jxvat$w@7-Vjs+Mw;3+{4NLnY3vcSu~EO%$N3TwN!Iv9v&(^%_46g;u87N^0uMmC(M?Y3z5(H|V;zGuj((&z=M',
    b'vYCIP`iiGwQz^PvPP`*F*_e_|KkX{>BshVD=>S-D$tbIS;%%gE$UH(@uLIy=P{C3b3IyN+m2u3D7gAY{8pg-Zwf-*ghO*`$YxpI2>M4>B9n056vYU)T0k#{!cu-cfm7Ef(3u$-0D$3m+@7be?YD9uyO*ZMs>b<`kRV~`%O0YLDww%W9p=Jbp&lwaN=sun+EzXkH@pge*RO_$1`dS8Iu6+I=6-gJXUnj}y5DbOHdEJycuuDHb%6|}WWf4EKv7krKzV_c@rqYB>6F`?3eXEgwrg*$u6S?P1@gE^}&s8Ob!H1z!_S2m?G<?*5PZ{~Tt1CM*cSHZ(k4SO%C6C&TaLQ^(_cng%bmgGIBCg6vD6tGA2XzGwYs2P(3-#PXxFp9u~d$Pi(7^mZVfNU_Tjiu$8=11S#)s-tF#D)}>txg)|c&`D4!7_1FExLcsw6d7XgMxp^=^_PhFFqS<Psh&-HRGuD%2X|=p5BVlXfyr$bkXfAQjoxC23&X;CE8PWzGC($1e9Ys@d+J}{!Ct4C_4?i#KhD0m|IlR@TRjnwAxbiaUJVf0t96vq0_DrPc@UClyyIlH596+W%?6nVmaR7IKr@II*LdqP|>9F@qS!7?VIaLdm7K?P^Z=M&iP5}(wyvbT6OClsl%^CkWe#)Dhy#e9<6g#v88o#+QTOe$bHpZ?Poi+tF=;HEuYWJH%tN<{h4x{DfH$>T(rIXNI@QP`?I#odnmSTU-LQ#*h)w5poK*T@pj+Ou>UKQm1IA$hopX5_U2nc4cCJO(i`6W@zxP)JiSok_fre3{~kHPy}w16R@FD$)biVUmivnDpIhEuO{_@|OHMsxcRG1q<GN}8xN=oVnq(60cehTLkb^m#($sdZ5i9N@?DKoQHM*7J#9b*>#fmmvjYrjt#bj%?YfaA;&kprXUoiDD0iB}f^jM;)Exr;_u-kmX4E4@PU-eNm0U{*_ehUz_+v9(Jq8EjPnqUT#W=AbKnnwWZ$vocuh&h_uBig^}pz$ag8Opw$DtOYME+1htw#gUm;=;cFBan',
    b'GL&pcfkzwUYL{>_`s8h;!;+A8_lHx^plU)#|AYY{YP(aPx$bV(ZD)y?MdlfAf>VvQ_cj(4NpZB-ulD(Z?21h&00vE|VB^}+xBx8MHxrudckz{36=&R*kMoR{h1p#SubzyAGCqCD{1yWUx7&GWatwHCRskF9Jv=%4u_HJD-lE+crc5(<?2`saVVz5Qo&C?#-oEXFAR{`R-u{!tZ8y#Cac*wu2z5NM&4KmP4sfBoBE{^!rXeR%h{a8VidKHgtR|L~cgG%^$257H;vXZl+px|;3!c-uqqvGc$GwXc@iGuvTUz(0Ua(PuO>oA<XW_pl|T@E#dRs_i({WcxE&QKbuZ(7bz8FL4b)<+xP_tvi>oEtSs$jh;B?>f<O_f#N+%2Ya`*E#DGQ{Vc<w>rE<q@H=`=Z|hw>>{oltTj%lwr&&A0KIA@iHm$d+le8D^dlhUj<+*BXz^zlP`vu)IrdB?((xz6bv2@8%`Wv`~nBwIlVMwTE?Us&H8yt2IZISV-7?@JEs61`?8hdqy%12Yh7%KX1E<0JtHDJ83khV>~=3#LMK{;kGr2C~vel35gM*n0-|70XO&5A{J_Eh^+TSSG(!LCLn*ObdeFn0e=%#nlYLbR~@HgRng`-<fzVa4WZysl-)u>+56=$p?*-h2^}?>^rI<Ti0*1_^Z|JjPWIs}>TkXyoqrsvsI&Iu6RH<i~FY>JZldD<5k29a~4&Tnf7<_u{q5Tdn0IZ7Vr2Y#N`Gt&704i6j32gzrAWY+$4^I7AI)f91wLKu)!Axj&(p5+M6Osv&?jby4LLd!>y4XNzB*Qo2RTm|2(hDA#joD0dNQ6TBa<;L|Ff#gWotyRI3Z$0*<|gF`0PZIyS^H)Cmq((*<5Yg8-g7kx)2b*QCh(N__*bgImmgf8x9Z^Y9Y+E*)&tz?as1ed}dQCEKn1{B{YngR4oN8=`_<V5%Hvxyd4Vc#{#5j5#`V_2XTP&LQvvGltzl%u0&3sTn)7oizhOJuyBQ#EUgG#}9)075@K-K&+5N`8<YwD6j4{!FN_3m',
    b'$B577xOOmbdH&32j}Emo6do)Q?<K^a3fL8`3=kyPD@u$59;BsMkR$eCH7h+7_>_N}(Ma6$xmmwF{(KZ9|&!uGjoI1ysbWjy01*y?WHm{t5O`;!K5&%wn&P$wV6FH5<0eR&#9`z6I~ag&q|f7K_Epv8sznJ=FMsc~=qQ>w0qgGVJTo^*uJ)uB?F2s6(yXR=%LR3s#8q-kiyY2aadfBL{;)Zz-m)4x~!9#g>et9Aq5fto&D}_j1`Vtj<yesnTj1;UnjJPz1|FD|%=}U%h2qi&nFj4-ZW|;`5`Uol4K_$HZ!!U$)HEs|13TfwXXUl&^wQs_azukb#txE?bcUusYFAMd4OOL+3PB`frE#JE&LK#%&NeJl91%-BV873s}v`roOd7EbLY?Jr`MmsXLX!5+?N$FjGdR&relye@h`MqP=QbUWq&%DyZc5A-_k6Ijl35g(kNvF;?y}F8rrNaZ{7o)MSjPX~$s?7xlc5dfYFI5WwS>g37vKTm(@Y(FqCOJzYhx{a{lS*=sx9C~m$^wvf)xOg|!A6W@83uF3@+NpZ?D{q(5Jrd=T|-JJ^wxRR|6GJVR?MA?+ujR%B);<s!QbD8X~N?k-LhL#PyO48O=f^JHz-dc5`Vin(}1kDyUeNfF;+4@Fv!wrkHofMTBU051bz0edmxZf0yX#{?YS<n60W=V5YJ-uG0LPK@ZqUs&dJ-E+Wv=WE{`Sn$I%T?xxA5hh01|++}(q7ckB6MyKbxltA-53`ykHqhubxlXu-YI5NX4r^VL-&AEpth^DPUY;Q5wH-c3$x8R1oc*LQe5M^gR`Zg933j+Hq^>1R6&;BT)Dy?NS|Vg0hZgp7Bv*~J?71-T|u{H+{tY*v){i@U{sQ0h3n1A&M259>@>lw|J^SVwN$DC=~|DL()VCX!^a<!dOC#doK9<=iZtD5D`*Qa?@Dy`<cMQUMZxp8FYVA?ZV2T}e(vKI?<-y8ZmmATH|J4jS*88Ktx72Ss}wmZ5+qaka@QPQd-e#0Qd@g_k&K3cXtNE+ATM;+iFf2zJ',
    b'dr(ERaS`JW8~UP!vV|!5)ExWr)*MdDs@u$($E2bq9MK+e4*t*v#nFbF=eRxNPnswB=UrFku6qF5lJzVxO)FBcfnF)bg7*x?as7Mp6u)RM1<w@J_BnJc~_n4WWxhipVxh^+K$Qkk|ay(`LjlLR*+Gpp(>;4o|E(%Sb1Kp0}oaK$ndUh9O$+6so<6>b6c}uU%#W(+CSbR@)l~(Pl@arb4;f2oj<KvW+OciQ<dw^K)!ohR2}a70G?u1TI-Ury+@cx_7IS5I#uA)fDUOZGAa&cjy3F<-dy2u@bixbt!oQIs9x2C&A1IRCdq6Fps_NTP!{Ay*}3PKR_57#H4v*W(GUamxZ&;nxB0*M_m(^}aPX{W6y|v&Sfeew$8HoI8gqwBiM%NnsP)W*7|2x6tVx<xj5{{Bn=0W_Q~EUov`P|;7P;H~0yhC^FCv9Yp&FIe&D{0vpEbAA<8mhbN_IT}RYNbZUmhBt*SOl4g145sqSdjXi!$t^z+yjWgJQTrF=169Or62%kXKBrCh!<@%P@}(e;Y$vcZGKg3yih{;l7)>5;U@+!fCcdeoG%F#7P-aaCf|;l@DGox}p^}lfcue=I_xrr?w!qpsr;=i;9;-$!eUAA1{n~Pi29h4b|z^Wp%wRO)9%wT$igF4k`{wPD|M@J{n<+V|w<nu~chDCQt>teV$UFbM>1I-=mJf7C|?07Dt)0(>o~p=krgNa)ZVox}LXN?Z(X|a|?TM)B|)d=)J8&!1`IR(i&xD)T2$gJ}xFQ^hE7)g#vW+v@+BV98uakrO<678QLg9bnd`kR&~%VFng@bMg-dHn>^F)_j3#O+tLrL!y=BDu6@?-+vK-@^0MGB|EsSOvpS?f+$N!F(f{=!1QWnf%bD`Pcia?9OSbB4YH{zhJ}f*GX&H@Ms)uZ;tSA_)TXl$9qob;fK1A8P0<#Y-SlZ-+s21zJitH<G9>pt)mo`Z$mFbs6D0GxAnTDgYMRR=X?QNK`Y^TwChOCj_z>&Er8MqnL)o<RZ^t)#T(>&d&u<iv+FXl+U+q4n!',
    b'BTYItMbxsT^4<{pf9(O_a?tt)_^7z;5Y^A}dpEs~m$vPeT9|v@oH%r+8O>^mIeyerd4D&n5WUn3Muo{y5m7UpuJrvBp)71t8r8mvuL|`H-11~~QTs?seqn`ro5>~X3=vC}neyYLt&bm6&8e`6!})yKr^mdiL!wZ@ITWg;7`biBBz^BQ2Ee1C;q<KON|Z9O?R}@^&g>bBxgZ*C2RPGD+xVEiwJuj$$>Xh&)_$7P@KNpE;slHw0Si#$#@@=eYWueGsPu#?>vO&`PI&u`^{Jd&HeZzE47+a)?G;d}r_4_Ozj{!t*Exr(m4X%D7A4`p=%9T>R6J{aIDmS}qhxv4p0i)sb*`4UN(|LTS7MB-y%kdG>5;N#>zvZB!_xN_)+*?va4LJY-BevOikd&ht6kMh((U?DP7og6-ruO=JW#Emy6fVK`v$$Mqd2%m_lsquPPw4+ixMD-FL%FB(p^E+R?>0%tSY-kB@fc3RF!US8=$Xlg%alOy!oOEbd%A%hx@SHhQU~tuLhjn6N}cGZk@nPKC151uQ_gSY?x|aymyKSmKpXcn74LleB~A;%kE&{*U%g4>3#3K5p65Z^SArMPOG&eLa-a+a@0FNv@4>Zt+uk95~InKSv?!n&gr$j^p7ApHbhBP>au!T*BEMVt}9osOLK3ywc0p=dGK@VkY-)`Jw*HC3aV10M^F}wUdT4xg7Vg*b^6o?rfT%Vtq!1U*l8*y)M3<SNXL;qa@Kxiup_nyavapT`TK5ThXTs%8<A|l#=1O^#8l3U-vQRSJwZ72s(mfLv9bE0_W7G~^KQ!3=45nvKk6vb;D6IqTnLeag@B`zdyF)JCad259zk4UD(~v+>cE+bXAHcfvE`W!J_g0}JM9I^ysL&Xjcu?%tCpeFix=k~GGneo+Y`BQL$A}L7IO1)<&NtVUhTK;A!!yrsit3{vT%8a^e13vChwEU7V27{?L?*IhHa3}NjW98bd&0tXwwX`dVYv-EH-gK+Nr0J_?2fMt>EX*J+F^;Bq(>JR)R~=BStOLwpfbO=%qT',
    b'`vUn^>CaXRi<}C{M?)DxO%(;g6o^{S1-F`|FdJ7?wma36+)CuhUSl5xwt*LcFnf>+ViPL$ejvZ1t>z5U-;F;JRg$m*nQK9wfn(|_}eMC?~qSM*Z(^qE^NWF*lu0gX<Io9_g_3o{=P!Gm`uE6-6Y+YJUw&7%~PP0h)MA`4VWbtc6q`pvsPEDS(rW1Qo$FEf2_m^TSJ~+(!ILxLff5IP(x5aHE`iaaIdMR=4nH^Oz*k=!qs_Dv;r$?>BQb#TR$B$&1?pNzF(J-_S5N}L0$8NP{=uf$oW9si2qO)p0pRzhk1!hy##a3dj`_k$62Y5t&Bnw?={BF@o%3iQ|IVv~(h#K{+^#Rk`au18)s2>V(*p|}<Te&LG32iAF+H#&s!%%gyz~u?l%9k%g4yUx5U%%7SAYv*PdZ*F?GB5`+Jpdxktm0)39_xp)H;uJC22L@m`*0nlNBU#Q&1+2{k5GF26yCs3p5l-H`_7zu|67^=7`_YNoqgbGB)NMrcDTqO(kru-)$YGI2nI+^elE*blek~LS$H1F_={0}nQ`zZFkJ<rL}U`rcr!nc<0>FZa&mc#8gY!D$;?zFr@pB6k7q8b2Eq@5qQ=<%#b^84-F_S!1I-n`BB9M|k-b=r*)PQP@<^5BHMw5;C-A=;&r1wf{pn;ck5P7fA<s(#D^P^%<u#Z`!u=Je=Wp*9Wjgq!M)zN4B)=KKA1!)@-C^oGjl%c#_`T{Rd~k^d2H{_BQv1m*QZG2J-9*Urx})09Y(4$jQEl^N@_oj%AKDo&JF6X;j{QDk+QvitLPOa-9;`N%Z8q$gXCKQS*lkYj&+%%b<X*eWYs{1H)?@oP`3nwXM;S_T+k|gDCr_3}`ZEWBhgX^L>}WhXJ%8f!Ri^fOX0U3Q^3NQE>+R5cooTpexFB=3zBUeLNQA3G#m>RT&W#m}!NmYyXzHrR5|xnnz2`nAtF+XvO8+CpxC{*A{>G~)=y%k^*!;n4k|!7Q8HHsF%<iU*=9{TOEOy<^#-}O+|39Zh_!Bw4TU6hY0d)j7+RviB4p2',
    b'XT?xtUm`nuMU`O>u4hS5rz>yI^;UX#K)&b0SQb{#l|>3;h~%0}V!^>U25LD_;1i1&*S^<YMn9>uDE1Q6O1k!Gd3*jg6;QP5|cpjLBi^Q!=I;YgZ8W_V#<L5s@L;}rbHe0Np?6p1xf|GRbQIoZVgWhgFp(x3kE*T4S>cm3zz-fz5J4IPd9h%zAIFVwfvwKrk+L~%v$&wH*Z$OMtcOUv`+hON=Yz0)ut9mRgyG12%N7FSDP4LLJAwU>Aw0L>DDGP4}${!y0d7(@?@U*JZ<;?O@^yS%Xj$0EVf+Ak`d_Dd}<D+deg(X3iS{mrySV2D8BJampVMngf%2w)f%zU|vn*Yh&G{1~xu96I}LiuLDomaGj3>r*pkKkoY;QYcMtxTHBa%i%qinnZhLh?4C=ajRNZ5+fGfp78t&vA@*7Q~5@IVEFK-%xbqQpDn&uY4RcQnx~;U-qb>N8=zIsIpuFP0v@k|PPw;r-I#B+a<#HnS~X5-g_oXR4|mH{au6k3$>ES!z8cexC5llk)V3nb&AZOXS6*Y@KWx~urlokczf*fsL%}p$HV89^@8qlW9_pIS9R&MHG-oXo{O%Xugvp>f9e?{WmPhywYPH^Tz|pG7Wv6@9W1O~`+muTRmol8cQ-8t@KoKWO(D;dJ4bA6QryppjxHL_jTIEykv9#Vft937);0*H%Ff4?_-DHsygytrPvOnegZO9u!qXez&l1jLbP7dAjK|YULvNl6ntle>bC8mi6zmX#x8#SiBF|dN4NxT_L!K1XBwR6_`0yG>^NKiQotqN<$w{77Yp4PBu<bh3I*8~A~ia1^x<jm6s=Q-xKq|R3&cDwP7FU&^{c%v6#L}+rKoh%7~j=1uvD(CVW_Y*=VqR(2k&Dz~r%d0mrRyZ9_3YbyHE$q_xtF-{e8nn)h-zi(HFWDW%8$4Y2W9{wsdl>Zi5uJ|D>T5J(TM*-T75<EBg(<=ca{g9)8ES?#Tkm7Ay2|_7uwTA^^xnQb=ZzBjE)xNla=T0bTXLP+)Rf+NLod%C^1y6*rkC#n4s',
    b'@V5Fo~RBp4y`HF-?#CYji^o0iR=<^=EP{#7w(BWWt}=FddBrK}*GkofROenUtjjsdv@JCJ*YhY%b?-K<&^%u%pc4E@x%hxO;l$7ig)ESGQjEU|Qz-m!o7Oy+@6|;~59Y+DY<7rx(iNX>gvQ<1Cq?XUv}}`ToE>0j)2l0dE!EXpJ5QUx)L<K)QD^FF*qFq6*lpe6*1o<IiR@3HDhkb;A6;rLzC`gN4uULgX^mn}P!fqIW3pI%lllps&za+wb}JZ=qN1QAh-A@)OmpE0QIK7|h8x)C*}{dyB2VNc1E4kRn<{DMVBqAGvG@Ny^HyW~z1#_Q8z!GhyHl{NMlA|NH;^AHL83?|+^7Q|D7U(J7yX(|DRr^JzJ)r^Me(r$l@#Ii-Q$&M8f&G@sIPO6!TvC*sf0iSmht6OAXDPBfor;f(B*&!?PDnSW(I<>8daQ=U$FKIP?<*VAx54e2z{X~;MK9Zth|8m7}QpN8c$tf%pO8q;Z{)0j^q-`RK?CyuSBaXF1EZ<f=PP7|G`e42*S#5X>jruj52r)fRS=hK`{Go9vqnupUoo@Rco`7|%5dF4a(X-TJrPD?&5!)X~$%XC`!v6s`bp4RheO{bMkYd)>RX&q1NbXw=r%5Md)h2P7WU(A``%$Z-!ncvNsU(T7|&Y54&ncvTuuao!3*WnG~72+M@CE_jOHR3(;i%k3`6X8Ykb^J0Dzs<z2Gx7UO{6Z7I(ZsJb@jFfYQd5RS_&R>CiC=8uH=Fp?CVsbxUvA>JoA~u6ey3@G>G(Q+vx#4A;&+?)<tBc+iC=Hx_nY_yCw{|egpK(+e#wd7a^lyV_&q0n(TU%5;#ZycT_=9oX~JUQ>-c>qe&LDVc;Z){_?;(y>51QZ;@6(|y{8$ghOgsSpZMJ;e));te&W}k`28m~Kw=9dHbGjjy!bk{LSi!{wnJh=B(_9iQzW)UVq+w>Mq05Bu@HGBvOS3HL2M6Vdl1`$*dE08AhrjwJt$$t@^x$vVtWwVgV-L#_8_(gu{{V28!KCuHeZLujn$3ijrENMj',
    b'up=KAhrjwJt$)j@O5ktVtWwVgV-L#_8_(gu|0_GL2M5iuvz#zwg<62i0wga4`O=|+k@C1#P%Sz2aVWOd>z|^*dE08AhrjwJ&5f=Y!7045Zi+$Y(Kt^?Lll0VtWwVgV-L#_8_(gu|0_GK{NI#U&r<!wg<62i0wga4`O=|+k@C1#P*;C8=9|Udl1`$*dE08AhrjwJ&5f=Y!704(2AXpt<QU(?UC6YneCC;9+~Zt*&dngk=Y)Z?U56F1YgJY$ZU_y_Q-6H%=XA^kIeSSY>&+L$OLD?*Ree^+at3*GTS4wJu=%P!{xx|!0CwB;p^ac;CSG9;CkSD*dCefk=Y)Z?U4t#C%%sDk=Y)Z?UC6YneCC;9+~Zt*&dngkw^G3zK-pY*&dngk=Y)Z?UC6YneCC;9+~ZtCpbL5j_r}z9+~Zt*&dngk=Y)Z?UC6YneCBhct*aC?UC6YneCC;9+~Zt*&dngk=Y)Z?U5I_P`-}sk=Y)Z?UC6YneCC;9+~Zt*&dngkyrRyI9v9%Y>$EMF|a)bw#UHs7}y>I+hbsR3~Y}f!7KB1Y>$EMF|a)bw#UHs7}y>I+hbsR3~Y~q;J*1fg6rn%*d7DhV_<s>Y>$EMF|a)bw#Sg+;rTkY$H4X&*d7DhV_<s>2nL7-2nP}a_&NjvL;{2Y!~z5Zw#UHs7}y>I+hbsR3?l*tU&r<s*d7DhV_<s>Y>$EMF|a)bw#UHs7$!s(zK-oNussI0$H4X&*d7DhV_<s>Y>$EMG0X@-d>z|kV0#Q~kAdwmussI0$H4X&*d7DhV^|QY_&T=7!1fr}9s}EBV0#Q~kAdwmussI0$FL%(A*yj$V|$EjkCE*$vOPw&$H?{=*&ZX?V`O`b3E_{gV|$EjkCE*$vOPw&$H?{=*&ZX?V`O`bggD99u{}n%$H?{=*&ZX?V`O`bY>$!cF|s|zjDX75u{}n%$H?{=*&ZX?V`O`bY>$!cF|s|z0g;!lV|$EjkCE*$vOPvbXM|_OX9Q?OXbI7L9fCBXG{Q97V`O`bY>$!cF|s{Iw#PUj',
    b'mh*LNkCE*$vOPw&$H?{=*&ZX?V`O`bY>#n9(C6#e9wXahWP6NkkCE*$vOPw&$H?{=*&gG9?0~OhdyH(4k?k?EJw~?2$o3f79wXahWP6M&(g@@coJg=eCbq}K_L$fn6We2AdrWMPiS045J*I?&gRf(IOl*&d?J==ECbq}K_L$fn6We2AdrXAPgs)?JOl*&d?J==ECbq}K_L$fn6We2AdrTQA3}46gnAjc@+hbyTOl*&d?J==ECbq}K_Lv6bI(!}5V`6(uY>$cUF|j=+w#UTwnAjc@+hZD$6!CRzkBRLuu{|c_O-P)OIU#jI?u6t?vM0U{`4bW-w#UTwnAjc@+hbyTOl*&d?J><rv-mo;$HexS*d7zxV`6(uY>$cUF|j=+w#T#}ALHxT9uwPRVtY(%kBRLuu{|cX$HexS*dEi0#0{Anr*3SIne8#NJ!ZDY%=Vbs9y8lxW_!$Rk2xX5<LlTSGuvZkd(24T|5y5j651K7t2d*aQ<2in{IDxt)cW+Kc19x0zkZfvwoGbg`FcxgXZhZi)XqqFD@pB)MEE1Aoe6(qzDsIn!rxmZsh#n=*hq86JIa#O&bgk{&iFeMuI1m0zm@Sj_(kJyu{me9;XL4F^S$G5u_<S^<&3|@)|}a#Guv}!gU<L{Bdo&L@!y*HZ_WI-X8v0<{uWzy#@}Mw&TQOyg8BG5-f%P9cZPvsVVD>;hLK@qm|5)1*TK?k>X~glv$1Ek_RQv<+1@i7e1^prEC{}iZ9cQnXSVtbv-5_VVRzneGaG(p%g-xT4weqD9k%_#wqMxx3)_BS+b?YUg>ApE?H9KFlCZ}3I=217wqMxx3)_BS+b>vYZ2N`R)xx%42#b-gV>>Tw--YeCu>BUc-@-Oq*k%jcYe51%;%ls2zLxE^uw53m$-*{S*d`0xWMP{uY?Fm;vJ6<_d>z|lVVf*$lZ9=vuuT@W$-*{S*d`0xWErs+__}f0?jMPCAgNuDY&Ryg3!Z~*y|Aqpw)MicUf9;lgw4o*pKZOctrxcS!nR)6)(hKuVOuY3>xFH',
    b'-%s7Ygb!_Ve+Zy{C8yh<tTN`^Dn;Yj<Ik56|{2U8E$HLFC@N+Eu91B0kg6CjcFZ>+K3O4{hz>a{QW98>q`8ig8j+LKd<>y%WIaYp-m2JHyI1IjyZN0LsSGM)awqDuRE8BX78-WjDTdxGq!q>5_SGM)awqDuRE8BWyTd!>Em2JJUt=EiGHDAYeUfIqo+j(U>uWaX)?Yy#`SGM!Yc3uY@zxg`0^U8K!+0N?#_r=$-omaN=%64Aa&g-}(wJQ?BCnvS*i2vum&1+&E;WznjvyE4_@ya${*~aSxkbtja8?S8Rm2JGTjaRnu$~Iov#w*)+WgD+E94}wTHeT7rE8BQo()Rn=&MVt_Wjn8I=auceF7VR$8D9%W4o?nr0`LUz2>=vu=J4j?&c&a@p^HZsmo7eCoVs{*aqHsO#j(S)pOb_Dd;-q>4DWu1dq2a!pAjn02$pAr%QJwBGo1XHBx>Lj@abnb^E15m8Ls{ee|<&<x&@atobqxM!f!{y>Lk5<k^BdL9q#@NzkWslIRh3s0~a|XhMa+noB@rTfsLF8iF5b_;E^-%kuw02GZ2z9Ad)jMk~2V(Gf<K<V3PAFp%I@zP&p&2oDo*eh%0BHC1=1TXW%7g048T3Cg(|_DLw(v<P6m04A|rh+~f@4<P7BG4Cv$x?Bopa<UC6N#wUQEoB^PmfuNiLp`3xCoB^VofufuNqnv@GoEM4N_ynMoGq991z?3u4lr!LzGw_r%0F^Tkl`|lf^D1Fp;yi*mj3z08<OGrwNLC<ef#d~}7;KrrmKxv_k{n2OAnAeR2ZDMD>?ODt){i6!k|ju*AbEnIUjlzgsvrQEAYhU$2nHq~m?R96F-XcFIfEn(0)+_{CV7LPVFHH<9wxbiBo6|K2_`0>n4n?;iwQ0!z?dLo0*whaCV7M;5|T+sDj~UqAY=lO2}UL$nV@6>lL<~HK$#$A0+k6?CSaMMWdfH8UM7H<AZ7xY31%jsnV@C@n+a|vz?mRt0-XtVCg7Q%X9AxIekK5#AZP-i35F&hnxJR`qX~`%AT0',
    b'q&OMudlo+sdGm;5{dSW7_G5}>sNY%Kv?OF-8W;I#yNEs-SHus<NS1dJ^KWJ^HV5@5CjoGp<6pOPd};*^~Z4En&J4-ERipnog|o$!M9AaueT&*HCS8vtu=z@Zb2bOE9h$?n0AKA`ACf>pO;(Fw)|7@c783mTnB{tEUYjGsu{oQ+5)EDu1^32OwHbizW3fYJ%8sRflzSb8n6biz9QN?bZ&6C~_{Ju;oJEpDOdM1n)(u?38l$C(A4P9(52UMBn)iGkqBNGv4yG%o+<Aa%m-e=KEBa2~fgdm>4hW!)l-N}Pd{hgTUT(f}^LfvXez%qYtn0P+oBo#1&AT#p7>C-|cThXl|%!7C+_b&og!$8-U%6Fk%fxlV9Z3BD@9StWR@J$jwsz7qV`1;0*kWVZlzf>%o<y9QYG2Zo(UKx_HYASiYsfv-0hJJAP@ooM^daz(NeTwsC^OeFtAR3h1Ah)X2kwp=j@qMhIp6MSMK0lE>W1?@(p#s%<?2@W#BL#Di4fmps>f@lsmiCSHPpG-Mj{wCaI5`T%RT@r_hx?Muy4nTK;*GzDmlcWtKc?13+eWKkdZ|I+VHK~Aif_DYto#0}h33(^@>s!n_p@{e_&^u+xlmPFq@<rYfYZNA1fbW#$JxPiMn6iR>^A8u!r1OzU|7w}^uLgW4q(09?z7z7OghVPKlS)~<2HsQ_w}Dd!(y4@eDj}gtl2M_&pCqM1fj=RqN|IEe#-C8+zhL1B)&51CB2eBBG&~{G^78A1RP2&qC*))aNm)WxM)GInl}vIa{s404n>_n~#Sd8gfW;43{D8#|Sp2nM@gpY>=H$bXlOH+xAFCN93BHJ)*CYTVx?U3y#sr8l0b@)6852;(1eh@aXG{PZlOT=ge@%cI6R^gV<(I;V18z)NHZz<!5XY2dHzUhCnpBQ#{K&?SZ2ZW^k8J$N#*b{g%*Mwf8$Yt~=V#*y<?f?5{7AdCv^!xs@5sBqHt$a8Gx?E;cS2LkqnUR?SImz}y;J`B+&jy!f2G`ekZ1kna_>Q2Szqowh|!;yd!',
    b'vOb4Z+-7`n&?U_aMLJKKCBvm)_^z!}&{d??LYV$i1JCdrwF1edOMMG$j1zG^ZTt_<@2SDENVbA1L^Nf*&aOk%{jz@%hNak4*dzmWdBCeu)<&GnnPfWkT|ikbPKFmkFuKe(W+KSHWQ2k$WGx_mO)ax%ZKKAG!CDdmp*Cl6x;l?tSFmf4JNml{8pkkP;wlF-Vb5@@^8c7`n2aq(O>()eIcp%<>uhT6yuX+90GTM-qM{;YSjFB;iLAek9>X5`HA%Y7)L4N%*fvIhf@?$P2SD55j~3&f*hLo=D*LN+9@3!tgc7$Aqd-xC*!Q9iI@oLTUdJ!a`vz#GZo=MS>7Tf)Yg%nlETkB=CGC5Pc;ueI-zRC2)NukbNbveI?L+CGdSE5Pl^vekD+TC2)QvkbWhwekIU;CGdVF5Pu~wf2BcM{@`ap{*}P~l|cWM!2gv%0G7Z2mOufPzyX#(0+t3zzNO`F5#mLB0)7@$U<q7c31naiY+wm=U<rI+34~w?j9>|rU<sUH38Y|2f+Msm5~5=og*R%Jf8r#A)i6pc$}C?kAH$spnF%p7A!sIvnozY!qwFy(5X8+SjA4kJ386C~b|xVWljsS3iv)p-B&=bOxJb~rNJ1M1m5T)4ums|;B+OyscGkitumr#zmOvhsggp#;7YTwF3G`tJ{9y?MVhIdlNhri1dy$}fk%U8R6xK?itrXr$A+8kW%8j}bpOE96oaf{~ha~_Jj06>o1R}8nCb0x6u_Rn#kitmN!bl(!OTs1wIgA84u{6rt#S(xdMuH|r8s+U`2?(VabTN`}ia{A838@&gF%nqClF*7l9wP~_7z8pB6fzQs#gZ_KK_nwVB_ly5BS9x)kz2+xfLca^Tt*UlF$iWP@QW?Nyosd%-HZg`44e}Zh{kYGNRZG-!ZrpGjRX~q1R0G49gPGbjRYl)1SyR)iKoRHfSN{voJNA4MuMP5f}%!(q(*|KMuMnDf~rP>tVR;LF$ilUC~G9)8-un+g1AP4x<&%wSQ5rD2y7%MY$Ql*Bv6heaE>LAjwP^;C',
    b'7~UI&_<GQ3aO2Rc$yGU6C!FtNKJ^T2|+a>swRZhgt(dzSQ8>^LTF8soa%+dR9H-f##DGrg~)VcGQ}t4jS875)&L|p(j>1GYhaRhiZy^pn-FRfVr@dOO^CJ$;Wi=OCIsAsh?@{{6Jl;c&`pTCNy08vITB<!lEhsIb0i78kmg8;yh+G^5qlGYZ$k7<2)_yOHz5EgMBs!FoFoQArXxY8BSEMmAqXc#;e;@p5Qh^2ags<p<Cr$dzwjv>)+YI4xt>fCq$-kVONg+9s7Q!>gy=?yKm@>-B<Kqumn4`A0G9;BB>{3tg1HFbBFGA0l>}TR0a!^uRuZ6<1Z*V%TuDGz65y2td?m?<4-l3Fh$R7INdQ?AP?iLkB>`ti09q1|mISCJ0c%MBTN2Qg1h^#uZ%F`L(k#zDT@Efnjy200z(N4_l4f}a!fPwUw!&;H)V9KHE9ABtyDdH;{I*yKz+n=Am;@vy0g6e$Vv>L{7zaw22TB+SN|*>r7zs+42}&3WN|*{t7z;|63rZLaN|+2v7!6984N4deN{cK6JTGR15{85l8WWRrCSpz~Noyh|g%WxblQ3rioJotEo$z9T(IkL02`EhhOp_K#y5I(Y)FePP30O@6Sd)O(B)~NZcufLelYrPHKsE`OO#*0>fZA+l8Q?Z)k)zinKgDOU7#2ByUC%PWauUFt1T-fB&Pl*?5&)e9L?;2#Nx*c{B6p8v0azyi*GT|&5|Ev=$VqIKukf)^Xom4ItPKD>3Fu7%9FxT10+2~SWD+2m1WYCYlu3&`0+t1EnY73w2#vVBPOJ?8G)WxqBF}-fvCNmV44|3>tR^kOfGiZq!htL#$ijjwgyQlhg$!BPkT*JHd<?4u;7wZOiLgq5-=sxY#qZBDi;#@VTf@o$u9E=lq(x3_cx%9S5&)hAgeL*wNx*m#K%NAY=Qzs*&ob%I@H{FCFIG`Vz-SXd+60s~0j5pBX%m3j1f(_rs!hOZ6TsR8v^K4>Ht|vc*d`#h36O09W}5)oCZM(nux$cvn*iJ<Ah!w7Z31?i0Ny5`',
    b'w+ZlV0)CqS;HEXt1@p`*X92lCy!=%f_V1Zz0Q9C+`uDNRf%hf=zG;<1!u4>nif5DKg4~~+8RY&D82|+*V8IDsZ~_{f00*aayc|w|;3h!02^ebvh?{`ord8f4B7<a1KynkH+@ln6lp>B&#8HYkN)bmX;wVLY&-6VV>HCqszgqeZr;EUx5Sf!;afr=HNGqhafDoM!qZ5L3LX=Lc6emPC3GR25{8}m$*V+OicUq-ZA^J)1#)#f&m2~?UmpxMSBSk+_^dm(-QuHH5KT`D9NYM|4(qFG#Xcc4^UImdTt%B@|?5E_fBK#@JpCbJ!+MklYiu$L>e{TApA^-{nP>}!?4NwsQ6%|mC0Tmrk5dsw@P)LCyDiNX*Au3(ccS3X`L?=RYB19)bbRtA2I?I~EszP)kL?=RYB19)bbRtA2LUbZTCqiT*L>xlIAw(2HL?J{JLPQ}%6hcHHL=-|qAw(2HL?J{JI?Js=)d&%V5K#ybg%D8)5rq&@2oZ%4Q3w%*B%*xc@t;iag2#{YR>twEc>Jze$j0MGc`tI>lJ~N~?m8Y%=L#NA=k0qRcswCc5&|Wi<)_d`MWCef_ES7GG*S^L34xLjC<%d*5GV<Ok`O2ffszm?34xLjC<%d*5GV<Ok`O2ffszm?34xL%P>%8oq~WJ*7NYak1SG!z|HNqlVU!R?31O5FMhRh*5Jm}Mln_P<VU)1J2|Jvy#R+?yWRol1LWG@9Y3l}(i!pYMosdKncT>g1uM%Bc5yll|T#?3=Ic*u#mPu{wfFypEL`WB(zz%>%ptQ9FeKZVF+M0tt8iojtg3{I|Bwwvt!w|u1P`We>QQH24Uy4lv|3Ppd1P?-RA(XbRA^eR&hRWqD+8Tx^ZT&;I7<(pd-@`B1zK36cJ%fl$h{%M9Oo+$?4P*ieK!5=VH~;|%ARqw*D1bI8WGo~g0t7{5f+R9+n#i*Zx64oGn}#8RN-{wvnV^$QB9x4XOOQ(@(Mv|qC8#D-l26PsBab_nKN^Mz`pJ~!V^}_rP$tn(M(`ylDU(PkUmJ!9s>%d',
    b'eWrD6UCHXB_aFA9eXe$%Ml?m#~l;pQy!9ie|ps-9Lu`KH3cMU@X$z_~PNOYG)cv+N}MWcKc4dt`^6q7^JjU@9#SaYIQKFdfE7M=()Z;H&Td=J-Q^$AOW=#~R`At4J8@$zfS5D9Sr{5O3JFUeQ97x`;4z9jQY^4Da732RQYn}tO{Xas~uK=hkMz?q~$X_V{mG3in&TZTx~=Sb3jxizdqQK1tVI?<sMAv#fd7O7{Eq7$)aQF|7-XA!McT82nOphX2*WS~U{T7;lQp-v>~L<?HPphXQ@<e)_lS_Gj*5n3dnMST@5LnO-3+k3`36oF`x-j%zSArcwZMXV+LEMh1uOOcNj{b&)876oaMkQNPT5s?-ZX_1i@9cdAg7I=iDl|_t#btr18%QjYq&tfr(ys7wU(N$f<GV8dt43TI}3xXobs@Ikw0-8dAQwVqp0Z<_zDg;P{fT<8b6#}Y4LJACQheTi%=ng?;l|*G#JTatJ2`a0UH@*oOdBZP&Gla@2L1C4kuu4!@B~e$swhR$;Rtfs5lx18Fix66>goK|&s~RW`Wl<3qK8uae0v{74agh=iEph3ITI4!>7OPHxI>0srw2pw*5zsmUT1P<Z2xuJvts_C}fZ>opc0h3`%R|d~M1CJ!Aiz5k@DAV(34*uCi^b9dj7M4Gu)N(-rv1)a%Md|m@i@?gTZWFFp<i$SLPlcmiVhovt+f811O|}ibxD$?=#a*G(P2w8q=g#!%xTLnr0p2l0OC7Zd{2@}NN1|_%p!6jNTC<?B$<PB97@|KyzNQyg+-bn;j7?>;7Wd4&(Ie6CGS#sfi6?Y7Aug%^BpOWgmUqp@LP+k6)OSyLn>CHGlDbRBD^A^FoG}~A^hNtMy!NiL|+77#9p|61TX|%WPOOZh_wi{aQQf4Aj?BEMKDDyMJPoig_lPhMHodCg@;EBMF<TP7$iP678no%5&aPS5cUxD5cCl95b_Z55bzN15bhA|5bO})5a1Bs5Z(~o;QZmQ;iBPLvv?!E4*m!Z2_6Y92|kJKfiQ*uhWLf>h3J',
    b'Lgh1iAAg~)}Ng=mFfg)oIEg&>9SgusNjgs_Bwgm{EV1f^dH{X*v#62DORg}^WLeIbqpZ8RK+aUO;+8g$Vhi-xl>j>0$zLlO;&Xb?n$9vTk8I0NGd3@tQBq2c@sAvEZqK?V&fXpo|S;s6No!BG#2c`(0&njK{5U=9ZZI9R(u+KmDO$hMzdV8CEL=JGL?kJ&7YW??c5gISo%!dMojvM`i|nJkQCVIm6yS%bs|z77Eb5dt9sF#<t??SVNgjA3C43qx3#!NLd@Ca^Goh50LtUt#(R!&jKS!srzyuP}IpxhsraVd@G)SD3lN$Q358FmQ!=D~wxV+6u!~n6<*F6(+4PXoWc|j9H-)iXkh^SYgBp6IK|oLJbt-RhX{Aa24vGD1TzI3WHUsePXN%Q&p%TV5TZsLcrt{2B*9N0w55KOJQ0H_y=aCfPP?73fKqcq=0;2N(w_#n2`eNfe9%LNMSw-<58H7!f+I3qc9qU$tVm)VJ-?|QJ9LtP!wjOFcO7{s8K>UUx(n0=#B8r_P{6<CZRA0g*hmUL179CLr|E3!Uz;5pfCW1`6rA&Vfu-%p=9_8o(XUUVS5qw*C?4OehGfWM#0<@#-0ef4pUDUdcw>TMxJEiNn9BABZ=TPb|hg-680oz$%_%q;Lb4YBy&!Z@x!T2l5ZmElRC2G`?GwDuTJtW{AOf<Y$h3Pl01J)@!@5&+%o(uHa1~rQ<h{}ZX3=PyBk2oB<VUo0Op#eEfqleBcI@Vkz@cd4sH-LO&DpyL=*4{%ri}rm9OU`JC0|^M@!BhCOf7q$89;BuCh~PBo$v^E6NC@og-705xLJKnX;f?_gONDGRia3WT{a|YK$ZnIV=Eepwqyh(Qt!8#wl%;36u>^0+C5o0;sM4(I)}Mq#z-H?n$69fnc$E@j)aCGzeyk%=9eZN4|>RfQ*3#L2Bgt$Xf-i*(Jt9`pArhb`$vo3B~3!lVhRetYybS&568)khrJ`GP0Q>Lq>W-$ZsS^Mv`+)l7*{ikmtrezvRinOY}gZOvsuDX%iuD(sE@Ij-',
    b'w&hvt>e>rKQV++$)+d(;!K+pD+_Lxl6`ONa?(knUM1ll0HJ#M@aj!7z)4oBX6cb)&c&=N9Ih!wjzdW?o3DwYsoVWaz9An7rAt~%by8JBq58uWYC0k@-~Gg<d%fw@{&XovP>G}8IWhn_CT_ElSn_>sYA$INmAEJDov8pD!FvRe%anDypqrjkmSzrq>@16sgMPKWYi=?1hS=&6qB$K$gYyyShkkz4`j%}%$kr&(;)UiGL6(~ko_bn^_E@}a%<X>YuVelX|bAYFJwo9m<`zx@@yJpgNfy^!;otel5HBqcGzXew+RV14Prj*H00cbq?-mY;3^(OTnK!~ZTAjsIe3|SlO*QwUcz^vHt>*qaF>9S)Hvl=!873Z(e?`w2XSn_q~RpFJ@%dS2;n4un~5h9JK^n0^8b;Hlbiwu`L6s2e@$)%{tS0RayuV6IpMxYP6py9NxlWoDWO%8>sfvv|H1Ep6UT9Z<VLXz<<Kz6cf|SPci`+W%NL{M=!APG+&tm#2?q(nNrG^cAlyD}_YViZJpfiC4itnF1&!k6#m$SK7e_CTu;C#11QH+EbQFg#9v?0rCk(<7gK)+m95M)}3>xM2!R>Dkf`z?32^K648-&vaZI1$cWP28QC0L-SpOVy1<-~#d=_KzEixlS$!oh>&<N?A3!qJ0p_8=TS2&WIi@q<9PKp<Qo5H1i17YJkvv^~%&(M!S?&>grL96*lzJ)FPiBY!{g_djy}{v+Y+Ne%`#Df=Xcghv2vJmOa&+Tf)jAU9^J4$c|^XAOa~MzVE*cEDS6&DX(SL%?_lAP<4Rh9)^=;C+L?Msjwvq}&4BB<(5Sv#BQSB#~v3Y+b$>C2gn4$lFQ!S_Che<VUz0`MHt06Cfo*<I4r^Cc%~_adGk^{Mqey;n!!m0`Em^kKgx!dlS$m0^D>#zG;%>fHeIE`zA2zNCJ;Qh6J7v%`O={A*>PN+Lps3x=9X?5O>Mp39*h4>~2wTLgc&T@PrUZh=GJ4cunFF4JC<}Gwviejy(1zi6=zKS{6@}tV(&jZC!4eJ',
    b'Ryo+QhAzWnIdogK*b4sJ|yiqh^*Iaev;>(<$_W2d79)cU=1VI5`rxu+Fk&1n&fpX8$?dhwlm2~k^ETh40!YhPELr%gm5ewJ^FYqP&t7Bh!C4^Q+fg!&?ToQ1Z)B+&>k-*MDSa{oF>^OfVbu5*H6rx5Zy1JIU&qTE{|A$fz1g)pAhvWkw@q!#C`&)5P?*PKq^EC{)Ff+dHlTP<@iAAhtLTb1kLiDEr%yG8xb0fZeer+OA*Zi7r>oLo6#)L06Z#~iAdUvX1Q3>2W(2rei4Sh2$Nrg(J#X67h(8|F!@Co@giw6!hjcH!i&&mMABx2AumFkk)*=XD~ee!!mt-%+KVvmMVR*@415tLz6c{<G|MMoc}#r~#=Z!1UxdLg!sHi8s}W|u2*Y1APamK=+KmY9Mg(#q0yz<ZoQP&gp5?~G6^kzxXDr@WJ}drM9P;Lo@iFnqSk7oSBD5P3+KousjlfVuvn&gEW-t`dEDHl$8w^D>%gVs!215~np@_gxL|`Z)Fcc9OiU<rvB<)6EC?b#(5y*)MV_<|%BSNPUNv9FWiAXw)FiHi-EB%7fafG3;3knaXdI903QxMZ)gmE$WNO6&v8zT&kT|juk1Q}t3j4(q+7$Q6J_;4OS9C`ea#~*q8_i9VQkSt+JmX<Ad{kqPSFyekq;w!x=b^?!4U#ml<in+&=cLCj(?J6U8CrlgnG^%W<`yyVD+Im%-eEqgvMNihXh84o7^uA-|9&Nu%)CFwEX!Tv5jyY=&Z8x&?edkIoM_&YKmmA1iioOK<R|r$zfZJ)wA9y>VUs%cym;%3TVj+x#(;{hu*c!WWk>7&-xX7=;j>Kp<N&oO7zX!V#<Kcw)a9ZS7jncS<Phf9u1LCqf@m4X<O&I8wL>*JzT62q@r{A`>NR}>W`y#30B}o^s9lgkx9v70HWQ>b{WtR&r(<ME}L^>_vUzGG5bLzBgUPmV9@!G`sT+?%muG1op=$f8mjGdPGE<MLcJ7K2%(*Ck#=h7~{NP;UC6R#z1OR`+CnDo_2s(a1Om(7tax7qn3',
    b'+hx6E=a`4DX6J&p@3M1gyxwK!i=@wZI%&q1?0k_#T3oGU)8cF;rI!0%#l5auc0O)jT#|Fl-xCJ#Nm{cpf=}!DlAKF_Hb(IY9iD^^PeO+$t>Tpt?^c0T5b@CAN$Bt-ba>J#`SW)1a!t<RQsGSrZJ&g;PeR)#q3x5<_DN{_B(#0fD%Zixt0aI@&X7<5TVp<rFrP-4-zN<46Xw&7Bt4v@k4KXJD<<i*ZjAX)?<FT;VT8+)QO^gv$q7@UBvYdBr>opK0;f!c?s9bcU~N2_WsYW<qgm!?mN}Ybj%JyoS>|Y#`9tUL(~-abYWX{XT+U433EU_IZWQ{UMUfnyHf9t9WEU3m50IU}RY$^R{zWsPw6UZc89Z&2ElvuLOykJmj~xEU;g1~t$l;G1{>b5v9R63z;pZcVKQQ<v41Uw7d=3V0waCyWXLR82NA7;)?nmx^<nBlAe&p^)?tbL%*W7(Ma`z*5Kl*2m{Qbz^kNo|}-+$Ho{pf%{I^d5E_+K)EUyls_kD%ZxW2%rd{819F1i>MK;82f<E5`WYA>km!Kd$mkzK{(JzlgDYcttox_(Y8C!z0dOXub}nW?N%eAHEQy`tXDp)W@9uq1O7n)mjMxR<9`6N~;`mrNeAnxY!Gm`~a^)?i~pXbVvz8qy!~Wf)pu1i<BTnN>C#u$dMBCNC|?Z1VvI><=l&y4na~9MbdzZD~W8WA>&F=lA~4n*2J!2Sh1{_R%|QAU7xGxN~?r(Su60doszD!ZmaBUin?Yf-QiDUsNLZ;=CiCuS&p(EWkJe{6n=jY{6q0BL-8&{@h(H@E<@=qL&+#Z=`KU*E<?#EpJnnI=Se8tWhmX{O^5YSnb#~&xT(3`%Dm<ean}!-*L*MYD#`m->AViIt>sT4(BtoYvCeD8lZei1-fr_w=QZQ5GqeITv;y;XS4co`R~dJeaaS34m2p=Yca?Eh8F!U&R~dJeaaS34m2p=Yca?EhdAqASo!5+;%(%(C-6Y7Iwwr{)!dX@v=GM2H#N;|O_A)f~GBoxwH1;yIn=&-^GBox',
    b'wH1;wy_A)f~vWU^%>%3-2?qx_hWk~L2NbY4w?qx{sWk~Mjv#fgvG2ou_Sr)zAGiKW%xtF1~m!YheA*z=ls+S?<lp(5@MVt0s=QTrPFGFK5Lt`&PV=qHvFGFK5BTLFilQQz8j6^9TQ_4t{G9Dr05wbi&5)F-wq5z)|tVX~Z(9;~5d@z$whb+>s*$e-bx=7Lm|CPc>MHBopsb+E#ELU`Pgkk2MZ6=HE<_+CU77fl1zL}ha+K>btCYN$1!=^b2o67cS`5v4YpkY`xC)q9nD`x?m@d<1j*f%HnE!aFTafXdEjGR$%$*^*UnKSI1VdxA?XHnjq=o6LCFnosPbJ{5KOfs4*Ohg_VjY}%&ve97qWvsu91(>k{GnQb+8q8RP8LKd38D^}*4Va%U$UIgg$vc*3oCKwxY~>`Z+ABJb<$DLuV=ZSa=8V-W06ms;#+uGp)H%ufLCn0>P{JD5xM>pBN1>vG-H@>zGWJ8phPblRU}t1(jf}k^1U1+o89PKMYOqZ*_K7GAqrRi!^r&Ttq!M;g##YMMOBtI<kb3MWkzTnX_1IrI2_XLo)hCkmhG3mURY|_sQPpHRe^S+C+P-toDVec5Z<GEkTNSbOlJ;k8+<o4kv6C~la>icHB)=<(b(4VhnWQfRp`?b|U-)XSn7vd#NbUkjIov?nN?;e%KEo|!_yrU%5{c@PT#uWO6(&n;TVwcbvdUzcNfAh@K(f%J4kR~<#Xu4rrCx;14u_H9F|xP})a<et6;-<|FCTTgOmfGAtYmxwn;nIRjOty+z31(=VNF!<GKvkE<eo=a{KCtGPhfYUj+X%gWK{ApI$kn-Mn;267N>z)UPdu5lWZESKDdnxzmefMGCW6y>&WmOS)2!IdKum$!+m7<j|>Nr;XyK7NQMu|a3UFMYZ+c7!;NJ4k<F%9zVId)?j*yXWH^)zkCFiwWVn<JpOWEJvUnA7D*_q_Y9O$I;06L52y!6Mfe5jQ5}OcRiC>Y5pEwq&`AN}F1ldH9P0D`aS;V!7ZxQDr-bLJt_!p`EiGvXjvlR',
    b'fv#fXpDYJm8JNVG{6P|AQ(2b4k}>@KB1sRfFg5kDi2M#_QWX~fltuMuY>B|)hPin|eiBMwJAjtIAja+^rE2_KfY9Pv3)85B+|A;l7>Beg-{#S*V0DsEC86dgAaaudHJjz>I?xE}F6!jC1+M+DtO(M=@Xq)aH@M@X`SB}-_s#Q%u0n@GEf0}>A;Tv>VR#E~E1V=%lpBGGx1Gp?K}#TB8bp5cqKI3pAeGYaa*u^@abIP{?YILCsFAT8;k^qI(3Tep9%&1ZuQWG^Gi|Fpxw;{&o;8fjn~X{$j5kggg80O^LgJswC0Oofgz51WE@{csTJ&dHL*SK4#(u-(<F_UGh{n;W-D(pDm0;Zq1#KvMG{1d9k)(iS7JS6X1C)kWG`q+MleMUfViER>A+1op19m`HocAgwF%1oX8Ms<hpvoP0VcDnpGr(hVbBFoPID5_{Pfh_;e5OIHgzS+Zn?=wQjniZl8-G8#HEIyy31I`Sa)K(rmiAc(d|HgBa>g|ITC(<5&m!LOO*Tlh5y!RP?V=m5#+_sD4Y$mssa==#WmoJ-^^f)I?3kBp9wjE;|tj*pCvk3w#Bkk%diD&ek_MkD;DMNTvLO?O>9gUGMq^6g(_uvli%1HI|u$s!K=aWcrF5?Uz-WRVZOsRv}y5^d05&1g!=XiCXwO37#{&*&x30Jbxb?F?u;1KZ93w=>Y~40t;O-_8KIGZ5~K{*;Ucl?;eG1LKw#GN!GaCj;#k^8@nE!m5t4SqAc*rRN-#vyA?gj0TpB4wj6L?u?G^JPJ`M*5N3JM%+I-xidPs^C&?R^qMd<y12#8=;Y350ueK#n>(Ws<RaFZ(bX-sL}&Mf&oZOKyCk&C((`>Gw9IJ!-d-KlPj6IPGn&COn!)oZ=UnMVlXoE}T{&`GPXgn%z;V-}9xR8CMJ~oKkQ2!DB!CX_j1KWElBx27&~X2u<(fr6^`kLAi;}9m8*CKOQ~hYn&*&h}=pfG`ttzq&&{q|EKx7pg8%;GC-Q;<ccPe))FBSV^lolUcA#Yah7>cW#{IPt0xC',
    b'*@HQSKg%G9MDJS(I3BG+wjFvd(htdfNUr1X|@!ON$R*D}Q*UPm$M*R`gqy*NnFGi^^+8d%7gXXmH8WqAoc*`qc5WjJ}<WUiFM#^^9Ki3<Z;m%xi{vNrnPRhEmA({~*KN{sT%}-V4wI$$)V(zPiY~W~hT?sDfmCD}Q_sI0a5jAQjF+DlCd1PyxwM0Rh2$Mq^Kw&Ylm7`7A9y$Z+$<FrS4#SlH+*GOt-^>W{QOn%2{i)*tlphs^8IdEeA||I2De<w*pIzEnFZ<G22B?WimPwMRQD%j10{^I0Cx){e?}`X8hnbu{N6!?nk7?J-<?r1;eo|4@ScW2N|6a*k)E_!-H|KE=;SW)20ZKbV5lF-Cig(H>*8N2Y#c>PM!2Wa?j#sUKReM~}?|JvP%9@>u>_JvK=)x0WWGJW0MM(cretW|Glgy!0%dS!=YJB<cJvjW$O*f28wAI)9||M>>C`^G7=WNIHM0!XA^ff7D6ZuLRw<8CvO-XrG})PetB(Dv;w#4@KH?dr9Z<6%sP(n^0zG^EBQL(M~d{S{tIxaD;bLw9;ZBtrZ6(f8_8-4u9nEM-G4F@J9~+syY0j40|-s{Gpp?j{f+Anf}P%kNo|}-;ezL$ls6r{hQ_QhbrtJQ&}oclAj9%yWFAh8Joa@;qxTlz?GQdpXA$H`kt|2E=YWy1OUd?5oCAz%-}Qjk(R<|Y_3ZVpJkWbB=K2>+*$>uvP{LDB{?EV<TG}wp2>ee@)>)%hRJ7a@*mL&pC@st^5FB<R5OWd^%D3Dx3l5z%h~USmjqsHv}_!f8LsLMeZNcJm-g&EPr{`h<Ejyoxk(5YZ&LS3C>^)lUFhZDKP7jcga)sHtY#AGw0-hEf#ze{DAS7cJx{{hf|s<?J!d5x^<Dlxtsnd}@Z1@$J5Rz6vVL+_!jb1mm?l0bE8)^Je0qjc&+zIQZas@%|0J!%i@M9<@z%gh0~V4DkDuZ4uSq<7e}?nV@ctR@Us&UWkmfUs=O19Yb^IJ<73#|MGll0Xru#wG8F{&q!{=F&^>&u}eG1RNX',
    b'jN1AJmWVYs6e_%PDYR~k~S|$7zxo4WQ;BuJpV&JC0}9Pg8yZdtCx+7W(En)C<i4N0_z0I9l}He`3@l>f_}#(iO&%1z)4_`Q`bihpJ(BjN!JuUL(2nc{FcSbOfvsTwwUnGK;c7JhGyZQ;a|_|9H~1p_bdAh^goccBWo9SAP@q<SP%sf>X}*iX%INi!aS296LQN%pP?CYo33XFh*(lme8F6j(S!E=A|}OGELWx(<iACpo}o)3`R+}cE;KWerX$b2Wa&t9g=7XYCV3X)U`x$nol%ai@>AIWxV=&4y4MeqNvN9SS*(P^@jP8fCTB60>@l2?A#jpK|7rUzqW>f>S<cI-OG58N=w{~aKluE7s~?r;ZTsQ#B5rljkIJ)bMX~JWP4Q>Noo)wip_0PyTjU4$EsOjBzZ5JtVoYq#8M|}F_MByZ!mG?;bL`N0yG~fCgq8|h9(G^gJr;NmXr$yt`bv}hV8VZph{01pD<v=DDdeZ{5YS7>i+Bil14yRiMW{Sixeg!0tA=(;#tc@*3|3w?H7Q(-_YD=5yvX5bmG9wC!Usc6B|}doFWa>Jk*hc4>W2#K!9{a$(HvO&fwdo4`>SE?fkb_g@BW}f{T6KR&eNgt_KW1{5Ucy@JbjVW;n6&OkwYE!*CKiLU7ntoe96-xwU?o_cOdEqqW(fz`l0lCWa&qieq`xKmVRXEM@P)j5p#6J933%7{{C(9cRKW4|Ck*yy;G?K{dY;-*C+7H(551fn}#n~<<8;sASS<0-<w-uj{N<|-;ezL$ls6r{Z5-{!CDh-rcHdRPn&5O21A=^0jOVSGeO;|)Mi>_>7ujdqRq6(rn=K+T96OiX)`Ty%d@O<`p{-tWcQ)va*>?jqRq6(16;J3mQgO9Ked^bQ9iPMYBMd<`Cgl8!65`4FUv$n5`QG|uav|eYOF^Re<blo5`QG|M-qP|@dpThfbdZeJ};6m-zD(;J?y&e6foT+@cd8g&q*#>)dW5-vfGtr_XXmz*Jk$x6Jv|q#U%d}?X~eHf#-kXlp>8;a-b14W;x8r',
    b'@e93Ki?|5!5u(bxOqXVN(Q!-XErI8ME}sc}UL-p&6L@wjG7N-E;8>O|ZI`d|O#;vV9E=2h&2q_jB=GSB{?J`L68Iy5KN9#Ofj<)XBZ2>A6L{p5gq-rW0e+vqBipvx-;pivQ+FiCm*#gumP!~&CZwt)sp>551iKPb2sDcu!HZiq7X$|aM<gjJ1`P;<286tnBrnAzFk!NQkeAXTGIH2X7!;sIhC=yW0>2jc8h!t?$UMEwg~Iw<&brE1LVznj#jnIG#@s8RpPrV@pM7+~(<};jc<o3~39a;m1XU(h7rf@JlOFH-b7FN_FDU#r@p_q9rCA&<o>S6V8FZD5789$4#Fb>Ib%n(+)XHDUf3Qu@@-g`-zwPY{d|>muaJ}2LaJIM>myt}dt}=-_%XM(WNYg+5vu+cs{2%-YQOHJsS!F_f`vL9@vk|xiNj$`?p<@jRYyL5Q10u&t;`!nFw;5IZ8mx&`u7jCZ$qwa^Bq1S7$}YKzM2x&;@cf@6gKx;-4_VbCgKx{=alx}Pc&h`R|I++**}KRzv0KWTPSa}XH04$P(IV6oN!;&rnnL2Lm&9F?7~KDg`-p0*Uf5)+;;Ki=ex&S2%6?$(2j+hC%p5&4FO#qz>Z(V=en9RA<bFWzNB(}~@4s~Zo`IjXK<@m3<(j)^z^fVXYF;I47bk)Pzx<GHXlDd;736&uR6s5*k@qaP>@1lBHt4#2cG{5l|G&NS>22c%qWE|5*@xkdwY%(1H5Awg92v5ENPt480=*?j+Ed~0UjE+Ns#YRbELyZ^!9xT&;&5j<{PXz8nQ_I5jnfa~5%cb_whm$IB5Yya9oE*x+7fc-59*yTI~}2QM7_H%Wkglt{Y1Sx3z|n^6OR-1?yN|z)#yOIJL{51RXU^IqaM-g<26{MM?F!v`Hgx{|JR>V@9F=BSL%J3coS4%?uUBsh$s0<z2E3hB&4ziFxW`cyG{%g+*OJ7G_qF;>xwFR0a5RaH#qt9Exu9jy(Vqy><`p?uMf=zv_-Gzu!wQ*CjzlvsrU2;PgQ*80{NbPMIG+',
    b'>#K9->J$*x#c`Odc<a_#4zoMAV9ste_%hzK0mRhx8`G(~imTy?TVflvT8yDZW_{POIF8;rR-&<PM{~dmZ-GJOU3nIy89Q+Q|!5u?l8571B{0`Mo!tXF3;CJE}6S|nohcF<-CngLr^asob#0R{`Fg*-NhjcO~qzB=9iQda)LkJH-_Y%1mwga+*u)PHAh3Oz%FVT9TIK*&7{bwqe3=FeDU|x6)Xbo5m5@*0^3^H^ohpCgTp)+7J$cq7&K|%~%gaYU+0M7#WECA5b2aT$^gWV3PcLEqK0MY^|EdbL3I4uCw0!S?Y)zSwN=~YODF=`;DcOnNyAR7-i2yqG`0S5UmNPj{03zA=u`-0RLB)uT#1t~AcctOGo@?DVWf<zbOxgf&@2`<QQL3)c(OCsOt&Z%rNBN3R1kW74IA{!IQm>9-{FeU&6p(h9-L7)i2LJ$Ok7!ZVhAl3uX9f<2dL<eFy5W#`C4Gpz7)ZS2gL+uT<H`Lxx`@f3XTQb##+8b(bsJ#NUOaIkS1=>fQ*3Lour4*`!ZX;qFLE4DVMpOr)IN)%Hza1`dIK#;oj$>Zj-f(!s-wh`>eB5ws!><izHXOC^Uc-3}w>2Uy;IxLj8s2K727M}*UIec+4#jaX!=(&oGTg}Ubi+Lj$25G#@DIa13{Ntg!|)BmdyJFm$tO%lQ!Jx#>bQ)BUO&(j(p^W+ETfz(chku-%P5D+-E^qTGRjeM7v*inG8(~oQD;?3Ov@4))2ndy%nbrA5O76+8v-1<aX!Y;*mSa$-j_oy&a^nv;slFxe$MrEjvq&T@I={FhLvSyTG>{{jeiZcpQcYu74p97E6lMg{-t(K9f3iQooP~9lm?|eX--;`#-uH$rkr{if2ESCA!$dNkyfM;X~R??=n?5ls+Vn!sW+-Ssz0hjsz)lTFaCNK)OqQ>Y$J_*q_L4Sc9O;o5+kLNv6nP9lg4h+I`wxXnAuIsgw_cy6j~`*DyTkCeW3b4^?~XG)d#8%R3BV#$O1w2f$GC+^^}`_Kq1x+V*#*kocvV}(7opJOdh',
    b"xAJXXVvhP58WLeKNkHy-c$KmExoGTR!ntufo$jaE(+X0|nETVu90W?N&nHD+65wl!v3W45)Go}pi9bE(NXWh_)i3>LI@vPh*5tR$@fAZX=i<!I$-<!I$-<!I$-<!I$-<!I$-<!I$-<!I%TRyb@K9$PkQDx(5+z>|_(V40L8R;8)O5oU8zl4zC2{@u8zV%$@)UXvjb+*2{)(-`+tj6J+@4aGRl7SIsV2<o3~>Wy8!v8}g3(<v1U`jraQ@41JA&Ti`T(wT5GM$8x+H3K8TMI0l9jIqHtcK8Mk6#x+e32I$J$Qbu>j4i(r6UU;`V5bT|Ne)bMaFPR*9HitxCFd2KS8%-oaLK_-9z@iLsS#C^#y~^z)kwajT5U+aA^C>nYxwu^&(rVT3v|{&d1p>%ax8K04qnPo@6M&mzngidH;8x#LDLs55%11C-OzkP^9{{6G~dvCL-YSEnt!>z`u6kZFW(ZN|LySY>ihNk&F<sX_1l*O`oDhu^_QQnzDu@wxz3i{{qW)TZvXVSeV!HfeDQbd-NR=0w0~Id-f!j-H+hNs+s*!Vw^={zpYFDgGl^AR;&Jo5zg@rE%*ytKviF}pt<P&7@}-|1w%g~s`^~w)m>2lC-hJ3VJ#Tk&W#YW_Q4?mHezjD*uKi~+Wg`Sz5UaPuOe|}Tuq04rlbJ}_FiV<MS?8p&as~?Ff<)d%p<5QK9xG>(3ziy|C3D6KenBR0uhp_tt;x;=^HvM<COK(=ieYswQ5abl#1^b}L9%*$ok?bg*d>{~)y8G1TC1H27OXaxI=n~LUS2F)7+-x!U||9rN;1{fIg-harm<MMX5Kq0Uu%mak%ATGGv1LNIXiT=P`YZ&%tXqz=t5blY>lofOI3`MdHJkGIxkBUZPMM8h4NPE=8Da0u9(oy1oE~yY2{O?>gnuMuzI|l3RW2DOr&N7Iu$AppJ7q1V6FLNb5tmc7~(?df}Iu~urP*(ved#DT6k8@Jd5S?V`wZ(l*iD*V~S(w$Cuv$UEG@c')))\r\n_ROUTES={int(k):[_R10",
    b"8_DATA['actions'][i] for i in ids] for k,ids in _R108_DATA['routes'].items()}\r\n_R108_SHOP_ROUTES={tuple(r['shops']):r['route'] for r in _R108_DATA['shops']}\r\ndel _R108_DATA\r\n_SETTINGS={'hand_align': True, 'weed_repair': True, 'sell_lead': True, 'budget_guard': False, 'room_guard': False, 'clamp_sells': False, 'dead_stock': False, 'terminal_liquidation': False, 'front_run': False}\r\n\r\n# EXP241: fixed two-policy choice, learned with five whole-team held-out folds.\r\n# The full-fit tree and all five fold trees use only first-two-shop YARN_STORE count.\r\n# V39 handles yarn-specialized worlds; EXP240 handles the other shop combinations.\r\n_R110_OLD_SHOPS={('BAKERY', 'BAKERY'): 0, ('BAKERY', 'BRUNCH_SPOT'): 0, ('BAKERY', 'FARMERS_MARKET'): 0, ('BAKERY', 'ICE_CREAM_SHOP'): 0, ('BAKERY', 'PET_CAFE'): 0, ('BAKERY', 'PIZZA_SHOP'): 0, ('BAKERY', 'SMOOTHIE_SHOP'): 0, ('BAKERY', 'YARN_STORE'): 3, ('BRUNCH_SPOT', 'BAKERY'): 0, ('BRUNCH_SPOT', 'BRUNCH_SPOT'): 0, ('BRUNCH_SPOT', 'FARMERS_MARKET'): 0, ('BRUNCH_SPOT', 'ICE_CREAM_S",
    b"HOP'): 0, ('BRUNCH_SPOT', 'PET_CAFE'): 0, ('BRUNCH_SPOT', 'PIZZA_SHOP'): 0, ('BRUNCH_SPOT', 'SMOOTHIE_SHOP'): 0, ('BRUNCH_SPOT', 'YARN_STORE'): 3, ('FARMERS_MARKET', 'BAKERY'): 0, ('FARMERS_MARKET', 'BRUNCH_SPOT'): 0, ('FARMERS_MARKET', 'FARMERS_MARKET'): 0, ('FARMERS_MARKET', 'ICE_CREAM_SHOP'): 0, ('FARMERS_MARKET', 'PET_CAFE'): 0, ('FARMERS_MARKET', 'PIZZA_SHOP'): 0, ('FARMERS_MARKET', 'SMOOTHIE_SHOP'): 0, ('FARMERS_MARKET', 'YARN_STORE'): 5, ('ICE_CREAM_SHOP', 'BAKERY'): 0, ('ICE_CREAM_SHOP', 'BRUNCH_SPOT'): 0, ('ICE_CREAM_SHOP', 'FARMERS_MARKET'): 0, ('ICE_CREAM_SHOP', 'ICE_CREAM_SHOP'): 0, ('ICE_CREAM_SHOP', 'PET_CAFE'): 0, ('ICE_CREAM_SHOP', 'PIZZA_SHOP'): 0, ('ICE_CREAM_SHOP', 'SMOOTHIE_SHOP'): 0, ('ICE_CREAM_SHOP', 'YARN_STORE'): 6, ('PET_CAFE', 'BAKERY'): 0, ('PET_CAFE', 'BRUNCH_SPOT'): 0, ('PET_CAFE', 'FARMERS_MARKET'): 0, ('PET_CAFE', 'ICE_CREAM_SHOP'): 0, ('PET_CAFE', 'PET_CAFE'): 0, ('PET_CAFE', 'PIZZA_SHOP'): 0, ('PET_CAFE', 'SMOOTHIE_SHOP'): 0, ('PET_CAFE', 'YARN_STORE'): 11, ('PIZZA_SHOP', 'BA",
    b"KERY'): 0, ('PIZZA_SHOP', 'BRUNCH_SPOT'): 0, ('PIZZA_SHOP', 'FARMERS_MARKET'): 0, ('PIZZA_SHOP', 'ICE_CREAM_SHOP'): 0, ('PIZZA_SHOP', 'PET_CAFE'): 0, ('PIZZA_SHOP', 'PIZZA_SHOP'): 0, ('PIZZA_SHOP', 'SMOOTHIE_SHOP'): 0, ('PIZZA_SHOP', 'YARN_STORE'): 7, ('SMOOTHIE_SHOP', 'BAKERY'): 0, ('SMOOTHIE_SHOP', 'BRUNCH_SPOT'): 0, ('SMOOTHIE_SHOP', 'FARMERS_MARKET'): 0, ('SMOOTHIE_SHOP', 'ICE_CREAM_SHOP'): 0, ('SMOOTHIE_SHOP', 'PET_CAFE'): 0, ('SMOOTHIE_SHOP', 'PIZZA_SHOP'): 0, ('SMOOTHIE_SHOP', 'SMOOTHIE_SHOP'): 0, ('SMOOTHIE_SHOP', 'YARN_STORE'): 8, ('YARN_STORE', 'BAKERY'): 9, ('YARN_STORE', 'BRUNCH_SPOT'): 9, ('YARN_STORE', 'FARMERS_MARKET'): 3, ('YARN_STORE', 'ICE_CREAM_SHOP'): 9, ('YARN_STORE', 'PET_CAFE'): 10, ('YARN_STORE', 'PIZZA_SHOP'): 6, ('YARN_STORE', 'SMOOTHIE_SHOP'): 11, ('YARN_STORE', 'YARN_STORE'): 12}\r\n\r\n_V92_TABLE={('BAKERY', 'YARN_STORE'): 9, ('BRUNCH_SPOT', 'YARN_STORE'): 9, ('FARMERS_MARKET', 'YARN_STORE'): 9, ('ICE_CREAM_SHOP', 'YARN_STORE'): 9, ('PET_CAFE', 'YARN_STORE'): 9, ('PIZZA_SHOP', 'YARN_S",
    b"TORE'): 9, ('SMOOTHIE_SHOP', 'YARN_STORE'): 9, ('YARN_STORE', 'BAKERY'): 9, ('YARN_STORE', 'BRUNCH_SPOT'): 9, ('YARN_STORE', 'FARMERS_MARKET'): 9, ('YARN_STORE', 'ICE_CREAM_SHOP'): 9, ('YARN_STORE', 'PET_CAFE'): 9, ('YARN_STORE', 'PIZZA_SHOP'): 9, ('YARN_STORE', 'SMOOTHIE_SHOP'): 9, ('YARN_STORE', 'YARN_STORE'): 9}\r\n\r\ndef _router(observation,step,state):\r\n    if step==2:\r\n        try:\r\n            _rv=observation['farms'][1-int(observation['player'])]\r\n            state['rkey']=(round(float(_rv['money']),3), int(observation['market']['inventory']['WHEAT']))\r\n        except Exception:\r\n            state['rkey']=None\r\n    if step>=144 and not state.get('day6'):\r\n        shops=tuple((_get(_get(observation,'town',{}),'unlocked_shops',[]) or [])[:2])\r\n        use_new=shops.count('YARN_STORE')<=0\r\n        state['expert']='EXP240' if use_new else 'V39'\r\n        state['route']=_R108_SHOP_ROUTES.get(shops,100) if use_new else _R110_OLD_SHOPS.get(shops,0)\r\n        state['route']=_V92_TABLE.get(shops,state['route'])\r\n  ",
    b"      state['day6']=True\r\n    if step>=648 and not state.get('day27'):\r\n        state['route']=2\r\n        state['day27']=True\r\n    return state.get('route',0)\r\n\r\n_R42_OPENING=[['BUY_PRODUCT', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 30], ['SELL', 'WHEAT', 30]]\r\nfor _r42_tape in _ROUTES.values():\r\n    _r42_tape[0]=dict(_r42_tape[0],market=[list(o) for o in _R42_OPENING])\r\ndel _r42_tape\r\n_IMPL=make_agent(_ROUTES,router=_router,**_SETTINGS)\r\n_IMPL.chassis.diagnostics['terminal_rescue_errors']=0\r\n\r\ndef agent(observation,configuration=None):\r\n    try:\r\n        action=_IMPL(observation,configuration)\r\n        pass\r\n        return action\r\n    except Exception:\r\n        return {'farmer':['PASS'],'hands':[],'market':[]}\r\n\r\n_SHOP_PARENT=agent\r\ndel agent\r\n\r\ndef agent(observation,configuration=None):\r\n    action=_SHOP_PARENT(observation,configuration)\r\n    try:\r\n        if _step_of(observation)>=718:\r\n            view=_View(observation,_int(_get(observation,'player',0)),_IMPL.chassis.cfg)\r\n            units=[]\r\n           ",
    b' for i,pos in enumerate(view.positions):\r\n                units.append([\'DROP\'] if _shed_adjacent(pos,view.board) and view.inv(i) else [\'PASS\'])\r\n            action={\'farmer\':units[0],\'hands\':units[1:],\'market\':[]}\r\n            projected=_IMPL.chassis._projected_shed(action,view)\r\n            action[\'market\']=[[\'SELL\',item,projected.get(item,0)] for item in PRODUCTS if projected.get(item,0)>0]\r\n            action[\'market\'].sort(key=lambda o:-view.prices.get(o[1],0)*o[2])\r\n    except Exception:\r\n        _IMPL.chassis.diagnostics[\'terminal_rescue_errors\'] += 1\r\n    return action\r\n\r\n# EXP-154 modifications: Ahmed Berat Ozer; public capabilities credited below.\r\n# Dmitrii Gluzdov Seven Turn Rescue and Kaggle engine contributors, Apache-2.0.\r\n\r\n_UNIT_NS={"__name__":"v28_own_unit_model"}\r\nexec(\'# SPDX-License-Identifier: Apache-2.0\\n# Extracted Kaggle / kaggle-environments contributor code; see NOTICE.txt.\\n"""Exact deterministic unit/decay semantics extracted from kaggle-environments 1.32.7.\\nSource kaggriculture.',
    b'py SHA256 bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e.\\nNo interpreter, market RNG, policy controls, or replay content is included.\\n"""\\n\\nENGINE_VERSION = "1.32.7"\\nSOURCE_SHA256 = "bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e"\\n\\nCROPS = {\\n    "WHEAT":      {"seed": 10, "first_yield_day": 2, "max_yield_day": 4, "interval": 0, "max_yield": 6, "ongoing": False},\\n    "CARROT":     {"seed": 20, "first_yield_day": 2, "max_yield_day": 3, "interval": 0, "max_yield": 4, "ongoing": False},\\n    "TOMATO":     {"seed": 50, "first_yield_day": 8, "max_yield_day": 8, "interval": 1, "max_yield": 4, "ongoing": True},\\n    "STRAWBERRY": {"seed": 100, "first_yield_day": 10, "max_yield_day": 10, "interval": 2, "max_yield": 4, "ongoing": True},\\n    "MELON":      {"seed": 80, "first_yield_day": 10, "max_yield_day": 12, "interval": 0, "max_yield": 6, "ongoing": False},\\n}\\n\\nANIMALS = {\\n    "GOOSE": {"cost": 300, "structure": "COOP",    "first_yield_day": 4, "interval": 1, "max_held"',
    b': 4, "product": "EGG"},\\n    "COW":   {"cost": 400, "structure": "PASTURE", "first_yield_day": 8, "interval": 2, "max_held": 6, "product": "MILK"},\\n    "SHEEP": {"cost": 500, "structure": "PASTURE", "first_yield_day": 6, "interval": 3, "max_held": 6, "product": "WOOL"},\\n}\\n\\nPRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]\\n\\nFARMER_MOVES = {\\n    "NORTH": (0, -1),\\n    "SOUTH": (0, 1),\\n    "EAST":  (1, 0),\\n    "WEST":  (-1, 0),\\n}\\n\\ndef _shed_access_tiles(board_size):\\n    """Four inner-corner tiles around the shed, in NWSE order."""\\n    half = board_size // 2\\n    return [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]\\n\\ndef _is_shed_adjacent(pos, board_size):\\n    return tuple(pos) in {(x, y) for (x, y) in _shed_access_tiles(board_size)}\\n\\ndef _new_plant(crop, day, turns_per_day):\\n    cd = CROPS[crop]\\n    return {\\n        "kind": "PLANT",\\n        "crop": crop,\\n        "planted_day": day,\\n        "watered_today": False',
    b',\\n        "consecutive_unwatered": 1,  # planting day counts as unwatered\\n        "yield_units": 0 if cd["ongoing"] else 1,\\n        "max_lifespan_step": (-1 if cd["ongoing"] else (day + cd["max_yield_day"] + 1) * turns_per_day),\\n        "fertilized_until_day": -1,\\n    }\\n\\ndef _new_animal(animal, day):\\n    a = ANIMALS[animal]\\n    return {\\n        "kind": a["structure"],\\n        "animal": animal,\\n        "placed_day": day,\\n        "yield_units": 0,\\n        "consecutive_unfed": 0,\\n        "fed_today": False,\\n        "cared_today": False,\\n        "fertilizer_available": False,\\n        "pending_care_bonus": 0,\\n    }\\n\\ndef _farmer_position(farm, idx):\\n    """idx 0 = main farmer, 1+ = hand index."""\\n    if idx == 0:\\n        return farm["farmer"]\\n    return farm["hands"][idx - 1] if idx - 1 < len(farm["hands"]) else None\\n\\ndef _set_farmer_position(farm, idx, pos):\\n    if idx == 0:\\n        farm["farmer"] = list(pos)\\n    else:\\n        farm["hands"][idx - 1] = list(pos)\\n\\ndef _farmer_invento',
    b'ry(private, idx):\\n    """Inventories list is [main_farmer, *hands]; grow it if idx is past the end."""\\n    while len(private["inventories"]) <= idx:\\n        private["inventories"].append({})\\n    return private["inventories"][idx]\\n\\ndef _inv_add(inv, item, n=1):\\n    inv[item] = inv.get(item, 0) + n\\n\\ndef _inv_take(inv, item, n=1):\\n    if inv.get(item, 0) < n:\\n        return False\\n    inv[item] -= n\\n    if inv[item] == 0:\\n        del inv[item]\\n    return True\\n\\ndef _apply_unit_action(farm, private, idx, action, board_size, day, turns_per_day, shed_capacity=100):\\n    """Process one farmer/hand\\\'s action. Invalid / illegal actions are silent no-ops."""\\n    if not isinstance(action, list) or not action:\\n        return\\n    op = action[0]\\n    pos = _farmer_position(farm, idx)\\n    if pos is None:\\n        return\\n    fx, fy = pos[0], pos[1]\\n    inv = _farmer_inventory(private, idx)\\n\\n    if op in FARMER_MOVES:\\n        dx, dy = FARMER_MOVES[op]\\n        nx, ny = fx + dx, fy + dy\\n        if not ',
    b'(0 <= nx < board_size and 0 <= ny < board_size):\\n            return\\n        # Movement onto LOCKED tiles is allowed: a hand can spawn on a locked\\n        # shed-access tile, and blocking movement would strand it there forever.\\n        # Tile operations (PLANT, WATER, etc.) still no-op on LOCKED tiles.\\n        _set_farmer_position(farm, idx, (nx, ny))\\n        return\\n\\n    if op == "PASS":\\n        return\\n\\n    tile = farm["tiles"][fy][fx]\\n\\n    # Shed operations resolve before the LOCKED guard. They use the tile only as\\n    # a standing position -- the shed itself is always owned -- and three of the\\n    # four shed-access tiles start LOCKED, so guarding them first would make the\\n    # shed unreachable from those tiles.\\n    if op == "DROP":\\n        if not _is_shed_adjacent((fx, fy), board_size):\\n            return\\n        shed = private["shed"]\\n        for item, n in list(inv.items()):\\n            if n <= 0:\\n                del inv[item]\\n                continue\\n            room = max(0, sh',
    b'ed_capacity - sum(shed.values()))\\n            take = min(n, room)\\n            if take > 0:\\n                shed[item] = shed.get(item, 0) + take\\n            del inv[item]\\n        return\\n\\n    if op == "PICKUP":\\n        if not _is_shed_adjacent((fx, fy), board_size):\\n            return\\n        if len(action) < 2:\\n            return\\n        item = action[1]\\n        n = int(action[2]) if len(action) >= 3 else 1\\n        if n <= 0:\\n            return\\n        # Seeds live in private["seeds"] and are consumed directly by PLANT;\\n        # they never pass through farmer inventory or the shed.\\n        available = private["shed"].get(item, 0)\\n        n = min(n, available)\\n        if n <= 0:\\n            return\\n        private["shed"][item] -= n\\n        _inv_add(inv, item, n)\\n        return\\n\\n    if op == "PLACE":\\n        if len(action) < 2:\\n            return\\n        item = action[1]\\n        # Animal placement: standing on a matching unoccupied structure. A LOCKED\\n        # tile is the string',
    b' "LOCKED", never a dict, so this branch cannot match\\n        # there and PLACE falls through to the shed path below.\\n        if (\\n            item in ANIMALS\\n            and isinstance(tile, dict)\\n            and tile.get("kind") == ANIMALS[item]["structure"]\\n            and "animal" not in tile\\n        ):\\n            if _inv_take(inv, item, 1):\\n                farm["tiles"][fy][fx] = _new_animal(item, day)\\n            return\\n        # Shed drop: orthogonally adjacent to the shed; obeys shedCapacity.\\n        if _is_shed_adjacent((fx, fy), board_size):\\n            n = int(action[2]) if len(action) >= 3 else 1\\n            if n <= 0:\\n                return\\n            n = min(n, inv.get(item, 0))\\n            if n <= 0:\\n                return\\n            current = sum(private["shed"].values())\\n            room = max(0, shed_capacity - current)\\n            n = min(n, room)\\n            if n <= 0:\\n                return\\n            inv[item] -= n\\n            if inv[item] == 0:\\n             ',
    b'   del inv[item]\\n            private["shed"][item] = private["shed"].get(item, 0) + n\\n        return\\n\\n    # Everything below mutates the tile the unit stands on, so it requires that\\n    # tile to be owned.\\n    if tile == "LOCKED":\\n        return\\n\\n    if op == "PLANT":\\n        if len(action) < 2:\\n            return\\n        crop = action[1]\\n        if crop not in CROPS:\\n            return\\n        if tile is not None:\\n            return\\n        if private["seeds"].get(crop, 0) <= 0:\\n            return\\n        private["seeds"][crop] -= 1\\n        farm["tiles"][fy][fx] = _new_plant(crop, day, turns_per_day)\\n        return\\n\\n    if op == "WATER":\\n        if not (isinstance(tile, dict) and tile.get("kind") == "PLANT"):\\n            return\\n        if tile["watered_today"]:\\n            return\\n        tile["watered_today"] = True\\n        crop_data = CROPS[tile["crop"]]\\n        if not crop_data["ongoing"]:\\n            age_days = day - tile["planted_day"]\\n            window_start = (crop_data',
    b'["max_yield_day"] + 1) // 2\\n            if window_start <= age_days <= crop_data["max_yield_day"]:\\n                bonus = 2 if tile["fertilized_until_day"] >= day else 1\\n                tile["yield_units"] = min(crop_data["max_yield"], tile["yield_units"] + bonus)\\n        return\\n\\n    if op == "HARVEST":\\n        if not isinstance(tile, dict):\\n            return\\n        if tile.get("yield_units", 0) <= 0:\\n            return\\n        if tile.get("kind") == "PLANT":\\n            crop_data = CROPS[tile["crop"]]\\n            if day - tile["planted_day"] < crop_data["first_yield_day"]:\\n                # Ongoing crops only accumulate yield_units after first_yield_day,\\n                # so reaching here with yield_units > 0 indicates a bug.\\n                if crop_data["ongoing"]:\\n                    print(\\n                        f"WARNING: HARVEST on immature ongoing {tile[\\\'crop\\\']} "\\n                        f"(planted day {tile[\\\'planted_day\\\']}, current day {day}, "\\n                        f"fir',
    b'st_yield_day {crop_data[\\\'first_yield_day\\\']}, "\\n                        f"yield_units {tile[\\\'yield_units\\\']}); should never happen"\\n                    )\\n                return\\n            units = tile["yield_units"]\\n            tile["yield_units"] = 0\\n            _inv_add(inv, tile["crop"], units)\\n            if not crop_data["ongoing"]:\\n                farm["tiles"][fy][fx] = None\\n        elif "animal" in tile:\\n            units = tile["yield_units"]\\n            tile["yield_units"] = 0\\n            _inv_add(inv, ANIMALS[tile["animal"]]["product"], units)\\n        return\\n\\n    if op == "FERTILIZE":\\n        if not (isinstance(tile, dict) and tile.get("kind") == "PLANT"):\\n            return\\n        if not _inv_take(inv, "FERTILIZER", 1):\\n            return\\n        # Active for `day`, `day+1`, `day+2` (3 days inclusive).\\n        tile["fertilized_until_day"] = max(tile.get("fertilized_until_day", -1), day + 2)\\n        return\\n\\n    if op == "DIG":\\n        if tile is None:\\n            retur',
    b'n\\n        # Removes plants, weeds, empty coop/pasture. Does NOT remove a placed animal.\\n        if isinstance(tile, dict) and "animal" in tile:\\n            return\\n        farm["tiles"][fy][fx] = None\\n        return\\n\\n    if op == "BUILD_COOP":\\n        if tile is not None:\\n            return\\n        farm["tiles"][fy][fx] = {"kind": "COOP"}\\n        return\\n\\n    if op == "BUILD_PASTURE":\\n        if tile is not None:\\n            return\\n        farm["tiles"][fy][fx] = {"kind": "PASTURE"}\\n        return\\n\\n    if op == "FEED":\\n        if not (isinstance(tile, dict) and "animal" in tile):\\n            return\\n        if tile["fed_today"]:\\n            return\\n        if not _inv_take(inv, "WHEAT", 1):\\n            return\\n        tile["fed_today"] = True\\n        return\\n\\n    if op == "COLLECT_FERTILIZER":\\n        if not (isinstance(tile, dict) and "animal" in tile):\\n            return\\n        if not tile["fertilizer_available"]:\\n            return\\n        tile["fertilizer_available"] = False\\n',
    b'        _inv_add(inv, "FERTILIZER", 1)\\n        return\\n\\n    if op == "CARE":\\n        if not (isinstance(tile, dict) and "animal" in tile):\\n            return\\n        if tile["cared_today"]:\\n            return\\n        tile["cared_today"] = True\\n        return\\n\\ndef _decay_plants(farm, step):\\n    board_size = len(farm["tiles"])\\n    for y in range(board_size):\\n        for x in range(board_size):\\n            tile = farm["tiles"][y][x]\\n            if not isinstance(tile, dict) or tile.get("kind") != "PLANT":\\n                continue\\n            mls = tile["max_lifespan_step"]\\n            if mls < 0 or step < mls:\\n                continue\\n            if (step - mls) % 2 != 0:\\n                continue\\n            tile["yield_units"] -= 1\\n            if tile["yield_units"] <= 0:\\n                farm["tiles"][y][x] = {"kind": "WEED"}\\n\\n\',_UNIT_NS)\r\n_PLANNER_NS=dict(_UNIT_NS)\r\nexec(\'"""E182 modification: Shop0909 last-seven-turn physical closure planner.\\n\\nNo engine imports, policy tapes, repla',
    b'y fixtures, RNG or remote calls.\\nThe only supported market continuation is SELL; unknown execution abstains.\\n"""\\nfrom copy import deepcopy\\nfrom time import perf_counter\\nSTART, FINAL = (712, 718)\\nOPS = set(FARMER_MOVES) | {\\\'PASS\\\', \\\'DROP\\\', \\\'PICKUP\\\', \\\'PLACE\\\', \\\'PLANT\\\', \\\'WATER\\\', \\\'HARVEST\\\', \\\'FERTILIZE\\\', \\\'DIG\\\', \\\'BUILD_COOP\\\', \\\'BUILD_PASTURE\\\', \\\'FEED\\\', \\\'CARE\\\', \\\'COLLECT_FERTILIZER\\\'}\\nITEMS = tuple(PRODUCTS) + tuple(ANIMALS)\\n\\nclass Unsupported(ValueError):\\n    pass\\n\\ndef _get(obj, key, default=None):\\n    return obj.get(key, default) if isinstance(obj, dict) else getattr(obj, key, default)\\n\\ndef _settings(config):\\n    size, turns, last = (_get(config, k, d) for k, d in [(\\\'boardSize\\\', 10), (\\\'turnsPerDay\\\', 24), (\\\'episodeSteps\\\', 720)])\\n    if (size, turns, last) != (10, 24, 720):\\n        raise Unsupported(\\\'requires pinned 10x10/24/720 terminal window\\\')\\n    cap = int(_get(config, \\\'shedCapacity\\\', 100))\\n    orders = min(10, int(_get(config, \\\'maxMarketOrdersPerTurn\\\', 10)))',
    b'\\n    if cap < 1 or orders < 1:\\n        raise Unsupported(\\\'invalid capacity/order limit\\\')\\n    return (size, turns, cap, orders)\\n\\ndef physical_state(obs):\\n    """Comparable own physical state; market prices and bank are intentionally excluded."""\\n    seat = int(_get(obs, \\\'player\\\', 0))\\n    farm = _get(obs, \\\'farms\\\')[seat]\\n    return ({k: v for k, v in farm.items() if k != \\\'money\\\'}, _get(obs, \\\'private\\\'))\\n\\ndef _commands(action, n):\\n    return [action.get(\\\'farmer\\\', [\\\'PASS\\\']), *action.get(\\\'hands\\\', [])][:n] + [[\\\'PASS\\\'] for _ in range(max(0, n - 1 - len(action.get(\\\'hands\\\', []))))]\\n\\ndef _clone_state(farm, private):\\n    f = dict(farm)\\n    f[\\\'tiles\\\'] = [[dict(tile) if isinstance(tile, dict) else tile for tile in row] for row in farm[\\\'tiles\\\']]\\n    f[\\\'farmer\\\'] = list(farm[\\\'farmer\\\'])\\n    f[\\\'hands\\\'] = [list(pos) for pos in farm[\\\'hands\\\']]\\n    f[\\\'unlocked_quadrants\\\'] = list(farm[\\\'unlocked_quadrants\\\'])\\n    pr = dict(private)\\n    pr[\\\'shed\\\'], pr[\\\'seeds\\\'] = (dict(private[',
    b"\\'shed\\']), dict(private[\\'seeds\\']))\\n    pr[\\'inventories\\'] = [dict(inv) for inv in private[\\'inventories\\']]\\n    return (f, pr)\\n\\ndef _clone_schedule(schedule):\\n    result = []\\n    for action in schedule:\\n        value = dict(action)\\n        if \\'farmer\\' in action:\\n            value[\\'farmer\\'] = list(action[\\'farmer\\'])\\n        for key in [\\'hands\\', \\'market\\']:\\n            if key in action:\\n                value[key] = [list(command) for command in action[key]]\\n        result.append(value)\\n    return result\\n\\ndef _validate(schedule, n, orders):\\n    for action in schedule:\\n        if not isinstance(action, dict) or set(action) - {\\'farmer\\', \\'hands\\', \\'market\\'}:\\n            raise Unsupported(\\'unknown action shape\\')\\n        if not isinstance(action.get(\\'hands\\', []), list):\\n            raise Unsupported(\\'hands must be a list\\')\\n        for command in [action.get(\\'farmer\\', [\\'PASS\\']), *action.get(\\'hands\\', [])]:\\n            if not isinstance(command, list) or not command or",
    b' command[0] not in OPS:\\n                raise Unsupported(\\\'unknown/malformed unit operation\\\')\\n            if command[0] in {\\\'PICKUP\\\', \\\'PLACE\\\', \\\'PLANT\\\'}:\\n                if len(command) < 2 or command[1] not in ITEMS:\\n                    raise Unsupported(\\\'unknown unit item\\\')\\n                if len(command) > 2 and (not isinstance(command[2], int)):\\n                    raise Unsupported(\\\'noninteger unit quantity\\\')\\n        market = action.get(\\\'market\\\', [])\\n        if not isinstance(market, list) or len(market) > orders:\\n            raise Unsupported(\\\'market order shape/cap\\\')\\n        for order in market:\\n            if not isinstance(order, list) or len(order) != 3 or order[0] != \\\'SELL\\\' or (order[1] not in PRODUCTS) or (not isinstance(order[2], int)) or (order[2] <= 0):\\n                raise Unsupported(\\\'baseline market must contain positive integer SELL only\\\')\\n\\ndef liquidation(shed, inherited_market, max_orders=10):\\n    """Use actual post-unit stock; retain first parent item o',
    b'rdering, then stable product order."""\\n    items = []\\n    for order in inherited_market:\\n        if order[1] not in items:\\n            items.append(order[1])\\n    items += [item for item in PRODUCTS if item not in items]\\n    orders = [[\\\'SELL\\\', item, int(shed.get(item, 0))] for item in items if shed.get(item, 0) > 0]\\n    if len(orders) > min(10, max_orders):\\n        raise Unsupported(\\\'actual final stock exceeds order slots\\\')\\n    return orders\\n\\ndef shop_liquidation(farm, private, prices):\\n    """Exact original final worker/drop and market rule, with current stock/prices."""\\n    return liquidate(FarmView({\\\'player\\\': 0, \\\'farms\\\': [farm], \\\'private\\\': private, \\\'market\\\': {\\\'prices\\\': prices}}))\\n\\ndef simulate(obs, config, schedule, *, final_liquidate=False, detailed=False, preserve_final_commands=False):\\n    """Exact own unit/decay and SELL-stock transitions. No claim to simulate shared prices."""\\n    size, turns, cap, order_cap = _settings(config)\\n    step = int(_get(obs, \\\'step\\\', -1))\\n  ',
    b"  if step < START or step + len(schedule) - 1 > FINAL or (not schedule):\\n        raise Unsupported(\\'outside 712..718; no day boundary or terminal auto-drop\\')\\n    if any(((t + 1) % turns == 0 for t in range(step, step + len(schedule)))):\\n        raise Unsupported(\\'day boundary\\')\\n    farm0, private0 = physical_state(obs)\\n    farm, private = _clone_state(farm0, private0)\\n    n = 1 + len(farm[\\'hands\\'])\\n    if len(private[\\'inventories\\']) != n or n > 32:\\n        raise Unsupported(\\'invalid/unbounded worker inventory shape\\')\\n    _validate(schedule, n, order_cap)\\n    deposited = [dict() for _ in range(n)]\\n    sold = {}\\n    snapshots, rows, events = ([], [], [])\\n    executed = _clone_schedule(schedule)\\n    overflow = 0\\n    for offset, action in enumerate(executed):\\n        t = step + offset\\n        if detailed:\\n            snapshots.append(_clone_state(farm, private))\\n        if t == FINAL and (not preserve_final_commands):\\n            action = shop_liquidation(farm, private, _get(obs, \\'m",
    b"arket\\')[\\'prices\\'])\\n            executed[offset] = action\\n        all_commands = [action.get(\\'farmer\\', [\\'PASS\\']), *action.get(\\'hands\\', [])]\\n        demand = {}\\n        for command in all_commands:\\n            if command[0] == \\'PLANT\\':\\n                demand[command[1]] = demand.get(command[1], 0) + 1\\n        blocked = {item for item, count in demand.items() if count > private[\\'seeds\\'].get(item, 0)}\\n        for actor, command in enumerate(_commands(action, n)):\\n            if command[0] == \\'PLANT\\' and command[1] in blocked:\\n                command = [\\'PASS\\']\\n            pos = farm[\\'farmer\\'] if actor == 0 else farm[\\'hands\\'][actor - 1]\\n            xy = tuple(pos)\\n            inv = private[\\'inventories\\'][actor]\\n            before_inv = dict(inv) if command[0] in {\\'DROP\\', \\'HARVEST\\', \\'COLLECT_FERTILIZER\\'} else None\\n            before_shed = dict(private[\\'shed\\']) if command[0] in {\\'DROP\\', \\'PLACE\\'} else None\\n            _apply_unit_action(farm, private, actor, command",
    b", size, t // turns, turns, cap)\\n            if before_shed is not None:\\n                delta = {item: amount - before_shed.get(item, 0) for item, amount in private[\\'shed\\'].items() if amount > before_shed.get(item, 0)}\\n                for item, amount in delta.items():\\n                    deposited[actor][item] = deposited[actor].get(item, 0) + amount\\n                if delta:\\n                    events.append({\\'offset\\': offset, \\'actor\\': actor, \\'op\\': command[0], \\'xy\\': xy, \\'deposited\\': delta})\\n                if command[0] == \\'DROP\\':\\n                    overflow += sum((max(0, amount - inv.get(item, 0) - delta.get(item, 0)) for item, amount in before_inv.items()))\\n            if command[0] in {\\'HARVEST\\', \\'COLLECT_FERTILIZER\\'}:\\n                delta = {item: amount - before_inv.get(item, 0) for item, amount in inv.items() if amount > before_inv.get(item, 0)}\\n                if delta:\\n                    events.append({\\'offset\\': offset, \\'actor\\': actor, \\'op\\': command[0], \\'xy\\'",
    b": xy, \\'acquired\\': delta})\\n        pre_market = dict(private[\\'shed\\'])\\n        if t == FINAL and preserve_final_commands and final_liquidate:\\n            action[\\'market\\'] = liquidation(pre_market, [], order_cap)\\n            prices = _get(obs, \\'market\\')[\\'prices\\']\\n            action[\\'market\\'].sort(key=lambda order: -int(prices.get(order[1], 0)) * order[2])\\n        for _, item, requested in action.get(\\'market\\', []):\\n            quantity = min(requested, private[\\'shed\\'].get(item, 0), 99999)\\n            if quantity > 0:\\n                private[\\'shed\\'][item] -= quantity\\n                sold[item] = sold.get(item, 0) + quantity\\n        _decay_plants(farm, t)\\n        rows.append({\\'pre_market_shed\\': pre_market, \\'post_market_shed\\': dict(private[\\'shed\\']), \\'deposited_by_actor\\': [dict(v) for v in deposited], \\'sold\\': dict(sold)})\\n    if detailed:\\n        snapshots.append(_clone_state(farm, private))\\n    return {\\'rows\\': rows, \\'states\\': snapshots, \\'events\\': events, \\'actions\\': ",
    b'executed, \\\'overflow_units\\\': overflow, \\\'farm\\\': farm, \\\'private\\\': private, \\\'sold\\\': sold}\\n\\ndef _ge(left, right):\\n    return all((left.get(item, 0) >= value for item, value in right.items()))\\n\\ndef dominates(candidate, baseline):\\n    """Preserve every baseline worker\\\'s actual deposit prefixes and shed availability."""\\n    if candidate[\\\'overflow_units\\\']:\\n        return False\\n    for new, old in zip(candidate[\\\'rows\\\'], baseline[\\\'rows\\\']):\\n        if not _ge(new[\\\'pre_market_shed\\\'], old[\\\'pre_market_shed\\\']):\\n            return False\\n        if not _ge(new[\\\'sold\\\'], old[\\\'sold\\\']):\\n            return False\\n        if any((not _ge(a, b) for a, b in zip(new[\\\'deposited_by_actor\\\'], old[\\\'deposited_by_actor\\\']))):\\n            return False\\n    return True\\n\\ndef _value(run, prices):\\n    shed = run[\\\'private\\\'][\\\'shed\\\']\\n    return sum(((run[\\\'sold\\\'].get(item, 0) + shed.get(item, 0)) * prices[item] for item in PRODUCTS))\\n\\ndef _walk(start, end):\\n    x, y = start\\n    tx, ty = end\\n    re',
    b'turn [[\\\'EAST\\\']] * max(0, tx - x) + [[\\\'WEST\\\']] * max(0, x - tx) + [[\\\'SOUTH\\\']] * max(0, ty - y) + [[\\\'NORTH\\\']] * max(0, y - ty)\\n\\ndef _return(pos):\\n    targets = _shed_access_tiles(10)\\n    target = min(targets, key=lambda xy: (abs(pos[0] - xy[0]) + abs(pos[1] - xy[1]), targets.index(xy)))\\n    return _walk(pos, target) + [[\\\'DROP\\\']]\\n\\ndef _proposals(run, actor, prices, max_per_actor):\\n    """One/two resource bundles plus direct carry closure, replacing a baseline suffix."""\\n    owners = {}\\n    for event in run[\\\'events\\\']:\\n        if \\\'acquired\\\' in event:\\n            owners.setdefault((tuple(event[\\\'xy\\\']), event[\\\'op\\\']), set()).add(event[\\\'actor\\\'])\\n    proposals = []\\n    seen = set()\\n    horizon = len(run[\\\'rows\\\'])\\n    for offset in range(horizon):\\n        farm, private = run[\\\'states\\\'][offset]\\n        pos = tuple(farm[\\\'farmer\\\'] if actor == 0 else farm[\\\'hands\\\'][actor - 1])\\n        inventory = private[\\\'inventories\\\'][actor]\\n        carried = sum((prices.get(item, 0) * count fo',
    b"r item, count in inventory.items()))\\n        prefix_deposits = run[\\'rows\\'][offset - 1][\\'deposited_by_actor\\'][actor] if offset else {}\\n        future_deposits = run[\\'rows\\'][-1][\\'deposited_by_actor\\'][actor]\\n        obligation = sum((prices.get(item, 0) * (count - prefix_deposits.get(item, 0)) for item, count in future_deposits.items()))\\n        bundles = []\\n        for y, row in enumerate(farm[\\'tiles\\']):\\n            for x, tile in enumerate(row):\\n                if not isinstance(tile, dict):\\n                    continue\\n                xy, operations, value = ((x, y), [], 0)\\n                if tile.get(\\'yield_units\\', 0) > 0:\\n                    item = tile.get(\\'crop\\') if tile.get(\\'kind\\') == \\'PLANT\\' else ANIMALS.get(tile.get(\\'animal\\'), {}).get(\\'product\\')\\n                    mature = item and (\\'animal\\' in tile or (START + offset) // 24 - tile[\\'planted_day\\'] >= CROPS[item][\\'first_yield_day\\'])\\n                    if mature and (not owners.get((xy, \\'HARVEST\\'), set()) - {ac",
    b"tor}):\\n                        operations.append([\\'HARVEST\\'])\\n                        value += prices[item] * tile[\\'yield_units\\']\\n                if tile.get(\\'fertilizer_available\\') and \\'animal\\' in tile and (not owners.get((xy, \\'COLLECT_FERTILIZER\\'), set()) - {actor}):\\n                    operations.append([\\'COLLECT_FERTILIZER\\'])\\n                    value += prices[\\'FERTILIZER\\']\\n                if operations:\\n                    distance = len(_walk(pos, xy)) + len(operations) + len(_return(xy))\\n                    if distance <= horizon - offset:\\n                        bundles.append((xy, operations, value, distance))\\n        bundles.sort(key=lambda b: (-b[2] / b[3], -b[2], b[0]))\\n        variants = [([], carried)] if carried else []\\n        for xy, ops, value, _ in bundles[:6]:\\n            variants.append(([(xy, ops)], carried + value))\\n        for first in bundles[:3]:\\n            for second in bundles[:3]:\\n                if first[0] != second[0]:\\n                    varian",
    b'ts.append(([(first[0], first[1]), (second[0], second[1])], carried + first[2] + second[2]))\\n        for stops, value in variants:\\n            route, cursor = ([], pos)\\n            for xy, ops in stops:\\n                route += _walk(cursor, xy) + ops\\n                cursor = xy\\n            route += _return(cursor)\\n            if len(route) > horizon - offset:\\n                continue\\n            route += [[\\\'PASS\\\']] * (horizon - offset - len(route))\\n            key = (offset, tuple((tuple(c) for c in route)))\\n            if key not in seen:\\n                seen.add(key)\\n                proposals.append((value - obligation, offset, route, len(stops)))\\n    proposals.sort(key=lambda p: (-p[0], p[1], p[2]))\\n    direct = [p for p in proposals if p[3] == 0 and p[0] > 0][:2]\\n    chosen = direct + [p for p in proposals if p not in direct]\\n    return chosen[:max_per_actor]\\n\\ndef plan_terminal(obs, config, baseline_remaining, *, max_simulations=64, passes=1, proposals_per_actor=4):\\n    """At 712 acc',
    b'ept seven actions; positive physical delivery is mandatory."""\\n    begun = perf_counter()\\n    fallback = {\\\'accepted\\\': False, \\\'reason\\\': \\\'\\\', \\\'actions\\\': None, \\\'simulations\\\': 0}\\n    try:\\n        if int(_get(obs, \\\'step\\\', -1)) != START or len(baseline_remaining) != FINAL - START + 1:\\n            raise Unsupported(\\\'planning requires step 712 and exactly seven actions through 718\\\')\\n        max_simulations = min(256, max(1, int(max_simulations)))\\n        passes = min(2, max(1, int(passes)))\\n        proposals_per_actor = min(16, max(1, int(proposals_per_actor)))\\n        baseline = simulate(obs, config, baseline_remaining, detailed=True)\\n        prices = {item: max(1, float(_get(obs, \\\'market\\\', {}).get(\\\'prices\\\', {}).get(item, 1))) for item in PRODUCTS}\\n        current, best = (_clone_schedule(baseline_remaining), baseline)\\n        baseline_value = best_value = _value(baseline, prices)\\n        changes, simulations = ([], 0)\\n        n = len(baseline[\\\'private\\\'][\\\'inventories\\\'])\\n        fo',
    b"r sweep in range(passes):\\n            improved = False\\n            for actor in range(n):\\n                winner = None\\n                for _, offset, route, bundle_count in _proposals(best, actor, prices, proposals_per_actor):\\n                    if simulations >= max_simulations:\\n                        break\\n                    trial = _clone_schedule(current)\\n                    for i, command in enumerate(route, offset):\\n                        if actor == 0:\\n                            trial[i][\\'farmer\\'] = command\\n                        else:\\n                            trial[i].setdefault(\\'hands\\', [])\\n                            while len(trial[i][\\'hands\\']) < n - 1:\\n                                trial[i][\\'hands\\'].append([\\'PASS\\'])\\n                            trial[i][\\'hands\\'][actor - 1] = command\\n                    evaluated = simulate(obs, config, trial)\\n                    simulations += 1\\n                    score = _value(evaluated, prices)\\n                    if s",
    b"core > best_value and dominates(evaluated, baseline):\\n                        required = {(tuple(e[\\'xy\\']), e[\\'op\\'], e[\\'actor\\']): e[\\'acquired\\'] for e in best[\\'events\\'] if \\'acquired\\' in e and e[\\'actor\\'] != actor}\\n                        acquired = {}\\n                        for e in evaluated[\\'events\\']:\\n                            if \\'acquired\\' in e:\\n                                key = (tuple(e[\\'xy\\']), e[\\'op\\'], e[\\'actor\\'])\\n                                dst = acquired.setdefault(key, {})\\n                                for item, amount in e[\\'acquired\\'].items():\\n                                    dst[item] = dst.get(item, 0) + amount\\n                        if all((_ge(acquired.get(k, {}), v) for k, v in required.items())):\\n                            winner, best_value = ((trial, offset, bundle_count), score)\\n                if winner:\\n                    current, offset, bundle_count = winner\\n                    best = simulate(obs, config, current, detailed=True)\\n  ",
    b"                  changes.append({\\'pass\\': sweep, \\'actor\\': actor, \\'from_step\\': START + offset, \\'resource_bundles\\': bundle_count, \\'estimated_stock_value\\': best_value})\\n                    improved = True\\n                if simulations >= max_simulations:\\n                    break\\n            if not improved or simulations >= max_simulations:\\n                break\\n        if not changes or best_value <= baseline_value:\\n            return {**fallback, \\'reason\\': \\'no positive physical delivery gain\\', \\'simulations\\': simulations, \\'changed_workers\\': [], \\'changes\\': [], \\'certificate\\': {\\'stock_value_gain_at_initial_prices\\': 0, \\'sold_unit_delta\\': dict.fromkeys(PRODUCTS, 0)}, \\'planning_ms\\': (perf_counter() - begun) * 1000}\\n        final = simulate(obs, config, current, final_liquidate=True, detailed=True)\\n        physical = simulate(obs, config, current)\\n        if not dominates(physical, baseline):\\n            raise Unsupported(\\'no zero-overflow dominating continuation\\')\\n        d",
    b"elta = {item: final[\\'sold\\'].get(item, 0) - baseline[\\'sold\\'].get(item, 0) for item in PRODUCTS}\\n        deposited_gain = any((final[\\'rows\\'][-1][\\'deposited_by_actor\\'][actor].get(item, 0) > baseline[\\'rows\\'][-1][\\'deposited_by_actor\\'][actor].get(item, 0) for actor in range(n) for item in PRODUCTS))\\n        worker_change = any((_commands(new, n) != _commands(old, n) for new, old in zip(final[\\'actions\\'], baseline[\\'actions\\'])))\\n        accepted = worker_change and deposited_gain and any((v > 0 for v in delta.values())) and all((v >= 0 for v in delta.values()))\\n        plan = {\\'accepted\\': accepted, \\'reason\\': \\'joint physical dominance\\' if accepted else \\'no improvement\\', \\'baseline\\': _clone_schedule(baseline_remaining), \\'actions\\': final[\\'actions\\'], \\'expected_states\\': final[\\'states\\'][:-1], \\'simulations\\': simulations, \\'changes\\': changes, \\'abandoned\\': False, \\'changed_workers\\': sorted({c[\\'actor\\'] for c in changes}), \\'certificate\\': {\\'baseline_rows\\': baseline[\\'rows\\'], \\'phy",
    b'sical_rows\\\': physical[\\\'rows\\\'], \\\'baseline_overflow\\\': baseline[\\\'overflow_units\\\'], \\\'candidate_overflow\\\': final[\\\'overflow_units\\\'], \\\'sold_unit_delta\\\': delta, \\\'stock_value_gain_at_initial_prices\\\': best_value - baseline_value, \\\'baseline_final_shed\\\': baseline[\\\'private\\\'][\\\'shed\\\'], \\\'final_shed\\\': final[\\\'private\\\'][\\\'shed\\\'], \\\'positive_physical_deposit_gain\\\': deposited_gain, \\\'markets_712_717_unchanged\\\': all((final[\\\'actions\\\'][i].get(\\\'market\\\', []) == baseline_remaining[i].get(\\\'market\\\', []) for i in range(FINAL - START)))}}\\n    except (Unsupported, KeyError, TypeError, ValueError, IndexError) as exc:\\n        plan = {**fallback, \\\'reason\\\': str(exc)}\\n    plan[\\\'planning_ms\\\'] = (perf_counter() - begun) * 1000\\n    return plan\\n\\ndef _effective_action(action, n):\\n    return (_commands(action, n), action.get(\\\'market\\\', []))\\n\\ndef _recover_observed(obs, config, parent_action, plan):\\n    """Bounded cargo salvage after deviation; never resume old positional commands."""\\n    farm, private =',
    b" physical_state(obs)\\n    positions = [farm[\\'farmer\\'], *farm[\\'hands\\']]\\n    remaining = FINAL - int(_get(obs, \\'step\\')) + 1\\n    room = max(0, int(_get(config, \\'shedCapacity\\', 100)) - sum(private[\\'shed\\'].values()))\\n    commands = []\\n    problems = []\\n    prices = _get(obs, \\'market\\', {}).get(\\'prices\\', {})\\n    for actor, (pos, inv) in enumerate(zip(positions, private[\\'inventories\\'])):\\n        command = [\\'PASS\\']\\n        if any((v > 0 for v in inv.values())):\\n            route = _return(pos)\\n            if len(route) > remaining:\\n                problems.append({\\'actor\\': actor, \\'reason\\': \\'unreachable cargo\\'})\\n            elif len(route) > 1:\\n                command = route[0]\\n            elif sum((max(0, q) for q in inv.values())) <= room:\\n                command = [\\'DROP\\']\\n                room -= sum((max(0, q) for q in inv.values()))\\n            else:\\n                items = [item for item in PRODUCTS if inv.get(item, 0) > 0]\\n                if room and items:\\n        ",
    b'            item = max(items, key=lambda i: (prices.get(i, 1) * min(inv[i], room), -PRODUCTS.index(i)))\\n                    quantity = min(inv[item], room)\\n                    command = [\\\'PLACE\\\', item, quantity]\\n                    room -= quantity\\n                else:\\n                    problems.append({\\\'actor\\\': actor, \\\'reason\\\': \\\'no shed capacity\\\'})\\n        commands.append(command)\\n    action = {\\\'farmer\\\': commands[0], \\\'hands\\\': commands[1:], \\\'market\\\': deepcopy(parent_action.get(\\\'market\\\', []))}\\n    if int(_get(obs, \\\'step\\\')) == FINAL:\\n        action[\\\'market\\\'] = []\\n        action = simulate(obs, config, [action], final_liquidate=True, preserve_final_commands=True)[\\\'actions\\\'][0]\\n    plan[\\\'recovery_steps\\\'] = plan.get(\\\'recovery_steps\\\', 0) + 1\\n    if problems:\\n        plan.setdefault(\\\'recovery_failures\\\', []).append({\\\'step\\\': int(_get(obs, \\\'step\\\')), \\\'problems\\\': problems})\\n    return action\\n\\ndef terminal_action(obs, config, parent_action, plan):\\n    """Canonical guar',
    b'd, pre-deviation abstention, observed recovery after deviation."""\\n    step = int(_get(obs, \\\'step\\\', -1))\\n    if not plan or not plan.get(\\\'accepted\\\') or (not START <= step <= FINAL):\\n        return parent_action\\n    if plan.get(\\\'abandoned\\\'):\\n        return _recover_observed(obs, config, parent_action, plan) if plan.get(\\\'deviated\\\') else parent_action\\n    index = step - START\\n    n = 1 + len(physical_state(obs)[0][\\\'hands\\\'])\\n    mismatch = physical_state(obs) != plan[\\\'expected_states\\\'][index] or _effective_action(parent_action, n) != _effective_action(plan[\\\'baseline\\\'][index], n)\\n    if mismatch:\\n        plan[\\\'abandoned\\\'] = True\\n        plan[\\\'abandon_step\\\'] = step\\n        plan[\\\'reason\\\'] = \\\'physical observation or effective baseline action diverged\\\'\\n        plan[\\\'safety_failure\\\'] = True\\n        return _recover_observed(obs, config, parent_action, plan) if plan.get(\\\'deviated\\\') else parent_action\\n    result = deepcopy(plan[\\\'actions\\\'][index])\\n    if step == FINAL:\\n        f',
    b"arm, private = physical_state(obs)\\n        result = shop_liquidation(farm, private, _get(obs, \\'market\\')[\\'prices\\'])\\n    if _commands(result, n) != _commands(parent_action, n):\\n        plan[\\'deviated\\'] = True\\n    return result',_PLANNER_NS)\r\n# EXP-154 integration by Ahmed Berat Ozer, derived from Dmitrii Gluzdov E182.\r\n# The preserved v27 parent is simulated on a private shadow only at step 712.\r\n_PRE_TERMINAL_AGENT=agent\r\ndel agent\r\n_TERMINAL_PLANS={}\r\n_TERMINAL_PREVIOUS={}\r\n_UPGRADE_STATS={'planning_calls':0,'accepted':0,'changed_steps':0,'aborted':0,'shadow_declines':0,'errors':0,'max_planning_ms':0.0}\r\n\r\ndef _parent_liquidate(farm, private, prices):\r\n    # Exactly v27's final projected DROP ordering, in the planner's private state.\r\n    view=_View({'player':0,'farms':[farm],'private':private,'market':{'prices':prices}},0,_IMPL.chassis.cfg)\r\n    commands=[['DROP'] if _shed_adjacent(pos,view.board) and view.inv(i) else ['PASS'] for i,pos in enumerate(view.positions)]\r\n    action={'farmer':commands[0",
    b"],'hands':commands[1:],'market':[]}\r\n    stock=_IMPL.chassis._projected_shed(action,view)\r\n    action['market']=[['SELL',item,stock.get(item,0)] for item in PRODUCTS if stock.get(item,0)>0]\r\n    action['market'].sort(key=lambda o:-view.prices.get(o[1],0)*o[2])\r\n    return action\r\n\r\n_PLANNER_NS['shop_liquidation']=_parent_liquidate\r\n\r\ndef _shadow_terminal(obs,config):\r\n    seat=int(obs['player']);chassis=_IMPL.chassis\r\n    state=chassis.players.get(seat)\r\n    if not state or state.get('last_step')!=711 or state.get('route')!=2:\r\n        return None\r\n    # No delayed weed/structure intervention may depend on an unmodeled future.\r\n    if state.get('pending'):\r\n        return None\r\n    shadow=copy.copy(chassis);shadow.players=copy.deepcopy(chassis.players)\r\n    shadow.diagnostics={k:0 for k in chassis.diagnostics}\r\n    projected=copy.deepcopy(obs);baseline=[];states=[]\r\n    for step in range(712,719):\r\n        projected['step']=step;projected['day']=step//24;projected['hour']=step%24\r\n        states.append(copy.d",
    b"eepcopy(shadow.players[seat]))\r\n        action=shadow.act(projected,config)\r\n        if step==718:\r\n            action=_parent_liquidate(projected['farms'][seat],projected['private'],projected['market']['prices'])\r\n        else:\r\n            market=action.get('market',[])\r\n            if len(market)!=9 or {o[1] for o in market}!=set(PRODUCTS) or any(o[0]!='SELL' or len(o)!=3 or type(o[2]) is not int or o[2]<100 for o in market):\r\n                return None\r\n        if any(shadow.diagnostics.values()):return None\r\n        run=_PLANNER_NS['simulate'](projected,config,[action])\r\n        if run['actions'][0]!=action:return None\r\n        baseline.append(action)\r\n        run['farm']['money']=projected['farms'][seat]['money']\r\n        projected['farms'][seat]=run['farm'];projected['private']=run['private']\r\n    return baseline,states\r\n\r\ndef agent(observation,configuration=None):\r\n    try:\r\n        step=int(observation['step']);seat=int(observation['player'])\r\n    except Exception:\r\n        return _PRE_TERMINAL_AGEN",
    b"T(observation,configuration)\r\n    previous=_TERMINAL_PREVIOUS.get(seat)\r\n    if step==0 or (previous is not None and step<=previous):_TERMINAL_PLANS.pop(seat,None)\r\n    _TERMINAL_PREVIOUS[seat]=step\r\n    plan=_TERMINAL_PLANS.get(seat)\r\n    if plan and plan.get('accepted') and 712<=step<=718:\r\n        if previous!=step-1:plan.update(abandoned=True,reason='nonconsecutive callback')\r\n        try:\r\n            result=_PLANNER_NS['terminal_action'](observation,configuration,plan['baseline'][step-712],plan)\r\n            if plan.get('abandoned'):\r\n                if not plan.get('abort_counted'):\r\n                    plan['abort_counted']=True;_UPGRADE_STATS['aborted']+=1\r\n                if not plan.get('deviated'):\r\n                    _IMPL.chassis.players[seat]=copy.deepcopy(plan['parent_states_before'][step-712])\r\n                    _TERMINAL_PLANS.pop(seat,None)\r\n                    return _PRE_TERMINAL_AGENT(observation,configuration)\r\n            _UPGRADE_STATS['changed_steps']+=int(result!=plan['baseline']",
    b"[step-712])\r\n            return result\r\n        except Exception:\r\n            _UPGRADE_STATS['errors']+=1\r\n            if plan.get('deviated'):\r\n                try:return _PLANNER_NS['_recover_observed'](observation,configuration,plan['baseline'][step-712],plan)\r\n                except Exception:return _parent_liquidate(observation['farms'][seat],observation['private'],observation['market']['prices'])\r\n    if step!=712:return _PRE_TERMINAL_AGENT(observation,configuration)\r\n    _UPGRADE_STATS['planning_calls']+=1\r\n    try:shadow=_shadow_terminal(observation,configuration)\r\n    except (ValueError,KeyError,TypeError,IndexError):shadow=None\r\n    if shadow is None:\r\n        _UPGRADE_STATS['shadow_declines']+=1\r\n        return _PRE_TERMINAL_AGENT(observation,configuration)\r\n    baseline,states=shadow\r\n    actual=_PRE_TERMINAL_AGENT(observation,configuration)\r\n    if actual!=baseline[0]:\r\n        _UPGRADE_STATS['shadow_declines']+=1;return actual\r\n    try:\r\n        plan=_PLANNER_NS['plan_terminal'](observation,con",
    b"figuration,baseline,max_simulations=64,passes=1,proposals_per_actor=4)\r\n        _UPGRADE_STATS['max_planning_ms']=max(_UPGRADE_STATS['max_planning_ms'],plan.get('planning_ms',0.0))\r\n        if not plan.get('accepted'):return actual\r\n        plan['parent_states_before']=states;_TERMINAL_PLANS[seat]=plan\r\n        _UPGRADE_STATS['accepted']+=1\r\n        result=_PLANNER_NS['terminal_action'](observation,configuration,actual,plan)\r\n        _UPGRADE_STATS['changed_steps']+=int(result!=actual)\r\n        return result\r\n    except Exception:\r\n        _UPGRADE_STATS['errors']+=1;return actual\r\n\r\nagent.telemetry=_UPGRADE_STATS\r\n\r\n# EXP-154: aurax7 Reactive v2 day-end storage guard, adapted to our v27 view.\r\n_PRE_ROOM_AGENT=agent\r\ndel agent\r\n_ROOM_STATS={'changed_turns':0,'added_units':0,'errors':0}\r\ndef agent(observation,configuration=None):\r\n    action=_PRE_ROOM_AGENT(observation,configuration)\r\n    try:\r\n        step=_step_of(observation)\r\n        if step%24!=23:return action\r\n        view=_View(observation,_int(_get(ob",
    b"servation,'player',0)),_IMPL.chassis.cfg)\r\n        carried=sum(max(0,int(n)) for inv in view.invs for n in inv.values())\r\n        needed=sum(view.shed.values())+carried-99\r\n        if needed<=0:return action\r\n        planned={}\r\n        for o in action.get('market',[]):\r\n            if o and o[0]=='SELL' and len(o)>=3:planned[o[1]]=planned.get(o[1],0)+max(0,int(o[2]))\r\n        result=copy.deepcopy(action);added=0\r\n        for item in sorted(PRODUCTS,key=lambda it:-int(view.prices.get(it,0))):\r\n            qty=min(needed,max(0,view.shed.get(item,0)-planned.get(item,0)))\r\n            if qty<=0:continue\r\n            if len(result['market'])>=10:break\r\n            result['market'].append(['SELL',item,qty]);needed-=qty;added+=qty\r\n            if needed<=0:break\r\n        if added:_ROOM_STATS['changed_turns']+=1;_ROOM_STATS['added_units']+=added\r\n        return result\r\n    except Exception:\r\n        _ROOM_STATS['errors']+=1;return action\r\n\r\nagent.telemetry=_ROOM_STATS\r\n\r\n# Incorporated upstream attribution and chang",
    b"e notice:\r\n# E182 Shop0909 + terminal physical closure (modified 2026-09-09)\r\n# \r\n# The active public parent is Yusuke Hayashi's yhay81/shop-router-0909 v3.\r\n# router_parent.py and actions.json are exact original bytes, not newly authored\r\n# routes. The parent credits aurax7's Reactive Router for sale timing and shed\r\n# projection; that attribution remains in router_parent.py. Original payload\r\n# LICENSE.txt is preserved unchanged (Apache License 2.0 text); it contains no\r\n# named copyright grantor and no separate NOTICE was supplied. No additional\r\n# ownership, endorsement, or upstream replay-data rights claim is made.\r\n# \r\n# Local changes: separate main.py/policy.py adapter; bounded start712 planner\r\n# copied from frozen E180/S78 and modified for seven callbacks, exact Shop final\r\n# liquidation, strict positive physical delivery/sale gain, and observation guards.\r\n# unit_model.py is an unchanged frozen E180 copy of Kaggle's extracted semantics.\r\n# The following original E180 notice is retained verbatim for ",
    b'attribution history.\r\n# Its references to Thomas files describe E180, not files supplied in this Shop\r\n# package: no Thomas tapes, trees or policy are included here.\r\n# \r\n# ----- Original E180 notice -----\r\n# Kaggriculture: Last-Mile Harvest Planner\r\n# Attribution and change notice\r\n# \r\n# Thomas Tschinkel is the author of the parent public state-router policy and its\r\n# published decision trees and action-route data. Source: Kaggriculture: 93.8% Win\r\n# Rate Public State Router, notebook version 3, scriptVersionId 347936183:\r\n# https://www.kaggle.com/code/thomastschinkel/kaggriculture-93-8-win-rate-public-state-router?scriptVersionId=347936183\r\n# The public notebook identifies its license as Apache License, Version 2.0.\r\n# Original published main.py SHA-256:\r\n# b87a27ed614a33329be85f1b662e51cf4078a019fee937afcebbbbf2f51f8522\r\n# \r\n# Changes to that source for this distribution: compressed route/tree literals\r\n# were decoded into readable tapes.json and trees.json; a read-only planned_action\r\n# helper was added;',
    b" descriptive headers and local data loading were adapted.\r\n# The original parent feature extraction, tree traversal and agent behavior are\r\n# retained. These public routes are not claimed as newly authored or trained by\r\n# the notebook distributor.\r\n# \r\n# unit_model.py contains deterministic unit-action and crop-decay definitions\r\n# extracted from Kaggle's kaggle-environments 1.32.7 Kaggriculture engine, licensed\r\n# under Apache License, Version 2.0. Credit: Kaggle and the kaggle-environments\r\n# contributors. Project: https://github.com/Kaggle/kaggle-environments\r\n# Source file: kaggle_environments/envs/kaggriculture/kaggriculture.py\r\n# Source SHA-256:\r\n# bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e\r\n# The extracted unit/decay definitions are not a newly authored game engine;\r\n# market price dynamics and the full interpreter are not part of this module.\r\n# \r\n# Additional work in this distribution: a bounded last-nine-action collection\r\n# and delivery planner, observation guards and recover",
    b"y, a settings-consuming\r\n# factory and entry point, standalone examples, and deterministic packaging.\r\n# The full Apache License, Version 2.0 is included as LICENSE.txt.\r\n# No endorsement by Thomas Tschinkel or Kaggle is implied.\r\n# \r\n# Data provenance limitation: Thomas's source refers to public replay data and\r\n# an upstream provenance.json. That original episode-level manifest, replay IDs\r\n# and individual replay-author identities were not supplied with the public\r\n# notebook/output used here. No names or episode lineage have been invented.\r\n# Notebook-level licensing does not independently establish the missing underlying\r\n# replay-data rights chain. The package supplies usable readable routes, not a\r\n# reproducible reconstruction of their original collection or training process.\r\n# \r\n# Packaging note: source inputs described as byte-exact above are\r\n# normalized to UTF-8/LF text with a final newline in this standalone\r\n# notebook package. Route JSON values and parent policy behavior are unchanged.\r\n\r\n# F",
    b"inal public-entry guard; measured separately and compared on captured observations.\r\n_V28_CORE=agent\r\ndel agent\r\n_IMPL.chassis.diagnostics['v28_entry_errors']=0\r\ndef agent(observation,configuration=None):\r\n    try:\r\n        return _V28_CORE(observation,configuration)\r\n    except Exception:\r\n        _IMPL.chassis.diagnostics['v28_entry_errors']+=1\r\n        return {'farmer':['PASS'],'hands':[],'market':[]}\r\nagent.telemetry=_ROOM_STATS\r\n\r\n# EXP-155: prvsiyan V221B finite tomato investment, adapted by Ahmed Berat Ozer.\r\n# Original public source is retained under research24/public; Apache-2.0.\r\nMAX_ORDERS=10\r\nclass FarmView(_View):\r\n    def __init__(self,obs):super().__init__(obs,int(obs['player']),_IMPL.chassis.cfg)\r\n    def inventory(self,actor):return self.inv(actor)\r\ndef projected_shed(action,view):return _IMPL.chassis._projected_shed(action,view)\r\n\r\nCROP_MIN_PRICE=70\r\n\r\n# V219: a finite late tomato investment with dedicated, observed workers.\r\n_V219_PARENT = agent\r\ndel agent\r\n_V219_FERTILIZE = True  # Builder",
    b" changes only this flag for the ablation.\r\n_V219_STATES = {}\r\n_V219_REPORT = {'commitments': 0, 'hire_requests': 0, 'confirmed_workers': 0,\r\n                'hire_shortfalls': 0, 'plant_requests': 0, 'confirmed_plants': 0,\r\n                'water_requests': 0, 'fertilize_requests': 0, 'harvest_requests': 0,\r\n                'confirmed_harvest_units': 0, 'drop_requests': 0,\r\n                'tomato_sale_requests': 0, 'budget_declines': 0, 'lost_plants': 0}\r\n\r\n\r\ndef _v219_fib(n):\r\n    a, b = 1, 1\r\n    for _ in range(n): a, b = b, a+b\r\n    return a\r\n\r\n\r\ndef _v219_native_day(native, day):\r\n    tape = _IMPL.chassis.routes[native['route']]\r\n    return tape[day*24:min((day+1)*24,719)]\r\n\r\n\r\ndef _v219_qualifies(obs, native):\r\n    farm=obs['farms'][obs['player']]\r\n    if len(farm['tiles']) != 10 or set(farm['unlocked_quadrants']) != {'NW','NE','SW'}:\r\n        return False\r\n    if farm['money'] < 12000 or obs['market']['prices']['TOMATO'] < CROP_MIN_PRICE:\r\n        return False\r\n    if sum(s in ('PIZZA_SHOP','FARMERS_MA",
    b"RKET') for s in obs['town']['unlocked_shops']) < 3:\r\n        return False\r\n    if any(farm['tiles'][y][x] != 'LOCKED' for y in (5,6) for x in range(5,10)):\r\n        return False\r\n    if obs['private']['seeds'].get('TOMATO',0) or obs['private']['shed'].get('TOMATO',0):\r\n        return False\r\n    if any(isinstance(t,dict) and t.get('crop')=='TOMATO' for row in farm['tiles'] for t in row):\r\n        return False\r\n    # The investment uses spare land and new worker indices. Avoid taking over\r\n    # any native tomato or land purchase obligation on the known own schedule.\r\n    for tape in _IMPL.chassis.routes.values():\r\n        for a in tape[432:719]:\r\n            if any(o and o[0]=='BUY_LAND' for o in a.get('market',[])):return False\r\n            if any(c==['PLANT','TOMATO'] for c in [a.get('farmer')]+a.get('hands',[])):return False\r\n    return True\r\n\r\n\r\ndef _v219_walk(pos, target):\r\n    x,y=pos;tx,ty=target\r\n    if x != tx:return ['EAST' if x < tx else 'WEST']\r\n    if y != ty:return ['SOUTH' if y < ty else 'NORTH'",
    b"]\r\n    return None\r\n\r\n\r\ndef _v219_home(pos):\r\n    return min(((4,4),(5,4),(4,5),(5,5)),key=lambda p:abs(pos[0]-p[0])+abs(pos[1]-p[1]))\r\n\r\n\r\ndef _v219_request(obs, action, state, native):\r\n    step=int(obs['step']);day=step//24;offset=step%24\r\n    farm=obs['farms'][obs['player']];private=obs['private']\r\n    # If the planting-day transaction could not complete, abandon investment.\r\n    # Later purchases would miss the finite day26..29 production window.\r\n    if not state.get('committed') and day!=18:return action\r\n    if state.get('requested_day')==day:return action\r\n    planned=_v219_native_day(native,day)\r\n    # EXP240: committed crops must wait for the native worker indices.\r\n    # New schedules finish native hiring at hour4 or6. The existing two/three\r\n    # crop-worker groups can still water/harvest their ten cells by midnight.\r\n    # Initial investment stays within hour3; ordinary schedules are unchanged.\r\n    latest_hire=max((i for i,a in enumerate(planned) if any(o and o[0]=='HIRE' for o in a.get('marke",
    b"t',[]))),default=-1)\r\n    deadline=6 if state.get('committed') and 3<latest_hire<=6 else 3\r\n    if offset>deadline:return action\r\n    remaining=planned[offset+1:]\r\n    if any(o and o[0]=='HIRE' for a in remaining for o in a.get('market',[])):\r\n        return action\r\n    parent_hires=sum(bool(o) and o[0]=='HIRE' for o in action['market'])\r\n    expected=max(len(a.get('hands',[])) for a in planned)\r\n    if len(farm['hands'])+parent_hires != expected:return action\r\n    fertilizer=bool(_V219_FERTILIZE and day in (24,27) and _r79_tomato_fertilizer_worthwhile(obs,action))\r\n    # One watering tour: at most 2 entry moves + 9 between tiles + 10 waters.\r\n    # A hire request by hour2 leaves at least21 callbacks after confirmation.\r\n    crop_workers=1 if day in (19,20,21,22,23,25) and offset<=2 else (3 if 26<=day<=28 else 2)\r\n    labor=_r53_labor_assignment(obs,action,fertilizer)\r\n    if labor is not None:crop_workers=labor['workers']\r\n    count=crop_workers+int(fertilizer and day==27 and labor is None)\r\n    extra=[]\r\n  ",
    b"  if not state.get('committed'):\r\n        extra += [['BUY_LAND'],['BUY_SEED','TOMATO',10]]\r\n    fertilizer_quantity=_r70_parent_fert_qty(obs,action,planned,offset) if fertilizer else 0\r\n    if fertilizer:extra.append(['BUY_PRODUCT','FERTILIZER',fertilizer_quantity])\r\n    extra += [['HIRE'] for _ in range(count)]\r\n    if len(action['market'])+len(extra)>MAX_ORDERS:return action\r\n    # No assumed sale proceeds. Reserve 3,000 for parent obligations and price\r\n    # movement; the qualification separately requires 12,000 initial liquidity.\r\n    budget=sum(_v219_fib(n) for n in range(farm['hires_today'],farm['hires_today']+parent_hires+count))\r\n    if not state.get('committed'):budget+=4500\r\n    if fertilizer:budget+=fertilizer_quantity*(obs['market']['prices']['FERTILIZER']+5)\r\n    for order in action['market']:\r\n        if not order:continue\r\n        if order[0]=='BUY_PRODUCT':budget+=int(order[2])*(int(obs['market']['prices'][order[1]])+10)\r\n        elif order[0]=='BUY_ANIMAL':budget+=int(order[2])*{'COW':400,'S",
    b"HEEP':500,'GOOSE':300}[order[1]]\r\n        elif order[0]=='BUY_SEED':budget+=int(order[2])*{'WHEAT':10,'CARROT':20,'TOMATO':50,'STRAWBERRY':100,'MELON':80}[order[1]]\r\n    if farm['money']<budget+3000:\r\n        _V219_REPORT['budget_declines']+=1;return action\r\n    state['pending']={'step':step,'first_actor':expected+1,'count':count,'crop_workers':crop_workers,'fertilizer':fertilizer,'labor':labor}\r\n    if labor is not None:\r\n        _R53_LABOR_REPORT['labor_requests']+=1;_R53_LABOR_REPORT['labor_hires_avoided']+=1;_R53_LABOR_REPORT['labor_day'+str(day)]+=1\r\n    state['requested_day']=day\r\n    _V219_REPORT['hire_requests']+=count\r\n    if not state.get('committed'):\r\n        state['committed']=True;_V219_REPORT['commitments']+=1\r\n    changed=copy.deepcopy(action);changed['market']+=extra\r\n    return changed\r\n\r\n\r\ndef _v219_worker(obs, state, actor, role):\r\n    day=int(obs['step'])//24;step=int(obs['step']);view=FarmView(obs)\r\n    pos=tuple(view.positions[actor]);inv=view.inventory(actor)\r\n    targets=role['targets",
    b"']\r\n    # Actual cargo differences, observed on the next callback, verify harvests.\r\n    previous=state['last_work'].get(actor)\r\n    if previous and previous['step']==step-1 and previous['command']==['HARVEST']:\r\n        _V219_REPORT['confirmed_harvest_units']+=max(0,int(inv.get('TOMATO',0))-previous['tomatoes'])\r\n    if role.get('needs_fertilizer') and not role.get('loaded'):\r\n        home=_v219_home(pos)\r\n        walk=_v219_walk(pos,home)\r\n        if walk:return walk\r\n        desired=role.get('fertilizer_quantity',10 if role['kind']=='fertilizer' else 5)\r\n        if inv.get('FERTILIZER',0)>=desired:role['loaded']=True\r\n        elif role.get('pickup_requested'):\r\n            # Never spend repeated turns waiting for stock that was not bought.\r\n            role['loaded']=True;role['fertilizer_available']=int(inv.get('FERTILIZER',0))\r\n        elif view.shed.get('FERTILIZER',0)>=desired:\r\n            role['pickup_requested']=True;return ['PICKUP','FERTILIZER',desired]\r\n        else:role['loaded']=True\r\n    todo=",
    b"[]\r\n    for target in targets:\r\n        x,y=target;tile=view.tiles[y][x]\r\n        tomato=isinstance(tile,dict) and tile.get('crop')=='TOMATO'\r\n        if tomato and target not in state['seen_plants']:\r\n            state['seen_plants'].add(target);_V219_REPORT['confirmed_plants']+=1\r\n        if target in state['seen_plants'] and not tomato and target not in state['lost']:\r\n            state['lost'].add(target);_V219_REPORT['lost_plants']+=1\r\n        command=None\r\n        if role['kind']=='fertilizer':\r\n            if tomato and tile.get('fertilized_until_day',-1)<day+2 and inv.get('FERTILIZER',0)>0:\r\n                command=['FERTILIZE']\r\n        elif day==18 and not tomato:\r\n            if tile is None and obs['private']['seeds'].get('TOMATO',0)>0:command=['PLANT','TOMATO']\r\n            elif isinstance(tile,dict) and tile.get('kind')=='WEED':command=['DIG']\r\n        elif tomato:\r\n            # No later production follows the final day, so watering then would\r\n            # consume time needed to harvest and d",
    b"eliver the final cargo.\r\n            if day<29 and not tile.get('watered_today'):command=['WATER']\r\n            elif role.get('needs_fertilizer') and tile.get('fertilized_until_day',-1)<day+2 and inv.get('FERTILIZER',0)>0:\r\n                command=['FERTILIZE']\r\n            elif tile.get('yield_units',0)>0:command=['HARVEST']\r\n        if command:todo.append((target,command))\r\n    # Final return has priority once only the exact distance plus DROP remains.\r\n    home=_v219_home(pos);distance=abs(pos[0]-home[0])+abs(pos[1]-home[1])\r\n    if step>=718-distance and inv.get('TOMATO',0):\r\n        return _v219_walk(pos,home) or ['PLACE','TOMATO',int(inv.get('TOMATO',0))]\r\n    if todo:\r\n        target,command=min(todo,key=lambda v:(abs(pos[0]-v[0][0])+abs(pos[1]-v[0][1]),targets.index(v[0])))\r\n        return _v219_walk(pos,target) or command\r\n    if inv.get('TOMATO',0):return _v219_walk(pos,home) or ['PLACE','TOMATO',int(inv['TOMATO'])]\r\n    if any(inv.values()):return _v219_walk(pos,home) or ['DROP']\r\n    return ['PASS",
    b"']\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    action=_V219_PARENT(observation,configuration)\r\n    step=int(observation['step']);player=int(observation['player']);day=step//24\r\n    state=_V219_STATES.get(player)\r\n    if state is None or step<=state['last_step']:\r\n        state={'last_step':step,'day':-1,'workers':{},'last_work':{},'seen_plants':set(),'lost':set(),\r\n               'targets':[(x,y) for y in (5,6) for x in range(5,10)]}\r\n        _V219_STATES[player]=state\r\n    state['last_step']=step\r\n    native=_IMPL.chassis.players[player]\r\n    if step==432:state['eligible']=_v219_qualifies(observation,native)\r\n    if not state.get('eligible') or day<18:return action\r\n    if state['day']!=day:\r\n        state['day']=day;state['workers']={};state['last_work']={}\r\n    farm=observation['farms'][player]\r\n    pending=state.pop('pending',None)\r\n    if pending:\r\n        if len(farm['hands'])+1 >= pending['first_actor']+pending['count'] and 'SE' in farm['unlocked_quadrants']:\r\n            for index in range(p",
    b"ending['count']):\r\n                fertilizer_worker=index==pending['crop_workers']\r\n                if fertilizer_worker:targets=state['targets']\r\n                elif pending['crop_workers']==1:targets=state['targets']\r\n                elif pending['crop_workers']==2:targets=state['targets'][index*5:index*5+5]\r\n                else:targets=[[(5,5),(6,5),(7,5)],[(8,5),(9,5),(9,6),(8,6)],[(5,6),(6,6),(7,6)]][index]\r\n                state['workers'][pending['first_actor']+index]={'kind':'fertilizer' if fertilizer_worker else 'crop','targets':targets,\r\n                    'needs_fertilizer':pending['fertilizer'] and (day==24 or fertilizer_worker)}\r\n                if pending.get('labor') is not None:\r\n                    role=state['workers'][pending['first_actor']+index]\r\n                    role['targets']=[tuple(p) for p in pending['labor']['paths'][index]]\r\n                    role['needs_fertilizer']=pending['labor']['fertilizer'];role['fertilizer_quantity']=len(role['targets'])\r\n                    if tup",
    b"le(farm['hands'][pending['first_actor']+index-1])!=tuple(pending['labor']['spawns'][index]):_R53_LABOR_REPORT['labor_spawn_errors']+=1\r\n                    if index==0:_R53_LABOR_REPORT['labor_confirmed']+=1\r\n            _V219_REPORT['confirmed_workers']+=pending['count']\r\n        else:_V219_REPORT['hire_shortfalls']+=pending['count']\r\n    action=_v219_request(observation,action,state,native)\r\n    if state['workers']:\r\n        commands=[action.get('farmer') or ['PASS']]+list(action.get('hands') or [])\r\n        commands += [['PASS'] for _ in range(len(farm['hands'])+1-len(commands))]\r\n        for actor,role in state['workers'].items():\r\n            if actor>=len(commands):continue\r\n            command=_v219_worker(observation,state,actor,role)\r\n            commands[actor]=command\r\n            name={'PLANT':'plant_requests','WATER':'water_requests','FERTILIZE':'fertilize_requests',\r\n                  'HARVEST':'harvest_requests','DROP':'drop_requests'}.get(command[0])\r\n            if name:_V219_REPORT[name]+=1\r",
    b"\n            state['last_work'][actor]={'step':step,'command':command,'tomatoes':observation['private']['inventories'][actor].get('TOMATO',0)}\r\n        action=copy.deepcopy(action);action['farmer'],action['hands']=commands[0],commands[1:]\r\n    if state.get('committed') and len(action['market'])<MAX_ORDERS and not any(o[:2]==['SELL','TOMATO'] for o in action['market']):\r\n        quantity=projected_shed(action,FarmView(observation)).get('TOMATO',0)\r\n        if quantity>0:\r\n            action=copy.deepcopy(action);action['market'].append(['SELL','TOMATO',quantity])\r\n            _V219_REPORT['tomato_sale_requests']+=quantity\r\n    return action\r\n\r\n\r\nagent.telemetry=_V219_REPORT\r\n\r\n# V221B: labor-only ablation of frozen V219G; not yet publicly scored.\r\n\r\n\r\n# Crop workers own their final routes after commitment. A private parent shadow\r\n# does not contain these obligations, so terminal rescue must abstain there.\r\n_ORIGINAL_SHADOW_TERMINAL=_shadow_terminal\r\ndef _shadow_terminal(obs,config):\r\n    if _V219_STATES.get(i",
    b"nt(obs['player']),{}).get('committed'):return None\r\n    return _ORIGINAL_SHADOW_TERMINAL(obs,config)\r\n\r\nAPPLY_TIMING=False\r\n\r\n_EXPERIMENT_PARENT=agent\r\ndel agent\r\n_V219_REPORT['extra_fertilizer_days']=0\r\n_V219_REPORT['reordered_market_turns']=0\r\n_V219_REPORT['errors']=0\r\ndef agent(observation,configuration=None):\r\n    try:\r\n        action=_EXPERIMENT_PARENT(observation,configuration)\r\n        if APPLY_TIMING and int(observation['step'])>=144:action=_v224_sales_first(action)\r\n        return action\r\n    except Exception:\r\n        _V219_REPORT['errors']+=1\r\n        return {'farmer':['PASS'],'hands':[],'market':[]}\r\nagent.telemetry=_V219_REPORT\r\n\r\ndef _v224_sales_first(action):\r\n    original=action.get('market',[])[:MAX_ORDERS]\r\n    orders=[list(o) for o in original if o and (o[0] in ('HIRE','BUY_LAND') or (len(o)>=3 and int(o[2])>0))]\r\n    for index in range(len(orders)):\r\n        order=orders[index]\r\n        if order[0]!='SELL':continue\r\n        cursor=index\r\n        while cursor>0:\r\n            previous=orders",
    b'[cursor-1]\r\n            if previous[0]==\'SELL\':break\r\n            if previous[0] in (\'BUY_PRODUCT\',\'BUY_ANIMAL\') and previous[1]==order[1]:break\r\n            orders[cursor-1],orders[cursor]=orders[cursor],orders[cursor-1]\r\n            cursor-=1\r\n    if orders==original:return action\r\n    _V219_REPORT[\'reordered_market_turns\']+=1\r\n    changed=copy.deepcopy(action);changed[\'market\']=orders\r\n    return changed\r\n_ORDER_PARENT=agent\r\ndel agent\r\n\r\ndef agent(observation,configuration=None):\r\n    try:\r\n        action=_ORDER_PARENT(observation,configuration)\r\n        if int(observation["step"])>=144:action=_v224_sales_first(action)\r\n        return action\r\n    except Exception:\r\n        _V219_REPORT["errors"]+=1\r\n        return {"farmer":["PASS"],"hands":[],"market":[]}\r\nagent.telemetry=_V219_REPORT\r\n\r\n_V31_CORE=agent\r\ndel agent\r\n_IMPL.chassis.diagnostics[\'production_errors\']=0\r\n_IMPL.chassis.diagnostics[\'v31_entry_errors\']=0\r\ndef agent(observation,configuration=None):\r\n    before=_V219_REPORT[\'errors\']\r\n    try:\r\n    ',
    b"    action=_V31_CORE(observation,configuration)\r\n        _IMPL.chassis.diagnostics['production_errors']+=_V219_REPORT['errors']-before\r\n        return action\r\n    except Exception:\r\n        _IMPL.chassis.diagnostics['v31_entry_errors']+=1\r\n        return {'farmer':['PASS'],'hands':[],'market':[]}\r\nagent.telemetry=_V219_REPORT\r\n\r\n# Apache-2.0; later cattle transfer from prvsiyan, Moon (2026-09-10).\r\n# Bounded livestock substitution; confirm owned animals before redirecting workers.\r\n_V231_PARENT=agent\r\n_V231_CAP=4\r\n_V231_STATES={}\r\n_V231_REPORT={}\r\n\r\ndef _v231_new_state():\r\n    return {'last':-1,'confirmed':0,'reserved':0,'pending_buy':None,\r\n            'carrying':{},'pending_places':[],'sites':{},'milk_credit':0,\r\n            'requested':0,'failed_purchase_units':0,'picked':0,'placed':0,\r\n            'failed_placements':0,'extra_milk_harvested':0,'extra_milk_sale_requests':0}\r\n\r\ndef _v231_controller(obs,action,state,cap):\r\n    step=int(obs['step']);seat=int(obs['player']);farm=obs['farms'][seat]\r\n    private",
    b"=obs['private'];shed=private['shed'];inventories=private['inventories']\r\n    positions=[farm['farmer'],*farm['hands']]\r\n    pending=state['pending_buy']\r\n    if pending is not None:\r\n        gained=max(0,int(shed.get('COW',0))-pending['before'])\r\n        confirmed=min(pending['quantity'],gained)\r\n        state['confirmed']+=confirmed;state['reserved']+=confirmed\r\n        state['failed_purchase_units']+=pending['quantity']-confirmed\r\n        state['pending_buy']=None\r\n    for pending in state['pending_places']:\r\n        x,y=pending['site'];tile=farm['tiles'][y][x]\r\n        if (isinstance(tile,dict) and tile.get('animal')=='COW'\r\n                and tile.get('placed_day')==pending['day']):\r\n            state['sites'][(x,y)]=pending['day'];state['placed']+=1\r\n            actor=pending['actor'];state['carrying'][actor]=max(0,state['carrying'].get(actor,0)-1)\r\n        else:state['failed_placements']+=1\r\n    state['pending_places']=[]\r\n    state['last']=step\r\n    result=copy.deepcopy(action)\r\n    workers=[result.ge",
    b"t('farmer') or ['PASS'],*(result.get('hands') or [])]\r\n    seen_harvest=set();cow_available=int(shed.get('COW',0));occupied=set()\r\n    for actor,work in enumerate(workers[:len(positions)]):\r\n        inventory=inventories[actor] if actor<len(inventories) else {}\r\n        x,y=positions[actor];tile=farm['tiles'][y][x];site=(x,y)\r\n        if (work==['HARVEST'] and site in state['sites'] and site not in seen_harvest\r\n                and isinstance(tile,dict) and tile.get('animal')=='COW'\r\n                and tile.get('placed_day')==state['sites'][site]):\r\n            units=max(0,int(tile.get('yield_units',0)))\r\n            state['milk_credit']+=units;state['extra_milk_harvested']+=units\r\n            seen_harvest.add(site)\r\n        if len(work)>=2 and work[:2]==['PICKUP','SHEEP']:\r\n            quantity=max(0,int(work[2]) if len(work)>2 else 1)\r\n            center=len(farm['tiles'])//2\r\n            if (quantity and state['reserved']>=quantity and cow_available>=quantity\r\n                    and x in (center-1,center",
    b") and y in (center-1,center)\r\n                    and not any(inventory.get(a,0) for a in ('COW','SHEEP','GOOSE'))):\r\n                work[1]='COW';state['reserved']-=quantity;cow_available-=quantity\r\n                state['carrying'][actor]=state['carrying'].get(actor,0)+quantity\r\n                state['picked']+=quantity\r\n        if (len(work)>=2 and work[:2]==['PLACE','SHEEP']\r\n                and state['carrying'].get(actor,0)>0 and inventory.get('COW',0)>0\r\n                and isinstance(tile,dict) and tile.get('kind')=='PASTURE'\r\n                and 'animal' not in tile and site not in occupied):\r\n            work[1]='COW'\r\n            state['pending_places'].append({'actor':actor,'site':site,'day':step//24})\r\n        if (len(work)>=2 and work[0]=='PLACE' and work[1] in ('COW','SHEEP','GOOSE')\r\n                and inventory.get(work[1],0)>0):occupied.add(site)\r\n    result['farmer'],result['hands']=workers[0],workers[1:]\r\n    market=result.get('market',[])\r\n    animal_orders=[o for o in market if len(o)>",
    b"=3 and o[0]=='BUY_ANIMAL']\r\n    shops=obs['town']['unlocked_shops'];prices=obs['market']['prices']\r\n    counts={'COW':0,'SHEEP':0}\r\n    for line in farm['tiles']:\r\n        for tile in line:\r\n            if isinstance(tile,dict) and tile.get('animal') in counts:counts[tile['animal']]+=1\r\n    cargo=sum(int(inv.get(a,0)) for inv in inventories for a in ('COW','SHEEP','GOOSE'))\r\n    stock_animals=sum(int(shed.get(a,0)) for a in ('COW','SHEEP','GOOSE'))\r\n    milk_shops=sum(shop in ('PIZZA_SHOP','ICE_CREAM_SHOP','SMOOTHIE_SHOP') for shop in shops)\r\n    if (216<=step<=227 and len(shops)>=3 and state['confirmed']<cap and not state['reserved']\r\n            and not any(state['carrying'].values()) and not state['pending_places']\r\n            and not cargo and not stock_animals and len(animal_orders)==1\r\n            and animal_orders[0][1]=='SHEEP' and milk_shops>=2 and 'YARN_STORE' not in shops\r\n            and int(prices.get('MILK',0))>=int(prices.get('WOOL',0))\r\n            and counts['COW']>=4 and counts['SHEEP']>=2)",
    b":\r\n        order=animal_orders[0];quantity=int(order[2])\r\n        if 1<=quantity<=2 and quantity<=cap-state['confirmed']:\r\n            order[1]='COW';state['requested']+=quantity\r\n            state['pending_buy']={'before':int(shed.get('COW',0)),'quantity':quantity}\r\n    # Sell only additional physically harvested production at an existing sale slot.\r\n    if state['milk_credit']>0:\r\n        stock=projected_shed(result,FarmView(obs))\r\n        total_planned=sum(max(0,int(o[2])) for o in market if len(o)>=3 and o[:2]==['SELL','MILK'])\r\n        extra=min(state['milk_credit'],max(0,int(stock.get('MILK',0))-total_planned))\r\n        if extra:\r\n            for order in market:\r\n                if len(order)>=3 and order[:2]==['SELL','MILK'] and int(order[2])>0:\r\n                    order[2]=int(order[2])+extra\r\n                    state['milk_credit']-=extra;state['extra_milk_sale_requests']+=extra\r\n                    break\r\n    result['market']=market\r\n    return result\r\n\r\ndef agent(observation,configuration=None):",
    b"\r\n    step=int(observation['step']);seat=int(observation['player'])\r\n    state=_V231_STATES.get(seat)\r\n    if state is None or step<=state['last']:\r\n        state=_V231_STATES[seat]=_v231_new_state()\r\n    action=_V231_PARENT(observation,configuration)\r\n    action=_v231_controller(observation,action,state,_V231_CAP)\r\n    _V231_REPORT.clear();_V231_REPORT.update(_V231_PARENT.telemetry)\r\n    for name in ('confirmed','reserved','requested','failed_purchase_units','picked','placed',\r\n                 'failed_placements','extra_milk_harvested','extra_milk_sale_requests','milk_credit'):\r\n        _V231_REPORT['cattle_'+name]=state[name]\r\n    _V231_REPORT['cattle_carried_pending']=sum(state['carrying'].values())\r\n    return action\r\n\r\nagent.telemetry=_V231_REPORT\r\n\r\n\r\n# EXP-167, adapted from Dmitrii Gluzdov's Two Coins, One Sheep (Apache-2.0).\r\n# Reserve only physically available stock after the final parent worker actions.\r\n_R36_SALE_PARENT=agent\r\n_R36_NATIVE_LEAD=Chassis._sell_lead\r\n_R36_NATIVE_SUPPRESS=Chassis._appl",
    b"y_suppression\r\n_R36_SALE_REPORT={}\r\n\r\ndef _r36_native_lead(self,action,view,projected,route,step,next_sup):\r\n    if step<288 or step>=696:\r\n        return _R36_NATIVE_LEAD(self,action,view,projected,route,step,next_sup)\r\n\r\ndef _r36_suppress(action,state,step):\r\n    _R36_NATIVE_SUPPRESS(action,state,step)\r\n    due=state.get('r36_debts',{}).pop(step,{})\r\n    for order in action.get('market',[]):\r\n        if len(order)>=3 and order[0]=='SELL':\r\n            removed=min(max(0,int(order[2])),due.get(order[1],0))\r\n            order[2]-=removed\r\n            due[order[1]]=due.get(order[1],0)-removed\r\n\r\nChassis._sell_lead=_r36_native_lead\r\nChassis._apply_suppression=staticmethod(_r36_suppress)\r\n\r\ndef _r36_reserve(obs,action):\r\n    step=int(obs['step'])\r\n    # The final planner forecasts its own parent, so keep its full window native.\r\n    if not 192<=step<696:return action\r\n    native=_IMPL.chassis.players[int(obs['player'])]\r\n    tape=_IMPL.chassis.routes[native['route']]\r\n    _v9_hz=_V9_ITEM_HZ.get(int(obs['player'])",
    b")\r\n    end=min(695,step+(max(_v9_hz.values()) if _v9_hz else _R37_HORIZONS.get(int(obs['player']),2)))\r\n    if end<=step:return action\r\n    commands=[action.get('farmer') or ['PASS'],*(action.get('hands') or [])]\r\n    view=FarmView(obs)\r\n    # This projection intentionally abstains on ambiguous animal depot returns.\r\n    if any(len(c)>1 and c[0]=='PLACE' and c[1] in ANIMAL_STRUCTURE\r\n           and view.inv(i).get(c[1],0)>0 for i,c in enumerate(commands[:len(view.positions)])):\r\n        return action\r\n    stock=projected_shed(action,view)\r\n    market=action.get('market',[])\r\n    blocked={o[1] for o in market if len(o)>1 and o[0] in ('SELL','BUY_PRODUCT')}\r\n    blocked.update(c[1] for c in commands if len(c)>1 and c[0]=='PICKUP')\r\n    blocked.update(c[1] for queue in native['pending'].values() for pos,c in queue\r\n                   if len(c)>1 and c[0]=='PICKUP')\r\n    debts=native['sell_state'].setdefault('r36_debts',{})\r\n    for item in PRODUCTS:\r\n        if item in blocked or view.prices.get(item,0)<2:contin",
    b"ue\r\n        available=max(0,int(stock.get(item,0)))\r\n        if not available or len(market)>=10:continue\r\n        reservations=[]\r\n        item_end=min(end,step+_v9_hz[item]) if _v9_hz and item in _v9_hz else end\r\n        for due_step in range(step+1,item_end+1):\r\n            future=tape[due_step]\r\n            work=[future.get('farmer') or ['PASS'],*(future.get('hands') or [])]\r\n            if any(len(c)>1 and c[:2]==['PICKUP',item] for c in work):break\r\n            if any(len(o)>1 and o[:2]==['BUY_PRODUCT',item] for o in future.get('market',[])):break\r\n            planned=sum(max(0,int(o[2])) for o in future.get('market',[]) if len(o)>=3 and o[:2]==['SELL',item])\r\n            amount=min(available,max(0,planned-debts.get(due_step,{}).get(item,0)))\r\n            if amount:\r\n                reservations.append((due_step,amount));available-=amount\r\n            if not available:break\r\n        qty=sum(q for _,q in reservations)\r\n        if qty:\r\n            market.append(['SELL',item,qty])\r\n            for due,q i",
    b"n reservations:\r\n                debt=debts.setdefault(due,{})\r\n                debt[item]=debt.get(item,0)+q\r\n            _R36_SALE_REPORT['sale_reserved_units']+=qty\r\n            _R36_SALE_REPORT['sale_reservations']+=1\r\n    return action\r\n\r\ndef agent(observation,configuration=None):\r\n    if int(observation.get('step',0))==0:\r\n        _R36_SALE_REPORT.update(sale_reserved_units=0,sale_reservations=0,sale_errors=0)\r\n    action=_R36_SALE_PARENT(observation,configuration)\r\n    try:\r\n        if configuration is None or all(configuration.get(k,v)==v for k,v in\r\n            [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):\r\n            action=_r36_reserve(observation,action)\r\n            if int(observation['step'])>=288:action=_v224_sales_first(action)\r\n    except Exception:\r\n        _R36_SALE_REPORT['sale_errors']=_R36_SALE_REPORT.get('sale_errors',0)+1\r\n    _R36_SALE_REPORT.update(_R36_SALE_PARENT.telemetry)\r\n    return action\r\n\r\nagent.telemetry=_R36_SALE_REPORT\r\n\r\n# Ens",
    b'ure the Kaggle-selected final callable is the exported policy.\r\nagent = globals().pop("agent")\r\n\r\n\r\n# Public capability transfer: lucifer19; Flexon is the same Two Coins asset set.\r\n# Apache-2.0; exact functions from Kaggle kaggle-environments 1.32.7.\r\n# https://github.com/Kaggle/kaggle-environments/tree/master/kaggle_environments/envs/kaggriculture\r\nimport math\r\n_R37_MARKET_PARAMS = {\'WHEAT\': {\'base\': 25, \'I0\': 10000, \'T\': 400, \'below_func\': \'sqrt\', \'below_target\': 0.8, \'above_func\': \'log\', \'above_target\': 0.2}, \'CARROT\': {\'base\': 35, \'I0\': 10000, \'T\': 450, \'below_func\': \'hinge\', \'below_target\': 1.0, \'above_func\': \'sqrt\', \'above_target\': 0.7}, \'TOMATO\': {\'base\': 60, \'I0\': 10000, \'T\': 200, \'below_func\': \'hinge\', \'below_target\': 0.4, \'above_func\': \'sqrt\', \'above_target\': 0.6}, \'STRAWBERRY\': {\'base\': 120, \'I0\': 10000, \'T\': 100, \'below_func\': \'sqrt\', \'below_target\': 0.7, \'above_func\': \'linear\', \'above_target\': 1.6}, \'MELON\': {\'base\': 250, \'I0\': 10000, \'T\': 300, \'below_func\': \'log\', \'below_target\': 0.2, \'above_fu',
    b'nc\': \'sq\', \'above_target\': 3.6}, \'EGG\': {\'base\': 50, \'I0\': 10000, \'T\': 332, \'below_func\': \'hinge\', \'below_target\': 0.4, \'above_func\': \'log\', \'above_target\': 0.2}, \'MILK\': {\'base\': 160, \'I0\': 10000, \'T\': 122, \'below_func\': \'sqrt\', \'below_target\': 0.6, \'above_func\': \'linear\', \'above_target\': 1.6}, \'WOOL\': {\'base\': 200, \'I0\': 10000, \'T\': 105, \'below_func\': \'log\', \'below_target\': 0.2, \'above_func\': \'sq\', \'above_target\': 3.2}, \'FERTILIZER\': {\'base\': 100, \'I0\': 10000, \'T\': 200, \'below_func\': \'linear\', \'below_target\': 0.4, \'above_func\': \'linear\', \'above_target\': 0.4}}\r\n_R37_PRICE_FLOOR = 1\r\n_R37_HINGE_GAIN = 8.0\r\ndef _r37_shape(func, x, T=None):\r\n    x = max(0.0, x)\r\n    if func == "linear": return x\r\n    if func == "sq":     return x * x\r\n    if func == "sqrt":   return math.sqrt(x)\r\n    if func == "log":    return math.log(1.0 + x)\r\n    if func == "log10":  return math.log10(1.0 + x)\r\n    if func == "hinge":\r\n        # Degenerates to linear if T is missing or non-positive.\r\n        if not T or T <= 0:\r\n           ',
    b' return x\r\n        u = x / T\r\n        return u + _R37_HINGE_GAIN * max(0.0, u - 1.0) ** 2\r\n    return x\r\n\r\ndef _r37_market_price(item, inventory, params=None):\r\n    """Floor at _R37_PRICE_FLOOR."""\r\n    p = (params or _R37_MARKET_PARAMS)[item]\r\n    base = p["base"]\r\n    I0 = p["I0"]\r\n    T = p["T"]\r\n    if inventory < I0:\r\n        f = p["below_func"]\r\n        amp = p["below_target"] * base / _r37_shape(f, T, T)\r\n        price = base + amp * _r37_shape(f, I0 - inventory, T)\r\n    else:\r\n        f = p["above_func"]\r\n        amp = p["above_target"] * base / _r37_shape(f, T, T)\r\n        price = base - amp * _r37_shape(f, inventory - I0, T)\r\n    return max(_R37_PRICE_FLOOR, int(round(price)))\r\n\r\ndef _r37_similarity(observation):\r\n    """Empty tiles cannot make two unrelated production layouts look alike."""\r\n    farms = observation[\'farms\']\r\n    own, rival = farms[observation[\'player\']], farms[1-observation[\'player\']]\r\n    if own[\'unlocked_quadrants\'] != rival[\'unlocked_quadrants\']:\r\n        return 0.0\r\n    matches',
    b' = total = 0\r\n    for a, b in zip([t for row in own[\'tiles\'] for t in row],\r\n                    [t for row in rival[\'tiles\'] for t in row]):\r\n        sa = (a.get(\'crop\'), a.get(\'animal\')) if isinstance(a, dict) else (None, None)\r\n        sb = (b.get(\'crop\'), b.get(\'animal\')) if isinstance(b, dict) else (None, None)\r\n        if sa != (None, None) or sb != (None, None):\r\n            total += 1\r\n            matches += sa == sb\r\n    return matches / total if total >= 8 else 0.0\r\n\r\n\r\ndef _r37_quote_priority(observation, order, stock):\r\n    """Revenue exposed to a small rival batch, not nominal headline revenue."""\r\n    item = order[1]\r\n    quantity = min(max(0, int(order[2])), stock.get(item, 0))\r\n    if not quantity or item not in _R37_MARKET_PARAMS:\r\n        return 0.0\r\n    inventory = observation[\'market\'][\'inventory\'][item]\r\n    params = {k: dict(v) for k, v in _R37_MARKET_PARAMS.items()}\r\n    for k, patch in observation[\'market\'].get(\'params\', {}).items():\r\n        if k in params:\r\n            params[k].upda',
    b'te(patch)\r\n    rival = observation[\'farms\'][1-observation[\'player\']]\r\n    crop_item = item if item in (\'WHEAT\',\'CARROT\',\'TOMATO\',\'STRAWBERRY\',\'MELON\') else None\r\n    animal = {\'EGG\':\'GOOSE\',\'MILK\':\'COW\',\'WOOL\':\'SHEEP\'}.get(item)\r\n    standing = sum(max(0, int(t.get(\'yield_units\', 0))) for row in rival[\'tiles\'] for t in row\r\n                   if isinstance(t, dict) and\r\n                   ((crop_item is not None and t.get(\'crop\') == crop_item) or\r\n                    (animal is not None and t.get(\'animal\') == animal)))\r\n    # Public fields do not reveal the rival shed. Eight units are a scenario,\r\n    # not a recovered hidden quantity; visible ripe yield increases the stress.\r\n    batch = min(24, max(8, standing))\r\n    now = sum(_r37_market_price(item, inventory+j, params) for j in range(quantity))\r\n    later = sum(_r37_market_price(item, inventory+batch+j, params) for j in range(quantity))\r\n    return now-later\r\n\r\n\r\ndef _r37_reorder_sales(observation, action):\r\n    """Keep quantities and purchase barriers; r',
    b'ank distinct contiguous sales."""\r\n    stock = projected_shed(action, FarmView(observation))\r\n    orders = [list(o) for o in action[\'market\']]\r\n    start = 0\r\n    while start < len(orders):\r\n        if orders[start][0] != \'SELL\':\r\n            start += 1\r\n            continue\r\n        end = start\r\n        while end < len(orders) and orders[end][0] == \'SELL\':\r\n            end += 1\r\n        block = orders[start:end]\r\n        if len({o[1] for o in block}) == len(block):\r\n            orders[start:end] = sorted(block, key=lambda o: _r37_quote_priority(observation, o, stock), reverse=True)\r\n        start = end\r\n    if orders != action[\'market\']:\r\n        _R37_STATS[\'quote_reordered_turns\'] += 1\r\n        action = dict(action, market=orders)\r\n    return action\r\n\r\n\r\n\r\n# EXP175: bounded public cash-response probe inspired by leoprovorov,\r\n# Two Coins Mirror Counter v1 (Apache-2.0). No hidden rival inventory.\r\n_R44_PROBES={}\r\n_R44_REPORT=dict(probe_matches=0,probe_four_turn_calls=0,probe_errors=0)\r\n\r\ndef _r44_before(obs)',
    b":\r\n    player=int(obs['player']);step=int(obs['step'])\r\n    st=_R44_PROBES.get(player)\r\n    if st is None or step<=st['step']:\r\n        st=_R44_PROBES[player]={'step':-1,'money':None,'probe':0,'matched':False}\r\n    if step==0:_R44_REPORT.update(probe_matches=0,probe_four_turn_calls=0,probe_errors=0)\r\n    money=tuple(float(obs['farms'][i]['money']) for i in (player,1-player))\r\n    if st['money'] is not None and st['probe']>=100 and _r37_similarity(obs)>=.90:\r\n        own=money[0]-st['money'][0];rival=money[1]-st['money'][1]\r\n        if own>0 and rival>0 and abs(own-rival)<=max(5.0,.05*st['probe']):\r\n            if not st['matched']:_R44_REPORT['probe_matches']+=1\r\n            st['matched']=True\r\n    st.update(step=step,money=money,probe=0)\r\n    return st\r\n\r\ndef _r44_after(obs,action,st):\r\n    step=int(obs['step']);player=int(obs['player'])\r\n    if not 336<=step<648 or st['matched']:return\r\n    # Positive all-sale probes avoid mistaking equal spending for preemption.\r\n    if not action['market'] or any(o and o[",
    b"0]!='SELL' for o in action['market']):return\r\n    debts=_IMPL.chassis.players[player]['sell_state'].get('r36_debts',{})\r\n    own=debts.get(step+3,{})\r\n    if own:st['probe']=sum(max(0,int(n))*int(obs['market']['prices'].get(item,0)) for item,n in own.items())\r\n\r\n_R37_ADAPTIVE = True\r\n_R37_QUOTE = True\r\n# EXP-168: adapted from lucifer19 / Harvest Nocturne, Apache-2.0.\r\n# All rivalry features use public occupied tiles; no private rival inventory.\r\n_R37_PARENT = agent\r\n_R37_PLAYERS = {}\r\n_R37_HORIZONS = {}\r\n_V9_ITEM_HZ = {}\r\n_R37_REPORT = {}\r\n_R37_STATS = dict(quote_reordered_turns=0, three_turn_calls=0, nocturne_errors=0)\r\ndel agent\r\n\r\ndef agent(observation, configuration=None):\r\n    player, step = int(observation['player']), int(observation['step'])\r\n    state = _R37_PLAYERS.get(player)\r\n    if state is None or step <= state['step']:\r\n        state = _R37_PLAYERS[player] = {'step': -1, 'streak': 0}\r\n    if step == 0:\r\n        _R37_STATS.update(quote_reordered_turns=0, three_turn_calls=0, nocturne_errors=0)\r\n  ",
    b"  state['step'] = step\r\n    _R37_HORIZONS[player] = 2\r\n    probe_state=_r44_before(observation)\r\n    try:\r\n        if _R37_ADAPTIVE and step < 648:\r\n            state['streak'] = state['streak'] + 1 if _r37_similarity(observation) >= .90 else 0\r\n            if 336 <= step < 648 and state['streak'] >= 6:\r\n                _R37_HORIZONS[player] = 3\r\n                _R37_STATS['three_turn_calls'] += 1\r\n    except Exception:\r\n        _R37_STATS['nocturne_errors'] += 1\r\n    if _R37_HORIZONS[player]==3 and probe_state['matched']:\r\n        _R37_HORIZONS[player]=4\r\n        _R44_REPORT['probe_four_turn_calls']+=1\r\n    # EXP179: four-turn reservation; retain stock, debt and purchase barriers.\r\n    if 288 <= step < 696:_R37_HORIZONS[player] = 4\r\n    action = _R37_PARENT(observation, configuration)\r\n    _r44_after(observation,action,probe_state)\r\n    if _R37_QUOTE and step >= 288:\r\n        try:\r\n            action = _r37_reorder_sales(observation, action)\r\n        except Exception:\r\n            _R37_STATS['nocturne_errors",
    b"'] += 1\r\n    _R37_REPORT.update(getattr(_R37_PARENT, 'telemetry', {}))\r\n    _R37_REPORT.update(_R37_STATS)\r\n    _R37_REPORT.update(_R44_REPORT)\r\n    return action\r\n\r\nagent.telemetry = _R37_REPORT\r\n\r\n# Export guard: normal decisions stay identical to the frozen screened policy.\r\n_RELEASE_PARENT=agent\r\n_RELEASE_REPORT={}\r\n_RELEASE_ERRORS=0\r\ndel agent\r\n\r\ndef agent(observation,configuration=None):\r\n    global _RELEASE_ERRORS\r\n    try:\r\n        result=_RELEASE_PARENT(observation,configuration)\r\n    except Exception:\r\n        _RELEASE_ERRORS+=1\r\n        count=0\r\n        try:\r\n            count=min(64,len(observation['farms'][int(observation['player'])]['hands']))\r\n        except Exception:\r\n            pass\r\n        result={'farmer':['PASS'],'hands':[['PASS'] for _ in range(count)],'market':[]}\r\n    _RELEASE_REPORT.update(getattr(_RELEASE_PARENT,'telemetry',{}))\r\n    _RELEASE_REPORT['release_errors']=_RELEASE_ERRORS\r\n    return result\r\n\r\nagent.telemetry=_RELEASE_REPORT\r\nagent=globals().pop('agent')\r\n\r\n# Adapted fro",
    b"m prvsiyan / The Soil Remembers Rain, Apache-2.0.\r\n# V233: bounded, financed six-sheep SE discovery investment.\r\n_V233_PARENT=agent\r\ndel agent\r\n_V233_STATES={}\r\n_V233_REPORT=dict(sheep_commit_requests=0,sheep_committed=0,sheep_hire_requests=0,\r\n    sheep_workers_confirmed=0,sheep_hire_shortfalls=0,sheep_budget_declines=0,\r\n    sheep_capacity_declines=0,sheep_purchase_shortfalls=0,sheep_feed_buy_requests=0,\r\n    sheep_wool_harvested=0,sheep_fert_collected=0,sheep_extra_wool_sales=0,\r\n    sheep_extra_fert_sales=0,sheep_rescue_feed_requests=0)\r\n\r\ndef _v233_eligible(obs,native):\r\n    farm=obs['farms'][obs['player']];prices=obs['market']['prices']\r\n    if len(farm['tiles'])!=10 or set(farm['unlocked_quadrants'])!={'NW','NE','SW'}:return False\r\n    if obs['town']['unlocked_shops'].count('YARN_STORE')<2 or prices['WOOL']<220 or prices['WHEAT']>45:return False\r\n    if any(farm['tiles'][y][x]!='LOCKED' for y in (5,6) for x in range(5,8)):return False\r\n    if obs['private']['shed'].get('SHEEP',0) or any(i.get('SHEEP',0",
    b") for i in obs['private']['inventories']):return False\r\n    for day in range(12,30):\r\n        for a in _v219_native_day(native,day):\r\n            if any(o and (o[0]=='BUY_LAND' or o[:2]==['BUY_ANIMAL','SHEEP']) for o in a.get('market',[])):return False\r\n            if any(c and c[0] in ('PICKUP','PLACE') and len(c)>1 and c[1]=='SHEEP' for c in [a.get('farmer')]+a.get('hands',[])):return False\r\n    return True\r\n\r\ndef _v233_request(obs,action,state,native):\r\n    step=int(obs['step']);day=step//24;hour=step%24\r\n    # EXP242: preserve native indices while servicing already committed sheep.\r\n    if state.get('requested_day')==day:return action\r\n    committed=state.get('committed')\r\n    if not committed and (hour>(3 if day==11 else 1) or day not in (11,12) or not _v233_eligible(obs,native)):return action\r\n    planned=_v219_native_day(native,day)\r\n    deadline=2 if committed else (3 if day==11 else 1)\r\n    if committed:\r\n        last_native_hire=max((h for h,a in enumerate(planned) if any(o and o[0]=='HIRE' for o in",
    b" a.get('market',[]))),default=0)\r\n        if 2<last_native_hire<=6:deadline=6\r\n    if hour>deadline:return action\r\n    if any(o and o[0]=='HIRE' for a in planned[hour+1:] for o in a.get('market',[])):return action\r\n    farm=obs['farms'][obs['player']];market=action.get('market',[])\r\n    parent_hires=sum(bool(o) and o[0]=='HIRE' for o in market)\r\n    expected=max(len(a.get('hands',[])) for a in planned)\r\n    if len(farm['hands'])+parent_hires!=expected:return action\r\n    initial=not state.get('committed')\r\n    extra=([['BUY_LAND'],['BUY_ANIMAL','SHEEP',6]] if initial else [])+[['BUY_PRODUCT','WHEAT',6],['HIRE'],['HIRE']]\r\n    if len(market)+len(extra)>MAX_ORDERS:return action\r\n    stock=projected_shed(action,FarmView(obs))\r\n    incoming=6+6*initial\r\n    budget=7000*initial+6*(int(obs['market']['prices']['WHEAT'])+10)\r\n    budget+=sum(_v219_fib(n) for n in range(farm['hires_today'],farm['hires_today']+parent_hires+2))\r\n    for o in market:\r\n        if not o:continue\r\n        if o[0]=='BUY_LAND':return action\r\n ",
    b"       if o[0]=='BUY_PRODUCT':\r\n            incoming+=int(o[2]);budget+=int(o[2])*(int(obs['market']['prices'][o[1]])+10)\r\n        elif o[0]=='BUY_ANIMAL':\r\n            incoming+=int(o[2]);budget+=int(o[2])*{'SHEEP':500,'COW':400,'GOOSE':300}[o[1]]\r\n        elif o[0]=='BUY_SEED':budget+=int(o[2])*{'WHEAT':10,'CARROT':20,'TOMATO':50,'STRAWBERRY':100,'MELON':80}[o[1]]\r\n    if sum(stock.values())+incoming>100:\r\n        _V233_REPORT['sheep_capacity_declines']+=1;return action\r\n    if farm['money']<budget+(3000 if initial else 1000):\r\n        _V233_REPORT['sheep_budget_declines']+=1;return action\r\n    state['requested_day']=day\r\n    state['pending']={'first':expected+1,'initial':initial}\r\n    _V233_REPORT['sheep_hire_requests']+=2;_V233_REPORT['sheep_feed_buy_requests']+=6\r\n    if initial:_V233_REPORT['sheep_commit_requests']+=1\r\n    result=copy.deepcopy(action);result['market']=market+extra\r\n    return result\r\n\r\ndef _v233_worker(obs,actor,targets):\r\n    farm=obs['farms'][obs['player']];private=obs['private'];step",
    b"=int(obs['step'])\r\n    pos=tuple(farm['hands'][actor-1]);inv=private['inventories'][actor]\r\n    access=((4,4),(5,4),(4,5),(5,5))\r\n    home=min(access,key=lambda p:(abs(pos[0]-p[0])+abs(pos[1]-p[1]),p))\r\n    distance=abs(pos[0]-home[0])+abs(pos[1]-home[1])\r\n    cargo=[item for item in ('WOOL','FERTILIZER') if inv.get(item,0)]\r\n    if cargo and step%24 >= (22 if step//24==29 else 23)-distance:\r\n        return _v219_walk(pos,home) or ['PLACE',cargo[0],inv[cargo[0]]]\r\n    missing=sum(not(isinstance(farm['tiles'][y][x],dict) and farm['tiles'][y][x].get('animal')=='SHEEP') for x,y in targets)\r\n    if missing and not inv.get('SHEEP',0) and private['shed'].get('SHEEP',0):\r\n        return _v219_walk(pos,home) or ['PICKUP','SHEEP',min(missing,private['shed']['SHEEP'])]\r\n    hungry=sum(not(isinstance(farm['tiles'][y][x],dict) and farm['tiles'][y][x].get('fed_today')) for x,y in targets)\r\n    if hungry and not inv.get('WHEAT',0) and private['shed'].get('WHEAT',0):\r\n        return _v219_walk(pos,home) or ['PICKUP','WHEAT'",
    b",min(hungry,private['shed']['WHEAT'])]\r\n    tasks=[]\r\n    for target in targets:\r\n        x,y=target;tile=farm['tiles'][y][x];command=None\r\n        if tile is None:command=['BUILD_PASTURE']\r\n        elif isinstance(tile,dict) and tile.get('kind')=='WEED':command=['DIG']\r\n        elif isinstance(tile,dict) and tile.get('kind')=='PASTURE' and not tile.get('animal'):\r\n            if inv.get('SHEEP',0):command=['PLACE','SHEEP']\r\n        elif isinstance(tile,dict) and tile.get('animal')=='SHEEP':\r\n            if not tile['fed_today'] and inv.get('WHEAT',0):command=['FEED']\r\n            elif not tile['cared_today']:command=['CARE']\r\n            elif tile['yield_units']:command=['HARVEST']\r\n\r\n        if command:tasks.append((abs(pos[0]-x)+abs(pos[1]-y),targets.index(target),target,command))\r\n    if tasks:\r\n        _,_,target,command=min(tasks);return _v219_walk(pos,target) or command\r\n    if cargo:return _v219_walk(pos,home) or ['PLACE',cargo[0],inv[cargo[0]]]\r\n    return ['PASS']\r\n\r\ndef _v234_rescue(obs,action,stat",
    b"e):\r\n    if not state['workers'] or int(obs['step'])%24>14:return action\r\n    orders=action.get('market',[])\r\n    if len(orders)>=MAX_ORDERS:return action\r\n    if any(o and (o[0] in ('HIRE','BUY_LAND','BUY_ANIMAL','BUY_PRODUCT','BUY_SEED') or (len(o)>1 and o[1]=='WHEAT')) for o in orders):return action\r\n    farm=obs['farms'][obs['player']];private=obs['private'];hungry=carried=0\r\n    commands=[action.get('farmer') or ['PASS']]+list(action.get('hands') or [])\r\n    for actor,targets in state['workers'].items():\r\n        command=commands[actor]\r\n        if command==['FEED'] or command[:2]==['PICKUP','WHEAT']:return action\r\n        carried+=private['inventories'][actor].get('WHEAT',0)\r\n        hungry+=sum(isinstance(farm['tiles'][y][x],dict) and farm['tiles'][y][x].get('animal')=='SHEEP' and not farm['tiles'][y][x].get('fed_today') for x,y in targets)\r\n    stock=projected_shed(action,FarmView(obs))\r\n    shortage=hungry-carried-stock.get('WHEAT',0)\r\n    if not 0<shortage<=6 or state.get('rescue_today',0)+shortage>",
    b"6:return action\r\n    quote=int(obs['market']['prices']['WHEAT'])\r\n    if quote<1 or farm['money']<1000+shortage*(quote+10) or sum(stock.values())+shortage>100:return action\r\n    result=copy.deepcopy(action);result['market'].append(['BUY_PRODUCT','WHEAT',shortage])\r\n    state['rescue_today']=state.get('rescue_today',0)+shortage\r\n    _V233_REPORT['sheep_rescue_feed_requests']+=shortage\r\n    return result\r\n\r\ndef agent(observation,configuration=None):\r\n    action=_V233_PARENT(observation,configuration)\r\n    step=int(observation['step']);player=int(observation['player']);day=step//24\r\n    state=_V233_STATES.get(player)\r\n    if state is None or step<=state['last_step']:\r\n        state={'last_step':step,'day':-1,'workers':{},'work':{},'credit':{'WOOL':0,'FERTILIZER':0}}\r\n        _V233_STATES[player]=state\r\n    state['last_step']=step\r\n    if configuration is not None and any(configuration.get(k,v)!=v for k,v in\r\n        (('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10))):return ",
    b"action\r\n    if day<11:return action\r\n    farm=observation['farms'][player];private=observation['private']\r\n    if state['day']!=day:state['day']=day;state['workers']={};state['work']={};state['rescue_today']=0\r\n    for actor,previous in state['work'].items():\r\n        if previous['step']!=step-1 or actor>=len(private['inventories']):continue\r\n        item={'HARVEST':'WOOL','COLLECT_FERTILIZER':'FERTILIZER'}.get(previous['command'][0])\r\n        if item:\r\n            gained=max(0,private['inventories'][actor].get(item,0)-previous['inventory'].get(item,0))\r\n            state['credit'][item]+=gained\r\n            _V233_REPORT['sheep_wool_harvested' if item=='WOOL' else 'sheep_fert_collected']+=gained\r\n    pending=state.pop('pending',None)\r\n    if pending:\r\n        funded='SE' in farm['unlocked_quadrants'] and (not pending['initial'] or private['shed'].get('SHEEP',0)>=6)\r\n        if not funded:_V233_REPORT['sheep_purchase_shortfalls']+=1\r\n        elif len(farm['hands'])<pending['first']+pending.get('count',2)-1:_V2",
    b"33_REPORT['sheep_hire_shortfalls']+=1\r\n        else:\r\n            if pending.get('count',2)==1:\r\n                state['workers'][pending['first']]=list(pending['targets'])\r\n                _SL_REPORT['confirmed']+=1\r\n            else:\r\n                for i in range(2):state['workers'][pending['first']+i]=[(x,5+i) for x in range(5,8)]\r\n            _V233_REPORT['sheep_workers_confirmed']+=pending.get('count',2)\r\n            if pending['initial']:state['committed']=True;_V233_REPORT['sheep_committed']+=1\r\n    action=_v233_request(observation,action,state,_IMPL.chassis.players[player])\r\n    if not state.get('committed'):return action\r\n    result=copy.deepcopy(action)\r\n    commands=[result.get('farmer') or ['PASS']]+list(result.get('hands') or [])\r\n    commands += [['PASS'] for _ in range(len(farm['hands'])+1-len(commands))]\r\n    state['work']={}\r\n    for actor,targets in state['workers'].items():\r\n        command=_v233_worker(observation,actor,targets);commands[actor]=command\r\n        state['work'][actor]={'ste",
    b"p':step,'command':command,'inventory':dict(private['inventories'][actor])}\r\n    result['farmer'],result['hands']=commands[0],commands[1:]\r\n    result=_v234_rescue(observation,result,state)\r\n    stock=projected_shed(result,FarmView(observation))\r\n    for item in ('WOOL','FERTILIZER'):\r\n        scheduled=sum(int(o[2]) for o in result['market'] if o[:2]==['SELL',item])\r\n        count=min(state['credit'][item],max(0,stock.get(item,0)-scheduled))\r\n        if count and len(result['market'])<MAX_ORDERS:\r\n            result['market'].append(['SELL',item,count]);state['credit'][item]-=count\r\n            _V233_REPORT['sheep_extra_wool_sales' if item=='WOOL' else 'sheep_extra_fert_sales']+=count\r\n    return result\r\n\r\n_R46_SHEEP_AGENT=agent\r\n_R46_SHADOW_PARENT=_shadow_terminal\r\n_R46_REPORT={}\r\ndef _shadow_terminal(obs,config):\r\n    if _V233_STATES.get(int(obs['player']),{}).get('committed'):return None\r\n    return _R46_SHADOW_PARENT(obs,config)\r\ndel agent\r\ndef agent(observation,configuration=None):\r\n    try:\r\n        if ",
    b"int(observation.get('step',-1))==0:\r\n            for k in _V233_REPORT:_V233_REPORT[k]=0\r\n        result=_R46_SHEEP_AGENT(observation,configuration)\r\n    except Exception:\r\n        _R46_REPORT['sheep_overlay_errors']=_R46_REPORT.get('sheep_overlay_errors',0)+1\r\n        result={'farmer':['PASS'],'hands':[],'market':[]}\r\n    _R46_REPORT.update(getattr(_V233_PARENT,'telemetry',{}))\r\n    _R46_REPORT.update(_V233_REPORT)\r\n    return result\r\nagent.telemetry=_R46_REPORT\r\nagent=globals().pop('agent')\r\n\r\n# EXP182: finite-harvest wheat/carrot input planner; original adaptation.\r\n_R51_INPUT_PARENT=agent\r\n_R51_INPUT_STATES={}\r\n_R51_INPUT_REPORT={}\r\n_R51_INPUT_MAX_WORKERS=2\r\n_R51_INPUT_CROPS={'WHEAT':(2,4,6),'CARROT':(2,3,4)}\r\n\r\ndef _r51_input_forecast(obs,route,expected):\r\n    step=int(obs['step']);day=step//24;farm=obs['farms'][obs['player']]\r\n    pos=[list(farm['farmer'])]+[list(p) for p in farm['hands'][:expected]];targets={}\r\n    for y,line in enumerate(farm['tiles']):\r\n        for x,tile in enumerate(line):\r\n       ",
    b"     if not isinstance(tile,dict) or tile.get('crop') not in _R51_INPUT_CROPS:continue\r\n            item=tile['crop'];first,last,cap=_R51_INPUT_CROPS[item]\r\n            if 1<=day-tile['planted_day']<last:\r\n                targets[(x,y)]={'crop':item,'birth':tile['planted_day'],'yield':tile['yield_units'],\r\n                    'until':tile.get('fertilized_until_day',-1),'watered':tile.get('watered_today',False),'water':[],'harvest':None,'first':first,'last':last,'cap':cap}\r\n    access=((4,4),(5,4),(4,5),(5,5));seen=set()\r\n    # Native continuation ends before the reactive terminal closure planner.\r\n    for t in range(step,min(712,(day+4)*24)):\r\n        tape=_IMPL.chassis.routes[2 if t>=648 else route];a=tape[t]\r\n        for actor,c in enumerate([a.get('farmer') or ['PASS'],*(a.get('hands') or [])][:len(pos)]):\r\n            if not c:continue\r\n            xy=tuple(pos[actor]);target=targets.get(xy)\r\n            if target is not None and target['harvest'] is None:\r\n                if c[0]=='WATER' and (t//24,xy) ",
    b"not in seen:\r\n                    seen.add((t//24,xy))\r\n                    if not(t//24==day and target['watered']) and target['first']<=t//24-target['birth']<=target['last']:target['water'].append(t)\r\n                if c[0]=='HARVEST':target['harvest']=t\r\n            if c[0] in MOVES:\r\n                dx,dy=MOVES[c[0]];pos[actor]=[max(0,min(9,pos[actor][0]+dx)),max(0,min(9,pos[actor][1]+dy))]\r\n        for o in a.get('market',[]):\r\n            if o and o[0]=='HIRE':\r\n                counts={p:sum(tuple(q)==p for q in pos) for p in access}\r\n                pos.append(list(min(access,key=lambda p:(counts[p],access.index(p)))))\r\n        if (t+1)%24==0:pos=[[4,4]]\r\n    return targets\r\n\r\ndef _r51_input_gain(target,arrival,day):\r\n    if target['harvest'] is None or target['harvest']<=arrival:return 0\r\n    extra=sum(arrival<t<=target['harvest'] and day<=t//24<=day+2 and t//24>target['until'] for t in target['water'])\r\n    baseline=target['yield']+sum(2 if t//24<=target['until'] else 1 for t in target['water'])\r\n  ",
    b"  return max(0,min(extra,target['cap']-baseline))\r\n\r\ndef _r51_input_path(obs,targets,action,index):\r\n    step=int(obs['step']);day=step//24\r\n    ready,start=_r62_input_start(obs,action,index)\r\n    prices={p:max(1,int(obs['market']['prices'][p])-2) for p in ('WHEAT','CARROT')}\r\n    fertilizer=max(1,_r37_market_price('FERTILIZER',obs['market']['inventory']['FERTILIZER']-16)+2)\r\n    # Tuple: penalized value, gross value, next free turn, position, path, used,\r\n    # wheat units, carrot units. No state reads from the rival's private farm.\r\n    beam=[(0,0,ready,start,(),frozenset(),0,0)]\r\n    best=None\r\n    for depth in range(8):\r\n        expanded=[]\r\n        for score,gross,now,pos,path,used,wheat,carrot in beam:\r\n            for xy,target in targets.items():\r\n                if xy in used:continue\r\n                arrival=now+abs(pos[0]-xy[0])+abs(pos[1]-xy[1])\r\n                if arrival>=day*24+23:continue\r\n                gain=_r51_input_gain(target,arrival,day)\r\n                if not gain:continue\r\n         ",
    b"       item=target['crop'];new_gross=gross+gain*prices[item]\r\n                new_path=path+((xy[0],xy[1],item,target['birth']),)\r\n                expanded.append((new_gross-1.5*fertilizer*len(new_path),new_gross,arrival+1,xy,new_path,\r\n                                 used|{xy},wheat+(gain if item=='WHEAT' else 0),carrot+(gain if item=='CARROT' else 0)))\r\n        if not expanded:break\r\n        expanded.sort(key=lambda s:(-s[0],-s[1],s[2],s[4]))\r\n        beam=expanded[:8]\r\n        if depth>=2:\r\n            candidate=beam[0]\r\n            if best is None or (-candidate[0],-candidate[1],candidate[2],candidate[4])<(-best[0],-best[1],best[2],best[4]):best=candidate\r\n    if best is None:return [],{'WHEAT':0,'CARROT':0}\r\n    return list(best[4]),{'WHEAT':best[6],'CARROT':best[7]}\r\n\r\ndef _r51_input_control(obs,action,state):\r\n    step=int(obs['step']);day=step//24;hour=step%24;player=int(obs['player']);farm=obs['farms'][player];private=obs['private']\r\n    native=_IMPL.chassis.players[player]\r\n    if state.get('day')!",
    b"=day:state.update(day=day,workers={},pending=None,placed=[])\r\n    for x,y in state['placed']:\r\n        tile=farm['tiles'][y][x]\r\n        if isinstance(tile,dict) and tile.get('fertilized_until_day',-1)>=day+2:_R51_INPUT_REPORT['input_confirmed_applications']+=1\r\n        else:_R51_INPUT_REPORT['input_application_errors']+=1\r\n    state['placed']=[]\r\n    if state.get('pending'):\r\n        pending=state.pop('pending')\r\n        for actor,plan in pending.items():\r\n            if len(farm['hands'])>=actor:state['workers'][actor]=plan;_R51_INPUT_REPORT['input_confirmed_hires']+=1\r\n            else:_R51_INPUT_REPORT['input_hire_errors']+=1\r\n    if state['workers']:\r\n        changed=copy.deepcopy(action)\r\n        for actor,plan in state['workers'].items():\r\n            inv=private['inventories'][actor];pos=tuple(farm['hands'][actor-1]);cmd=['PASS']\r\n            if not plan['loaded']:\r\n                stock=projected_shed(changed,FarmView(obs));q=min(plan['quantity'],max(0,stock.get('FERTILIZER',0)))\r\n                if ",
    b"q and _shed_adjacent(pos,10):\r\n                    cmd=['PICKUP','FERTILIZER',q];plan['loaded']=True;_R51_INPUT_REPORT['input_loaded_units']+=q\r\n                    if q<plan['quantity']:_R51_INPUT_REPORT['input_stock_shortfalls']+=plan['quantity']-q\r\n            elif inv.get('FERTILIZER',0):\r\n                while plan['path']:\r\n                    x,y,crop,birth=plan['path'][0];tile=farm['tiles'][y][x]\r\n                    if not isinstance(tile,dict) or tile.get('crop')!=crop or tile.get('planted_day')!=birth or tile.get('fertilized_until_day',-1)>=day+2:\r\n                        plan['path'].pop(0);continue\r\n                    cmd=_v219_walk(pos,(x,y)) or ['FERTILIZE']\r\n                    if cmd==['FERTILIZE']:state['placed'].append((x,y));plan['path'].pop(0);_R51_INPUT_REPORT['input_application_requests']+=1\r\n                    break\r\n            changed['hands'][actor-1]=cmd\r\n        return changed\r\n    if hour not in (1,2,3) or not 12<=day<=28:return action\r\n    planned=_v219_native_day(native,day);",
    b"expected=max(len(a.get('hands',[])) for a in planned)\r\n    if any(o and o[0]=='HIRE' for a in planned[hour:] for o in a.get('market',[])) or native['pending']:return action\r\n    parents=[_V219_STATES.get(player,{}),_V233_STATES.get(player,{})]\r\n    # A parent may retry after a full market queue; its headcount must remain native.\r\n    if day in (12,18) or any(p.get('committed') and p.get('requested_day')!=day for p in parents):return action\r\n    if any(p.get('pending') for p in parents) or any(o and o[0]=='HIRE' for o in action.get('market',[])):return action\r\n    owned=set(range(1,expected+1))\r\n    for p in parents:\r\n        actors=set(p.get('workers',{}))\r\n        if owned&actors:return action\r\n        owned|=actors\r\n    if owned!=set(range(1,len(farm['hands'])+1)):return action\r\n    targets=_r51_input_forecast(obs,native['route'],expected);plans=[];total_q=0;total_cost=0;all_units={'WHEAT':0,'CARROT':0}\r\n    stock=projected_shed(action,FarmView(obs));purchases=sum(max(0,int(o[2])) for o in action.get('marke",
    b"t',[]) if len(o)>2 and o[0] in ('BUY_PRODUCT','BUY_ANIMAL'))\r\n    # Units act before market orders. Preserve the native next-turn pickup,\r\n    # after the current parent's actual sales/purchases, before buying tour inputs.\r\n    available=max(0,stock.get('FERTILIZER',0))\r\n    for o in action.get('market',[]):\r\n        if len(o)>=3 and o[:2]==['SELL','FERTILIZER']:available=max(0,available-max(0,int(o[2])))\r\n        elif len(o)>=3 and o[:2]==['BUY_PRODUCT','FERTILIZER']:available+=max(0,int(o[2]))\r\n    next_native=planned[hour+1];native_pickups=sum(max(0,int(c[2]) if len(c)>2 else 1) for c in [next_native.get('farmer') or ['PASS'],*(next_native.get('hands') or [])] if len(c)>1 and c[:2]==['PICKUP','FERTILIZER'])\r\n    topup=max(0,native_pickups-available)\r\n    plans,total_q,total_cost,all_units=_r68_joint_plans(obs,action,targets,stock,purchases,topup)\r\n    if not plans:return action\r\n    state['pending']={len(farm['hands'])+1+i:plan for i,plan in enumerate(plans)}\r\n    _R51_INPUT_REPORT['input_hire_requests']+=",
    b"len(plans);_R51_INPUT_REPORT['input_purchase_requests']+=total_q+topup\r\n    _R51_INPUT_REPORT['input_forecast_wheat']+=all_units['WHEAT'];_R51_INPUT_REPORT['input_forecast_carrot']+=all_units['CARROT']\r\n    changed=copy.deepcopy(action);changed['market'] += [['BUY_PRODUCT','FERTILIZER',total_q+topup]]+[['HIRE'] for _ in plans];return changed\r\n\r\ndef agent(observation,configuration=None):\r\n    try:\r\n        step=int(observation['step']);player=int(observation['player']);state=_R51_INPUT_STATES.get(player)\r\n        if state is None or step<=state['step']:\r\n            state=_R51_INPUT_STATES[player]={'step':-1}\r\n            _R51_INPUT_REPORT.update(input_hire_requests=0,input_confirmed_hires=0,input_hire_errors=0,input_purchase_requests=0,\r\n                input_loaded_units=0,input_stock_shortfalls=0,input_application_requests=0,input_confirmed_applications=0,\r\n                input_application_errors=0,input_errors=0,input_forecast_wheat=0,input_forecast_carrot=0)\r\n        state['step']=step;action=_R51_INPUT_",
    b"PARENT(observation,configuration)\r\n        if configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):\r\n            action=_r51_input_control(observation,action,state)\r\n        _R51_INPUT_REPORT.update(getattr(_R51_INPUT_PARENT,'telemetry',{}));return action\r\n    except Exception:\r\n        _R51_INPUT_REPORT['input_errors']=_R51_INPUT_REPORT.get('input_errors',0)+1\r\n        return {'farmer':['PASS'],'hands':[],'market':[]}\r\nagent.telemetry=_R51_INPUT_REPORT\r\nagent=globals().pop('agent')\r\n\r\n# EXP182: project the final hour's actual worker actions before automatic deposit.\r\n_R51_WAREHOUSE_PARENT=agent\r\n_R51_WAREHOUSE_REPORT={}\r\n\r\ndef _r51_close_warehouse(obs,action):\r\n    step=int(obs['step']);day=step//24\r\n    if step%24!=23 or not 12<=day<=28:return action\r\n    # No speculative product purchase/worker count model: these hours abstain.\r\n    if any(o and o[0] not in ('SELL',) for o in action.get('market',[])):retu",
    b"rn action\r\n    farm,private=_PLANNER_NS['_clone_state'](obs['farms'][obs['player']],obs['private'])\r\n    commands=[action.get('farmer') or ['PASS'],*(action.get('hands') or [])]\r\n    demand={}\r\n    for c in commands:\r\n        if len(c)>1 and c[0]=='PLANT':demand[c[1]]=demand.get(c[1],0)+1\r\n    blocked={k for k,q in demand.items() if q>private['seeds'].get(k,0)}\r\n    for actor,c in enumerate(commands[:len(private['inventories'])]):\r\n        if len(c)>1 and c[0]=='PLANT' and c[1] in blocked:c=['PASS']\r\n        _PLANNER_NS['_apply_unit_action'](farm,private,actor,c,10,day,24,100)\r\n    post=dict(private['shed'])\r\n    for o in action.get('market',[]):\r\n        if len(o)>=3 and o[0]=='SELL':post[o[1]]=max(0,post.get(o[1],0)-max(0,int(o[2])))\r\n    needed=sum(post.values())+sum(max(0,q) for inv in private['inventories'] for q in inv.values())-100\r\n    if needed<=0:return action\r\n    result=copy.deepcopy(action);orders=result['market']\r\n    # Grain and fertilizer have native input obligations; other products do not.\r\n",
    b"    # Additional commodity sales are bounded by actual post-action physical stock.\r\n    for item in sorted((p for p in PRODUCTS if p not in ('WHEAT','FERTILIZER')),key=lambda p:-obs['market']['prices'].get(p,0)):\r\n        qty=min(needed,post.get(item,0))\r\n        if not qty:continue\r\n        existing=next((o for o in orders if len(o)>=3 and o[:2]==['SELL',item]),None)\r\n        if existing is not None:existing[2]=max(0,int(existing[2]))+qty\r\n        elif len(orders)<10:orders.append(['SELL',item,qty])\r\n        else:continue\r\n        needed-=qty;post[item]-=qty;_R51_WAREHOUSE_REPORT['warehouse_extra_sales']+=qty\r\n        if needed<=0:break\r\n    if needed>0:\r\n        native=_IMPL.chassis.players[int(obs['player'])];reserve=0\r\n        for t in range(step+1,719):\r\n            future=_IMPL.chassis.routes[2 if t>=648 else native['route']][t]\r\n            for c in [future.get('farmer') or ['PASS'],*(future.get('hands') or [])]:\r\n                if len(c)>1 and c[:2]==['PICKUP','WHEAT']:reserve+=max(0,int(c[2]) if len",
    b"(c)>2 else 1)\r\n            if any(len(o)>1 and o[:2]==['BUY_PRODUCT','WHEAT'] for o in future.get('market',[])):break\r\n        incoming=sum(max(0,inv.get('WHEAT',0)) for inv in private['inventories'])\r\n        others=sum(q for p,q in post.items() if p!='WHEAT')+sum(max(0,q) for inv in private['inventories'] for p,q in inv.items() if p!='WHEAT')\r\n        # Even if every other carried item deposits first, this grain reserve fits.\r\n        qty=min(needed,post.get('WHEAT',0),max(0,post.get('WHEAT',0)+incoming-reserve)) if 100-others>=reserve else 0\r\n        existing=next((o for o in orders if len(o)>=3 and o[:2]==['SELL','WHEAT']),None)\r\n        if qty and (existing is not None or len(orders)<10):\r\n            if existing is not None:existing[2]=max(0,int(existing[2]))+qty\r\n            else:orders.append(['SELL','WHEAT',qty])\r\n            needed-=qty;_R51_WAREHOUSE_REPORT['warehouse_extra_sales']+=qty\r\n    _R51_WAREHOUSE_REPORT['warehouse_projected_unresolved']+=max(0,needed)\r\n    if result!=action:_R51_WAREHOUSE",
    b"_REPORT['warehouse_changed_turns']+=1\r\n    return result\r\n\r\ndef agent(observation,configuration=None):\r\n    result=_R51_WAREHOUSE_PARENT(observation,configuration)\r\n    try:\r\n        if int(observation['step'])==0:_R51_WAREHOUSE_REPORT.update(warehouse_changed_turns=0,warehouse_extra_sales=0,warehouse_projected_unresolved=0,warehouse_errors=0)\r\n        if configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):result=_r51_close_warehouse(observation,result)\r\n    except Exception:_R51_WAREHOUSE_REPORT['warehouse_errors']=_R51_WAREHOUSE_REPORT.get('warehouse_errors',0)+1\r\n    _R51_WAREHOUSE_REPORT.update(getattr(_R51_WAREHOUSE_PARENT,'telemetry',{}));return result\r\nagent.telemetry=_R51_WAREHOUSE_REPORT\r\nagent=globals().pop('agent')\r\n\r\nfrom itertools import permutations as _r53_permutations\r\n_R53_LABOR_REPORT=dict(labor_requests=0,labor_hires_avoided=0,labor_spawn_errors=0,labor_confirmed=0,labor_day26=0,labor_day",
    b"27=0,labor_day28=0)\r\n\r\ndef _r53_labor_assignment(obs,action,fertilizer):\r\n    step=int(obs['step']);day=step//24;farm=obs['farms'][obs['player']]\r\n    if day not in (26,27,28) or step%24>2:return None\r\n    # Do not preempt a later price-gated fertilizer request with a smaller unfertilized team.\r\n    if day==27 and not fertilizer:return None\r\n    count=3 if fertilizer else 2\r\n    positions=[list(farm['farmer'])]+[list(p) for p in farm['hands']]\r\n    for i,c in enumerate([action.get('farmer') or ['PASS'],*(action.get('hands') or [])][:len(positions)]):\r\n        if c and c[0] in MOVES:\r\n            dx,dy=MOVES[c[0]];positions[i]=[max(0,min(9,positions[i][0]+dx)),max(0,min(9,positions[i][1]+dy))]\r\n    access=((4,4),(5,4),(4,5),(5,5));spawns=[]\r\n    native_hires=sum(bool(o) and o[0]=='HIRE' for o in action.get('market',[]))\r\n    for i in range(native_hires+count):\r\n        chosen=min(access,key=lambda p:(sum(tuple(q)==p for q in positions),access.index(p)));positions.append(list(chosen))\r\n        if i>=native_hire",
    b's:spawns.append(chosen)\r\n    groups=(((5,5),(6,5),(7,5),(8,5)),((9,5),(9,6),(8,6)),((5,6),(6,6),(7,6))) if fertilizer else (tuple((x,5) for x in range(5,10)),tuple((x,6) for x in range(5,10)))\r\n    choices=[];remaining=23-step%24\r\n    for assignment in _r53_permutations(groups):\r\n        costs=[]\r\n        for start,path in zip(spawns,assignment):\r\n            distance=abs(start[0]-path[0][0])+abs(start[1]-path[0][1])\r\n            distance+=sum(abs(a[0]-b[0])+abs(a[1]-b[1]) for a,b in zip(path,path[1:]))\r\n            distance+=min(abs(path[-1][0]-x)+abs(path[-1][1]-y) for x,y in access)\r\n            costs.append(distance+(3 if fertilizer else 2)*len(path)+1+int(fertilizer))\r\n        if max(costs)<=remaining:choices.append((max(costs),sum(costs),assignment))\r\n    if not choices:return None\r\n    _,_,assignment=min(choices)\r\n    return dict(paths=assignment,spawns=spawns,remaining=remaining,workers=count,fertilizer=fertilizer)\r\n\r\n_R53_LABOR_PARENT=agent\r\ndef agent(observation,configuration=None):\r\n    if isinstan',
    b"ce(observation,dict) and observation.get('step')==0:\r\n        for k in _R53_LABOR_REPORT:_R53_LABOR_REPORT[k]=0\r\n    result=_R53_LABOR_PARENT(observation,configuration)\r\n    _R53_LABOR_COMBINED.update(getattr(_R53_LABOR_PARENT,'telemetry',{}));_R53_LABOR_COMBINED.update(_R53_LABOR_REPORT)\r\n    return result\r\n_R53_LABOR_COMBINED={}\r\nagent.telemetry=_R53_LABOR_COMBINED\r\nagent=globals().pop('agent')\r\n\r\n# EXP193: deterministic HIRE spawn after native unit actions, then next-turn pickup.\r\ndef _r62_input_start(obs,action,index):\r\n    farm=obs['farms'][obs['player']]\r\n    positions=[list(farm['farmer'])]+[list(p) for p in farm['hands']]\r\n    for actor,c in enumerate([action.get('farmer') or ['PASS'],*(action.get('hands') or [])][:len(positions)]):\r\n        if c and c[0] in MOVES:\r\n            dx,dy=MOVES[c[0]]\r\n            positions[actor]=[max(0,min(9,positions[actor][0]+dx)),max(0,min(9,positions[actor][1]+dy))]\r\n    access=((4,4),(5,4),(4,5),(5,5))\r\n    native_hires=sum(bool(o) and o[0]=='HIRE' for o in action.ge",
    b"t('market',[]))\r\n    for _ in range(native_hires+index+1):\r\n        chosen=min(access,key=lambda p:(sum(tuple(q)==p for q in positions),access.index(p)))\r\n        positions.append(list(chosen))\r\n    return int(obs['step'])+2,chosen\r\n\r\nagent=globals().pop('agent')\r\n\r\ndef _r68_joint_plans(obs,action,targets,stock,purchases,topup):\r\n    farm=obs['farms'][obs['player']];choices=[]\r\n    for mode,first_crop in enumerate((None,'WHEAT','CARROT')):\r\n        remaining=dict(targets);plans=[];total_q=0;total_cost=0;total_value=0\r\n        all_units={'WHEAT':0,'CARROT':0}\r\n        for i in range(_R51_INPUT_MAX_WORKERS):\r\n            subset={xy:t for xy,t in remaining.items() if t['crop']==first_crop} if i==0 and first_crop else remaining\r\n            path,units=_r51_input_path(obs,subset,action,i);q=len(path)\r\n            if q<3 or len(action.get('market',[]))+2+i>10 or sum(stock.values())+purchases+total_q+q+topup>95:break\r\n            quote=_r37_market_price('FERTILIZER',obs['market']['inventory']['FERTILIZER']-total_q-q",
    b"-topup)\r\n            cost=(q+(topup if i==0 else 0))*(quote+2)+_v219_fib(int(farm['hires_today'])+i)\r\n            value=sum(n*max(1,_r37_market_price(item,obs['market']['inventory'][item]+all_units[item]+n)-2) for item,n in units.items())\r\n            if value<1.5*cost+50 or farm['money']<total_cost+cost+3000:break\r\n            plans.append({'path':path,'quantity':q,'loaded':False});total_q+=q;total_cost+=cost;total_value+=value\r\n            for item,n in units.items():all_units[item]+=n\r\n            for x,y,_,_ in path:remaining.pop((x,y),None)\r\n        score=(total_value-total_cost,total_value,-total_cost,-len(plans),-mode)\r\n        choices.append((score,plans,total_q,total_cost,all_units))\r\n    _,plans,total_q,total_cost,all_units=max(choices,key=lambda v:v[0])\r\n    return plans,total_q,total_cost,all_units\r\n\r\nagent=globals().pop('agent')\r\n\r\n_R70_STATES={}\r\n_R70_REPORT={}\r\n\r\ndef _r70_parent_fert_qty(obs,action,planned,offset):\r\n    stock=dict(projected_shed(action,FarmView(obs)))\r\n    for order in action.g",
    b"et('market',[]):\r\n        if len(order)<3:continue\r\n        op,item,quantity=order[:3];quantity=max(0,int(quantity))\r\n        if op=='SELL':stock[item]=max(0,stock.get(item,0)-quantity)\r\n        elif op in ('BUY_PRODUCT','BUY_ANIMAL'):\r\n            stock[item]=stock.get(item,0)+min(quantity,max(0,100-sum(stock.values())))\r\n    next_action=planned[offset+1] if offset+1<len(planned) else {}\r\n    commands=[next_action.get('farmer') or ['PASS'],*(next_action.get('hands') or [])]\r\n    native_need=sum(max(0,int(c[2]) if len(c)>2 else 1) for c in commands if len(c)>1 and c[:2]==['PICKUP','FERTILIZER'])\r\n    quantity=max(10,10+native_need-max(0,stock.get('FERTILIZER',0)))\r\n    if quantity>max(0,100-sum(stock.values())):\r\n        _R70_REPORT['parent_input_capacity_declines']+=1\r\n        return 10\r\n    if quantity>10:\r\n        _R70_REPORT['parent_input_guard_turns']+=1\r\n        _R70_REPORT['parent_input_guard_extra_units']+=quantity-10\r\n    return quantity\r\n\r\ndef _r70_before(obs):\r\n    player=int(obs['player']);step=in",
    b"t(obs['step']);state=_R70_STATES.get(player)\r\n    if state is None or step<=state['step']:\r\n        state=_R70_STATES[player]={'step':-1,'pending':[],'roles':set()}\r\n        _R70_REPORT.update(parent_input_guard_turns=0,parent_input_guard_extra_units=0,\r\n            parent_input_requests=0,parent_input_confirmed=0,parent_input_shortfalls=0,\r\n            parent_input_errors=0,parent_input_capacity_declines=0)\r\n    state['step']=step\r\n    for request in state['pending']:\r\n        actor,quantity,old=request\r\n        actual=max(0,int(obs['private']['inventories'][actor].get('FERTILIZER',0))-old)\r\n        _R70_REPORT['parent_input_confirmed']+=min(quantity,actual)\r\n        _R70_REPORT['parent_input_shortfalls']+=max(0,quantity-actual)\r\n    state['pending']=[]\r\n    return state\r\n\r\ndef _r70_after(obs,action,state):\r\n    player=int(obs['player']);day=int(obs['step'])//24\r\n    for actor,role in _V219_STATES.get(player,{}).get('workers',{}).items():\r\n        if not role.get('needs_fertilizer'):continue\r\n        key=(da",
    b"y,actor);inv=obs['private']['inventories'][actor].get('FERTILIZER',0)\r\n        if key not in state['roles']:\r\n            state['roles'].add(key)\r\n            desired=role.get('fertilizer_quantity',10 if role['kind']=='fertilizer' else 5)\r\n            if role.get('loaded') and not role.get('pickup_requested') and inv<desired:\r\n                _R70_REPORT['parent_input_shortfalls']+=desired-inv\r\n        command=action.get('hands',[])[actor-1] if actor<=len(action.get('hands',[])) else ['PASS']\r\n        if len(command)>1 and command[:2]==['PICKUP','FERTILIZER']:\r\n            quantity=max(0,int(command[2]) if len(command)>2 else 1)\r\n            _R70_REPORT['parent_input_requests']+=quantity\r\n            state['pending'].append((actor,quantity,int(inv)))\r\n\r\n_R70_PARENT=agent\r\n\r\ndef agent(observation,configuration=None):\r\n    state=None\r\n    try:state=_r70_before(observation)\r\n    except Exception:_R70_REPORT['parent_input_errors']=_R70_REPORT.get('parent_input_errors',0)+1\r\n    result=_R70_PARENT(observation,conf",
    b"iguration)\r\n    try:\r\n        if state is not None:_r70_after(observation,result,state)\r\n    except Exception:_R70_REPORT['parent_input_errors']=_R70_REPORT.get('parent_input_errors',0)+1\r\n    _R70_REPORT.update(getattr(_R70_PARENT,'telemetry',{}))\r\n    return result\r\n\r\nagent.telemetry=_R70_REPORT\r\nagent=globals().pop('agent')\r\n\r\ndef _r79_tomato_fertilizer_worthwhile(obs,action):\r\n    if obs['market']['prices']['FERTILIZER']<=30:return True\r\n    farm=obs['farms'][obs['player']];day=int(obs['step'])//24;bonus=0\r\n    for y in (5,6):\r\n        for x in range(5,10):\r\n            tile=farm['tiles'][y][x]\r\n            if not isinstance(tile,dict) or tile.get('crop')!='TOMATO':continue\r\n            birth=tile['planted_day'];until=tile.get('fertilized_until_day',-1)\r\n            bonus+=sum(until<d and 8<=d+1-birth<=11 for d in range(day,day+3))\r\n    if not bonus:return False\r\n    inventory=obs['market']['inventory']\r\n    price=max(1,_r37_market_price('TOMATO',inventory['TOMATO']+bonus+10)-2)\r\n    fertilizer=max(1,_r37",
    b'_market_price(\'FERTILIZER\',inventory[\'FERTILIZER\']-10)+2)\r\n    native_hires=sum(bool(o) and o[0]==\'HIRE\' for o in action.get(\'market\',[]))\r\n    extra_labor=_v219_fib(int(farm[\'hires_today\'])+native_hires+3)\r\n    return bonus*price>=2*(10*fertilizer+extra_labor)+100\r\n\r\nagent=globals().pop(\'agent\')\r\n\r\n# EXP216: original adaptation of economic feed and fertilizer-sale concepts.\r\n# Conceptual credit: Steven Lee Hans, "Lord Momo Returns", September12 snapshot.\r\n_R85_FEED = True\r\n_R85_FERT = True\r\n_R85_PARENT = agent\r\n_R85_STATES = {}\r\n_R85_REPORT = {}\r\n\r\ndef _r85_feed(obs, action):\r\n    step=int(obs[\'step\']);day=step//24\r\n    if not 10<=day<=28 or step%24>21:return action\r\n    player=int(obs[\'player\']);native=_IMPL.chassis.players[player]\r\n    tape=_v219_native_day(native,day)\r\n    expected=max(len(a.get(\'hands\',[])) for a in tape)\r\n    farm=obs[\'farms\'][player];positions=[farm[\'farmer\'],*farm[\'hands\']]\r\n    commands=[action.get(\'farmer\') or [\'PASS\'],*(action.get(\'hands\') or [])]\r\n    prices=obs[\'market\'][\'prices\'',
    b"];changed=False\r\n    for actor,command in enumerate(commands[:expected+1]):\r\n        if command!=['FEED'] or actor>=len(positions):continue\r\n        tile=_tile_at(farm['tiles'],positions[actor])\r\n        if not isinstance(tile,dict) or tile.get('animal') not in ('GOOSE','COW','SHEEP'):continue\r\n        if tile.get('fed_today') or int(tile.get('consecutive_unfed',0))!=0:continue\r\n        if int(obs['private']['inventories'][actor].get('WHEAT',0))<=0:continue\r\n        item={'GOOSE':'EGG','COW':'MILK','SHEEP':'WOOL'}[tile['animal']]\r\n        bonus=_r88_feed_bonus_cost(tile,day)\r\n        if bonus*(float(prices[item])+5)*1.25>=float(prices['WHEAT']):continue\r\n        if not _r86_next_feed(obs,positions[actor]):continue\r\n        commands[actor]=['PASS'];changed=True\r\n        _R85_REPORT['feed_skips']+=1\r\n    if not changed:return action\r\n    result=copy.deepcopy(action);result['farmer'],result['hands']=commands[0],commands[1:]\r\n    return result\r\n\r\ndef _r85_reserve(obs, state):\r\n    step=int(obs['step']);player=int",
    b"(obs['player']);native=_IMPL.chassis.players[player]\r\n    route=native['route'];key=(route,step)\r\n    cache=state.setdefault('native_reserves',{})\r\n    if route not in cache:\r\n        # Backward recurrence preserves field-before-market order within a turn.\r\n        reserve=[0]*720\r\n        for t in range(718,-1,-1):\r\n            a=_IMPL.chassis.routes[2 if t>=648 else route][t]\r\n            pickup=sum(max(0,int(c[2]) if len(c)>2 else 1) for c in [a.get('farmer') or ['PASS'],*(a.get('hands') or [])] if len(c)>1 and c[:2]==['PICKUP','FERTILIZER'])\r\n            purchase=sum(max(0,int(o[2])) for o in a.get('market',[]) if len(o)>2 and o[:2]==['BUY_PRODUCT','FERTILIZER'])\r\n            reserve[t]=pickup+max(0,reserve[t+1]-purchase)\r\n        cache[route]=reserve\r\n    dedicated=0\r\n    for parent in (_V219_STATES.get(player,{}),_V233_STATES.get(player,{})):\r\n        for actor,role in parent.get('workers',{}).items():\r\n            if not isinstance(role,dict) or not role.get('needs_fertilizer') or role.get('loaded'):co",
    b"ntinue\r\n            desired=role.get('fertilizer_quantity',10 if role.get('kind')=='fertilizer' else 5)\r\n            carried=obs['private']['inventories'][actor].get('FERTILIZER',0)\r\n            dedicated+=max(0,desired-carried)\r\n        pending=parent.get('pending') or {}\r\n        if pending.get('fertilizer'):dedicated+=10\r\n    inputs=_R51_INPUT_STATES.get(player,{})\r\n    for actor,plan in {**inputs.get('workers',{}),**(inputs.get('pending') or {})}.items():\r\n        if not plan.get('loaded'):dedicated+=max(0,int(plan['quantity']))\r\n    return max(14,cache[route][min(719,step+1)]+dedicated)\r\n\r\ndef _r85_fertilizer(obs, action, state):\r\n    step=int(obs['step']);day=step//24\r\n    if not 6<=day<=28:return action\r\n    market=action.get('market',[])\r\n    if len(market)>=MAX_ORDERS or any(o and o[0]!='SELL' for o in market):return action\r\n    stock=projected_shed(action,FarmView(obs))\r\n    held=max(0,int(stock.get('FERTILIZER',0)))\r\n    sold=sum(max(0,int(o[2])) for o in market if len(o)>2 and o[:2]==['SELL','FERT",
    b"ILIZER'])\r\n    extra=held-sold-_r85_reserve(obs,state)\r\n    if extra<=0:return action\r\n    result=copy.deepcopy(action);result['market'].append(['SELL','FERTILIZER',extra])\r\n    _R85_REPORT['fert_sale_turns']+=1;_R85_REPORT['fert_sale_units']+=extra\r\n    return result\r\n\r\ndef agent(observation, configuration=None):\r\n    result=_R85_PARENT(observation,configuration)\r\n    try:\r\n        step=int(observation['step']);player=int(observation['player'])\r\n        state=_R85_STATES.get(player)\r\n        if state is None or step<=state['step']:\r\n            state=_R85_STATES[player]={'step':-1}\r\n            _R85_REPORT.update(feed_skips=0,fert_sale_turns=0,fert_sale_units=0,economic_overlay_errors=0)\r\n        state['step']=step\r\n        if configuration is not None and any(configuration.get(k,v)!=v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):return result\r\n        if _R85_FEED:result=_r85_feed(observation,result)\r\n        if _R85_FERT:result=_r85_fertilizer(observa",
    b"tion,result,state)\r\n        if step%24==23:result=_r51_close_warehouse(observation,result)\r\n    except Exception:\r\n        _R85_REPORT['economic_overlay_errors']=_R85_REPORT.get('economic_overlay_errors',0)+1\r\n    _R85_REPORT.update(getattr(_R85_PARENT,'telemetry',{}))\r\n    return result\r\n\r\nagent.telemetry=_R85_REPORT\r\nagent=globals().pop('agent')\r\n\r\n# EXP217: planned next-day service is required before discretionary feed cuts.\r\n_R86_FEED_CACHE = {}\r\n\r\ndef _r86_next_feed(obs, target):\r\n    step=int(obs['step']);day=step//24\r\n    if day==28:return True  # No second dawn follows before game termination.\r\n    player=int(obs['player']);native=_IMPL.chassis.players[player]\r\n    tomorrow=day+1;route=2 if tomorrow>=27 else native['route'];key=(route,tomorrow)\r\n    if key not in _R86_FEED_CACHE:\r\n        positions=[(4,4)];wheat=[0];access=((4,4),(5,4),(4,5),(5,5));feeds=set()\r\n        for hour in range(24):\r\n            a=_IMPL.chassis.routes[route][tomorrow*24+hour]\r\n            commands=[a.get('farmer') or ['PASS']",
    b",*(a.get('hands') or [])]\r\n            for actor,command in enumerate(commands[:len(positions)]):\r\n                if not command:continue\r\n                pos=positions[actor];op=command[0]\r\n                if op in MOVES:\r\n                    dx,dy=MOVES[op];positions[actor]=(max(0,min(9,pos[0]+dx)),max(0,min(9,pos[1]+dy)))\r\n                elif command[:2]==['PICKUP','WHEAT'] and pos in access:\r\n                    wheat[actor]+=max(0,int(command[2]) if len(command)>2 else 1)\r\n                elif op=='FEED' and wheat[actor]>0:\r\n                    wheat[actor]-=1\r\n                    if hour<=21:feeds.add(pos)\r\n                elif op=='DROP' and pos in access:wheat[actor]=0\r\n                elif command[:2]==['PLACE','WHEAT'] and pos in access:\r\n                    wheat[actor]=max(0,wheat[actor]-max(0,int(command[2]) if len(command)>2 else 1))\r\n            for order in a.get('market',[]):\r\n                if order and order[0]=='HIRE':\r\n                    chosen=min(access,key=lambda p:(positions.count",
    b"(p),access.index(p)))\r\n                    positions.append(chosen);wheat.append(0)\r\n        _R86_FEED_CACHE[key]=frozenset(feeds)\r\n    return tuple(target) in _R86_FEED_CACHE[key]\r\n\r\nagent=globals().pop('agent')\r\n\r\n# EXP219: charge care credits only when this feeding decision can affect them.\r\n_R88_PHASE = True\r\n_R88_HORIZON = True\r\n_R88_ANIMAL_DAYS = {'GOOSE': (4, 1), 'COW': (8, 2), 'SHEEP': (6, 3)}\r\n\r\n\r\ndef _r88_feed_bonus_cost(tile, day):\r\n    first, interval = _R88_ANIMAL_DAYS[tile['animal']]\r\n    first += int(tile['placed_day'])\r\n    tomorrow = day + 1\r\n    produces = tomorrow >= first and (tomorrow - first) % interval == 0\r\n    pending = max(0, int(tile.get('pending_care_bonus', 0)))\r\n    if _R88_PHASE and not produces:\r\n        pending = 0  # It remains banked on non-production dawns.\r\n    care = 1  # Conservative: charge one possible CARE even if not yet observed.\r\n    if _R88_HORIZON:\r\n        # Today's care is added AFTER tomorrow's production; its first possible\r\n        # payout is a later produc",
    b"tion dawn, which must occur before game end.\r\n        next_use = first\r\n        if next_use <= tomorrow:\r\n            next_use += ((tomorrow - next_use) // interval + 1) * interval\r\n        if next_use > 29:\r\n            care = 0\r\n    return pending + care\r\n\r\n\r\nagent = globals().pop('agent')\r\n\r\n# EXP226: retain physical grain for two complete days before trimming a buy.\r\n_R95_PARENT = agent\r\n_R95_REPORT = {}\r\n_R95_RESERVES = {}\r\n\r\ndef _r95_reserve(obs):\r\n    step=int(obs['step']);player=int(obs['player'])\r\n    native=_IMPL.chassis.players[player];route=native['route']\r\n    key=(route,step)\r\n    if key not in _R95_RESERVES:\r\n        demand=6  # Physical buffer beyond every scheduled pickup and sale.\r\n        for t in range(step+1,min(719,step+49)):\r\n            a=_IMPL.chassis.routes[2 if t>=648 else route][t]\r\n            for c in [a.get('farmer') or ['PASS'],*(a.get('hands') or [])]:\r\n                if c[:2]==['PICKUP','WHEAT']:\r\n                    demand+=max(0,int(c[2]) if len(c)>2 else 1)\r\n            f",
    b"or o in a.get('market',[]):\r\n                if len(o)>2 and o[:2]==['SELL','WHEAT']:\r\n                    demand+=max(0,int(o[2]))\r\n        _R95_RESERVES[key]=demand\r\n    demand=_R95_RESERVES[key]\r\n    # Reserve full feed for a possible southeast sheep commitment. Do not\r\n    # rely on its future discretionary buy, eligibility, or existing cargo.\r\n    if obs['town']['unlocked_shops'].count('YARN_STORE')>=2:\r\n        demand+=6*len({t//24 for t in range(step+1,step+49) if t//24>=12})\r\n    return demand\r\n\r\ndef _r95_replenish(obs,action):\r\n    step=int(obs['step'])\r\n    if not 10<=step//24<=11:return action\r\n    orders=action.get('market') or []\r\n    if not any(len(o)>2 and o[:2]==['BUY_PRODUCT','WHEAT'] and int(o[2])>0 for o in orders):return action\r\n    # Preserve all same-turn grain trading/arbitrage sequences unchanged.\r\n    if any(o[:2]==['SELL','WHEAT'] for o in orders):return action\r\n    farm,private=_PLANNER_NS['_clone_state'](obs['farms'][obs['player']],obs['private'])\r\n    commands=[action.get('farmer'",
    b") or ['PASS'],*(action.get('hands') or [])]\r\n    for actor,c in enumerate(commands[:len(private['inventories'])]):\r\n        _PLANNER_NS['_apply_unit_action'](farm,private,actor,c,10,step//24,24,100)\r\n    held=max(0,int(private['shed'].get('WHEAT',0)))\r\n    reserve=_r95_reserve(obs);result=None;removed=0\r\n    for i,o in enumerate(orders):\r\n        if len(o)<3 or o[:2]!=['BUY_PRODUCT','WHEAT']:continue\r\n        quantity=max(0,int(o[2]));retained=min(quantity,max(0,reserve-held))\r\n        held+=retained\r\n        if retained<quantity:\r\n            if result is None:result=copy.deepcopy(action)\r\n            result['market'][i][2]=retained  # Zero keeps every later order slot.\r\n            removed+=quantity-retained\r\n    if result is None:return action\r\n    _R95_REPORT['replenishment_trim_turns']+=1\r\n    _R95_REPORT['replenishment_trim_units']+=removed\r\n    return result\r\n\r\ndef agent(observation,configuration=None):\r\n    result=_R95_PARENT(observation,configuration)\r\n    try:\r\n        if int(observation['step'])==0",
    b":\r\n            _R95_REPORT.update(replenishment_trim_turns=0,replenishment_trim_units=0,replenishment_errors=0)\r\n        if configuration is not None and any(configuration.get(k,v)!=v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):return result\r\n        result=_r95_replenish(observation,result)\r\n    except Exception:\r\n        _R95_REPORT['replenishment_errors']=_R95_REPORT.get('replenishment_errors',0)+1\r\n    _R95_REPORT.update(getattr(_R95_PARENT,'telemetry',{}))\r\n    return result\r\n\r\nagent.telemetry=_R95_REPORT\r\nagent=globals().pop('agent')\r\n\r\n# EXP231: protect inputs using funded current orders without unassigned cash padding from observed physical resources.\r\n_R97_PARENT=agent\r\n_R97_REPORT={}\r\n_R97_LAST={}\r\n\r\ndef _r97_market_stock(shed,orders):\r\n    stock=dict(shed);buys={};sales={}\r\n    for index,order in enumerate(orders):\r\n        if len(order)<3:continue\r\n        op,item,n=order[:3];n=max(0,int(n))\r\n        if op=='SELL':\r\n            q=min(n,max(0",
    b",stock.get(item,0)));stock[item]=stock.get(item,0)-q;sales[index]=q\r\n        elif op in ('BUY_PRODUCT','BUY_ANIMAL'):\r\n            q=min(n,max(0,100-sum(stock.values())));stock[item]=stock.get(item,0)+q;buys[index]=q\r\n    return stock,buys,sales\r\n\r\ndef _r97_delivery(stock,private,night):\r\n    stock=dict(stock);lost={}\r\n    if night:\r\n        for inv in private['inventories']:\r\n            for item,q in inv.items():\r\n                q=max(0,int(q));take=min(q,max(0,100-sum(stock.values())))\r\n                stock[item]=stock.get(item,0)+take\r\n                if q>take:lost[item]=lost.get(item,0)+q-take\r\n    return stock,lost\r\n\r\ndef _r97_budget(obs,orders):\r\n    farm=obs['farms'][obs['player']];cost=0;hires=int(farm['hires_today'])\r\n    # At most ten 100-unit purchases per opponent turn. The additional 1000\r\n    # own units give an intentionally conservative upper bound on buy quotes.\r\n    prices={p:_r37_market_price(p,obs['market']['inventory'][p]-2000) for p in ('WHEAT','FERTILIZER')}\r\n    for order in orders",
    b":\r\n        if not order:continue\r\n        op=order[0]\r\n        if op=='HIRE':cost+=_v219_fib(hires);hires+=1\r\n        elif op=='BUY_LAND':cost+=4000\r\n        elif len(order)>2:\r\n            item=order[1];q=max(0,int(order[2]))\r\n            if op=='BUY_PRODUCT':cost+=q*prices[item]\r\n            elif op=='BUY_ANIMAL':cost+=q*{'GOOSE':300,'COW':400,'SHEEP':500}[item]\r\n            elif op=='BUY_SEED':cost+=q*{'WHEAT':10,'CARROT':20,'TOMATO':50,'STRAWBERRY':100,'MELON':80}[item]\r\n    return cost<=farm['money']  # No current sale proceeds are assumed.\r\n\r\ndef _r97_supply(obs,action):\r\n    step=int(obs['step']);player=int(obs['player']);day=step//24\r\n    if not 144<=step<695:return action\r\n    native=_IMPL.chassis.players[player]\r\n    future=_IMPL.chassis.routes[2 if step+1>=648 else native['route']][step+1]\r\n    commands=[future.get('farmer') or ['PASS'],*(future.get('hands') or [])]\r\n    following=_IMPL.chassis.routes[2 if step+2>=648 else native['route']][step+2]\r\n    next_orders=future.get('market') or []\r\n    pr",
    b"efund=0\r\n    if len(next_orders)==10 and not any(o[:2] in (['BUY_PRODUCT','WHEAT'],['SELL','WHEAT']) for o in next_orders):\r\n        later=[following.get('farmer') or ['PASS'],*(following.get('hands') or [])]\r\n        demand=lambda cs:sum(max(0,int(c[2]) if len(c)>2 else 1) for c in cs if c[:2]==['PICKUP','WHEAT'])\r\n        if demand(later):prefund=demand(commands)+demand(later)\r\n    if not prefund and not any(c[:2]==['PICKUP','WHEAT'] for c in commands):return action\r\n    orders=action.get('market') or []\r\n    if len(orders)>10 or not _r97_budget(obs,orders):\r\n        _R97_REPORT['supply_budget_declines']+=1;return action\r\n    farm,private=_PLANNER_NS['_clone_state'](obs['farms'][player],obs['private'])\r\n    for actor,c in enumerate([action.get('farmer') or ['PASS'],*(action.get('hands') or [])][:len(private['inventories'])]):\r\n        _PLANNER_NS['_apply_unit_action'](farm,private,actor,c,10,day,24,100)\r\n    positions=[tuple(farm['farmer']),*map(tuple,farm['hands'])];night=step%24==23\r\n    access=((4,4),(5,",
    b"4),(4,5),(5,5))\r\n    if night:positions=[(4,4)]\r\n    else:\r\n        for order in orders:\r\n            if order and order[0]=='HIRE':positions.append(min(access,key=lambda p:(positions.count(p),access.index(p))))\r\n    need=sum(max(0,int(c[2]) if len(c)>2 else 1) for pos,c in zip(positions,commands) if pos in access and c[:2]==['PICKUP','WHEAT'])\r\n    need=max(need,prefund)\r\n    if not need:return action\r\n    original_stock,original_buys,_=_r97_market_stock(private['shed'],orders)\r\n    original_final,original_loss=_r97_delivery(original_stock,private,night)\r\n    if original_final.get('WHEAT',0)>=need:return action\r\n    result=copy.deepcopy(action);proposed=result['market'];blocked=False\r\n    def project(candidate):\r\n        stock,buys,sales=_r97_market_stock(private['shed'],candidate)\r\n        final,loss=_r97_delivery(stock,private,night)\r\n        safe=all(buys.get(i,0)>=q for i,q in original_buys.items()) and all(q<=original_loss.get(item,0) for item,q in loss.items())\r\n        return final,sales,safe\r\n    # H",
    b"old an existing grain sale first. Preserve all order indices and every\r\n    # originally funded buy; no extra overnight overflow may be introduced.\r\n    for index in range(len(proposed)-1,-1,-1):\r\n        if proposed[index][:2]!=['SELL','WHEAT']:continue\r\n        final,sales,safe=project(proposed);shortage=max(0,need-final.get('WHEAT',0))\r\n        if not shortage:break\r\n        sold=sales.get(index,0)\r\n        if not sold:continue\r\n        old=proposed[index][2];proposed[index][2]=max(0,sold-shortage)\r\n        after,_,safe=project(proposed)\r\n        if not safe or after.get('WHEAT',0)<=final.get('WHEAT',0):proposed[index][2]=old\r\n    final,_,safe=project(proposed);shortage=max(0,need-final.get('WHEAT',0))\r\n    if shortage:\r\n        last_sale=max((i for i,o in enumerate(proposed) if o[:2]==['SELL','WHEAT']),default=-1)\r\n        index=next((i for i in range(len(proposed)-1,last_sale,-1) if proposed[i][:2]==['BUY_PRODUCT','WHEAT']),None)\r\n        if index is not None:proposed[index][2]=max(0,int(proposed[index][",
    b"2]))+shortage\r\n        elif len(proposed)<10:proposed.append(['BUY_PRODUCT','WHEAT',shortage])\r\n        else:_R97_REPORT['supply_slot_declines']+=1;return action\r\n    final,_,safe=project(proposed)\r\n    if not safe or final.get('WHEAT',0)<need:\r\n        _R97_REPORT['supply_capacity_declines']+=1;return action\r\n    if not _r97_budget(obs,proposed):\r\n        _R97_REPORT['supply_budget_declines']+=1;return action\r\n    if prefund:\r\n        _R97_REPORT['supply_prefund_changes']+=1\r\n        _R97_REPORT['supply_prefund_units']+=max(0,final.get('WHEAT',0)-original_final.get('WHEAT',0))\r\n    if step<288:\r\n        _R97_REPORT['supply_early_changes']+=1\r\n        _R97_REPORT['supply_early_units']+=max(0,final.get('WHEAT',0)-original_final.get('WHEAT',0))\r\n    _R97_REPORT['supply_guard_changes']+=1\r\n    _R97_REPORT['supply_grain_protected']+=final.get('WHEAT',0)-original_final.get('WHEAT',0)\r\n    _R97_REPORT['supply_buy_units']+=sum(max(0,int(o[2])) for o in proposed if o[:2]==['BUY_PRODUCT','WHEAT'])-sum(max(0,int(o[2]))",
    b" for o in orders if o[:2]==['BUY_PRODUCT','WHEAT'])\r\n    return result\r\n\r\ndef agent(observation,configuration=None):\r\n    result=_R97_PARENT(observation,configuration)\r\n    try:\r\n        player=int(observation['player']);step=int(observation['step'])\r\n        if player not in _R97_LAST or step<=_R97_LAST[player]:\r\n            _R97_REPORT.update(supply_guard_changes=0,supply_grain_protected=0,supply_buy_units=0,supply_early_changes=0,supply_early_units=0,supply_prefund_changes=0,supply_prefund_units=0,supply_slot_declines=0,supply_capacity_declines=0,supply_budget_declines=0,supply_errors=0)\r\n        _R97_LAST[player]=step\r\n        if configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10),('farmHandCostMult',1)]):result=_r97_supply(observation,result)\r\n    except Exception:_R97_REPORT['supply_errors']=_R97_REPORT.get('supply_errors',0)+1\r\n    _R97_REPORT.update(getattr(_R97_PARENT,'telemetry',{}))\r\n    return res",
    b"ult\r\n\r\nagent.telemetry=_R97_REPORT\r\nagent=globals().pop('agent')\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# v9 COURIER: deliver premium cargo before midnight and sell it the same day.\r\n#\r\n# Roughly 40% of the tape's strawberries and milk are still in workers' hands\r\n# when the day ends; the engine drops them into the shed *after* the market,\r\n# so every sibling of this policy sells them the next morning.  No town draw\r\n# happens between hour 20 and the next dawn's market, so an evening sale gets\r\n# the morning's quote a day earlier than a rival who waits for the auto-drop.\r\n#\r\n# From hour V9_COURIER_FROM_HOUR, a tape worker carrying premium goods whose\r\n# every remaining command today is a PASS or a move (moves are free: workers\r\n# respawn at the shed at dawn) walks to the nearest shed-access tile, drops,\r\n# and the delivered units are offered in the first market slot.\r\n# ---------------------------------------------------------------------------\r\nV9_COURIER_ITEMS = ",
    b'("STRAWBERRY", "MILK", "WOOL", "MELON")\r\nV9_COURIER_FROM_HOUR = 12\r\n_V9_COURIER_ACCESS = ((4, 4), (5, 4), (4, 5), (5, 5))\r\n_V9_COURIER_IDLE = frozenset({"PASS", "NORTH", "SOUTH", "EAST", "WEST", "DROP"})\r\n_V9_COURIER = {}\r\n_V9_COURIER_REPORT = dict(courier_trips=0, courier_units=0, courier_errors=0)\r\n\r\n\r\ndef _v9_courier_walk(pos, target):\r\n    x, y = pos\r\n    tx, ty = target\r\n    return ([["EAST"]] * max(0, tx - x) + [["WEST"]] * max(0, x - tx)\r\n            + [["SOUTH"]] * max(0, ty - y) + [["NORTH"]] * max(0, y - ty))\r\n\r\n\r\ndef _v9_courier_plan(tape, unit, pos, commands, step, end):\r\n    """Walk-and-drop route for an idle tape worker, or None if it has work left today."""\r\n    if commands[unit] and commands[unit][0] not in _V9_COURIER_IDLE:\r\n        return None\r\n    for t in range(step + 1, end + 1):\r\n        a = tape[t] if t < len(tape) else {}\r\n        units = [a.get("farmer") or ["PASS"]] + list(a.get("hands") or [])\r\n        command = units[unit] if unit < len(units) else ["PASS"]\r\n        if command and ',
    b'command[0] not in _V9_COURIER_IDLE:\r\n            return None\r\n    target = min(_V9_COURIER_ACCESS, key=lambda a: abs(a[0] - pos[0]) + abs(a[1] - pos[1]))\r\n    walk = _v9_courier_walk(pos, target)\r\n    return walk + [["DROP"]] if len(walk) <= end - step else None\r\n\r\n\r\ndef _v9_courier(obs, action, st):\r\n    step = int(obs["step"])\r\n    player = int(obs["player"])\r\n    if step >= 718 or step % 24 < V9_COURIER_FROM_HOUR:\r\n        return action\r\n    day = step // 24\r\n    if st.get("day") != day:\r\n        st["day"] = day\r\n        st["plans"] = {}\r\n    native = _IMPL.chassis.players.get(player)\r\n    if not native or native.get("route") not in _IMPL.chassis.routes:\r\n        return action\r\n    tape = _IMPL.chassis.routes[native["route"]]\r\n    farm = obs["farms"][player]\r\n    positions = [tuple(farm["farmer"])] + [tuple(p) for p in farm["hands"]]\r\n    inventories = obs["private"]["inventories"]\r\n    commands = [list(action.get("farmer") or ["PASS"])] + [list(c) for c in (action.get("hands") or [])]\r\n    commands += [["',
    b'PASS"]] * (len(positions) - len(commands))\r\n    end = day * 24 + 23\r\n    plans = st["plans"]\r\n    # Only tape workers: overlay-dedicated hands are appended after the tape\'s crew.\r\n    crew = 1 + max(len(tape[t].get("hands") or []) for t in range(day * 24, min(len(tape), end + 1)))\r\n    delivered = {}\r\n    changed = False\r\n    for unit, pos in enumerate(positions[:crew]):\r\n        inventory = inventories[unit] if unit < len(inventories) else {}\r\n        cargo = {k: int(v) for k, v in inventory.items() if k in V9_COURIER_ITEMS and int(v) > 0}\r\n        plan = plans.get(unit)\r\n        if plan is None:\r\n            if not cargo or native.get("pending", {}).get(unit):\r\n                continue\r\n            route = _v9_courier_plan(tape, unit, pos, commands, step, end)\r\n            if route is None:\r\n                continue\r\n            plan = plans[unit] = {"route": route, "start": step}\r\n            _V9_COURIER_REPORT["courier_trips"] += 1\r\n        index = step - plan["start"]\r\n        if index >= len(plan["route',
    b'"]):\r\n            continue\r\n        command = plan["route"][index]\r\n        if command == ["DROP"]:\r\n            if pos not in _V9_COURIER_ACCESS:\r\n                plans[unit] = {"route": [], "start": step}\r\n                continue\r\n            for item, n in cargo.items():\r\n                delivered[item] = delivered.get(item, 0) + n\r\n        commands[unit] = command\r\n        changed = True\r\n    if not changed:\r\n        return action\r\n    result = dict(action)\r\n    result["farmer"], result["hands"] = commands[0], commands[1:]\r\n    if delivered:\r\n        market = [list(o) for o in action.get("market") or []]\r\n        prices = obs["market"]["prices"]\r\n        for item, n in sorted(delivered.items(), key=lambda kv: -int(prices.get(kv[0], 0)) * kv[1]):\r\n            if int(prices.get(item, 0)) < 2:\r\n                continue\r\n            existing = next((o for o in market if o and o[0] == "SELL" and len(o) >= 3 and o[1] == item), None)\r\n            if existing is not None:\r\n                existing[2] = int(exist',
    b'ing[2]) + n\r\n                market.remove(existing)\r\n                market.insert(0, existing)\r\n            elif len(market) < MAX_ORDERS:\r\n                market.insert(0, ["SELL", item, n])\r\n            _V9_COURIER_REPORT["courier_units"] += n\r\n        result["market"] = market\r\n    return result\r\n\r\n\r\n_V9_COURIER_PARENT = agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    player, step = int(observation["player"]), int(observation["step"])\r\n    st = _V9_COURIER.get(player)\r\n    if st is None or step <= st["step"]:\r\n        st = _V9_COURIER[player] = {"step": -1}\r\n        if step == 0:\r\n            _V9_COURIER_REPORT.update(courier_trips=0, courier_units=0, courier_errors=0)\r\n    st["step"] = step\r\n    action = _V9_COURIER_PARENT(observation, configuration)\r\n    try:\r\n        return _v9_courier(observation, action, st)\r\n    except Exception:\r\n        _V9_COURIER_REPORT["courier_errors"] += 1\r\n        return action\r\n\r\n\r\nagent.telemetry = _V9_COURIER_REPORT\r\nagent = globals().pop("agent")\r\n\r\n\r\n# ----',
    b"-----------------------------------------------------------------------\r\n# v9 CARROT: plant carrots instead of wheat when the carrot book pays for it.\r\n#\r\n# The tape replants wheat on the same short cycle a carrot needs (water daily,\r\n# harvest at age 2-4), so the swap keeps every worker's schedule intact.  A\r\n# watered wheat plant yields 4 and a carrot 3, for $10 more seed, so the swap\r\n# only pays once the carrot quote clears V9_CARROT_RATIO x the wheat quote --\r\n# which pet cafes and farmers markets make happen by draining the hinge book.\r\n# Wheat is also feed, so the swap stops while the shed holds less than\r\n# V9_CARROT_WHEAT_RESERVE wheat.\r\n# ---------------------------------------------------------------------------\r\nV9_CARROT_RATIO = 2.0\r\nV9_CARROT_FIRST_DAY = 10\r\nV9_CARROT_LAST_DAY = 23\r\nV9_CARROT_WHEAT_RESERVE = 40\r\nV9_CARROT_BOOM_RATIO = 4.5   # from this ratio the feed reserve is bought instead of grown\r\nV9_CARROT_BOOM_RESERVE = 10\r\n_V9_CARROT_REPORT = dict(carrot_swaps=0, carrot_seed_swaps=0, car",
    b'rot_errors=0)\r\n\r\n\r\ndef _v9_carrot(obs, action, st):\r\n    step = int(obs["step"])\r\n    day = step // 24\r\n    prices = obs["market"]["prices"]\r\n    private = obs["private"]\r\n    commands = [list(action.get("farmer") or ["PASS"])] + [list(c) for c in (action.get("hands") or [])]\r\n    market = [list(o) for o in action.get("market") or []]\r\n    changed = False\r\n    if st.get("tiles"):\r\n        # swapped carrots die at the start of age 4, but the wheat tape may harvest at age 4:\r\n        # harvest them on the age-3 watering visit instead\r\n        _farm = obs["farms"][int(obs["player"])]\r\n        _pos = [_farm["farmer"]] + list(_farm["hands"])\r\n        for _i, c in enumerate(commands[:len(_pos)]):\r\n            if c != ["WATER"]:\r\n                continue\r\n            _p = tuple(_pos[_i])\r\n            if st["tiles"].get(_p) != day - 3:\r\n                continue\r\n            _t = _farm["tiles"][_p[1]][_p[0]]\r\n            if isinstance(_t, dict) and _t.get("crop") == "CARROT" and int(_t.get("planted_day", -9)) == day -',
    b' 3 and int(_t.get("yield_units", 0)) > 0:\r\n                commands[_i] = ["HARVEST"]\r\n                changed = True\r\n                _V9_CARROT_REPORT["carrot_rescues"] = _V9_CARROT_REPORT.get("carrot_rescues", 0) + 1\r\n    wheat_held = int(private["shed"].get("WHEAT", 0)) + sum(int(i.get("WHEAT", 0)) for i in private["inventories"])\r\n    ratio = int(prices.get("CARROT", 0)) / max(1, int(prices.get("WHEAT", 99)))\r\n    boom = ratio >= V9_CARROT_BOOM_RATIO\r\n    reserve = V9_CARROT_BOOM_RESERVE if boom else V9_CARROT_WHEAT_RESERVE\r\n    if boom and V9_CARROT_FIRST_DAY <= day <= V9_CARROT_LAST_DAY and wheat_held < V9_CARROT_WHEAT_RESERVE:\r\n        # Carrots are worth several wheat each: buy the feed the swap no longer grows.\r\n        topup = V9_CARROT_WHEAT_RESERVE - wheat_held\r\n        budget = float(obs["farms"][int(obs["player"])]["money"]) - 1500\r\n        qty = min(topup, int(budget // max(1, int(prices.get("WHEAT", 99)) + 5)))\r\n        if qty > 0 and len(market) < MAX_ORDERS and not any(o[:2] == ["BUY_PRODUC',
    b'T", "WHEAT"] for o in market if len(o) >= 2):\r\n            market.append(["BUY_PRODUCT", "WHEAT", qty])\r\n            changed = True\r\n    if (V9_CARROT_FIRST_DAY <= day <= V9_CARROT_LAST_DAY and wheat_held >= reserve\r\n            and ratio >= V9_CARROT_RATIO):\r\n        carrot_seeds = int(private["seeds"].get("CARROT", 0)) - sum(1 for c in commands if c[:2] == ["PLANT", "CARROT"])\r\n        _farm = obs["farms"][int(obs["player"])]\r\n        _pos = [_farm["farmer"]] + list(_farm["hands"])\r\n        for _i, c in enumerate(commands):\r\n            if c[:2] == ["PLANT", "WHEAT"] and carrot_seeds > 0:\r\n                c[1] = "CARROT"\r\n                carrot_seeds -= 1\r\n                changed = st["swapped"] = True\r\n                _V9_CARROT_REPORT["carrot_swaps"] += 1\r\n                if _i < len(_pos):\r\n                    st.setdefault("tiles", {})[tuple(_pos[_i])] = day\r\n        for o in market:\r\n            if len(o) >= 3 and o[:2] == ["BUY_SEED", "WHEAT"]:\r\n                o[1] = "CARROT"\r\n                changed',
    b' = st["swapped"] = True\r\n                _V9_CARROT_REPORT["carrot_seed_swaps"] += int(o[2])\r\n    result = dict(action)\r\n    result["farmer"], result["hands"], result["market"] = commands[0], commands[1:], market\r\n    if st.get("swapped") and day < 24:\r\n        # Swapped carrots have no planned sale on the tape before its own carrot days.\r\n        stock = projected_shed(result, FarmView(obs)).get("CARROT", 0)\r\n        selling = sum(int(o[2]) for o in market if len(o) >= 3 and o[:2] == ["SELL", "CARROT"])\r\n        if stock > selling and len(market) < MAX_ORDERS and int(prices.get("CARROT", 0)) >= 2:\r\n            market.insert(0, ["SELL", "CARROT", stock - selling])\r\n            changed = True\r\n    return result if changed else action\r\n\r\n\r\n_V9_CARROT = {}\r\n\r\n\r\n_V9_CARROT_PARENT = agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    player, step = int(observation["player"]), int(observation["step"])\r\n    st = _V9_CARROT.get(player)\r\n    if st is None or step <= st["step"]:\r\n        st = _V9_CARROT[player]',
    b' = {"step": -1}\r\n        if step == 0:\r\n            _V9_CARROT_REPORT.update(carrot_swaps=0, carrot_seed_swaps=0, carrot_errors=0)\r\n    st["step"] = step\r\n    action = _V9_CARROT_PARENT(observation, configuration)\r\n    try:\r\n        return _v9_carrot(observation, action, st)\r\n    except Exception:\r\n        _V9_CARROT_REPORT["carrot_errors"] += 1\r\n        return action\r\n\r\n\r\nagent.telemetry = _V9_CARROT_REPORT\r\nagent = globals().pop("agent")\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# v9 HERD: raise the tape\'s day-10 geese as sheep or cows when the draw pays.\r\n#\r\n# A cared goose lays 2 eggs a day into a $50 book; a cared sheep grows 4 wool\r\n# every 3 days into a $200 book, a cow 3 milk every 2 days into a $160 book.\r\n# The tape handles its geese exactly like pasture animals (build, pick up,\r\n# place, feed, care, collect fertilizer, harvest), so swapping the species at\r\n# the first goose purchase keeps every worker\'s schedule.  Wool needs a yarn\r\n# store to hold its pric',
    b'e and milk needs pizza / ice cream / smoothie shops;\r\n# egg shops keep the geese.  The swapped herd\'s extra product has no planned\r\n# sale on the tape, so anything beyond the tape\'s remaining planned sales is\r\n# sold as it reaches the shed.\r\n# ---------------------------------------------------------------------------\r\nV9_HERD_MIN_WOOL = 150         # wool quote needed to swap to sheep\r\nV9_HERD_MIN_MILK = 150         # milk quote needed to swap to cows\r\nV9_HERD_MAX_EGG_SHOPS = 1         # sheep swap: at most this many BAKERY + BRUNCH_SPOT\r\nV9_HERD_MAX_EGG_SHOPS_COW = 0  # cow swap: at most this many\r\nV9_HERD_MIN_MILK_SHOPS = 3      # cow swap: at least this many PIZZA / ICE_CREAM / SMOOTHIE\r\n_V9_HERD_PRODUCT = {"SHEEP": "WOOL", "COW": "MILK"}\r\n_V9_HERD = {}\r\n_V9_HERD_REPORT = dict(herd_species="", herd_rewrites=0, herd_extra_sold=0, herd_errors=0)\r\n\r\n\r\ndef _v9_herd_choose(obs):\r\n    shops = obs["town"]["unlocked_shops"]\r\n    prices = obs["market"]["prices"]\r\n    egg_shops = sum(s in ("BAKERY", "BRUNCH_SPOT") ',
    b'for s in shops)\r\n    if (egg_shops <= V9_HERD_MAX_EGG_SHOPS and "YARN_STORE" in shops\r\n            and int(prices.get("WOOL", 0)) >= V9_HERD_MIN_WOOL):\r\n        return "SHEEP"\r\n    milk_shops = sum(s in ("PIZZA_SHOP", "ICE_CREAM_SHOP", "SMOOTHIE_SHOP") for s in shops)\r\n    if (egg_shops <= V9_HERD_MAX_EGG_SHOPS_COW and milk_shops >= V9_HERD_MIN_MILK_SHOPS\r\n            and int(prices.get("MILK", 0)) >= V9_HERD_MIN_MILK):\r\n        return "COW"\r\n    return None\r\n\r\n\r\ndef _v9_herd(obs, action, st):\r\n    step = int(obs["step"])\r\n    orders = action.get("market") or []\r\n    if st.get("species") is None and not st.get("decided"):\r\n        if step >= 216 and any(len(o) >= 2 and o[:2] == ["BUY_ANIMAL", "GOOSE"] for o in orders):\r\n            st["decided"] = True\r\n            st["species"] = _v9_herd_choose(obs)\r\n            _V9_HERD_REPORT["herd_species"] = st["species"] or ""\r\n    species = st.get("species")\r\n    if not species:\r\n        return action\r\n    product = _V9_HERD_PRODUCT[species]\r\n    commands = [list(acti',
    b'on.get("farmer") or ["PASS"])] + [list(c) for c in (action.get("hands") or [])]\r\n    market = [list(o) for o in orders]\r\n    for c in commands:\r\n        if c and c[0] == "BUILD_COOP":\r\n            c[0] = "BUILD_PASTURE"\r\n            _V9_HERD_REPORT["herd_rewrites"] += 1\r\n        elif len(c) >= 2 and c[0] in ("PICKUP", "PLACE") and c[1] == "GOOSE":\r\n            c[1] = species\r\n            _V9_HERD_REPORT["herd_rewrites"] += 1\r\n    rewritten = []\r\n    for o in market:\r\n        if len(o) >= 3 and o[:2] == ["BUY_ANIMAL", "GOOSE"]:\r\n            rewritten.append(["BUY_ANIMAL", species, o[2]])\r\n        elif len(o) >= 2 and o[:2] == ["SELL", "EGG"] and not any(isinstance(t_, dict) and t_.get("animal") == "GOOSE" for r_ in obs["farms"][int(obs["player"])]["tiles"] for t_ in r_):\r\n            continue\r\n        else:\r\n            rewritten.append(o)\r\n    result = dict(action)\r\n    result["farmer"], result["hands"], result["market"] = commands[0], commands[1:], rewritten\r\n    player = int(obs["player"])\r\n    native = _IM',
    b'PL.chassis.players.get(player)\r\n    if native and native.get("route") in _IMPL.chassis.routes and step < 718:\r\n        stock = projected_shed(result, FarmView(obs)).get(product, 0)\r\n        selling = sum(int(o[2]) for o in rewritten if len(o) >= 3 and o[:2] == ["SELL", product])\r\n        planned = _IMPL.chassis.future_sells(native["route"], product, step + 1)\r\n        extra = stock - selling - planned\r\n        if extra > 0 and len(rewritten) < MAX_ORDERS and int(obs["market"]["prices"].get(product, 0)) >= 2:\r\n            rewritten.insert(0, ["SELL", product, extra])\r\n            _V9_HERD_REPORT["herd_extra_sold"] += extra\r\n    return result\r\n\r\n\r\n_V9_HERD_PARENT = agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    player, step = int(observation["player"]), int(observation["step"])\r\n    st = _V9_HERD.get(player)\r\n    if st is None or step <= st["step"]:\r\n        st = _V9_HERD[player] = {"step": -1}\r\n        if step == 0:\r\n            _V9_HERD_REPORT.update(herd_species="", herd_rewrites=0, herd_extra_s',
    b'old=0, herd_errors=0)\r\n    st["step"] = step\r\n    action = _V9_HERD_PARENT(observation, configuration)\r\n    try:\r\n        return action\r\n    except Exception:\r\n        _V9_HERD_REPORT["herd_errors"] += 1\r\n        return action\r\n\r\n\r\nagent.telemetry = _V9_HERD_REPORT\r\nagent = globals().pop("agent")\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# v9 FERT: spend carried fertilizer on young wheat and carrots.\r\n#\r\n# Workers that tend animals carry collected fertilizer back to the shed, where\r\n# the tape sells it into a book that falls from $55 on day 14 to $10 by day 27.\r\n# On a short crop the same unit is worth far more: a watered wheat plant ends at\r\n# 4 units, but fertilized for its yield window it caps at 6.  A worker that\r\n# carries fertilizer and is about to water a wheat or carrot plant one day\r\n# after planting (watered on planting day, not yet fertilized) fertilizes it\r\n# instead; the tape waters it again on the following days.  Fertilizer the\r\n# worker\'s own tape comm',
    b'ands still spend today is left alone, and the layer only\r\n# acts from day 16, once the fertilizer book is worth less than two wheat.\r\n# ---------------------------------------------------------------------------\r\nV9_FERT_CROPS = ("WHEAT", "CARROT")\r\nV9_FERT_AGES = (1,)\r\nV9_FERT_FIRST_DAY = 14\r\n_V9_FERT_REPORT = dict(fert_applied=0, fert_errors=0)\r\n\r\n\r\ndef _v9_fert(obs, action):\r\n    step = int(obs["step"])\r\n    day = step // 24\r\n    if day < V9_FERT_FIRST_DAY or step >= 700:\r\n        return action\r\n    player = int(obs["player"])\r\n    farm = obs["farms"][player]\r\n    positions = [tuple(farm["farmer"])] + [tuple(p) for p in farm["hands"]]\r\n    inventories = obs["private"]["inventories"]\r\n    commands = [list(action.get("farmer") or ["PASS"])] + [list(c) for c in (action.get("hands") or [])]\r\n    changed = False\r\n    native = _IMPL.chassis.players.get(player)\r\n    tape = _IMPL.chassis.routes.get(native.get("route")) if native else None\r\n    planned = {}\r\n    if tape is not None:\r\n        # fertilizer this worke',
    b'r\'s own tape commands still spend today\r\n        for t in range(step, min(len(tape), day * 24 + 24)):\r\n            a = tape[t] or {}\r\n            for u, c in enumerate([a.get("farmer") or ["PASS"]] + list(a.get("hands") or [])):\r\n                if c and c[0] == "FERTILIZE":\r\n                    planned[u] = planned.get(u, 0) + 1\r\n    carried = {}\r\n    targeted = set()\r\n    for unit, command in enumerate(commands[:len(positions)]):\r\n        if not command or command[0] != "WATER":\r\n            continue\r\n        x, y = positions[unit]\r\n        tile = farm["tiles"][y][x]\r\n        if not (isinstance(tile, dict) and tile.get("kind") == "PLANT" and tile.get("crop") in V9_FERT_CROPS):\r\n            continue\r\n        if (day - int(tile["planted_day"])) not in V9_FERT_AGES or tile.get("watered_today"):\r\n            continue\r\n        if int(tile.get("consecutive_unwatered", 1)) != 0 or int(tile.get("fertilized_until_day", -1)) >= day:\r\n            continue\r\n        have = carried.setdefault(unit, int((inventories[unit]',
    b' if unit < len(inventories) else {}).get("FERTILIZER", 0))\r\n                                  - planned.get(unit, 0))\r\n        if have <= 0 or (x, y) in targeted:\r\n            continue\r\n        commands[unit] = ["FERTILIZE"]\r\n        carried[unit] = have - 1\r\n        targeted.add((x, y))\r\n        changed = True\r\n        _V9_FERT_REPORT["fert_applied"] += 1\r\n    if not changed:\r\n        return action\r\n    result = dict(action)\r\n    result["farmer"], result["hands"] = commands[0], commands[1:]\r\n    return result\r\n\r\n\r\n_V9_FERT_PARENT = agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    if int(observation["step"]) == 0:\r\n        _V9_FERT_REPORT.update(fert_applied=0, fert_errors=0)\r\n    action = _V9_FERT_PARENT(observation, configuration)\r\n    try:\r\n        return _v9_fert(observation, action)\r\n    except Exception:\r\n        _V9_FERT_REPORT["fert_errors"] += 1\r\n        return action\r\n\r\n\r\nagent.telemetry = _V9_FERT_REPORT\r\nagent = globals().pop("agent")\r\n\r\n\r\n# ---------------------------------------------',
    b'------------------------------\r\n# v9 OPENING: a cash-safe step-0 wheat trade.\r\n#\r\n# Every route tape opens with a wheat round trip\r\n#     step 0: BUY 13, BUY 30, SELL 30      step 1: SELL 13, BUY 5   (net +5)\r\n# and then runs days 0-9 with only ~$6 of cash slack (minimum at step 32).\r\n# The round trip wins cash from rivals whose own opening buys into it, but\r\n# against openings that dump wheat in the same slots (e.g. BUY 43 / SELL 20 /\r\n# SELL 22 or BUY 30 / SELL all) it ends step 1 up to $75 short; the day-1\r\n# wheat and hire orders then fail, the herd goes unfed, and by days 5-8 the\r\n# strawberry seed orders fail too (18-21 plants instead of 33, -20k..-65k).\r\n#\r\n# Replacing it with BUY 10, SELL 5 at step 0 (still net +5, nothing at\r\n# step 1) leaves >= $1,050 after step 1 against all 6,648 recorded openings\r\n# in the metav2 / M&M replays (exact market simulation, build/v9/opensim.py),\r\n# and $2 more than the round trip against the V38/V39 tape lineage.\r\n# ----------------------------------------------------',
    b'-----------------------\r\nV9_OPENING_STEP0 = (("BUY_PRODUCT", "WHEAT", 20), ("SELL", "WHEAT", 15))\r\nV9_OPENING_TAPE = ((("BUY_PRODUCT", "WHEAT", 13), ("BUY_PRODUCT", "WHEAT", 30), ("SELL", "WHEAT", 30)),\r\n                   (("SELL", "WHEAT", 13), ("BUY_PRODUCT", "WHEAT", 5)))\r\n\r\n\r\ndef _v9_opening(obs, action):\r\n    step = int(obs["step"])\r\n    if step > 1:\r\n        return action\r\n    native = _IMPL.chassis.players.get(int(obs["player"]))\r\n    if not native or native.get("route") not in _IMPL.chassis.routes:\r\n        return action\r\n    market = [list(o) for o in action.get("market") or []]\r\n    wheat = [o for o in market if len(o) >= 3 and o[0] in ("BUY_PRODUCT", "SELL") and o[1] == "WHEAT"]\r\n    if tuple((o[0], o[1], int(o[2])) for o in wheat) != V9_OPENING_TAPE[step]:\r\n        return action  # the tape\'s opening was changed upstream; leave it alone\r\n    rest = [o for o in market if o not in wheat]\r\n    result = dict(action)\r\n    result["market"] = ([list(o) for o in V9_OPENING_STEP0] if step == 0 else []) + ',
    b'rest\r\n    return result\r\n\r\n\r\n_V9_OPENING_PARENT = agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    action = _V9_OPENING_PARENT(observation, configuration)\r\n    try:\r\n        return _v9_opening(observation, action)\r\n    except Exception:\r\n        return action\r\n\r\n\r\nagent = globals().pop("agent")\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# v9 RACE: fit the sale-reservation horizon to the rival\'s observed sale lead.\r\n#\r\n# Premium books crash within ~60 units of glut, so the first seller of a lot\r\n# takes the price and the second sells into the crash.  The parent reserves\r\n# planned tape sales a fixed four turns ahead; any deeper fixed horizon beats\r\n# it head-to-head, but selling earlier than necessary gives away town-demand\r\n# recovery against a rival that does not race.\r\n#\r\n# Everything here is public.  Each turn the rival\'s executed sales are\r\n#\r\n#   rival_sold[p] = inventory\'[p] - inventory[p] + town_draw[p] - own_sold[p]\r\n#\r\n# (exact above the',
    b" $1 floor, where sales never enter inventory).  When the\r\n# rival sells a product while we still hold stock that our tape sells later,\r\n# and the sale is not a late fill of our previous lot, the turns until our next\r\n# planned sale are the rival's lead.  One horizon serves every product:\r\n#\r\n#   horizon = clamp(largest lead seen + RACE_MARGIN, RACE_DEFAULT, RACE_MAX)\r\n#\r\n# and it also lifts the parent's 72-turn block bound (patched into the\r\n# parent's reservation by the release builder).\r\n#\r\n# v9/2: the shipped horizon is 40 turns with a 12-turn margin (was 6 and 4).\r\n# Mirrors are the ladder's real opponent, and the premium books are first-come\r\n# races, so reserve depth is the whole decision.  Against the 6/4 build the new\r\n# setting wins 73-7 (+308) over 80 mirror games; against the shipped 32/12 build\r\n# it wins 74-6 (+358).  Deeper is not better: 44/12 beats 40 head to head but\r\n# drops the shallow-baseline record to 65-15, and 48/12 (a constant 48) falls to\r\n# 14-26 against it, because a horizon past t",
    b"he rival's next lot gives up\r\n# town-demand recovery for nothing.  The frozen top-30 stream panel is unchanged\r\n# inside its noise (52.2% / +2,797 over 178 games vs 52.8% / +2,929 at 32/12) and\r\n# the public field still goes 8-0.\r\n#\r\n# v9/2b: the reservation window starts at step 192 (day 8) instead of 288 (day 12),\r\n# a one-line change in the parent's `_r36_reserve` gate made by the release builder.\r\n# The herd's first milk and wool lots land on days 8-11 and the same-day sale\r\n# reservation is what takes their price before a rival's lot lands; the two middle\r\n# days of the tape's own schedule give that up.  Three fresh mirror blocks against\r\n# the step-288 build: 32-0 (+147), 30-2 (+11), 30-2 (+10); and against the shallow\r\n# 6/4 baseline the new build is if anything stronger, not weaker: 28-4 (+556) vs\r\n# 28-4 (+406), 29-3 (+272), 31-1 (+359) vs 31-1 (+349).  Per-item horizons instead of\r\n# one global horizon change nothing once the window starts at 192 (28 of 32 games\r\n# identical), so the one-line gate i",
    b's the whole change.\r\n# ---------------------------------------------------------------------------\r\nV9_RACE_DEFAULT = 44\r\nV9_RACE_MAX = 48\r\nV9_RACE_MARGIN = 12\r\nV9_RACE_GAP = 3            # a sale this soon after our previous planned lot is a late fill\r\nV9_RACE_WINDOW = 30        # turns of tape searched for planned sales around a rival sale\r\nV9_RACE_ITEMS = ("CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL")\r\n_V9_SHOP_ITEMS = {\r\n    "BAKERY": ("EGG", "WHEAT"), "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),\r\n    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"), "YARN_STORE": ("WOOL",),\r\n    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"), "PET_CAFE": ("CARROT",),\r\n    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),\r\n    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),\r\n}\r\n_V9_RACE = {}\r\n_V9_RACE_REPORT = dict(rival_sales=0, leads=0, race_errors=0)\r\n\r\n\r\ndef _v9_town_draw(shops, step):\r\n    """Units each shop instance and the town centre remove after `step`\'s market."""\r\n    draw = dict.fromkey',
    b's(V9_RACE_ITEMS, 0)\r\n    if step % 4 == 0:\r\n        for shop in shops:\r\n            items = _V9_SHOP_ITEMS.get(shop, ())\r\n            for item in items:\r\n                if item in draw:\r\n                    draw[item] += 2 if len(items) == 1 else 1\r\n    if step % 24 == 0:\r\n        for item in draw:\r\n            draw[item] += 1\r\n    return draw\r\n\r\n\r\ndef _v9_planned_sells(tape, item, step):\r\n    """Turns within V9_RACE_WINDOW of `step` at which the tape sells `item`."""\r\n    out = []\r\n    for t in range(max(0, step - V9_RACE_WINDOW), min(len(tape), step + V9_RACE_WINDOW + 1)):\r\n        if any(o and o[0] == "SELL" and len(o) >= 3 and o[1] == item and int(o[2]) > 0\r\n               for o in tape[t].get("market") or []):\r\n            out.append(t)\r\n    return out\r\n\r\n\r\ndef _v9_race_update(obs, st):\r\n    """Recover last turn\'s rival sales and record the leads that prove racing."""\r\n    prev = st["prev"]\r\n    step = int(obs["step"])\r\n    if not prev or prev["step"] != step - 1:\r\n        return\r\n    native = _IMPL.cha',
    b'ssis.players.get(int(obs["player"]))\r\n    if not native or native.get("route") not in _IMPL.chassis.routes:\r\n        return\r\n    tape = _IMPL.chassis.routes[native["route"]]\r\n    inventory = obs["market"]["inventory"]\r\n    draw = _v9_town_draw(prev["shops"], prev["step"])\r\n    t = prev["step"]\r\n    for item in V9_RACE_ITEMS:\r\n        if prev["prices"].get(item, 0) <= 3:\r\n            continue\r\n        sold = inventory[item] - prev["inventory"][item] + draw[item] - prev["own"].get(item, 0)\r\n        if sold < 2:\r\n            continue\r\n        _V9_RACE_REPORT["rival_sales"] += 1\r\n        if prev["left"].get(item, 0) <= 0:\r\n            continue\r\n        planned = _v9_planned_sells(tape, item, t)\r\n        after = [s for s in planned if s >= t]\r\n        before = [s for s in planned if s < t]\r\n        if not after or (before and t - before[-1] < V9_RACE_GAP):\r\n            continue\r\n        st["lead"] = max(st["lead"], after[0] - t)\r\n        _V9_RACE_REPORT["leads"] += 1\r\n\r\n\r\n_V9_RACE_PARENT = agent\r\n\r\n\r\ndef agent(obs',
    b'ervation, configuration=None):\r\n    player, step = int(observation["player"]), int(observation["step"])\r\n    st = _V9_RACE.get(player)\r\n    if st is None or step <= st["step"]:\r\n        st = _V9_RACE[player] = {"step": -1, "lead": -V9_RACE_MARGIN, "prev": None}\r\n        if step == 0:\r\n            _V9_RACE_REPORT.update(rival_sales=0, leads=0, race_errors=0)\r\n    st["step"] = step\r\n    try:\r\n        _v9_race_update(observation, st)\r\n        horizon = min(V9_RACE_MAX, max(V9_RACE_DEFAULT, st["lead"] + V9_RACE_MARGIN))\r\n        _V9_ITEM_HZ[player] = dict.fromkeys(V9_RACE_ITEMS, horizon)\r\n    except Exception:\r\n        _V9_RACE_REPORT["race_errors"] += 1\r\n        _V9_ITEM_HZ.pop(player, None)\r\n    action = _V9_RACE_PARENT(observation, configuration)\r\n    try:\r\n        stock = projected_shed(action, FarmView(observation))\r\n        own = {}\r\n        for o in action.get("market") or []:\r\n            if o and o[0] == "SELL" and len(o) >= 3 and o[1] in V9_RACE_ITEMS:\r\n                n = min(max(0, int(o[2])), max(0, ',
    b'stock.get(o[1], 0) - own.get(o[1], 0)))\r\n                own[o[1]] = own.get(o[1], 0) + n\r\n        st["prev"] = {"step": step, "inventory": dict(observation["market"]["inventory"]),\r\n                      "prices": dict(observation["market"]["prices"]), "own": own,\r\n                      "left": {item: stock.get(item, 0) - own.get(item, 0) for item in V9_RACE_ITEMS},\r\n                      "shops": list(observation["town"]["unlocked_shops"])}\r\n    except Exception:\r\n        _V9_RACE_REPORT["race_errors"] += 1\r\n        st["prev"] = None\r\n    return action\r\n\r\n\r\nagent.telemetry = _V9_RACE_REPORT\r\nagent = globals().pop("agent")\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# v9 RACEPX: lead-sell a planned lot early only while its book is not glutted.\r\n#\r\n# The engine prices a product off the market inventory: below I0 the quote is above\r\n# base, above I0 it is below.  The lead sale moves a lot one turn ahead of the tape\'s\r\n# own plan, which is what takes the price when both s',
    b"ides hold the same lot -- but\r\n# when the book is already above I0 the same units only fetch a lower price, and the\r\n# town drain between the two turns is not enough to pay for it.\r\n#\r\n# Measured against the frozen top-20 streams the shipped build realizes below-base\r\n# prices on exactly the products it floods (strawberry $107-123 vs a $120 base, milk\r\n# $98-107 vs $160, wool $129-139 vs $200, melon $212 vs $250, fertilizer $43 vs $100)\r\n# and above base on the ones the town drains (wheat $38 vs $25, carrot $54 vs $35,\r\n# tomato $125 vs $60, egg $52 vs $50).\r\n#\r\n# This layer keeps the lead sale for products quoted at or above base + MARGIN and\r\n# leaves the rest to the tape's own schedule.  The reservation (`_r36_reserve`) is\r\n# untouched, so the RACE horizon still governs how far ahead lots may be pulled.\r\n#\r\n# Evidence (v9/3): mirror against the shipped build 35-13 (+220) over 48 fresh games;\r\n# against the 6/4 ancestor 41-7 (+620) where the shipped build is 45-3 (+449); frozen\r\n# top-20 panel 69/120 (+3,58",
    b'6) against 66/120 (+3,364); second panel 63/120 (+2,264)\r\n# against 63/120 (+1,882); public field pool unchanged (14-2 vs cdb and fh11, 16-0\r\n# vs fa103/fa141/fa238/wd14).\r\n# ---------------------------------------------------------------------------\r\nV9_RACEPX_BASE = {"WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120, "MELON": 250,\r\n                  "EGG": 50, "MILK": 160, "WOOL": 200, "FERTILIZER": 100}\r\nV9_RACEPX_MARGIN = 0\r\n_v9_racepx_report = dict(racepx_skipped=0, racepx_sold=0, racepx_errors=0)\r\n_V9_RACEPX_LEAD = Chassis._sell_lead\r\n\r\n\r\ndef _v9_racepx_lead(self, action, view, projected, route, step, next_sup):\r\n    prices = view.prices\r\n    blocked = {i for i in V9_RACEPX_BASE\r\n               if prices.get(i, 0) <= V9_RACEPX_BASE[i] + V9_RACEPX_MARGIN}\r\n    if not blocked:\r\n        _v9_racepx_report["racepx_sold"] += 1\r\n        return _V9_RACEPX_LEAD(self, action, view, projected, route, step, next_sup)\r\n    nxt = step + 1\r\n    unlock_period = 3 * self.cfg["turns_per_day"]\r\n    if nxt > LAST_',
    b'ACT_STEP or nxt % unlock_period == 0 or step % 4 == 0:\r\n        return\r\n    tape = self.routes[route]\r\n    future = tape[nxt] if nxt < len(tape) and isinstance(tape[nxt], dict) else {}\r\n    planned = {}\r\n    for o in future.get("market") or []:\r\n        if o and o[0] == "SELL" and len(o) >= 3 and o[1] in PRODUCTS:\r\n            planned[o[1]] = planned.get(o[1], 0) + max(0, int(o[2]))\r\n    already = {o[1] for o in action.get("market") or [] if o and o[0] == "SELL" and len(o) > 1}\r\n    for item in PRODUCTS:\r\n        if item in blocked or item in already or planned.get(item, 0) <= 0:\r\n            continue\r\n        qty = min(projected.get(item, 0), planned[item])\r\n        if qty <= 0 or prices.get(item, 0) < self.cfg["min_sell_price"]:\r\n            continue\r\n        if not self._add_sell(action, item, qty, self.cfg["max_orders"], merge=False):\r\n            break\r\n        projected[item] -= qty\r\n        next_sup["suppress"][item] = next_sup["suppress"].get(item, 0) + qty\r\n        _v9_racepx_report["racepx_skipped"]',
    b' += 1\r\n    if next_sup["suppress"]:\r\n        next_sup["due_step"] = nxt\r\n\r\n\r\n_V9_RACEPX_PARENT = agent\r\nChassis._sell_lead = _v9_racepx_lead\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    if int(observation["step"]) == 0:\r\n        _v9_racepx_report.update(racepx_skipped=0, racepx_sold=0, racepx_errors=0)\r\n    try:\r\n        return _V9_RACEPX_PARENT(observation, configuration)\r\n    except Exception:\r\n        _v9_racepx_report["racepx_errors"] += 1\r\n        return {"farmer": ["PASS"], "hands": [], "market": []}\r\n\r\n\r\nagent.telemetry = _v9_racepx_report\r\nagent = globals().pop("agent")\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# v9 RACEGATE: the same glut gate for the *reservation* (the 40-turn pull-forward).\r\n#\r\n# `_r36_reserve` moves tape-planned sales up to the RACE horizon forward, item by\r\n# item.  RACEPX only gated the one-turn lead sale; this layer gates the reservation\r\n# too: an item whose quote is at or below its base price is left to the tape\'s own\r\n# sche',
    b'dule.  The debt ledger is untouched for the items that are still reserved, so\r\n# suppression stays consistent.\r\n#\r\n# Evidence: see README (v9/3 round).\r\n# ---------------------------------------------------------------------------\r\nV9_RACEGATE_BASE = {"WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120, "MELON": 250,\r\n                    "EGG": 50, "MILK": 160, "WOOL": 200, "FERTILIZER": 100}\r\nV9_RACEGATE_MARGIN = 0\r\n_v9_racegate_report = dict(racegate_reserved_units=0, racegate_errors=0)\r\n_V9_RACEGATE_RESERVE = _r36_reserve\r\n\r\n\r\ndef _r36_reserve(obs, action):\r\n    step = int(obs["step"])\r\n    if not 192 <= step < 696:\r\n        return action\r\n    prices = obs["market"]["prices"]\r\n    glutted = {i for i in V9_RACEGATE_BASE\r\n               if prices.get(i, 0) <= V9_RACEGATE_BASE[i] + V9_RACEGATE_MARGIN}\r\n    if not glutted:\r\n        return _V9_RACEGATE_RESERVE(obs, action)\r\n    native = _IMPL.chassis.players[int(obs["player"])]\r\n    tape = _IMPL.chassis.routes[native["route"]]\r\n    _v9_hz = _V9_ITEM_HZ.g',
    b'et(int(obs["player"]))\r\n    end = min(695, step + (max(_v9_hz.values()) if _v9_hz else _R37_HORIZONS.get(int(obs["player"]), 2)))\r\n    if end <= step:\r\n        return action\r\n    commands = [action.get("farmer") or ["PASS"], *(action.get("hands") or [])]\r\n    view = FarmView(obs)\r\n    if any(len(c) > 1 and c[0] == "PLACE" and c[1] in ANIMAL_STRUCTURE\r\n           and view.inv(i).get(c[1], 0) > 0 for i, c in enumerate(commands[:len(view.positions)])):\r\n        return action\r\n    stock = projected_shed(action, view)\r\n    market = action.get("market", [])\r\n    blocked = {o[1] for o in market if len(o) > 1 and o[0] in ("SELL", "BUY_PRODUCT")}\r\n    blocked.update(c[1] for c in commands if len(c) > 1 and c[0] == "PICKUP")\r\n    blocked.update(c[1] for queue in native["pending"].values() for pos, c in queue\r\n                   if len(c) > 1 and c[0] == "PICKUP")\r\n    debts = native["sell_state"].setdefault("r36_debts", {})\r\n    for item in PRODUCTS:\r\n        if item in blocked or item in glutted or view.prices.get(ite',
    b'm, 0) < 2:\r\n            continue\r\n        available = max(0, int(stock.get(item, 0)))\r\n        if not available or len(market) >= 10:\r\n            continue\r\n        reservations = []\r\n        item_end = min(end, step + _v9_hz[item]) if _v9_hz and item in _v9_hz else end\r\n        for due_step in range(step + 1, item_end + 1):\r\n            future = tape[due_step]\r\n            work = [future.get("farmer") or ["PASS"], *(future.get("hands") or [])]\r\n            if any(len(c) > 1 and c[:2] == ["PICKUP", item] for c in work):\r\n                break\r\n            if any(len(o) > 1 and o[:2] == ["BUY_PRODUCT", item] for o in future.get("market", [])):\r\n                break\r\n            planned = sum(max(0, int(o[2])) for o in future.get("market", [])\r\n                          if len(o) >= 3 and o[:2] == ["SELL", item])\r\n            amount = min(available, max(0, planned - debts.get(due_step, {}).get(item, 0)))\r\n            if amount:\r\n                reservations.append((due_step, amount))\r\n                available',
    b' -= amount\r\n            if not available:\r\n                break\r\n        qty = sum(q for _, q in reservations)\r\n        if qty:\r\n            market.append(["SELL", item, qty])\r\n            for due, q in reservations:\r\n                debt = debts.setdefault(due, {})\r\n                debt[item] = debt.get(item, 0) + q\r\n            _R36_SALE_REPORT["sale_reserved_units"] += qty\r\n            _R36_SALE_REPORT["sale_reservations"] += 1\r\n            _v9_racegate_report["racegate_reserved_units"] += qty\r\n    return action\r\n\r\n\r\n_V9_RACEGATE_PARENT = agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    if int(observation["step"]) == 0:\r\n        _v9_racegate_report.update(racegate_reserved_units=0, racegate_errors=0)\r\n    try:\r\n        return _V9_RACEGATE_PARENT(observation, configuration)\r\n    except Exception:\r\n        _v9_racegate_report["racegate_errors"] += 1\r\n        return {"farmer": ["PASS"], "hands": [], "market": []}\r\n\r\n\r\nagent.telemetry = _v9_racegate_report\r\nagent = globals().pop("agent")\r\n\r\n\r\n# ===',
    b'= layer ctrtable.py \r\n_P_ctrtable_336053 = agent\r\n\r\n# ---------------------------------------------------------------------------\r\n# v9/3 CTRTABLE: rival-specific early wheat counters.\r\n# Under the BUY 20 | SELL 15 opening, a rival tape\'s turn-2 observation (rival money, market wheat)\r\n# identifies it exactly.  For known cash-tight tapes we trade alongside their own early wheat orders\r\n# in the same market slots (checked unique over 4,604 recorded games):\r\n#   feel the agi (979.0, 9989): steps 7-8  BUY 5 slot 0, SELL 5 slot 1\r\n#   Mother-Goose (33.0, 9990): step 3      BUY 20 slot 0, SELL 20 slot 1\r\n# ---------------------------------------------------------------------------\r\nCT_TABLE = {\r\n    (979.0, 9989): ((7, 0, ("BUY_PRODUCT", "WHEAT", 5)), (7, 1, ("SELL", "WHEAT", 5)),\r\n                    (8, 0, ("BUY_PRODUCT", "WHEAT", 5)), (8, 1, ("SELL", "WHEAT", 5))),\r\n    (33.0, 9990): ((3, 0, ("BUY_PRODUCT", "WHEAT", 20)), (3, 1, ("SELL", "WHEAT", 20))),\r\n}\r\n_CT = {}\r\n_CT_REPORT = dict(ct_fired=0, ct_errors=0)\r\n',
    b'\r\n\r\ndef _ct_apply(obs, action, st):\r\n    step = int(obs["step"])\r\n    if step == 2:\r\n        rival = obs["farms"][1 - int(obs["player"])]\r\n        st["plan"] = CT_TABLE.get((round(float(rival["money"]), 3), int(obs["market"]["inventory"]["WHEAT"])))\r\n        if st["plan"]:\r\n            _CT_REPORT["ct_fired"] += 1\r\n    plan = st.get("plan")\r\n    if not plan:\r\n        return action\r\n    items = sorted((slot, list(o)) for t, slot, o in plan if t == step)\r\n    if not items:\r\n        return action\r\n    market = [list(o) for o in action.get("market") or []]\r\n    for slot, o in items:\r\n        while len(market) < slot:\r\n            market.append(["SELL", "WHEAT", 0])\r\n        market.insert(slot, o)\r\n    result = dict(action)\r\n    result["market"] = market[:MAX_ORDERS]\r\n    return result\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    player, step = int(observation["player"]), int(observation["step"])\r\n    st = _CT.get(player)\r\n    if st is None or step <= st["step"]:\r\n        st = _CT[player] = {"step": -1}\r\n ',
    b'   st["step"] = step\r\n    action = _P_ctrtable_336053(observation, configuration)\r\n    try:\r\n        return _ct_apply(observation, action, st)\r\n    except Exception:\r\n        _CT_REPORT["ct_errors"] += 1\r\n        return action\r\n\r\n\r\nagent.telemetry = _CT_REPORT\r\nagent = globals().pop("agent")\r\n\r\n\r\n# ==== layer overflow.py \r\n_P_overflow_338343 = agent\r\n\r\n# ---------------------------------------------------------------------------\r\n# v9/3 OVERFLOW (ported from public V43 R148, Ahmed Berat Ozer, Apache-2.0):\r\n# at hour 23 the workers\' cargo drops into a 100-unit shed; cargo that does not fit\r\n# is destroyed.  Sell exactly the shed stock that the destroyed cargo would replace,\r\n# so the complete post-dawn stock vector is unchanged and the sold units are extra.\r\n# ---------------------------------------------------------------------------\r\n_OV_REPORT = dict(ov_turns=0, ov_units=0, ov_errors=0)\r\n\r\n\r\ndef _ov_fields(obs, action):\r\n    farm, private = _PLANNER_NS[\'_clone_state\'](obs[\'farms\'][obs[\'player\']], obs[\'priva',
    b"te'])\r\n    commands = [action.get('farmer') or ['PASS'], *(action.get('hands') or [])]\r\n    demand = {}\r\n    for c in commands:\r\n        if len(c) > 1 and c[0] == 'PLANT':\r\n            demand[c[1]] = demand.get(c[1], 0) + 1\r\n    blocked = {p for p, n in demand.items() if n > private['seeds'].get(p, 0)}\r\n    for actor, c in enumerate(commands[:len(private['inventories'])]):\r\n        if len(c) > 1 and c[0] == 'PLANT' and c[1] in blocked:\r\n            continue\r\n        _PLANNER_NS['_apply_unit_action'](farm, private, actor, c, 10, int(obs['step']) // 24, 24, 100)\r\n    return farm, private\r\n\r\n\r\ndef _ov_same(a, b):\r\n    return all(int(a.get(p, 0)) == int(b.get(p, 0)) for p in set(a) | set(b))\r\n\r\n\r\ndef _ov_apply(obs, action):\r\n    if int(obs['step']) % 24 != 23:\r\n        return action\r\n    orders = action.get('market') or []\r\n    if len(orders) >= 10 or not _r97_budget(obs, orders):\r\n        return action\r\n    _, private = _ov_fields(obs, action)\r\n    stock, _, _ = _r97_market_stock(private['shed'], orders)\r\n    or",
    b"iginal, loss = _r97_delivery(stock, private, True)\r\n    if not loss:\r\n        return action\r\n    remaining = max(0, 100 - sum(stock.values()))\r\n    tail = []\r\n    for bag in private['inventories']:\r\n        for item, n in bag.items():\r\n            n = max(0, int(n)); take = min(n, remaining); remaining -= take\r\n            if n > take:\r\n                tail.extend([item] * (n - take))\r\n    released = {}; best = None\r\n    for item in tail:\r\n        released[item] = released.get(item, 0) + 1\r\n        if item not in obs['market']['prices'] or released[item] > stock.get(item, 0):\r\n            break\r\n        if len(orders) + len(released) > 10:\r\n            break\r\n        proposed = list(orders) + [['SELL', p, n] for p, n in released.items()]\r\n        after, _, _ = _r97_market_stock(private['shed'], proposed)\r\n        final, _ = _r97_delivery(after, private, True)\r\n        if _ov_same(original, final):\r\n            best = (proposed, dict(released))\r\n    if best is None:\r\n        return action\r\n    _OV_REPORT['ov_t",
    b'urns\'] += 1\r\n    _OV_REPORT[\'ov_units\'] += sum(best[1].values())\r\n    return dict(action, market=best[0])\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    action = _P_overflow_338343(observation, configuration)\r\n    try:\r\n        return _ov_apply(observation, action)\r\n    except Exception:\r\n        _OV_REPORT[\'ov_errors\'] += 1\r\n        return action\r\n\r\n\r\nagent.telemetry = _OV_REPORT\r\nagent = globals().pop("agent")\r\n\r\n\r\n# One-worker non-harvest service for the inherited six-sheep project.\r\n# All feeding and care are mandatory. Optional fertilizer collection only uses\r\n# leftover time; setup, wool harvests, and late-start days keep the old crew.\r\n_SL_REQUEST = _v233_request\r\n_SL_WORKER = _v233_worker\r\n_SL_REPORT = dict(compact_days=0, confirmed=0, collect=0)\r\n_SL_TILES = ((5,5),(6,5),(7,5),(7,6),(6,6),(5,6))\r\n\r\n\r\ndef _sl_dist(a,b):\r\n    return abs(a[0]-b[0])+abs(a[1]-b[1])\r\n\r\n\r\ndef _sl_path(pos,targets):\r\n    return min(_r53_permutations(targets),key=lambda path:(_sl_dist(pos,path[0])+sum(_sl_dist(a,b) for',
    b" a,b in zip(path,path[1:])),path)) if targets else ()\r\n\r\n\r\ndef _v233_request(obs,action,state,native):\r\n    result=_SL_REQUEST(obs,action,state,native)\r\n    pending=state.get('pending')\r\n    if result is action or not pending or pending['initial']:\r\n        return result\r\n    farm=obs['farms'][obs['player']];hour=int(obs['step'])%24\r\n    tiles=[farm['tiles'][y][x] for x,y in _SL_TILES]\r\n    if not all(isinstance(t,dict) and t.get('animal')=='SHEEP' and t.get('yield_units',0)==0 for t in tiles):\r\n        return result\r\n    # Exact spawn after native commands/hires, and a full feed pickup turn.\r\n    ready,spawn=_r62_input_start(obs,action,0)\r\n    path=_sl_path(spawn,_SL_TILES)\r\n    travel=_sl_dist(spawn,path[0])+sum(_sl_dist(a,b) for a,b in zip(path,path[1:]))\r\n    mandatory=sum(not t.get('fed_today') for t in tiles)+sum(not t.get('cared_today') for t in tiles)\r\n    available=min((int(obs['step'])//24+1)*24,719)-ready\r\n    if travel+mandatory>available:\r\n        return result\r\n    # Collection can be interleave",
    b"d along the essential route. Charge the\r\n    # discarded fertilizer against the saved wage, including final delivery.\r\n    native_hires=sum(o and o[0]=='HIRE' for o in action.get('market',[]))\r\n    saved=_v219_fib(farm['hires_today']+native_hires+1)\r\n    delivery=_sl_dist(path[-1],_v219_home(path[-1]))+1 if int(obs['step'])//24==29 else 0\r\n    possible=max(0,min(6,available-travel-mandatory-delivery))\r\n    if saved<=(6-possible)*obs['market']['prices']['FERTILIZER']:\r\n        return result\r\n    out=copy.deepcopy(result)\r\n    assert out['market'][-2:]==[['HIRE'],['HIRE']]\r\n    out['market'].pop()\r\n    pending.update(count=1,targets=path)\r\n    _V233_REPORT['sheep_hire_requests']-=1\r\n    _SL_REPORT['compact_days']+=1\r\n    return out\r\n\r\n\r\ndef _v233_worker(obs,actor,targets):\r\n    if len(targets)!=6:\r\n        return _SL_WORKER(obs,actor,targets)\r\n    farm=obs['farms'][obs['player']];private=obs['private'];step=int(obs['step'])\r\n    pos=tuple(farm['hands'][actor-1]);inv=private['inventories'][actor]\r\n    needed=[];",
    b"hungry=0\r\n    for target in targets:\r\n        x,y=target;t=farm['tiles'][y][x]\r\n        if not isinstance(t,dict) or t.get('animal')!='SHEEP':\r\n            return _SL_WORKER(obs,actor,targets)\r\n        if not t['fed_today']:hungry+=1\r\n        if not t['fed_today'] or not t['cared_today']:needed.append(target)\r\n    home=_v219_home(pos)\r\n    if hungry>inv.get('WHEAT',0):\r\n        return _v219_walk(pos,home) or ['PICKUP','WHEAT',min(hungry,private['shed'].get('WHEAT',0))]\r\n    if needed:\r\n        path=_sl_path(pos,needed)\r\n        current=farm['tiles'][pos[1]][pos[0]]\r\n        if pos in targets and isinstance(current,dict) and current.get('fed_today') and current.get('cared_today') and current.get('fertilizer_available'):\r\n            travel=_sl_dist(pos,path[0])+sum(_sl_dist(a,b) for a,b in zip(path,path[1:]))\r\n            work=sum(not farm['tiles'][y][x]['fed_today'] for x,y in path)+sum(not farm['tiles'][y][x]['cared_today'] for x,y in path)\r\n            delivery=_sl_dist(path[-1],_v219_home(path[-1]))+1 if s",
    b"tep//24==29 else 0\r\n            remaining=min((step//24+1)*24,719)-step\r\n            if 1+travel+work+delivery<=remaining:\r\n                _SL_REPORT['collect']+=1\r\n                return ['COLLECT_FERTILIZER']\r\n        target=path[0];t=farm['tiles'][target[1]][target[0]]\r\n        return _v219_walk(pos,target) or (['FEED'] if not t['fed_today'] else ['CARE'])\r\n    # Essential work is complete. Collect what can still reach the shed on\r\n    # the final day; on earlier days the normal midnight deposit is sufficient.\r\n    remaining=719-step if step//24==29 else 24-step%24\r\n    tasks=[]\r\n    for target in targets:\r\n        t=farm['tiles'][target[1]][target[0]]\r\n        if not t.get('fertilizer_available'):continue\r\n        dist=_sl_dist(pos,target)\r\n        ret=_sl_dist(target,_v219_home(target))+1 if step//24==29 else 0\r\n        if dist+1+ret<=remaining:tasks.append((dist,tuple(target)))\r\n    if tasks:\r\n        _,target=min(tasks)\r\n        command=_v219_walk(pos,target) or ['COLLECT_FERTILIZER']\r\n        _SL_REP",
    b"ORT['collect']+=command==['COLLECT_FERTILIZER']\r\n        return command\r\n    if inv.get('FERTILIZER',0):\r\n        return _v219_walk(pos,home) or ['PLACE','FERTILIZER',inv['FERTILIZER']]\r\n    return ['PASS']\r\n\r\n\r\nagent=globals().pop('agent')\r\n\r\n\r\n# v9/4 VE: commit the six-sheep expansion on day 11 when two of the first three\r\n# shops are yarn stores. The melon sale lands on day 11, and sheep placed that\r\n# day produce on days 17, 20, 23, 26 and 29 instead of 18, 21, 24 and 27: a\r\n# fifth wool harvest for one more day of feed and labour.\r\n# Day 11 waits until the tape's own land purchase (hour 1) and only ignores a\r\n# native sheep that the tape itself picks up later today (units act before the\r\n# market, and the project's hands spawn a step after its purchase). It commits\r\n# only when cash also covers every purchase the tape still plans through day 12\r\n# (its day-11 strawberry seeds above all); otherwise day 12 decides as before.\r\n_VE_ELIGIBLE = _v233_eligible\r\n_VE_REPORT = dict(ve_day11_checks=0, ve_day11_budg",
    b"et_declines=0, ve_day11_ok=0)\r\n_VE_SEED = {'WHEAT': 10, 'CARROT': 20, 'TOMATO': 50, 'STRAWBERRY': 100, 'MELON': 80}\r\n_VE_ANIMAL = {'SHEEP': 500, 'COW': 400, 'GOOSE': 300}\r\n\r\n\r\ndef _v233_eligible(obs, native):\r\n    step = int(obs['step'])\r\n    if step // 24 != 11:\r\n        return _VE_ELIGIBLE(obs, native)\r\n    tape = _IMPL.chassis.routes[native['route']]\r\n    private = obs['private']\r\n    held = int(private['shed'].get('SHEEP', 0)) + sum(int(i.get('SHEEP', 0)) for i in private['inventories'])\r\n    pickups = 0\r\n    for t in range(step, 12 * 24):\r\n        for c in [tape[t].get('farmer')] + tape[t].get('hands', []):\r\n            if c and c[0] == 'PICKUP' and len(c) > 1 and c[1] == 'SHEEP':\r\n                pickups += int(c[2]) if len(c) > 2 else 1\r\n    if held > pickups:\r\n        return False\r\n    clean = dict(private, shed=dict(private['shed'], SHEEP=0),\r\n                 inventories=[{k: v for k, v in i.items() if k != 'SHEEP'} for i in private['inventories']])\r\n    if not _VE_ELIGIBLE(dict(obs, private=clean),",
    b" native):\r\n        return False\r\n    _VE_REPORT['ve_day11_checks'] += 1\r\n    prices = obs['market']['prices']\r\n    spend = 0\r\n    hires = {}\r\n    for t in range(step + 1, min(len(tape), 13 * 24)):\r\n        for o in tape[t].get('market', []):\r\n            if not o:\r\n                continue\r\n            if o[0] == 'BUY_LAND' or o[:2] == ['BUY_ANIMAL', 'SHEEP']:\r\n                return False\r\n            if o[0] == 'BUY_SEED':\r\n                spend += int(o[2]) * _VE_SEED.get(o[1], 100)\r\n            elif o[0] == 'BUY_PRODUCT':\r\n                spend += int(o[2]) * (int(prices.get(o[1], 50)) + 10)\r\n            elif o[0] == 'BUY_ANIMAL':\r\n                spend += int(o[2]) * _VE_ANIMAL.get(o[1], 500)\r\n            elif o[0] == 'HIRE':\r\n                d = t // 24\r\n                spend += _v219_fib(hires.get(d, 0))\r\n                hires[d] = hires.get(d, 0) + 1\r\n    if obs['farms'][obs['player']]['money'] < 7000 + 3000 + spend:\r\n        _VE_REPORT['ve_day11_budget_declines'] += 1\r\n        return False\r\n    _VE_R",
    b"EPORT['ve_day11_ok'] += 1\r\n    return True\r\n\r\n\r\nagent.telemetry = _VE_REPORT\r\nagent = globals().pop('agent')\r\n\r\n\r\n# v9/4 VT: the sheep expansion's last two days.\r\n# No refresh follows day 29, so feeding, caring and buying feed that day are\r\n# worthless: only wool already grown is worth a hand. Day 29 hires nothing when\r\n# no sheep holds wool, and otherwise one harvest-only hand that delivers the\r\n# wool before the final market. A care on day 28 adds to the bonus after the\r\n# last production refresh has already consumed it, so day-28 hands skip CARE.\r\n_VT_REQUEST = _v233_request\r\n_VT_WORKER = _v233_worker\r\n_VT_RESCUE = _v234_rescue\r\n_VT_REPORT = dict(vt_no_hire_days=0, vt_single_harvest_days=0, vt_skipped_care=0)\r\n_VT_TILES = ((5, 5), (6, 5), (7, 5), (5, 6), (6, 6), (7, 6))\r\n\r\n\r\ndef _vt_wool_tiles(obs):\r\n    farm = obs['farms'][obs['player']]\r\n    return [xy for xy in _VT_TILES if isinstance(farm['tiles'][xy[1]][xy[0]], dict)\r\n            and farm['tiles'][xy[1]][xy[0]].get('animal') == 'SHEEP' and farm['tiles",
    b"'][xy[1]][xy[0]].get('yield_units', 0) > 0]\r\n\r\n\r\ndef _v233_request(obs, action, state, native):\r\n    result = _VT_REQUEST(obs, action, state, native)\r\n    if result is action or int(obs['step']) // 24 != 29 or not state.get('committed'):\r\n        return result\r\n    pending = state.get('pending')\r\n    if not pending or pending.get('initial'):\r\n        return result\r\n    wool = _vt_wool_tiles(obs)\r\n    extra = result['market'][len(action.get('market', [])):]\r\n    if not wool:\r\n        state.pop('pending', None)\r\n        _V233_REPORT['sheep_hire_requests'] -= sum(o == ['HIRE'] for o in extra)\r\n        _V233_REPORT['sheep_feed_buy_requests'] -= 6\r\n        _VT_REPORT['vt_no_hire_days'] += 1\r\n        return action\r\n    out = copy.deepcopy(action)\r\n    out['market'] = list(out.get('market', [])) + [['HIRE']]\r\n    _V233_REPORT['sheep_hire_requests'] -= sum(o == ['HIRE'] for o in extra) - 1\r\n    _V233_REPORT['sheep_feed_buy_requests'] -= 6\r\n    pending.update(count=1, targets=_sl_path(_r62_input_start(obs, action, 0)[",
    b"1], wool))\r\n    _VT_REPORT['vt_single_harvest_days'] += 1\r\n    return out\r\n\r\n\r\ndef _v233_worker(obs, actor, targets):\r\n    step = int(obs['step'])\r\n    day = step // 24\r\n    if day not in (28, 29):\r\n        return _VT_WORKER(obs, actor, targets)\r\n    farm = obs['farms'][obs['player']]\r\n    private = obs['private']\r\n    pos = tuple(farm['hands'][actor - 1])\r\n    inv = private['inventories'][actor]\r\n    home = _v219_home(pos)\r\n    distance = abs(pos[0] - home[0]) + abs(pos[1] - home[1])\r\n    cargo = [item for item in ('WOOL', 'FERTILIZER') if inv.get(item, 0)]\r\n    last = 717 if day == 29 else day * 24 + 23\r\n    if cargo and step >= last - distance:\r\n        return _v219_walk(pos, home) or ['PLACE', cargo[0], inv[cargo[0]]]\r\n    sheep = [(x, y) for x, y in targets if isinstance(farm['tiles'][y][x], dict) and farm['tiles'][y][x].get('animal') == 'SHEEP']\r\n    tasks = []\r\n    if day == 28:\r\n        hungry = sum(not farm['tiles'][y][x]['fed_today'] for x, y in sheep)\r\n        if hungry and not inv.get('WHEAT', 0) ",
    b"and private['shed'].get('WHEAT', 0):\r\n            return _v219_walk(pos, home) or ['PICKUP', 'WHEAT', min(hungry, private['shed']['WHEAT'])]\r\n    for x, y in sheep:\r\n        tile = farm['tiles'][y][x]\r\n        command = None\r\n        if day == 28 and not tile['fed_today'] and inv.get('WHEAT', 0):\r\n            command = ['FEED']\r\n        elif tile['yield_units']:\r\n            command = ['HARVEST']\r\n        elif day == 28 and tile['fertilizer_available']:\r\n            command = ['COLLECT_FERTILIZER']\r\n        if day == 28 and not tile['cared_today'] and command is None:\r\n            _VT_REPORT['vt_skipped_care'] += 1\r\n        if command:\r\n            tasks.append((abs(pos[0] - x) + abs(pos[1] - y), (x, y), command))\r\n    if tasks:\r\n        _, target, command = min(tasks)\r\n        return _v219_walk(pos, target) or command\r\n    if cargo:\r\n        return _v219_walk(pos, home) or ['PLACE', cargo[0], inv[cargo[0]]]\r\n    return ['PASS']\r\n\r\n\r\ndef _v234_rescue(obs, action, state):\r\n    if int(obs['step']) // 24 == 29:\r",
    b'\n        return action\r\n    return _VT_RESCUE(obs, action, state)\r\n\r\n\r\nagent.telemetry = _VT_REPORT\r\nagent = globals().pop(\'agent\')\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# Claude CARROT2 layer: carrot instead of wheat when the carrot book pays. Own implementation.\r\n# Built by tools/claude_build_carrot.py (derivation there).\r\n# ---------------------------------------------------------------------------\r\n_CA_FROM = 10\r\n_CA_TO = 28\r\n_CA_MARGIN = -20.0\r\n_CA_DROP = 0.0\r\n_CA_BUFFER = 8\r\n_CA_FEED_DAYS = 1\r\n_CA_CASH = 800\r\n_CA_RESCUE = True\r\n_CA_MOVES = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}\r\n_CA_CROP = {"WHEAT": (4, 6), "CARROT": (3, 4)}\r\n_CA_STATE = {}\r\n_CA_REPORT = {"ca_swaps": 0, "ca_rescues": 0, "ca_harvested": 0, "ca_sold": 0, "ca_seed_bought": 0,\r\n              "ca_wheat_seed_saved": 0, "ca_carrot_seed_saved": 0, "ca_feed_block": 0, "ca_errors": 0,\r\n              "ca_min_wheat": 999}\r\n\r\n\r\ndef _ca_tape(seat, t):\r\n    native = _IMPL.chas',
    b'sis.players.get(seat)\r\n    if not native or t > 719:\r\n        return {}\r\n    tape = _IMPL.chassis.routes[2 if t >= 648 else native["route"]]\r\n    return tape[t] if t < len(tape) and isinstance(tape[t], dict) else {}\r\n\r\n\r\ndef _ca_spawn(positions, board):\r\n    half = board // 2\r\n    access = [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]\r\n    occ = {a: 0 for a in access}\r\n    for p in positions:\r\n        if tuple(p) in occ:\r\n            occ[tuple(p)] += 1\r\n    return list(min(access, key=lambda a: (occ[a], access.index(a))))\r\n\r\n\r\ndef _ca_visits(obs, action, pos, t_end, start=None):\r\n    """Non-move commands issued on tile ``pos`` from this step (with ``action``) until ``t_end``."""\r\n    seat = int(obs["player"])\r\n    step = int(obs["step"])\r\n    farm = obs["farms"][seat]\r\n    board = len(farm["tiles"])\r\n    half = board // 2\r\n    positions = [list(farm["farmer"])] + [list(h) for h in farm["hands"]]\r\n    out = []\r\n    for t in range(step, min(t_end, 719) + 1):\r\n        act = action if t',
    b' == step else _ca_tape(seat, t)\r\n        units = [act.get("farmer") or ["PASS"]] + list(act.get("hands") or [])\r\n        for i in range(len(positions)):\r\n            cmd = units[i] if i < len(units) and units[i] else ["PASS"]\r\n            if cmd[0] in _CA_MOVES:\r\n                dx, dy = _CA_MOVES[cmd[0]]\r\n                nx, ny = positions[i][0] + dx, positions[i][1] + dy\r\n                if 0 <= nx < board and 0 <= ny < board:\r\n                    positions[i] = [nx, ny]\r\n            elif tuple(positions[i]) == pos and (start is None or t >= start):\r\n                out.append((t, i, cmd[0]))\r\n        for _ in range(sum(1 for o in (act.get("market") or []) if o and o[0] == "HIRE")):\r\n            positions.append(_ca_spawn(positions, board))\r\n        if t % 24 == 23:\r\n            positions = [[half - 1, half - 1]]\r\n    return out\r\n\r\n\r\ndef _ca_decays(mls, a, b):\r\n    """Decay events at steps s in [max(a, mls), b) with (s - mls) even."""\r\n    a = max(a, mls)\r\n    if b <= a:\r\n        return 0\r\n    first = a if ',
    b'(a - mls) % 2 == 0 else a + 1\r\n    return 0 if first >= b else (b - 1 - first) // 2 + 1\r\n\r\n\r\ndef _ca_yield_path(crop, planted, visits, y0=1, fert_until=-1, watered_day=-1, now_step=0):\r\n    """Return (harvest_units, best_rescue_units, rescue_step) for a crop following ``visits``.\r\n    ``y0`` is the yield observed at ``now_step`` (decay before that step already included)."""\r\n    myd, cap = _CA_CROP[crop]\r\n    lo = (myd + 1) // 2\r\n    mls = (planted + myd + 1) * 24\r\n    y = y0\r\n    best_rescue, rescue_t = 0, None\r\n    for t, i, op in visits:\r\n        day = t // 24\r\n        age = day - planted\r\n        dec = _ca_decays(mls, now_step, t)\r\n        now = y - dec\r\n        if now <= 0 and t > mls:\r\n            return 0, best_rescue, rescue_t\r\n        if op == "HARVEST":\r\n            return (max(0, now) if age >= 2 else 0), best_rescue, rescue_t\r\n        if op in ("PLANT", "DIG", "BUILD_COOP", "BUILD_PASTURE"):\r\n            return 0, best_rescue, rescue_t\r\n        if age >= 2 and now > best_rescue and t > now_step:\r\n',
    b'            best_rescue, rescue_t = now, t\r\n        if op == "WATER" and lo <= age <= myd and day != watered_day:\r\n            watered_day = day\r\n            y = min(cap, y + (2 if fert_until >= day else 1))\r\n    return 0, best_rescue, rescue_t\r\n\r\n\r\ndef _ca_wheat_total(obs):\r\n    priv = obs["private"]\r\n    return int(priv["shed"].get("WHEAT", 0)) + sum(int(inv.get("WHEAT", 0)) for inv in priv["inventories"])\r\n\r\n\r\ndef _ca_feed_need(seat, step, days):\r\n    need = 0\r\n    for t in range(step, min(719, step + 24 * days) + 1):\r\n        act = _ca_tape(seat, t)\r\n        units = [act.get("farmer") or ["PASS"]] + list(act.get("hands") or [])\r\n        need += sum(1 for c in units if c and c[0] == "FEED")\r\n    return need\r\n\r\n\r\n_CA_PARENT = agent\r\ndel agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    action = _CA_PARENT(observation, configuration)\r\n    try:\r\n        step = int(observation["step"])\r\n        seat = int(observation["player"])\r\n        st = _CA_STATE.get(seat)\r\n        if step == 0 or st is None or ',
    b'step <= st["step"]:\r\n            st = _CA_STATE[seat] = {"step": -1, "tiles": {}, "spare_wheat": 0, "spare_carrot": 0, "credit": 0}\r\n            if step == 0:\r\n                _CA_REPORT.update(ca_swaps=0, ca_rescues=0, ca_harvested=0, ca_sold=0, ca_seed_bought=0,\r\n                                  ca_wheat_seed_saved=0, ca_carrot_seed_saved=0, ca_feed_block=0,\r\n                                  ca_errors=0, ca_min_wheat=999)\r\n        st["step"] = step\r\n        if not isinstance(action, dict) or step > 717:\r\n            return action\r\n        day = step // 24\r\n        farm = observation["farms"][seat]\r\n        tiles = farm["tiles"]\r\n        priv = observation["private"]\r\n        prices = observation["market"]["prices"]\r\n        p_c, p_w = int(prices.get("CARROT", 0)), int(prices.get("WHEAT", 0))\r\n        positions = [tuple(farm["farmer"])] + [tuple(h) for h in farm["hands"]]\r\n        units = [list(action.get("farmer") or ["PASS"])] + [list(c) for c in (action.get("hands") or [])]\r\n        market = [list(o) fo',
    b'r o in (action.get("market") or [])]\r\n        changed = False\r\n        if _CA_FROM <= day <= _CA_TO + 4:\r\n            _CA_REPORT["ca_min_wheat"] = min(_CA_REPORT["ca_min_wheat"], _ca_wheat_total(observation))\r\n        # 1. bookkeeping + rescue of swapped carrots\r\n        for pos, planted in list(st["tiles"].items()):\r\n            tile = tiles[pos[1]][pos[0]]\r\n            if not (isinstance(tile, dict) and tile.get("crop") == "CARROT" and int(tile.get("planted_day", -9)) == planted):\r\n                st["tiles"].pop(pos, None)\r\n                continue\r\n            here = [i for i, p in enumerate(positions) if p == pos and i < len(units)]\r\n            if not here:\r\n                continue\r\n            i = here[0]\r\n            cmd = units[i]\r\n            yu = int(tile.get("yield_units", 0))\r\n            if cmd and cmd[0] == "HARVEST":\r\n                if day - planted >= 2 and yu > 0:\r\n                    st["credit"] += yu\r\n                    _CA_REPORT["ca_harvested"] += yu\r\n                    st["tiles"].',
    b'pop(pos, None)\r\n                continue\r\n            if not _CA_RESCUE or (cmd and cmd[0] in _CA_MOVES) or day - planted < 2 or yu <= 0:\r\n                continue\r\n            visits = _ca_visits(observation, action, pos, (planted + 5) * 24)\r\n            harvest, later, _ = _ca_yield_path("CARROT", planted, visits, y0=yu,\r\n                                               fert_until=int(tile.get("fertilized_until_day", -1)),\r\n                                               watered_day=day if tile.get("watered_today") else -1,\r\n                                               now_step=step)\r\n            if yu > max(harvest, later):\r\n                units[i] = ["HARVEST"]\r\n                st["credit"] += yu\r\n                _CA_REPORT["ca_harvested"] += yu\r\n                _CA_REPORT["ca_rescues"] += 1\r\n                st["tiles"].pop(pos, None)\r\n                changed = True\r\n        # 2. swaps\r\n        pays_now = 3 * (p_c - _CA_DROP) - 20 > 4 * p_w - 10 + _CA_MARGIN\r\n        if _CA_FROM <= day <= _CA_TO and pays_',
    b'now:\r\n            seeds_c = min(st["spare_carrot"],\r\n                          int(priv["seeds"].get("CARROT", 0)) - sum(1 for c in units if c[:2] == ["PLANT", "CARROT"]))\r\n            wheat_ok = None\r\n            for i, cmd in enumerate(units):\r\n                if cmd[:2] != ["PLANT", "WHEAT"] or i >= len(positions) or seeds_c <= 0:\r\n                    continue\r\n                pos = positions[i]\r\n                if tiles[pos[1]][pos[0]] is not None:\r\n                    continue\r\n                if wheat_ok is None:\r\n                    wheat_ok = _ca_wheat_total(observation) >= _ca_feed_need(seat, step, _CA_FEED_DAYS)\r\n                if not wheat_ok:\r\n                    _CA_REPORT["ca_feed_block"] += 1\r\n                    break\r\n                visits = _ca_visits(observation, action, pos, (day + 6) * 24, start=step + 1)\r\n                wu, _, _ = _ca_yield_path("WHEAT", day, visits)\r\n                ch, cr, _ = _ca_yield_path("CARROT", day, visits)\r\n                cu = max(ch, cr if _CA_RESCUE else ',
    b'0)\r\n                if cu * (p_c - _CA_DROP) - 20 > wu * p_w - 10 + _CA_MARGIN:\r\n                    units[i] = ["PLANT", "CARROT"]\r\n                    seeds_c -= 1\r\n                    st["spare_carrot"] -= 1\r\n                    st["tiles"][pos] = day\r\n                    st["spare_wheat"] += 1\r\n                    _CA_REPORT["ca_swaps"] += 1\r\n                    changed = True\r\n        # 3. seeds\r\n        new_market = []\r\n        for o in market:\r\n            if len(o) >= 3 and o[0] == "BUY_SEED" and o[1] in ("WHEAT", "CARROT"):\r\n                key = "spare_wheat" if o[1] == "WHEAT" else "spare_carrot"\r\n                cut = min(int(o[2]), st[key])\r\n                if cut > 0:\r\n                    st[key] -= cut\r\n                    _CA_REPORT["ca_wheat_seed_saved" if o[1] == "WHEAT" else "ca_carrot_seed_saved"] += cut\r\n                    changed = True\r\n                    if int(o[2]) - cut <= 0:\r\n                        continue\r\n                    o = [o[0], o[1], int(o[2]) - cut]\r\n            new_',
    b'market.append(o)\r\n        market = new_market\r\n        if _CA_FROM <= day <= _CA_TO - 1 and pays_now and len(market) < 10:\r\n            have = int(priv["seeds"].get("CARROT", 0)) - sum(1 for c in units if c[:2] == ["PLANT", "CARROT"])\r\n            buying = sum(int(o[2]) for o in market if len(o) >= 3 and o[:2] == ["BUY_SEED", "CARROT"])\r\n            q = _CA_BUFFER - have - buying\r\n            if q > 0 and int(farm.get("money", 0)) >= _CA_CASH + 20 * q:\r\n                market.append(["BUY_SEED", "CARROT", q])\r\n                st["spare_carrot"] += q\r\n                _CA_REPORT["ca_seed_bought"] += q\r\n                changed = True\r\n        # 4. sell credited carrots\r\n        if st["credit"] > 0 and p_c >= 2 and len(market) < 10:\r\n            view_action = {"farmer": units[0], "hands": units[1:], "market": market}\r\n            stock = int(projected_shed(view_action, FarmView(observation)).get("CARROT", 0))\r\n            selling = sum(int(o[2]) for o in market if len(o) >= 3 and o[:2] == ["SELL", "CARROT"])\r\n   ',
    b'         q = min(st["credit"], stock - selling)\r\n            if q > 0:\r\n                market.insert(0, ["SELL", "CARROT", q])\r\n                st["credit"] -= q\r\n                _CA_REPORT["ca_sold"] += q\r\n                changed = True\r\n        if changed:\r\n            action = dict(action)\r\n            action["farmer"] = units[0]\r\n            action["hands"] = units[1:]\r\n            action["market"] = market[:10]\r\n    except Exception:\r\n        _CA_REPORT["ca_errors"] += 1\r\n    return action\r\n\r\n\r\nagent.telemetry = _CA_REPORT\r\nagent = globals().pop(\'agent\')\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# Claude ORDERPRI2 layer: sales ordered by the rival\'s estimated sellable stock. Own implementation.\r\n# Built by tools/claude_build_orderpri2.py (derivation there).\r\n# ---------------------------------------------------------------------------\r\n_OR2_CAP = 24\r\n_OR2_SN_K = 0\r\n_OR2_SN_H = 24\r\n_OR2_SLOT_H = 6\r\n_OR2_SLOT_MARGIN = 12.0\r\n_OR2_SN_ITEMS = ("MILK", "STRAWBERRY", "',
    b'WOOL", "MELON", "EGG")\r\n_OR2_ITEMS = ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL")\r\n_OR2_ONGOING = ("TOMATO", "STRAWBERRY")\r\n_OR2_ANIMAL = {"GOOSE": "EGG", "COW": "MILK", "SHEEP": "WOOL"}\r\n_OR2_SHOPS = {"BAKERY": ("EGG", "WHEAT"), "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),\r\n              "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"), "YARN_STORE": ("WOOL",),\r\n              "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"), "PET_CAFE": ("CARROT",),\r\n              "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),\r\n              "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY")}\r\n_OR2_STATE = {}\r\n_OR2_REPORT = {"or2_reordered": 0, "or2_changed_vs_v39": 0, "or2_rival_harvest": 0, "or2_rival_sold": 0,\r\n               "or2_errors": 0}\r\n\r\n\r\ndef _or2_draw(shops, step):\r\n    draw = {}\r\n    if step % 4 == 0:\r\n        for name in shops:\r\n            products = _OR2_SHOPS.get(name, ())\r\n            for item in products:\r\n                draw[item] = draw.get(item, 0) + (2 if len(products)',
    b' == 1 else 1)\r\n    if step % 24 == 0:\r\n        for item in _OR2_ITEMS:\r\n            draw[item] = draw.get(item, 0) + 1\r\n    return draw\r\n\r\n\r\ndef _or2_tiles(farm):\r\n    out = {}\r\n    for y, row in enumerate(farm["tiles"]):\r\n        for x, t in enumerate(row):\r\n            if not isinstance(t, dict):\r\n                continue\r\n            if t.get("kind") == "PLANT" and t.get("crop"):\r\n                out[(x, y)] = ("P", t["crop"], int(t.get("planted_day", -1)), int(t.get("yield_units", 0)))\r\n            elif t.get("animal") in _OR2_ANIMAL:\r\n                out[(x, y)] = ("A", _OR2_ANIMAL[t["animal"]], int(t.get("placed_day", -1)), int(t.get("yield_units", 0)))\r\n    return out\r\n\r\n\r\ndef _or2_exposure(observation, item, qty, batch):\r\n    if qty <= 0 or batch <= 0 or item not in _R37_MARKET_PARAMS:\r\n        return 0.0\r\n    params = {k: dict(v) for k, v in _R37_MARKET_PARAMS.items()}\r\n    for k, patch in (observation["market"].get("params") or {}).items():\r\n        if k in params:\r\n            params[k].update(patc',
    b'h)\r\n    inv = int(observation["market"]["inventory"][item])\r\n    return float(sum(_r37_market_price(item, inv + j, params) - _r37_market_price(item, inv + batch + j, params)\r\n                     for j in range(qty)))\r\n\r\n\r\n_OR2_PARENT = agent\r\ndel agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    action = _OR2_PARENT(observation, configuration)\r\n    try:\r\n        step = int(observation["step"])\r\n        seat = int(observation["player"])\r\n        st = _OR2_STATE.get(seat)\r\n        if step == 0 or st is None or step <= st.get("step", -1):\r\n            st = _OR2_STATE[seat] = {"step": -1, "stock": {i: 0 for i in _OR2_ITEMS}, "prev": None}\r\n            if step == 0:\r\n                for k in _OR2_REPORT:\r\n                    _OR2_REPORT[k] = 0\r\n        rival = observation["farms"][1 - seat]\r\n        tiles = _or2_tiles(rival)\r\n        inv_now = {i: int(observation["market"]["inventory"].get(i, 0)) for i in _OR2_ITEMS}\r\n        prev = st["prev"]\r\n        if prev and prev["step"] == step - 1:\r\n            ',
    b'stock = st["stock"]\r\n            for pos, old in prev["tiles"].items():\r\n                kind, item, born, y = old\r\n                if y <= 0:\r\n                    continue\r\n                new = tiles.get(pos)\r\n                got = 0\r\n                if kind == "P" and item not in _OR2_ONGOING:\r\n                    if (new is None and not _OR2_WEED(rival, pos)) or (new is not None and new[2] != born):\r\n                        got = y\r\n                elif new is not None and new[0] == kind and new[1] == item and new[2] == born and new[3] < y:\r\n                    if step % 24 != 0:\r\n                        got = y - new[3]\r\n                    elif new[3] == 0:\r\n                        got = y\r\n                if got > 0:\r\n                    stock[item] = stock.get(item, 0) + got\r\n                    _OR2_REPORT["or2_rival_harvest"] += got\r\n            draw = _or2_draw(prev["shops"], prev["step"])\r\n            for item in _OR2_ITEMS:\r\n                if prev["prices"].get(item, 0) <= 1:\r\n                  ',
    b'  continue\r\n                moved = inv_now[item] - prev["inv"][item] + draw.get(item, 0) - prev["own"].get(item, 0)\r\n                if moved > 0:\r\n                    stock[item] = max(0, stock.get(item, 0) - moved)\r\n                    _OR2_REPORT["or2_rival_sold"] += moved\r\n        own = {}\r\n        if isinstance(action, dict):\r\n            orders = [list(o) for o in (action.get("market") or [])]\r\n            proj = dict(projected_shed(action, FarmView(observation)))\r\n            if _OR2_SN_K > 0 and 24 <= step < 694:\r\n                native = _IMPL.chassis.players.get(seat)\r\n                debts = native["sell_state"].setdefault("r36_debts", {}) if native else None\r\n                for item in _OR2_SN_ITEMS:\r\n                    if debts is None or int(st["stock"].get(item, 0)) < _OR2_SN_K:\r\n                        continue\r\n                    if int(observation["market"]["prices"].get(item, 0)) < 2 or len(orders) >= 10:\r\n                        continue\r\n                    if any(len(o) >= 2 and o[0]',
    b' in ("BUY_PRODUCT", "BUY_ANIMAL") and o[1] == item for o in orders):\r\n                        continue\r\n                    selling = sum(max(0, int(o[2])) for o in orders if len(o) >= 3 and o[0] == "SELL" and o[1] == item)\r\n                    avail = int(proj.get(item, 0)) - selling\r\n                    take = 0\r\n                    for t in range(step + 1, min(694, step + _OR2_SN_H) + 1):\r\n                        if take >= avail:\r\n                            break\r\n                        tape = _IMPL.chassis.routes[2 if t >= 648 else native["route"]]\r\n                        act = tape[t] if t < len(tape) and isinstance(tape[t], dict) else {}\r\n                        planned = sum(max(0, int(o[2])) for o in (act.get("market") or [])\r\n                                      if len(o) >= 3 and o[0] == "SELL" and o[1] == item)\r\n                        planned -= debts.get(t, {}).get(item, 0)\r\n                        q = min(planned, avail - take)\r\n                        if q > 0:\r\n                           ',
    b' debts.setdefault(t, {})[item] = debts.get(t, {}).get(item, 0) + q\r\n                            take += q\r\n                    if take > 0:\r\n                        for o in orders:\r\n                            if len(o) >= 3 and o[0] == "SELL" and o[1] == item:\r\n                                o[2] = int(o[2]) + take\r\n                                break\r\n                        else:\r\n                            orders.append(["SELL", item, take])\r\n                        _OR2_REPORT["or2_sellnow"] = _OR2_REPORT.get("or2_sellnow", 0) + take\r\n                        action = dict(action)\r\n                        action["market"] = orders\r\n            if _OR2_SLOT_H > 0 and 288 <= step < 694 and len(orders) >= 10:\r\n                native = _IMPL.chassis.players.get(seat)\r\n                debts = native["sell_state"].setdefault("r36_debts", {}) if native else None\r\n                sells = [o for o in orders if len(o) >= 3 and o[0] == "SELL"]\r\n                selling_items = {o[1] for o in sells}\r\n            ',
    b'    bought_items = {o[1] for o in orders if len(o) >= 2 and o[0] in ("BUY_PRODUCT", "BUY_ANIMAL")}\r\n                best = None\r\n                for item in _OR2_SN_ITEMS:\r\n                    if debts is None or item in selling_items or item in bought_items:\r\n                        continue\r\n                    avail = int(proj.get(item, 0))\r\n                    if avail <= 0 or int(observation["market"]["prices"].get(item, 0)) < 2:\r\n                        continue\r\n                    plan, take = [], 0\r\n                    for t in range(step + 1, min(694, step + _OR2_SLOT_H) + 1):\r\n                        if take >= avail:\r\n                            break\r\n                        tape = _IMPL.chassis.routes[2 if t >= 648 else native["route"]]\r\n                        act = tape[t] if t < len(tape) and isinstance(tape[t], dict) else {}\r\n                        planned = sum(max(0, int(o[2])) for o in (act.get("market") or [])\r\n                                      if len(o) >= 3 and o[0] == "SELL" and ',
    b'o[1] == item)\r\n                        q = min(planned - debts.get(t, {}).get(item, 0), avail - take)\r\n                        if q > 0:\r\n                            plan.append((t, q))\r\n                            take += q\r\n                    if take <= 0:\r\n                        continue\r\n                    b = min(_OR2_CAP, int(st["stock"].get(item, 0)))\r\n                    value = _or2_exposure(observation, item, take, max(1, b))\r\n                    if best is None or value > best[0]:\r\n                        best = (value, item, take, plan)\r\n                if best is not None and sells:\r\n                    def sval(o):\r\n                        q = min(max(0, int(o[2])), max(0, int(proj.get(o[1], 0))))\r\n                        return _or2_exposure(observation, o[1], q, max(1, min(_OR2_CAP, int(st["stock"].get(o[1], 0)))))\r\n                    weakest = min(sells, key=sval)\r\n                    if best[0] > sval(weakest) + _OR2_SLOT_MARGIN and weakest[1] not in ("WHEAT", "FERTILIZER")              ',
    b'               or best[0] > sval(weakest) + _OR2_SLOT_MARGIN and int(weakest[2]) <= 2:\r\n                        # drop the weakest sale; give its booked debts back (nearest due first)\r\n                        refund = max(0, int(weakest[2]))\r\n                        for t in range(step + 1, step + 49):\r\n                            if refund <= 0:\r\n                                break\r\n                            owed = debts.get(t, {}).get(weakest[1], 0)\r\n                            back = min(owed, refund)\r\n                            if back > 0:\r\n                                debts[t][weakest[1]] = owed - back\r\n                                refund -= back\r\n                        orders.remove(weakest)\r\n                        orders.append(["SELL", best[1], best[2]])\r\n                        for t, q in best[3]:\r\n                            debts.setdefault(t, {})[best[1]] = debts.get(t, {}).get(best[1], 0) + q\r\n                        _OR2_REPORT["or2_slot_swaps"] = _OR2_REPORT.get("or2_slot_swaps",',
    b' 0) + 1\r\n                        action = dict(action)\r\n                        action["market"] = orders\r\n            left = dict(proj)\r\n            movable, fixed, bought = [], [], set()\r\n            for idx, o in enumerate(orders):\r\n                if len(o) >= 2 and o[0] in ("BUY_PRODUCT", "BUY_ANIMAL"):\r\n                    bought.add(o[1])\r\n                if len(o) >= 3 and o[0] == "SELL" and int(o[2]) > 0 and o[1] not in bought:\r\n                    movable.append((idx, o))\r\n                else:\r\n                    fixed.append((idx, o))\r\n            if movable and step >= 1:\r\n                def score(io):\r\n                    item, qty = io[1][1], min(int(io[1][2]), max(0, int(proj.get(io[1][1], 0))))\r\n                    b = min(_OR2_CAP, int(st["stock"].get(item, 0)))\r\n                    return (-_or2_exposure(observation, item, qty, b),\r\n                            -_r37_quote_priority(observation, io[1], proj), io[0])\r\n                scored = sorted(movable, key=score)\r\n                new =',
    b' [o for _, o in scored] + [o for _, o in fixed]\r\n                if new != orders:\r\n                    v39 = sorted(movable, key=lambda io: (-_r37_quote_priority(observation, io[1], proj), io[0]))\r\n                    if [o for _, o in v39] != [o for _, o in scored]:\r\n                        _OR2_REPORT["or2_changed_vs_v39"] += 1\r\n                    _OR2_REPORT["or2_reordered"] += 1\r\n                    action = dict(action)\r\n                    action["market"] = new\r\n                    orders = new\r\n            for o in orders[:10]:\r\n                if len(o) >= 3 and o[0] == "SELL" and o[1] in _OR2_ITEMS:\r\n                    got = min(max(0, int(o[2])), max(0, int(left.get(o[1], 0))))\r\n                    left[o[1]] = left.get(o[1], 0) - got\r\n                    own[o[1]] = own.get(o[1], 0) + got\r\n        st["prev"] = {"step": step, "tiles": tiles, "inv": inv_now, "own": own,\r\n                      "prices": dict(observation["market"]["prices"]),\r\n                      "shops": list((observation.get("t',
    b'own") or {}).get("unlocked_shops") or [])}\r\n        st["step"] = step\r\n    except Exception:\r\n        _OR2_REPORT["or2_errors"] += 1\r\n    return action\r\n\r\n\r\ndef _OR2_WEED(farm, pos):\r\n    t = farm["tiles"][pos[1]][pos[0]]\r\n    return isinstance(t, dict) and t.get("kind") == "WEED"\r\n\r\n\r\nagent.telemetry = _OR2_REPORT\r\nagent = globals().pop(\'agent\')\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# Claude CAPHARV layer: harvest animals that would overflow tonight. Own implementation.\r\n# Built by tools/claude_build_capharv.py (derivation there).\r\n# ---------------------------------------------------------------------------\r\n_CH_SHED = 100\r\n_CH_SELL = True\r\n_CH_ANIMALS = {"GOOSE": ("EGG", 4, 4, 1), "COW": ("MILK", 6, 8, 2), "SHEEP": ("WOOL", 6, 6, 3)}\r\n_CH_MOVES = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}\r\n_CH_STATE = {}\r\n_CH_REPORT = {"ch_collect_swaps": 0, "ch_care_swaps": 0, "ch_saved": 0, "ch_sold": 0, "ch_shed_block": 0,\r\n              "ch_errors"',
    b': 0}\r\n\r\n\r\ndef _ch_tape(seat, t):\r\n    native = _IMPL.chassis.players.get(seat)\r\n    if not native or t > 719:\r\n        return {}\r\n    tape = _IMPL.chassis.routes[2 if t >= 648 else native["route"]]\r\n    return tape[t] if t < len(tape) and isinstance(tape[t], dict) else {}\r\n\r\n\r\ndef _ch_visits_today(obs, action):\r\n    """{pos: [(t, op)]} for non-move commands from the next step to the end of today."""\r\n    seat = int(obs["player"])\r\n    step = int(obs["step"])\r\n    farm = obs["farms"][seat]\r\n    board = len(farm["tiles"])\r\n    half = board // 2\r\n    access = [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]\r\n    positions = [list(farm["farmer"])] + [list(h) for h in farm["hands"]]\r\n    out = {}\r\n    for t in range(step, (step // 24 + 1) * 24):\r\n        act = action if t == step else _ch_tape(seat, t)\r\n        units = [act.get("farmer") or ["PASS"]] + list(act.get("hands") or [])\r\n        for i in range(len(positions)):\r\n            cmd = units[i] if i < len(units) and units[i] else ["PASS',
    b'"]\r\n            if cmd[0] in _CH_MOVES:\r\n                dx, dy = _CH_MOVES[cmd[0]]\r\n                nx, ny = positions[i][0] + dx, positions[i][1] + dy\r\n                if 0 <= nx < board and 0 <= ny < board:\r\n                    positions[i] = [nx, ny]\r\n            elif t > step:\r\n                out.setdefault(tuple(positions[i]), []).append((t, cmd[0]))\r\n        for _ in range(sum(1 for o in (act.get("market") or []) if o and o[0] == "HIRE")):\r\n            occ = {a: 0 for a in access}\r\n            for p in positions:\r\n                if tuple(p) in occ:\r\n                    occ[tuple(p)] += 1\r\n            positions.append(list(min(access, key=lambda a: (occ[a], access.index(a)))))\r\n    return out\r\n\r\n\r\n_CH_PARENT = agent\r\ndel agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    action = _CH_PARENT(observation, configuration)\r\n    try:\r\n        step = int(observation["step"])\r\n        seat = int(observation["player"])\r\n        st = _CH_STATE.get(seat)\r\n        if step == 0 or st is None or step <= st',
    b'["step"]:\r\n            st = _CH_STATE[seat] = {"step": -1, "credit": {}}\r\n            if step == 0:\r\n                for k in _CH_REPORT:\r\n                    _CH_REPORT[k] = 0\r\n        st["step"] = step\r\n        if not isinstance(action, dict) or step > 717:\r\n            return action\r\n        day = step // 24\r\n        farm = observation["farms"][seat]\r\n        priv = observation["private"]\r\n        prices = observation["market"]["prices"]\r\n        positions = [tuple(farm["farmer"])] + [tuple(h) for h in farm["hands"]]\r\n        units = [list(action.get("farmer") or ["PASS"])] + [list(c) for c in (action.get("hands") or [])]\r\n        market = [list(o) for o in (action.get("market") or [])]\r\n        changed = False\r\n        visits = None\r\n        for i, pos in enumerate(positions[:len(units)]):\r\n            cmd = units[i]\r\n            if not cmd or cmd[0] not in ("CARE", "COLLECT_FERTILIZER"):\r\n                continue\r\n            tile = farm["tiles"][pos[1]][pos[0]]\r\n            if not (isinstance(tile, dict',
    b') and tile.get("animal") in _CH_ANIMALS):\r\n                continue\r\n            product, cap, first, interval = _CH_ANIMALS[tile["animal"]]\r\n            since = day + 1 - int(tile.get("placed_day", 99)) - first\r\n            if since < 0 or since % interval != 0:\r\n                continue\r\n            y = int(tile.get("yield_units", 0))\r\n            if visits is None:\r\n                visits = _ch_visits_today(observation, action)\r\n            later = visits.get(pos, [])\r\n            if any(op == "HARVEST" for t, op in later):\r\n                continue\r\n            fed = bool(tile.get("fed_today")) or any(op == "FEED" for t, op in later)\r\n            prod = 1 + (int(tile.get("pending_care_bonus", 0)) if fed else 0)\r\n            overflow = y + prod - cap\r\n            if overflow <= 0 or y <= 0:\r\n                continue\r\n            quote = int(prices.get(product, 0))\r\n            if cmd[0] == "COLLECT_FERTILIZER":\r\n                if overflow * quote <= int(prices.get("FERTILIZER", 0)):\r\n                    c',
    b'ontinue\r\n            else:\r\n                if any(op == "COLLECT_FERTILIZER" for t, op in later) or overflow <= 1:\r\n                    continue\r\n            carried = sum(int(v) for inv in priv["inventories"] for v in inv.values())\r\n            if sum(int(v) for v in priv["shed"].values()) + carried + y >= _CH_SHED:\r\n                _CH_REPORT["ch_shed_block"] += 1\r\n                continue\r\n            units[i] = ["HARVEST"]\r\n            _CH_REPORT["ch_collect_swaps" if cmd[0] == "COLLECT_FERTILIZER" else "ch_care_swaps"] += 1\r\n            saved = overflow if cmd[0] == "COLLECT_FERTILIZER" else overflow - 1\r\n            _CH_REPORT["ch_saved"] += saved\r\n            st["credit"][product] = st["credit"].get(product, 0) + saved\r\n            changed = True\r\n        if _CH_SELL and any(v > 0 for v in st["credit"].values()):\r\n            view_action = {"farmer": units[0], "hands": units[1:], "market": market}\r\n            stock = dict(projected_shed(view_action, FarmView(observation)))\r\n            for product, c',
    b'redit in list(st["credit"].items()):\r\n                if credit <= 0 or len(market) >= 10 or int(prices.get(product, 0)) < 2:\r\n                    continue\r\n                selling = sum(int(o[2]) for o in market if len(o) >= 3 and o[:2] == ["SELL", product])\r\n                q = min(credit, int(stock.get(product, 0)) - selling)\r\n                if q > 0:\r\n                    for o in market:\r\n                        if len(o) >= 3 and o[:2] == ["SELL", product]:\r\n                            o[2] = int(o[2]) + q\r\n                            break\r\n                    else:\r\n                        market.insert(0, ["SELL", product, q])\r\n                    st["credit"][product] = credit - q\r\n                    _CH_REPORT["ch_sold"] += q\r\n                    changed = True\r\n        if changed:\r\n            action = dict(action)\r\n            action["farmer"] = units[0]\r\n            action["hands"] = units[1:]\r\n            action["market"] = market[:10]\r\n    except Exception:\r\n        _CH_REPORT["ch_errors"] +=',
    b' 1\r\n    return action\r\n\r\n\r\nagent.telemetry = _CH_REPORT\r\nagent = globals().pop(\'agent\')\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# Claude SHEDROOM layer: sell shed goods before the night drop overflows. Own implementation.\r\n# Built by tools/claude_build_shedroom.py (derivation there).\r\n# ---------------------------------------------------------------------------\r\n_SR_MARGIN = 8\r\n_SR_HOURS = (21, 22, 23)\r\n_SR_PRODUCTS = ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER")\r\n_SR_REPORT = {"sr_turns": 0, "sr_units": 0, "sr_errors": 0}\r\n\r\n\r\ndef _sr_tape(seat, t):\r\n    native = _IMPL.chassis.players.get(seat)\r\n    if not native or t > 719:\r\n        return {}\r\n    tape = _IMPL.chassis.routes[2 if t >= 648 else native["route"]]\r\n    return tape[t] if t < len(tape) and isinstance(tape[t], dict) else {}\r\n\r\n\r\n_SR_PARENT = agent\r\ndel agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    action = _SR_PARENT(observation, configuration)\r\n ',
    b'   try:\r\n        step = int(observation["step"])\r\n        if step == 0:\r\n            for k in _SR_REPORT:\r\n                _SR_REPORT[k] = 0\r\n        if step % 24 not in _SR_HOURS or step >= 717 or not isinstance(action, dict):\r\n            return action\r\n        seat = int(observation["player"])\r\n        farm = observation["farms"][seat]\r\n        priv = observation["private"]\r\n        view = FarmView(observation)\r\n        proj = dict(projected_shed(action, view))\r\n        market = [list(o) for o in (action.get("market") or [])]\r\n        left = dict(proj)\r\n        night_shed = sum(max(0, int(v)) for v in proj.values())\r\n        for o in market:\r\n            if len(o) >= 3 and o[0] == "SELL":\r\n                got = min(max(0, int(o[2])), max(0, int(left.get(o[1], 0))))\r\n                left[o[1]] = left.get(o[1], 0) - got\r\n                night_shed -= got\r\n            elif len(o) >= 3 and o[0] in ("BUY_PRODUCT", "BUY_ANIMAL"):\r\n                night_shed += max(0, int(o[2]))\r\n        units = [action.get("farm',
    b'er") or ["PASS"]] + list(action.get("hands") or [])\r\n        carried = 0\r\n        for i, pos in enumerate(view.positions):\r\n            inv = priv["inventories"][i] if i < len(priv["inventories"]) else {}\r\n            held = sum(max(0, int(v)) for v in inv.values())\r\n            cmd = units[i] if i < len(units) and units[i] else ["PASS"]\r\n            tile = farm["tiles"][pos[1]][pos[0]] if isinstance(pos, (list, tuple)) else None\r\n            op = cmd[0]\r\n            if op == "DROP" and _shed_adjacent(pos, view.board):\r\n                held = 0\r\n            elif op == "HARVEST" and isinstance(tile, dict):\r\n                held += max(0, int(tile.get("yield_units", 0)))\r\n            elif op == "COLLECT_FERTILIZER" and isinstance(tile, dict) and tile.get("fertilizer_available"):\r\n                held += 1\r\n            elif op in ("FEED", "FERTILIZE") and held > 0:\r\n                held -= 1\r\n            elif op == "PICKUP" and len(cmd) >= 2 and _shed_adjacent(pos, view.board):\r\n                held += max(1, in',
    b't(cmd[2]) if len(cmd) >= 3 else 1)\r\n            carried += held\r\n        cap = int((configuration or {}).get("shedCapacity", 100)) if isinstance(configuration, dict) else 100\r\n        overflow = night_shed + carried - cap + _SR_MARGIN\r\n        if overflow <= 0:\r\n            return action\r\n        need = {"WHEAT": 0, "FERTILIZER": 0}\r\n        for t in range(step + 1, min(719, step + 25)):\r\n            act = _sr_tape(seat, t)\r\n            for c in [act.get("farmer") or ["PASS"]] + list(act.get("hands") or []):\r\n                if not c:\r\n                    continue\r\n                if c[0] == "FEED":\r\n                    need["WHEAT"] += 1\r\n                elif c[0] == "FERTILIZE":\r\n                    need["FERTILIZER"] += 1\r\n        prices = observation["market"]["prices"]\r\n        cands = []\r\n        for item in _SR_PRODUCTS:\r\n            spare = int(left.get(item, 0)) - need.get(item, 0)\r\n            if spare > 0 and int(prices.get(item, 0)) >= 2:\r\n                cands.append((int(prices.get(item, 0)), it',
    b'em, spare))\r\n        cands.sort()\r\n        sold_now = 0\r\n        for price, item, spare in cands:\r\n            if overflow <= 0:\r\n                break\r\n            q = min(spare, overflow)\r\n            for o in market:\r\n                if len(o) >= 3 and o[:2] == ["SELL", item]:\r\n                    o[2] = int(o[2]) + q\r\n                    break\r\n            else:\r\n                if len(market) >= 10:\r\n                    continue\r\n                market.append(["SELL", item, q])\r\n            overflow -= q\r\n            sold_now += q\r\n        if sold_now:\r\n            _SR_REPORT["sr_turns"] += 1\r\n            _SR_REPORT["sr_units"] += sold_now\r\n            action = dict(action)\r\n            action["market"] = market\r\n    except Exception:\r\n        _SR_REPORT["sr_errors"] += 1\r\n    return action\r\n\r\n\r\nagent.telemetry = _SR_REPORT\r\nagent = globals().pop(\'agent\')\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# Claude HERD2 layer: goose / cow / sheep choice at the tape\'s goos',
    b'e purchase.\r\n# Own implementation. Built by tools/claude_build_herd2.py (derivation there).\r\n# ---------------------------------------------------------------------------\r\n_HD2_FROM = 192\r\n_HD2_TO = 360\r\n_HD2_RATIO = 1.3\r\n_HD2_MIN_GAIN = 600.0\r\n_HD2_LOOKBACK = 3\r\n_HD2_OPTIONS = (\'COW\', \'SHEEP\')\r\n_HD2_CARE = 0.8\r\n_HD2_FUTURE = 0.0\r\n\r\n_HD2_SPEC = {\r\n    "GOOSE": {"cost": 300, "first": 4, "interval": 1, "per": 2, "product": "EGG", "structure": "COOP"},\r\n    "COW": {"cost": 400, "first": 8, "interval": 2, "per": 3, "product": "MILK", "structure": "PASTURE"},\r\n    "SHEEP": {"cost": 500, "first": 6, "interval": 3, "per": 4, "product": "WOOL", "structure": "PASTURE"},\r\n}\r\n_HD2_SHOP_TYPES = {\r\n    "BAKERY": ("EGG", "WHEAT"), "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),\r\n    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"), "YARN_STORE": ("WOOL",),\r\n    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"), "PET_CAFE": ("CARROT",),\r\n    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"), "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "S',
    b'TRAWBERRY"),\r\n}\r\n_HD2_STATE = {}\r\n_HD2_REPORT = {"hd2_decision": "", "hd2_ev": "", "hd2_rewrites": 0, "hd2_credit_units": 0,\r\n               "hd2_sold_units": 0, "hd2_errors": 0}\r\n\r\n\r\ndef _hd2_daily_shop_demand(item, shops):\r\n    total = 0.0\r\n    for name in shops:\r\n        products = _HD2_SHOP_TYPES.get(name, ())\r\n        if item in products:\r\n            total += 6.0 * (2 if len(products) == 1 else 1)\r\n    return total\r\n\r\n\r\ndef _hd2_future_demand_per_day(item):\r\n    """Expected extra daily demand of one more uniformly drawn shop instance."""\r\n    return sum(6.0 * (2 if len(p) == 1 else 1) for p in _HD2_SHOP_TYPES.values() if item in p) / len(_HD2_SHOP_TYPES)\r\n\r\n\r\ndef _hd2_schedule(animal, placed_day, day_from):\r\n    """Units an animal of this type produces on each day >= day_from (daily care assumed,\r\n    scaled by _HD2_CARE)."""\r\n    spec = _HD2_SPEC[animal]\r\n    out = {}\r\n    for d in range(max(day_from, placed_day + spec["first"]), 30):\r\n        if (d - placed_day - spec["first"]) % spec["interval"] == 0',
    b':\r\n            out[d] = out.get(d, 0.0) + 1.0 + (spec["per"] - 1) * _HD2_CARE\r\n    return out\r\n\r\n\r\ndef _hd2_ev(option, k, obs, st):\r\n    """Margin value of k new animals of `option`: their own revenue, plus the price change\r\n    their supply causes on (our existing future units - rival existing future units)."""\r\n    spec = _HD2_SPEC[option]\r\n    item = spec["product"]\r\n    animal_of = {"EGG": "GOOSE", "MILK": "COW", "WOOL": "SHEEP"}[item]\r\n    day = int(obs["step"]) // 24\r\n    seat = int(obs["player"])\r\n    market = obs["market"]\r\n    params = {key: dict(v) for key, v in _R37_MARKET_PARAMS.items()}\r\n    for key, patch in (market.get("params") or {}).items():\r\n        if key in params and isinstance(patch, dict):\r\n            params[key].update(patch)\r\n    # existing supply, per farm, per day\r\n    existing = [dict(), dict()]\r\n    for farm_index, farm in enumerate(obs["farms"]):\r\n        for row in farm["tiles"]:\r\n            for tile in row:\r\n                if isinstance(tile, dict) and tile.get("animal") ==',
    b' animal_of:\r\n                    for d, u in _hd2_schedule(animal_of, int(tile.get("placed_day", day)), day + 1).items():\r\n                        existing[farm_index][d] = existing[farm_index].get(d, 0.0) + u\r\n    shed = (obs.get("private") or {}).get("shed") or {}\r\n    carried = sum(int(inv.get(animal_of, 0)) for inv in (obs.get("private") or {}).get("inventories") or [])\r\n    for _ in range(int(shed.get(animal_of, 0)) + carried):\r\n        for d, u in _hd2_schedule(animal_of, day + 1, day + 1).items():\r\n            existing[seat][d] = existing[seat].get(d, 0.0) + u\r\n    # Purchases the tape already plans after this turn; a family-similar rival runs the same tape.\r\n    native = _IMPL.chassis.players.get(seat)\r\n    if native:\r\n        similar = _r37_similarity(obs) >= 0.9\r\n        for t in range(int(obs["step"]) + 1, 696):\r\n            tape = _IMPL.chassis.routes[2 if t >= 648 else native["route"]]\r\n            if t >= len(tape) or not isinstance(tape[t], dict):\r\n                continue\r\n            for orde',
    b'r in tape[t].get("market", []) or []:\r\n                if len(order) >= 3 and order[0] == "BUY_ANIMAL" and order[1] == animal_of:\r\n                    for _ in range(max(0, int(order[2]))):\r\n                        for d, u in _hd2_schedule(animal_of, t // 24 + 1, day + 1).items():\r\n                            existing[seat][d] = existing[seat].get(d, 0.0) + u\r\n                            if similar:\r\n                                existing[1 - seat][d] = existing[1 - seat].get(d, 0.0) + u\r\n    ours_existing, rival_existing = existing[seat], existing[1 - seat]\r\n    new = {}\r\n    for d, u in _hd2_schedule(option, day + 1, day + 1).items():\r\n        new[d] = u * k\r\n    shops = list((obs.get("town") or {}).get("unlocked_shops") or [])\r\n    unlocks_left = max(0, 8 - len(shops))\r\n    base_demand = _hd2_daily_shop_demand(item, shops) + 1.0\r\n    extra_demand = _hd2_future_demand_per_day(item) * _HD2_FUTURE\r\n\r\n    def path(with_new):\r\n        inv = float(market["inventory"][item])\r\n        prices, revenue = {}, 0.0\r',
    b'\n        for d in range(day + 1, 30):\r\n            opened = min(unlocks_left, max(0, (d // 3) - (day // 3)))\r\n            inv -= base_demand + extra_demand * opened\r\n            inv += ours_existing.get(d, 0.0) + rival_existing.get(d, 0.0)\r\n            prices[d] = _r37_market_price(item, int(round(inv)), params)\r\n            if with_new:\r\n                units = int(round(new.get(d, 0.0)))\r\n                for _ in range(units):\r\n                    price = _r37_market_price(item, int(round(inv)), params)\r\n                    revenue += price\r\n                    if price > 1:\r\n                        inv += 1\r\n        return prices, revenue\r\n\r\n    base_prices, _ = path(False)\r\n    new_prices, revenue = path(True)\r\n    swing = sum((new_prices[d] - base_prices[d]) * (ours_existing.get(d, 0.0) - rival_existing.get(d, 0.0))\r\n                for d in base_prices)\r\n    return revenue + swing - spec["cost"] * k, revenue\r\n\r\n\r\ndef _hd2_decide(obs, action, st):\r\n    market = action.get("market") or []\r\n    buys = [o f',
    b'or o in market if len(o) >= 3 and o[0] == "BUY_ANIMAL" and o[1] == "GOOSE"]\r\n    if not buys:\r\n        return\r\n    st["decided"] = True\r\n    step = int(obs["step"])\r\n    if not (_HD2_FROM <= step < _HD2_TO):\r\n        return\r\n    farm = obs["farms"][int(obs["player"])]\r\n    if any(isinstance(t, dict) and t.get("kind") == "COOP" for row in farm["tiles"] for t in row):\r\n        _HD2_REPORT["hd2_decision"] = "skip:coop_exists"\r\n        return\r\n    k = sum(max(0, int(o[2])) for o in buys)\r\n    k_plan = max(k, 3)\r\n    evs = {opt: _hd2_ev(opt, k_plan, obs, st)[0] for opt in ("GOOSE",) + tuple(_HD2_OPTIONS)}\r\n    _HD2_REPORT["hd2_ev"] = ",".join("%s:%d" % (o, v) for o, v in sorted(evs.items()))\r\n    best = max(_HD2_OPTIONS, key=lambda o: evs[o]) if _HD2_OPTIONS else None\r\n    goose = evs["GOOSE"]\r\n    if best is None:\r\n        return\r\n    gain = evs[best] - goose\r\n    ok_ratio = evs[best] >= _HD2_RATIO * max(goose, 1.0)\r\n    extra_cost = (_HD2_SPEC[best]["cost"] - 300) * k\r\n    cash = float(farm.get("money", 0))\r\n   ',
    b' if gain >= _HD2_MIN_GAIN and ok_ratio and cash >= 300 * k + extra_cost + 50:\r\n        st["mode"] = best\r\n        _HD2_REPORT["hd2_decision"] = "%s@%d" % (best, step)\r\n    else:\r\n        _HD2_REPORT["hd2_decision"] = "keep@%d" % step\r\n\r\n\r\ndef _hd2_rewrite(obs, action, st):\r\n    mode = st["mode"]\r\n    spec = _HD2_SPEC[mode]\r\n    item = spec["product"]\r\n    seat = int(obs["player"])\r\n    farm = obs["farms"][seat]\r\n    private = obs["private"]\r\n    positions = [farm["farmer"]] + list(farm["hands"])\r\n    market = action.get("market") or []\r\n    cash = float(farm.get("money", 0))\r\n    seed_cost = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}\r\n    for order in market:\r\n        if len(order) >= 3 and order[0] == "BUY_ANIMAL" and order[1] == "GOOSE":\r\n            n = max(0, int(order[2]))\r\n            affordable = int(max(0.0, cash) // spec["cost"])\r\n            order[1] = mode\r\n            order[2] = min(n, affordable)\r\n            cash -= order[2] * spec["cost"]\r\n            _HD2_REPORT[',
    b'"hd2_rewrites"] += 1\r\n        elif len(order) >= 3 and order[0] in ("BUY_ANIMAL", "BUY_SEED", "BUY_PRODUCT"):\r\n            # money spent by earlier orders of the same list is not available to the swap (DS-5 #6)\r\n            q = max(0, int(order[2]))\r\n            if order[0] == "BUY_ANIMAL":\r\n                cash -= q * {"GOOSE": 300, "COW": 400, "SHEEP": 500}.get(order[1], 0)\r\n            elif order[0] == "BUY_SEED":\r\n                cash -= q * seed_cost.get(order[1], 0)\r\n            else:\r\n                cash -= q * int((obs["market"]["prices"] or {}).get(order[1], 0))\r\n        elif order and order[0] == "BUY_LAND":\r\n            cash -= 4000\r\n    workers = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])\r\n    for actor, work in enumerate(workers[:len(positions)]):\r\n        if not work:\r\n            continue\r\n        x, y = positions[actor]\r\n        tile = farm["tiles"][y][x]\r\n        if work[0] == "BUILD_COOP":\r\n            workers[actor] = ["BUILD_PASTURE"]\r\n            _HD2_REPORT["hd',
    b'2_rewrites"] += 1\r\n        elif len(work) >= 2 and work[0] in ("PICKUP", "PLACE") and work[1] == "GOOSE":\r\n            workers[actor] = [work[0], mode] + list(work[2:])\r\n            _HD2_REPORT["hd2_rewrites"] += 1\r\n            if work[0] == "PLACE":\r\n                st["pending"].append((x, y, int(obs["step"]) // 24))\r\n        elif len(work) >= 2 and work[0] == "PLACE" and work[1] == "EGG":\r\n            workers[actor] = ["PLACE", item] + list(work[2:])\r\n            _HD2_REPORT["hd2_rewrites"] += 1\r\n        elif work == ["HARVEST"] and (x, y) in st["sites"] and isinstance(tile, dict) \\\r\n                and tile.get("animal") == mode:\r\n            units = max(0, int(tile.get("yield_units", 0)))\r\n            st["credit"] += units\r\n            _HD2_REPORT["hd2_credit_units"] += units\r\n    action["farmer"] = workers[0]\r\n    action["hands"] = workers[1:]\r\n    if st["credit"] > 0:\r\n        try:\r\n            stock = projected_shed(action, FarmView(obs))\r\n        except Exception:\r\n            stock = dict(private.ge',
    b't("shed") or {})\r\n        planned = sum(max(0, int(o[2])) for o in market if len(o) >= 3 and o[:2] == ["SELL", item])\r\n        extra = min(st["credit"], max(0, int(stock.get(item, 0)) - planned))\r\n        if extra > 0:\r\n            for order in market:\r\n                if len(order) >= 3 and order[:2] == ["SELL", item]:\r\n                    order[2] = int(order[2]) + extra\r\n                    break\r\n            else:\r\n                if len(market) < 10:\r\n                    market.insert(0, ["SELL", item, extra])\r\n                else:\r\n                    extra = 0\r\n            st["credit"] -= extra\r\n            _HD2_REPORT["hd2_sold_units"] += extra\r\n    action["market"] = market\r\n\r\n\r\ndef _hd2_confirm(obs, st):\r\n    farm = obs["farms"][int(obs["player"])]\r\n    keep = []\r\n    for x, y, day in st["pending"]:\r\n        tile = farm["tiles"][y][x]\r\n        if isinstance(tile, dict) and tile.get("animal") == st["mode"] and tile.get("placed_day") == day:\r\n            st["sites"][(x, y)] = day\r\n        elif int(ob',
    b's["step"]) // 24 <= day + 1:\r\n            keep.append((x, y, day))\r\n    st["pending"] = keep\r\n\r\n\r\n_HD2_PARENT = agent\r\ndel agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    action = _HD2_PARENT(observation, configuration)\r\n    try:\r\n        seat = int(observation["player"])\r\n        step = int(observation["step"])\r\n        st = _HD2_STATE.get(seat)\r\n        if st is None or step <= st["step"]:\r\n            st = _HD2_STATE[seat] = {"step": -1, "inv": {}, "decided": False, "mode": None,\r\n                                     "pending": [], "sites": {}, "credit": 0}\r\n            if step == 0:\r\n                _HD2_REPORT.update(hd2_decision="", hd2_ev="", hd2_rewrites=0, hd2_credit_units=0,\r\n                                   hd2_sold_units=0, hd2_errors=0)\r\n        st["step"] = step\r\n        day = step // 24\r\n        if day not in st["inv"]:\r\n            st["inv"][day] = dict(observation["market"]["inventory"])\r\n        if not isinstance(action, dict):\r\n            return action\r\n        if st["mode"]:',
    b'\r\n            _hd2_confirm(observation, st)\r\n        elif not st["decided"]:\r\n            _hd2_decide(observation, action, st)\r\n        if st["mode"]:\r\n            action = copy.deepcopy(action)\r\n            _hd2_rewrite(observation, action, st)\r\n    except Exception:\r\n        _HD2_REPORT["hd2_errors"] += 1\r\n    return action\r\n\r\n\r\nagent.telemetry = _HD2_REPORT\r\nagent = globals().pop(\'agent\')\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# Claude COWSWAP layer: cow / goose / sheep at the tape\'s first cow purchase. Own implementation.\r\n# Built by tools/claude_build_cowswap.py (derivation there). Requires HERD2 above.\r\n# ---------------------------------------------------------------------------\r\n_CS_FROM = 144\r\n_CS_TO = 192\r\n_CS_RATIO = 1.3\r\n_CS_MIN_GAIN = 600.0\r\n_CS_OPTIONS = (\'GOOSE\',)\r\n_CS_SHOP_RULE = \'nomilk\'\r\n_CS_MOVES = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}\r\n_CS_STATE = {}\r\n_CS_REPORT = {"cs_decision": "", "cs_ev": "", "cs_rewrites": 0, ',
    b'"cs_broken": 0, "cs_credit": 0,\r\n              "cs_sold": 0, "cs_errors": 0}\r\n\r\n\r\ndef _cs_tape(seat, t):\r\n    native = _IMPL.chassis.players.get(seat)\r\n    if not native or t > 719:\r\n        return {}\r\n    tape = _IMPL.chassis.routes[2 if t >= 648 else native["route"]]\r\n    return tape[t] if t < len(tape) and isinstance(tape[t], dict) else {}\r\n\r\n\r\ndef _cs_spawn(positions, board):\r\n    half = board // 2\r\n    access = [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]\r\n    occ = {a: 0 for a in access}\r\n    for p in positions:\r\n        if tuple(p) in occ:\r\n            occ[tuple(p)] += 1\r\n    return list(min(access, key=lambda a: (occ[a], access.index(a))))\r\n\r\n\r\ndef _cs_plan(obs, action, k, mode):\r\n    seat = int(obs["player"])\r\n    step = int(obs["step"])\r\n    farm = obs["farms"][seat]\r\n    board = len(farm["tiles"])\r\n    positions = [list(farm["farmer"])] + [list(h) for h in farm["hands"]]\r\n    end = (step // 24 + 1) * 24 - 1\r\n    builds, pickups, places = [], [], []\r\n    for t in range(st',
    b'ep, end + 1):\r\n        act = action if t == step else _cs_tape(seat, t)\r\n        units = [act.get("farmer") or ["PASS"]] + list(act.get("hands") or [])\r\n        for i in range(len(positions)):\r\n            cmd = units[i] if i < len(units) and units[i] else ["PASS"]\r\n            pos = tuple(positions[i])\r\n            if cmd[0] in _CS_MOVES:\r\n                dx, dy = _CS_MOVES[cmd[0]]\r\n                nx, ny = pos[0] + dx, pos[1] + dy\r\n                if 0 <= nx < board and 0 <= ny < board:\r\n                    positions[i] = [nx, ny]\r\n            elif cmd[0] == "BUILD_PASTURE":\r\n                builds.append((t, i, pos))\r\n            elif len(cmd) >= 2 and cmd[0] == "PICKUP" and cmd[1] == "COW":\r\n                pickups.append((t, i, max(1, int(cmd[2]) if len(cmd) > 2 else 1)))\r\n            elif len(cmd) >= 2 and cmd[0] == "PLACE" and cmd[1] == "COW":\r\n                places.append((t, i, pos))\r\n        hires = sum(1 for o in (act.get("market") or []) if o and o[0] == "HIRE")\r\n        for _ in range(hires):\r\n ',
    b'           positions.append(_cs_spawn(positions, board))\r\n    chosen, tiles = [], set()\r\n    for t, i, pos in places:\r\n        if pos not in tiles:\r\n            chosen.append((t, i, pos))\r\n            tiles.add(pos)\r\n        if len(chosen) == k:\r\n            break\r\n    if len(chosen) < k:\r\n        return None\r\n    plan = {"builds": {}, "pickups": {}, "places": {}}\r\n    buying_land = any(o and o[0] == "BUY_LAND" for o in (action.get("market") or []))\r\n    unlocked = list(farm.get("unlocked_quadrants") or ["NW"])\r\n    next_quadrant = [q for q in ("NE", "SW", "SE") if q not in unlocked][:1]\r\n    half = board // 2\r\n    for t, i, pos in chosen:\r\n        x, y = pos\r\n        tile = farm["tiles"][y][x]\r\n        if mode == "GOOSE":\r\n            quadrant = ("N" if y < half else "S") + ("W" if x < half else "E")\r\n            locked_but_bought = tile == "LOCKED" and buying_land and quadrant in next_quadrant\r\n            if tile is not None and not locked_but_bought:\r\n                return None  # already built (or occup',
    b'ied): cannot become a coop without extra turns\r\n            prior = [(tb, ib) for tb, ib, pb in builds if pb == pos and tb < t]\r\n            if not prior:\r\n                return None\r\n            tb, ib = prior[-1]\r\n            plan["builds"][(tb, ib)] = pos\r\n        carrier = [(tp, ip, q) for tp, ip, q in pickups if ip == i and tp < t]\r\n        if not carrier:\r\n            return None\r\n        tp, ip, q = carrier[-1]\r\n        plan["pickups"][(tp, ip)] = plan["pickups"].get((tp, ip), 0) + 1\r\n        plan["places"][(t, i)] = pos\r\n    for key, n in plan["pickups"].items():\r\n        q = next(q for tp, ip, q in pickups if (tp, ip) == key)\r\n        if q != n:\r\n            return None  # a pickup that also carries unswapped cows cannot be split\r\n    return plan\r\n\r\n\r\n_CS_PARENT = agent\r\ndel agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    action = _CS_PARENT(observation, configuration)\r\n    try:\r\n        seat = int(observation["player"])\r\n        step = int(observation["step"])\r\n        st = _CS_STATE.ge',
    b't(seat)\r\n        if st is None or step <= st["step"]:\r\n            st = _CS_STATE[seat] = {"step": -1, "decided": False, "mode": None, "plan": None,\r\n                                    "broken": False, "sites": {}, "pending": [], "credit": 0}\r\n            if step == 0:\r\n                _CS_REPORT.update(cs_decision="", cs_ev="", cs_rewrites=0, cs_broken=0, cs_credit=0,\r\n                                  cs_sold=0, cs_errors=0)\r\n        st["step"] = step\r\n        if not isinstance(action, dict):\r\n            return action\r\n        farm = observation["farms"][seat]\r\n        positions = [farm["farmer"]] + list(farm["hands"])\r\n        market = [list(o) for o in (action.get("market") or [])]\r\n        # confirm placements\r\n        if st["pending"]:\r\n            keep = []\r\n            for x, y, day in st["pending"]:\r\n                tile = farm["tiles"][y][x]\r\n                if isinstance(tile, dict) and tile.get("animal") == st["mode"] and tile.get("placed_day") == day:\r\n                    st["sites"][(x, y)] = ',
    b'day\r\n                elif step // 24 <= day:\r\n                    keep.append((x, y, day))\r\n            st["pending"] = keep\r\n        if not st["decided"] and _CS_FROM <= step < _CS_TO:\r\n            buys = [o for o in market if len(o) >= 3 and o[0] == "BUY_ANIMAL" and o[1] == "COW" and int(o[2]) > 0]\r\n            shops_now = list((observation.get("town") or {}).get("unlocked_shops") or [])\r\n            shop_ok = True\r\n            if _CS_SHOP_RULE:\r\n                # research/claude_20260913/RESULTS.md: over 48 seeds the swap only paid with an egg shop\r\n                # open and no milk shop open (+1,254 mean over 9 seeds); with a milk shop it lost.\r\n                no_milk = not any(s in ("PIZZA_SHOP", "ICE_CREAM_SHOP", "SMOOTHIE_SHOP") for s in shops_now)\r\n                if _CS_SHOP_RULE == "nomilk":\r\n                    # 14 seeds without a milk shop and without a yarn store: +1,203 mean\r\n                    shop_ok = no_milk and "YARN_STORE" not in shops_now\r\n                else:\r\n                    sh',
    b'op_ok = no_milk and any(s in ("BAKERY", "BRUNCH_SPOT") for s in shops_now)\r\n            if buys and not shop_ok:\r\n                st["decided"] = True\r\n                _CS_REPORT["cs_decision"] = "shoprule@%d" % step\r\n            elif buys:\r\n                st["decided"] = True\r\n                k = sum(int(o[2]) for o in buys)\r\n                evs = {opt: _hd2_ev(opt, k, observation, st)[0] for opt in ("COW",) + tuple(_CS_OPTIONS)}\r\n                _CS_REPORT["cs_ev"] = ",".join("%s:%d" % (o, v) for o, v in sorted(evs.items()))\r\n                best = max(_CS_OPTIONS, key=lambda o: evs[o])\r\n                cow = evs["COW"]\r\n                if evs[best] - cow >= _CS_MIN_GAIN and evs[best] >= _CS_RATIO * max(cow, 1.0):\r\n                    plan = _cs_plan(observation, action, k, best)\r\n                    if plan is not None:\r\n                        st["mode"], st["plan"] = best, plan\r\n                        _CS_REPORT["cs_decision"] = "%s@%d" % (best, step)\r\n                        for o in market:\r\n        ',
    b'                    if len(o) >= 3 and o[0] == "BUY_ANIMAL" and o[1] == "COW":\r\n                                o[1] = best\r\n                                _CS_REPORT["cs_rewrites"] += 1\r\n                    else:\r\n                        _CS_REPORT["cs_decision"] = "noplan@%d" % step\r\n                else:\r\n                    _CS_REPORT["cs_decision"] = "keep@%d" % step\r\n        if st["mode"] and st["plan"] and not st["broken"]:\r\n            plan = st["plan"]\r\n            units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])\r\n            for i in range(min(len(units), len(positions))):\r\n                cmd = units[i] or ["PASS"]\r\n                pos = (int(positions[i][0]), int(positions[i][1]))\r\n                key = (step, i)\r\n                if key in plan["builds"]:\r\n                    if cmd == ["BUILD_PASTURE"] and pos == plan["builds"][key]:\r\n                        units[i] = ["BUILD_COOP"]\r\n                        _CS_REPORT["cs_rewrites"] += 1\r\n                    else:\r\n ',
    b'                       st["broken"] = True\r\n                if key in plan["pickups"]:\r\n                    if len(cmd) >= 2 and cmd[0] == "PICKUP" and cmd[1] == "COW":\r\n                        units[i] = ["PICKUP", st["mode"]] + list(cmd[2:])\r\n                        _CS_REPORT["cs_rewrites"] += 1\r\n                    else:\r\n                        st["broken"] = True\r\n                if key in plan["places"]:\r\n                    if len(cmd) >= 2 and cmd[0] == "PLACE" and cmd[1] == "COW" and pos == plan["places"][key]:\r\n                        units[i] = ["PLACE", st["mode"]] + list(cmd[2:])\r\n                        st["pending"].append((pos[0], pos[1], step // 24))\r\n                        _CS_REPORT["cs_rewrites"] += 1\r\n                    else:\r\n                        st["broken"] = True\r\n            if st["broken"]:\r\n                _CS_REPORT["cs_broken"] += 1\r\n            action = dict(action)\r\n            action["farmer"] = units[0]\r\n            action["hands"] = units[1:]\r\n        # credit harvests',
    b' on swapped tiles and sell them as they reach the shed\r\n        if st["sites"]:\r\n            product = _HD2_SPEC[st["mode"]]["product"]\r\n            units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])\r\n            for i in range(min(len(units), len(positions))):\r\n                x, y = int(positions[i][0]), int(positions[i][1])\r\n                tile = farm["tiles"][y][x]\r\n                if units[i] == ["HARVEST"] and (x, y) in st["sites"] and isinstance(tile, dict) \\\r\n                        and tile.get("animal") == st["mode"]:\r\n                    n = max(0, int(tile.get("yield_units", 0)))\r\n                    st["credit"] += n\r\n                    _CS_REPORT["cs_credit"] += n\r\n            if st["credit"] > 0:\r\n                stock = projected_shed(action, FarmView(observation))\r\n                planned = sum(max(0, int(o[2])) for o in market if len(o) >= 3 and o[:2] == ["SELL", product])\r\n                extra = min(st["credit"], max(0, int(stock.get(product, 0)) - planned))\r\n  ',
    b'              if extra > 0:\r\n                    for o in market:\r\n                        if len(o) >= 3 and o[:2] == ["SELL", product]:\r\n                            o[2] = int(o[2]) + extra\r\n                            break\r\n                    else:\r\n                        if len(market) < 10:\r\n                            market.insert(0, ["SELL", product, extra])\r\n                        else:\r\n                            extra = 0\r\n                    st["credit"] -= extra\r\n                    _CS_REPORT["cs_sold"] += extra\r\n        action = dict(action)\r\n        action["market"] = market\r\n    except Exception:\r\n        _CS_REPORT["cs_errors"] += 1\r\n    return action\r\n\r\n\r\nagent.telemetry = _CS_REPORT\r\nagent = globals().pop(\'agent\')\r\n\r\n\r\n\r\n\r\n# EXP283 adaptive arm: clone-gated sale pre-emption with drop-time race escalation.\r\n# Original mechanism by Ahmed Berat Ozer\'s project. Live top-band replays (research155) show\r\n# rivals executing the same public route tape and quoting the same product batches at t',
    b"he\r\n# same drop turns. While the rival is observed executing our tape, V43's R36 native-tape\r\n# reservation runs with horizon 8; if the rival is then observed selling a race product at the\r\n# very turn the same product was dropped into our shed while we did not sell it (public market\r\n# inventory change beyond town consumption; no own realized sale; own shed stock rose), the rival\r\n# quotes at the drop and the horizon escalates to 24 for the rest of the game.\r\n_RACE_PARENT=agent\r\n_RACE_HORIZON_CLONE=8\r\n_RACE_HORIZON_ESCALATED=24\r\n_RACE_HORIZON_MIRROR=24\r\n_RACE_ITEMS=('CARROT','TOMATO','STRAWBERRY','MELON','EGG','MILK','WOOL')\r\n_RACE_SHOPS={'BAKERY':('EGG','WHEAT'),'PIZZA_SHOP':('MILK','TOMATO','WHEAT'),'BRUNCH_SPOT':('EGG','WHEAT','STRAWBERRY'),'YARN_STORE':('WOOL',),\r\n             'ICE_CREAM_SHOP':('STRAWBERRY','MILK','WHEAT'),'PET_CAFE':('CARROT',),'SMOOTHIE_SHOP':('STRAWBERRY','MILK'),'FARMERS_MARKET':('WHEAT','CARROT','TOMATO','STRAWBERRY')}\r\n_RACE_STATE={}\r\n_RACE_REPORT=dict(race_clone_turns=0,race_horiz",
    b'on_turns=0,race_lost_races=0,race_escalations=0,race_errors=0)\r\n_RACE_ORIG_RESERVE=_r36_reserve\r\n\r\ndef _race_positions_equal(farms,player):\r\n    own,rival=farms[player],farms[1-player]\r\n    return len(own[\'hands\'])>0 and own[\'hands\']==rival[\'hands\'] and own[\'farmer\']==rival[\'farmer\']\r\n\r\ndef _race_clone(observation,state):\r\n    farms=observation[\'farms\'];player=int(observation[\'player\'])\r\n    if len(farms[player][\'hands\'])>0:\r\n        state[\'hist\'].append(_race_positions_equal(farms,player))\r\n        if len(state[\'hist\'])>6:state[\'hist\'].pop(0)\r\n    return len(state[\'hist\'])>=4 and sum(state[\'hist\'])>=4 and _r37_similarity(observation)>=.95\r\n\r\ndef _race_town(step,shops):\r\n    out={}\r\n    if step%4==0:\r\n        for shop in shops:\r\n            items=_RACE_SHOPS.get(shop,())\r\n            for item in items:out[item]=out.get(item,0)+(2 if len(items)==1 else 1)\r\n    if step%24==0:\r\n        for item in _RACE_ITEMS:out[item]=out.get(item,0)+1\r\n    return out\r\n\r\ndef _race_lost(observation,state):\r\n    """EXP293: True w',
    b'hen the rival sold a race product at the previous turn while we held it unsold, the common tape\r\n    has no sale of it within the lineage\'s own lead/reservation window (5 turns) and sells it within the following\r\n    24 turns: the rival pre-empts the plan\'s own sale ahead of us."""\r\n    prev=state.get(\'prev\');prev_action=state.get(\'prev_action\')\r\n    if prev is None or prev_action is None:return False\r\n    step=int(observation[\'step\']);player=int(observation[\'player\'])\r\n    if step!=prev[\'step\']+1 or step%24==0:return False\r\n    inv=observation[\'market\'][\'inventory\'];pinv=prev[\'inventory\'];prices=prev[\'prices\']\r\n    town=_race_town(step-1,prev[\'shops\'])\r\n    sold={}\r\n    for order in prev_action.get(\'market\',[]):\r\n        if len(order)>=3 and order[0]==\'SELL\' and order[1] in _RACE_ITEMS:sold[order[1]]=1\r\n    native=_IMPL.chassis.players[player]\r\n    for item in _RACE_ITEMS:\r\n        before=int(prev[\'view\'].shed.get(item,0))\r\n        if before<=0 or item in sold or prices.get(item,0)<=1:continue\r\n        rival',
    b"=int(inv[item])-int(pinv[item])+town.get(item,0)\r\n        if rival<=0:continue\r\n        def planned(t):\r\n            future=_IMPL.chassis.routes[2 if t>=648 else native['route']][t]\r\n            return any(len(o)>=3 and o[0]=='SELL' and o[1]==item for o in future.get('market',[]))\r\n        # every member of this lineage sells at the scheduled turn, one turn early (sale lead) or up to four turns early\r\n        # (the base reservation): only a sale further ahead of the plan is a race\r\n        if any(planned(t) for t in range(step-1,min(719,step+5))):continue\r\n        if any(planned(t) for t in range(step+5,min(719,step+24))):return True\r\n    return False\r\n\r\ndef _race_snapshot(observation):\r\n    market=observation['market']\r\n    return dict(step=int(observation['step']),inventory=dict(market['inventory']),prices=dict(market['prices']),shops=list(observation['town'].get('unlocked_shops',[])),view=FarmView(observation))\r\n\r\ndef _r36_reserve(obs,action):\r\n    player=int(obs['player']);h=_RACE_STATE.get(player,{}).ge",
    b"t('horizon',0)\r\n    if h>_R37_HORIZONS.get(player,2):\r\n        saved=_R37_HORIZONS.get(player);_R37_HORIZONS[player]=h\r\n        try:return _RACE_ORIG_RESERVE(obs,action)\r\n        finally:\r\n            if saved is None:_R37_HORIZONS.pop(player,None)\r\n            else:_R37_HORIZONS[player]=saved\r\n    return _RACE_ORIG_RESERVE(obs,action)\r\n\r\ndef agent(observation,configuration=None):\r\n    state=None\r\n    try:\r\n        player=int(observation['player']);step=int(observation['step'])\r\n        state=_RACE_STATE.get(player)\r\n        if state is None or step<=state['step']:\r\n            state=_RACE_STATE[player]={'step':-1,'hist':[],'horizon':0,'level':_RACE_HORIZON_CLONE,'prev':None,'prev_action':None}\r\n        if step==0:_RACE_REPORT.update(race_clone_turns=0,race_horizon_turns=0,race_lost_races=0,race_escalations=0,race_errors=0)\r\n        state['step']=step;state['horizon']=0\r\n        # EXP288 mirror gate: a rival whose cash after the first turn equals ours executed the same first-turn\r\n        # round trip (a copy",
    b" of this agent); against a copy the sale race is won only by pre-empting the whole day.\r\n        if step==1:\r\n            try:\r\n                farms=observation['farms'];rival=farms[1-player]['money'];own=farms[player]['money']\r\n                state['level']=_RACE_HORIZON_MIRROR if (abs(float(rival)-float(own))<0.5 and _RACE_HORIZON_MIRROR>state['level']) else state['level']\r\n                _RACE_REPORT['race_mirror']=int(abs(float(rival)-float(own))<0.5)\r\n            except Exception:_RACE_REPORT['race_errors']+=1\r\n        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)])\r\n        if standard and 216<=step<696 and _race_clone(observation,state):\r\n            _RACE_REPORT['race_clone_turns']+=1\r\n            if state['level']<_RACE_HORIZON_ESCALATED and _race_lost(observation,state):\r\n                _RACE_REPORT['race_lost_races']+=1;state['level']=_RACE_HORIZON_ESCALATED;_RACE_REPORT['race_es",
    b'calations\']+=1\r\n            state[\'horizon\']=state[\'level\'];_RACE_REPORT[\'race_horizon_turns\']+=1\r\n    except Exception:_RACE_REPORT[\'race_errors\']+=1\r\n    snapshot=None\r\n    try:\r\n        if state is not None and 215<=int(observation[\'step\'])<696:snapshot=_race_snapshot(observation)\r\n    except Exception:_RACE_REPORT[\'race_errors\']+=1\r\n    action=_RACE_PARENT(observation,configuration)\r\n    try:\r\n        if state is not None:state[\'prev\']=snapshot;state[\'prev_action\']=action if snapshot is not None else None\r\n    except Exception:_RACE_REPORT[\'race_errors\']+=1\r\n    _RACE_REPORT.update(getattr(_RACE_PARENT,\'telemetry\',{}))\r\n    return action\r\nagent.telemetry=_RACE_REPORT\r\nagent=globals().pop(\'agent\')\r\n\r\n\r\n"""Fund urgent grain at the first quote slot; avoid unwatered last-hour plants.\r\nOriginal safety contracts by Ahmed Berat Ozer, EXP258.\r\n"""\r\n_R127_PARENT=agent\r\n_R127_STATES={}\r\n_R127_REPORT={}\r\n\r\ndef _r127_fields(obs,action):\r\n    farm,private=_PLANNER_NS[\'_clone_state\'](obs[\'farms\'][obs[\'player\']],obs[\'pr',
    b"ivate'])\r\n    commands=[action.get('farmer') or ['PASS'],*(action.get('hands') or [])]\r\n    demand={}\r\n    for c in commands:\r\n        if len(c)>1 and c[0]=='PLANT':demand[c[1]]=demand.get(c[1],0)+1\r\n    blocked={p for p,n in demand.items() if n>private['seeds'].get(p,0)}\r\n    for actor,c in enumerate(commands[:len(private['inventories'])]):\r\n        if len(c)>1 and c[0]=='PLANT' and c[1] in blocked:continue\r\n        _PLANNER_NS['_apply_unit_action'](farm,private,actor,c,10,int(obs['step'])//24,24,100)\r\n    return farm,private\r\n\r\ndef _r127_last_hour(obs,action):\r\n    if int(obs['step'])%24!=23:return action\r\n    commands=[action.get('farmer') or ['PASS'],*(action.get('hands') or [])]\r\n    if not any(c and c[0]=='PLANT' for c in commands):return action\r\n    result=copy.deepcopy(action);changed=False\r\n    # Removing rejected requests can unblock the engine's atomic crop batch.\r\n    # Recompute until every retained request ends the turn with a watered crop.\r\n    for _ in range(len(commands)+1):\r\n        farm,_=_",
    b"r127_fields(obs,result);original=obs['farms'][obs['player']]\r\n        positions=[original['farmer'],*original['hands']];drop=[]\r\n        for actor,c in enumerate(commands):\r\n            if not c or c[0]!='PLANT':continue\r\n            tile=None\r\n            if actor<len(positions):\r\n                x,y=positions[actor];tile=farm['tiles'][y][x]\r\n            if not (isinstance(tile,dict) and tile.get('kind')=='PLANT' and tile.get('crop')==c[1] and tile.get('planted_day')==int(obs['step'])//24 and tile.get('watered_today')):drop.append(actor)\r\n        if not drop:break\r\n        for actor in drop:commands[actor]=['PASS']\r\n        result['farmer']=commands[0];result['hands']=commands[1:];changed=True\r\n        _R127_REPORT['last_hour_plants_dropped']+=len(drop)\r\n    return result if changed else action\r\n\r\ndef _r127_prefix_bound(obs,quantity):\r\n    inventory=int(obs['market']['inventory']['WHEAT'])\r\n    # Both players quote one unit before either commits. Before own unit j,\r\n    # at most j-1 own and j-1 opponent whe",
    b"at purchases have depleted inventory.\r\n    return sum(_r37_market_price('WHEAT',inventory-(2*j-1)) for j in range(1,quantity+1))\r\n\r\ndef _r127_priority(obs,action,state):\r\n    step=int(obs['step']);player=int(obs['player'])\r\n    if not 144<=step<695:return action\r\n    orders=action.get('market') or []\r\n    if len(orders)>9 or any(o[:2] in (['BUY_PRODUCT','WHEAT'],['SELL','WHEAT']) for o in orders):return action\r\n    native=_IMPL.chassis.players[player]\r\n    future=_IMPL.chassis.routes[2 if step+1>=648 else native['route']][step+1]\r\n    commands=[future.get('farmer') or ['PASS'],*(future.get('hands') or [])]\r\n    if not any(c[:2]==['PICKUP','WHEAT'] for c in commands):return action\r\n    if not _r97_budget(obs,orders):return action\r\n    farm,private=_r127_fields(obs,action);night=step%24==23\r\n    access=((4,4),(5,4),(4,5),(5,5));positions=[tuple(farm['farmer']),*map(tuple,farm['hands'])]\r\n    if night:positions=[(4,4)]\r\n    else:\r\n        for o in orders:\r\n            if o and o[0]=='HIRE':positions.append(min(a",
    b"ccess,key=lambda p:(positions.count(p),access.index(p))))\r\n    need=sum(max(0,int(c[2]) if len(c)>2 else 1) for pos,c in zip(positions,commands) if pos in access and c[:2]==['PICKUP','WHEAT'])\r\n    stock,buys,_=_r97_market_stock(private['shed'],orders);before,loss=_r97_delivery(stock,private,night)\r\n    shortage=max(0,need-before.get('WHEAT',0))\r\n    if not shortage or shortage>100-sum(private['shed'].values()):return action\r\n    cost=_r127_prefix_bound(obs,shortage)\r\n    budget=dict(obs,farms=[dict(f) for f in obs['farms']]);budget['farms'][player]['money']-=cost\r\n    if not _r97_budget(budget,orders):return action\r\n    proposed=[['BUY_PRODUCT','WHEAT',shortage],*copy.deepcopy(orders)]\r\n    stock,after_buys,_=_r97_market_stock(private['shed'],proposed);after,after_loss=_r97_delivery(stock,private,night)\r\n    if after.get('WHEAT',0)<need or any(after_buys.get(i+1,0)<q for i,q in buys.items()) or any(q>loss.get(p,0) for p,q in after_loss.items()):return action\r\n    state['pending_grain']=(step+1,after.get('WHE",
    b"AT',0),shortage)\r\n    _R127_REPORT['priority_grain_orders']+=1;_R127_REPORT['priority_grain_units']+=shortage\r\n    return dict(action,market=proposed)\r\n\r\ndef agent(observation,configuration=None):\r\n    result=_R127_PARENT(observation,configuration)\r\n    try:\r\n        player=int(observation['player']);step=int(observation['step']);state=_R127_STATES.get(player)\r\n        if state is None or step<=state['step']:\r\n            state=_R127_STATES[player]={'step':-1}\r\n            _R127_REPORT.update(last_hour_plants_dropped=0,priority_grain_orders=0,priority_grain_units=0,priority_grain_confirmed=0,priority_grain_shortfalls=0,priority_contract_errors=0)\r\n        pending=state.pop('pending_grain',None)\r\n        if pending and step==pending[0]:\r\n            if observation['private']['shed'].get('WHEAT',0)>=pending[1]:_R127_REPORT['priority_grain_confirmed']+=pending[2]\r\n            else:_R127_REPORT['priority_grain_shortfalls']+=1\r\n        state['step']=step\r\n        standard=configuration is None or all(configuration",
    b".get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10),('farmHandCostMult',1)])\r\n        if standard:result=_r127_priority(observation,_r127_last_hour(observation,result),state)\r\n    except Exception:_R127_REPORT['priority_contract_errors']=_R127_REPORT.get('priority_contract_errors',0)+1\r\n    _R127_REPORT.update(getattr(_R127_PARENT,'telemetry',{}))\r\n    return result\r\nagent.telemetry=_R127_REPORT\r\nagent=globals().pop('agent')\r\n\r\n\r\n# ---- v44y pre-guard: quote at hour 21,22 what the hour-23 day-end storage guard (EXP-154) would dump ----\r\n# The guard sells shed stock by price desc once shed+carried exceeds 99 at hour 23. Selling those lots\r\n# a step earlier stays in the same town-consumption price window and quotes before a same-tape rival.\r\n_PG_HOST=[v for v in list(globals().values()) if callable(v)][-1]\r\n_Y_HOURS=(21, 22)\r\n_Y_ITEMS=('MILK', 'STRAWBERRY', 'MELON', 'WOOL', 'TOMATO')\r\n_Y_MIN_DAY=1\r\n_Y_MARGIN=-6\r\n_PG_REPORT={'preguard_turns':0,'preguard",
    b"_units':0,'preguard_errors':0}\r\n\r\ndef _y_preguard(obs,action):\r\n    step=int(obs['step'])\r\n    if step%24 not in _Y_HOURS or step//24<_Y_MIN_DAY or step>=696:return action\r\n    orders=[list(o) for o in (action.get('market') or [])]\r\n    if len(orders)>=10:return action\r\n    farm,private=_r127_fields(obs,action)\r\n    stock,_,_=_r97_market_stock(private['shed'],orders)\r\n    carried=sum(max(0,int(n)) for bag in private['inventories'] for n in bag.values())\r\n    needed=sum(max(0,int(v)) for v in stock.values())+carried-99-_Y_MARGIN\r\n    if needed<=0:return action\r\n    prices=obs['market']['prices'];extra=[]\r\n    for item in sorted(PRODUCTS,key=lambda it:-int(prices.get(it,0))):\r\n        avail=max(0,int(stock.get(item,0)));qty=min(needed,avail)\r\n        if qty<=0:continue\r\n        if item in _Y_ITEMS and int(prices.get(item,0))>=2:extra.append(['SELL',item,qty])\r\n        needed-=qty\r\n        if needed<=0:break\r\n    if not extra or len(orders)+len(extra)>10:return action\r\n    _PG_REPORT['preguard_turns']+=1;_PG_REP",
    b"ORT['preguard_units']+=sum(o[2] for o in extra)\r\n    return dict(action,market=orders+extra)\r\n\r\ndef agent_v44y_preguard(observation,configuration=None):\r\n    action=_PG_HOST(observation,configuration)\r\n    try:\r\n        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)])\r\n        if standard:action=_y_preguard(observation,action)\r\n    except Exception:_PG_REPORT['preguard_errors']+=1\r\n    return action\r\nagent_v44y_preguard.telemetry=_PG_REPORT\r\n\r\n\r\n# ---- v44y: clone-mode race horizon override ----\r\n_RACE_HORIZON_CLONE = 9\r\n\r\n# ---- v44y: exact lockstep best-response SELL ordering against a detected clone ----\r\n_V44Y_HOST = [v for v in list(globals().values()) if callable(v)][-1]\r\n_V44Y_REORDER_GATE = False\r\n_V44Y_REPORT = dict(v44y_reorder_turns=0, v44y_reorder_gain=0.0, v44y_errors=0)\r\nimport itertools as _v44y_it\r\n\r\ndef _v44y_price(item, inventory, params):\r\n    return _r37_market_price(item, in",
    b'ventory, params)\r\n\r\ndef _v44y_params(obs):\r\n    params = {k: dict(v) for k, v in _R37_MARKET_PARAMS.items()}\r\n    for k, patch in (obs[\'market\'].get(\'params\') or {}).items():\r\n        if k in params and isinstance(patch, dict): params[k].update(patch)\r\n    return params\r\n\r\ndef _v44y_lockstep(orders_me, orders_opp, inv0, stock_me, stock_opp, params):\r\n    """Replay the engine\'s per-slot / per-unit lockstep for SELL and BUY_PRODUCT orders (money-unbounded).\r\n    Returns (revenue_me, revenue_opp)."""\r\n    inv = dict(inv0); stock = [dict(stock_me), dict(stock_opp)]; rev = [0.0, 0.0]\r\n    queues = [list(orders_me), list(orders_opp)]\r\n    for i in range(max(len(queues[0]), len(queues[1]))):\r\n        rem = [None, None]\r\n        for p in (0, 1):\r\n            if i < len(queues[p]):\r\n                o = queues[p][i]\r\n                if o and len(o) >= 3 and o[0] in (\'SELL\', \'BUY_PRODUCT\') and o[1] in params:\r\n                    try: n = int(o[2])\r\n                    except Exception: n = 0\r\n                    if n >',
    b" 0: rem[p] = [o[0], o[1], n]\r\n        guard = 0\r\n        while True:\r\n            guard += 1\r\n            if guard > 5000: break\r\n            quoted = [None, None]\r\n            for p in (0, 1):\r\n                r = rem[p]\r\n                if r is None or r[2] <= 0: continue\r\n                if r[0] == 'SELL':\r\n                    quoted[p] = ('SELL', r[1], _v44y_price(r[1], inv[r[1]], params))\r\n                elif r[1] in ('WHEAT', 'FERTILIZER'):\r\n                    quoted[p] = ('BUY_PRODUCT', r[1], _v44y_price(r[1], inv[r[1]] - 1, params))\r\n                else:\r\n                    rem[p] = None\r\n            if quoted[0] is None and quoted[1] is None: break\r\n            committed = False\r\n            for p in (0, 1):\r\n                q = quoted[p]\r\n                if q is None: continue\r\n                op, item, price = q\r\n                if op == 'SELL':\r\n                    if stock[p].get(item, 0) <= 0:\r\n                        rem[p] = None; continue\r\n                    stock[p][item] -= 1; rev[p] +",
    b"= price\r\n                    if price > 1: inv[item] += 1\r\n                else:\r\n                    stock[p][item] = stock[p].get(item, 0) + 1; rev[p] -= price; inv[item] -= 1\r\n                rem[p][2] -= 1; committed = True\r\n            if not committed: break\r\n    return rev[0], rev[1]\r\n\r\ndef _v44y_factor_margin(opp, inv0, stock, params):\r\n    # EXP298: cache independent item schedules, preserving the donor's exact search/ties.\r\n    # The donor model has no shared cash/capacity constraint; per-item revenues add.\r\n    cache = {}\r\n    opp_schedules = {}\r\n    for i, order in enumerate(opp):\r\n        if order and len(order) >= 3 and order[0] in ('SELL', 'BUY_PRODUCT') and order[1] in params:\r\n            item = order[1]\r\n            padded = opp_schedules.setdefault(item, [[] for _ in opp])\r\n            padded[i] = order\r\n    def margin(cand):\r\n        schedules = {item: [] for item in opp_schedules}\r\n        for i, order in enumerate(cand):\r\n            if order and len(order) >= 3 and order[0] in ('SELL', ",
    b"'BUY_PRODUCT') and order[1] in params:\r\n                schedules.setdefault(order[1], []).append((i, order[0], int(order[2])))\r\n        total = 0.0\r\n        for item, schedule in schedules.items():\r\n            key = (item, tuple(schedule))\r\n            value = cache.get(key)\r\n            if value is None:\r\n                mine = [[] for _ in cand]\r\n                for i, op, n in schedule: mine[i] = [op, item, n]\r\n                theirs = opp_schedules.get(item, [[] for _ in opp])\r\n                a, b = _v44y_lockstep(mine, theirs, {item: inv0[item]},\r\n                                      {item: stock.get(item, 0)}, {item: stock.get(item, 0)},\r\n                                      {item: params[item]})\r\n                value = a - b\r\n                cache[key] = value\r\n            total += value\r\n        return total\r\n    return margin\r\n\r\ndef _v44y_reorder(obs, action):\r\n    market = action.get('market') or []\r\n    if len(market) < 2: return action\r\n    orders = [list(o) if isinstance(o, (list, tuple)) e",
    b"lse o for o in market]\r\n    blocks = []; i = 0\r\n    while i < len(orders):\r\n        o = orders[i]\r\n        if o and o[0] == 'SELL':\r\n            j = i\r\n            while j < len(orders) and orders[j] and orders[j][0] == 'SELL': j += 1\r\n            if 2 <= j - i <= 6: blocks.append((i, j))\r\n            i = j\r\n        else: i += 1\r\n    if not blocks: return action\r\n    view = FarmView(obs)\r\n    stock = projected_shed(action, view)\r\n    stock = {k: max(0, int(v)) for k, v in stock.items()}\r\n    params = _v44y_params(obs)\r\n    inv0 = {k: int(v) for k, v in obs['market']['inventory'].items()}\r\n    opp = [list(o) for o in orders]\r\n    margin = _v44y_factor_margin(opp, inv0, stock, params)\r\n    base = margin(orders); best = base; best_orders = None\r\n    for (i, j) in blocks:\r\n        blk = orders[i:j]; n = j - i\r\n        seen = set()\r\n        for perm in _v44y_it.permutations(range(n)):\r\n            key = tuple((blk[p][1], int(blk[p][2])) for p in perm)\r\n            if key in seen: continue\r\n            seen.add(key",
    b")\r\n            cand = orders[:i] + [blk[p] for p in perm] + orders[j:]\r\n            v = margin(cand)\r\n            if v > best + 0.5: best = v; best_orders = cand\r\n        if best_orders is not None:\r\n            orders = best_orders; best_orders = None\r\n    if best <= base + 0.5: return action\r\n    _V44Y_REPORT['v44y_reorder_turns'] += 1; _V44Y_REPORT['v44y_reorder_gain'] += best - base\r\n    out = dict(action); out['market'] = orders\r\n    return out\r\n\r\ndef _v44y_clone_gate(obs):\r\n    player = int(obs['player']); step = int(obs['step'])\r\n    st = _RACE_STATE.get(player) or {}\r\n    if st.get('horizon', 0) > 0: return True\r\n    if step >= 696 and len(st.get('hist', [])) >= 4 and sum(st['hist']) >= 4:\r\n        return _r37_similarity(obs) >= .95\r\n    return False\r\n\r\ndef v44y_lockstep_agent(observation, configuration=None):\r\n    action = _V44Y_HOST(observation, configuration)\r\n    try:\r\n        step = int(observation.get('step', 0))\r\n        if step >= 216 and (not _V44Y_REORDER_GATE or _v44y_clone_gate(observation",
    b")):\r\n            action = _v44y_reorder(observation, action)\r\n    except Exception:\r\n        _V44Y_REPORT['v44y_errors'] += 1\r\n    return action\r\nv44y_lockstep_agent.telemetry = _V44Y_REPORT\r\n\r\n# ==== v44y shop-aware herd wrapper (appended after the public V44 file; host entry captured first) ====\r\n_Y_HOST=[v for v in list(globals().values()) if callable(v)][-1]\r\n_Y_CFG={'yarnsheep': True, 'yarngeese': True, 'days': (8, 11)}\r\nimport copy as _y_copy\r\n_Y_PRODUCT={'COW':'MILK','SHEEP':'WOOL','GOOSE':'EGG'}\r\n_Y_STRUCT={'COW':'PASTURE','SHEEP':'PASTURE','GOOSE':'COOP'}\r\n_Y_COST={'COW':400,'SHEEP':500,'GOOSE':300}\r\n_Y_SEED={'WHEAT':10,'CARROT':20,'TOMATO':50,'STRAWBERRY':100,'MELON':80}\r\n_Y_STATES={}\r\n_Y_REPORT={'swaps':0,'pick':0,'place':0,'harvest':0,'boost':0,'errors':0,'coop':0,'declined':0}\r\n\r\ndef _y_new_state():\r\n    return {'last':-1,'pending':[],'credit':{},'sites':{},'sale':{},'coop_swap':0}\r\n\r\ndef _y_target(kind,shops,cfg):\r\n    yarn='YARN_STORE' in shops\r\n    egg=('BAKERY' in shops) or ('BRUNCH_SPOT' in ",
    b"shops)\r\n    milk=sum(s in ('PIZZA_SHOP','ICE_CREAM_SHOP','SMOOTHIE_SHOP') for s in shops)\r\n    if kind=='SHEEP' and cfg.get('nosheep') and not yarn and milk>=cfg.get('min_milk',0):return 'COW'\r\n    if kind=='COW' and cfg.get('yarnsheep') and yarn:return 'SHEEP'\r\n    if kind=='GOOSE' and cfg.get('nogeese') and not egg:return 'SHEEP' if yarn else 'COW'\r\n    if kind=='GOOSE' and cfg.get('yarngeese') and yarn:return 'SHEEP'\r\n    return None\r\n\r\ndef _y_fib(n):\r\n    a,b=1,1\r\n    for _ in range(n):a,b=b,a+b\r\n    return a\r\n\r\ndef _y_cash(obs,action,market):\r\n    farm=obs['farms'][int(obs['player'])];prices=obs['market']['prices'];shed=obs['private']['shed']\r\n    try:shed=projected_shed({'farmer':action.get('farmer') or ['PASS'],'hands':action.get('hands') or [],'market':[]},FarmView(obs))\r\n    except Exception:pass\r\n    cash=float(farm['money']);cost=0.0;hires=int(farm.get('hires_today',0) or 0)\r\n    quads=len(farm.get('unlocked_quadrants',[]) or [])\r\n    for o in market:\r\n        if not o:continue\r\n        op=o[0]\r\n  ",
    b"      if op=='SELL' and len(o)>=3:\r\n            cash+=0.8*min(max(0,int(o[2])),int(shed.get(o[1],0)))*float(prices.get(o[1],0))\r\n        elif op=='BUY_ANIMAL' and len(o)>=3:cost+=int(o[2])*_Y_COST.get(o[1],500)\r\n        elif op=='BUY_PRODUCT' and len(o)>=3:cost+=int(o[2])*(float(prices.get(o[1],0))+10)\r\n        elif op=='BUY_SEED' and len(o)>=3:cost+=int(o[2])*_Y_SEED.get(o[1],100)\r\n        elif op=='BUY_LAND':cost+=(1000,2000,4000)[min(2,max(0,quads-1))]\r\n        elif op=='HIRE':cost+=_y_fib(hires);hires+=1\r\n    return cash-cost\r\n\r\ndef _y_controller(obs,action,state,cfg):\r\n    step=int(obs['step']);day=step//24;seat=int(obs['player'])\r\n    farm=obs['farms'][seat];private=obs['private'];shed=private['shed'];inventories=private.get('inventories',[])\r\n    shops=list((obs.get('town') or {}).get('unlocked_shops',[]) or [])\r\n    tiles=farm['tiles'];n=len(tiles);center=n//2\r\n    # 1. confirm last step's swapped purchases (physical shed gain)\r\n    gained={}\r\n    for p in state['pending']:\r\n        to=p['to']\r\n      ",
    b"  if to not in gained:gained[to]=max(0,int(shed.get(to,0))-p['before'])\r\n        got=min(p['qty'],gained[to]);gained[to]-=got\r\n        if got>0:\r\n            key=(p['from'],to);state['credit'][key]=state['credit'].get(key,0)+got\r\n            if _Y_STRUCT[p['from']]!=_Y_STRUCT[to]:state['coop_swap']+=got\r\n    state['pending']=[]\r\n    result=_y_copy.deepcopy(action)\r\n    market=result.get('market') or []\r\n    result['market']=market\r\n    # 2. purchase-point substitution by unlocked shops (cash-checked)\r\n    if cfg['days'][0]<=day<=cfg['days'][1]:\r\n        for o in market:\r\n            if len(o)>=3 and o[0]=='BUY_ANIMAL' and o[1] in _Y_COST and 1<=int(o[2])<=cfg.get('maxq',2):\r\n                to=_y_target(o[1],shops,cfg)\r\n                if to is None or to==o[1]:continue\r\n                trial=[list(x) if isinstance(x,list) else x for x in market]\r\n                for t in trial:\r\n                    if isinstance(t,list) and t==o:t[1]=to\r\n                if _y_cash(obs,result,trial)<cfg.get('margin',100):\r\n  ",
    b"                  _Y_REPORT['declined']+=1;continue\r\n                state['pending'].append({'from':o[1],'to':to,'qty':int(o[2]),'before':int(shed.get(to,0))})\r\n                _Y_REPORT['swaps']+=int(o[2]);o[1]=to\r\n    # 3. worker command rewrites (PICKUP / PLACE / BUILD_COOP) and harvest credit\r\n    workers=[result.get('farmer') or ['PASS'],*(result.get('hands') or [])]\r\n    positions=[farm['farmer'],*farm['hands']]\r\n    avail={k:int(shed.get(k,0)) for k in _Y_COST}\r\n    seen=set();occupied=set()\r\n    for actor,work in enumerate(workers[:len(positions)]):\r\n        if not work or not isinstance(work,list):continue\r\n        inv=inventories[actor] if actor<len(inventories) else {}\r\n        x,y=positions[actor];tile=tiles[y][x];site=(x,y);op=work[0]\r\n        if op=='PICKUP' and len(work)>=2 and work[1] in _Y_COST:\r\n            kind=work[1];qty=max(1,int(work[2])) if len(work)>2 else 1\r\n            if avail.get(kind,0)>=qty:avail[kind]-=qty;continue\r\n            if not (x in (center-1,center) and y in (center-1",
    b",center)):continue\r\n            if any(inv.get(a,0) for a in _Y_COST):continue\r\n            for (frm,to),c in list(state['credit'].items()):\r\n                if frm==kind and c>=qty and avail.get(to,0)>=qty:\r\n                    work[1]=to;state['credit'][(frm,to)]=c-qty;avail[to]-=qty;_Y_REPORT['pick']+=qty;break\r\n        elif op=='PLACE' and len(work)>=2 and work[1] in _Y_COST:\r\n            kind=work[1]\r\n            if inv.get(kind,0)>0:continue\r\n            for to in ('COW','SHEEP','GOOSE'):\r\n                if (to!=kind and inv.get(to,0)>0 and isinstance(tile,dict) and tile.get('kind')==_Y_STRUCT[to]\r\n                        and not tile.get('animal') and site not in occupied):\r\n                    work[1]=to;state['sites'][site]=to;occupied.add(site);_Y_REPORT['place']+=1;break\r\n        elif op=='BUILD_COOP' and state['coop_swap']>0:\r\n            work[0]='BUILD_PASTURE';_Y_REPORT['coop']+=1\r\n        elif op=='HARVEST' and site in state['sites'] and site not in seen:\r\n            if isinstance(tile,dict) ",
    b"and tile.get('animal')==state['sites'][site]:\r\n                units=max(0,int(tile.get('yield_units',0) or 0))\r\n                if units:\r\n                    prod=_Y_PRODUCT[tile['animal']];state['sale'][prod]=state['sale'].get(prod,0)+units;_Y_REPORT['harvest']+=units\r\n            seen.add(site)\r\n    result['farmer'],result['hands']=workers[0],workers[1:]\r\n    # 4. sell the extra production at the tape's own existing sale slots (never add new orders)\r\n    if cfg.get('boost',True):\r\n        for prod,credit in list(state['sale'].items()):\r\n            credit=min(credit,int(shed.get(prod,0)))\r\n            state['sale'][prod]=credit\r\n            if credit<=0:continue\r\n            planned=sum(max(0,int(o[2])) for o in market if len(o)>=3 and o[0]=='SELL' and o[1]==prod)\r\n            if planned<=0:continue\r\n            extra=min(credit,int(shed.get(prod,0))-planned)\r\n            if extra<=0:continue\r\n            for o in market:\r\n                if len(o)>=3 and o[0]=='SELL' and o[1]==prod and int(o[2])>0:\r\n    ",
    b"                o[2]=int(o[2])+extra;state['sale'][prod]=credit-extra;_Y_REPORT['boost']+=extra;break\r\n    return result\r\n\r\ndef _y_agent_shopherd(observation,configuration=None):\r\n    action=_Y_HOST(observation,configuration)\r\n    try:\r\n        seat=int(observation['player']);step=int(observation['step'])\r\n        state=_Y_STATES.get(seat)\r\n        if state is None or step<=state['last']:state=_Y_STATES[seat]=_y_new_state()\r\n        state['last']=step\r\n        return _y_controller(observation,action,state,_Y_CFG)\r\n    except Exception:\r\n        _Y_REPORT['errors']+=1\r\n        return action\r\n\r\n# V47 attribution and modification notice (17 September 2026):\r\n# Seyit Kaan Gunes, kaggle.com/code/seyitkaangunes/kaggriculture-2820-score:\r\n# pre-overflow sale guard, clone-market lockstep ordering/horizon, and shop-aware herd layers.\r\n# Ahmed Berat Ozer: transfer onto V46 EXP293, exact per-product score caching,\r\n# exact physical-prefix/resource reuse, terminal no-op/movement fast paths,\r\n# private reacting-opponent c",
    b"onfirmation and packaging.\r\n# Original source notices and license text above are retained.\r\n\r\n# EXP334: remove useless cash-product sale slots without moving purchases.\r\n_E334_ITEMS={'CARROT','TOMATO','STRAWBERRY','MELON','EGG','MILK','WOOL'}\r\n_E334_REPORT=dict(changed=0,removed=0,errors=0)\r\n_E334_BASE=_y_agent_shopherd\r\n\r\ndef _e334_compact(obs,action):\r\n    market=action.get('market') or []\r\n    if int(obs['step'])<144 or len(market)<2:return action\r\n    segments=[];i=0\r\n    while i<len(market):\r\n        if len(market[i])>=3 and market[i][0]=='SELL' and market[i][1] in _E334_ITEMS:\r\n            j=i+1\r\n            while j<len(market) and len(market[j])>=3 and market[j][0]=='SELL' and market[j][1] in _E334_ITEMS:j+=1\r\n            if j-i>=2:segments.append((i,j))\r\n            i=j\r\n        else:i+=1\r\n    if not segments:return action\r\n    _,private=_r127_fields(obs,action);remaining=dict(private['shed']);new=[list(o) for o in market];removed=0\r\n    for start,end in segments:\r\n        quantities={};order=[]\r\n    ",
    b"    for o in market[start:end]:\r\n            p=o[1]\r\n            if p not in quantities:order.append(p);quantities[p]=0\r\n            quantities[p]+=max(0,int(o[2]))\r\n        kept=[]\r\n        for p in order:\r\n            q=min(quantities[p],max(0,int(remaining.get(p,0))))\r\n            if q:kept.append(['SELL',p,q]);remaining[p]-=q\r\n        # Empty order slots are explicitly skipped by the pinned engine parser.\r\n        # Keep external order indices unchanged; do not pull BUY/HIRE forward.\r\n        replacement=kept+[[] for _ in range(end-start-len(kept))]\r\n        if replacement!=market[start:end]:removed+=end-start-len(kept);new[start:end]=replacement\r\n    if new==market:return action\r\n    _E334_REPORT['changed']+=1;_E334_REPORT['removed']+=removed\r\n    return dict(action,market=new)\r\n\r\ndef _e334_agent(observation,configuration=None):\r\n    if int(observation.get('step',0))==0:\r\n        for k in _E334_REPORT:_E334_REPORT[k]=0\r\n    action=_E334_BASE(observation,configuration)\r\n    try:return _e334_compact(observ",
    b"ation,action)\r\n    except Exception:\r\n        _E334_REPORT['errors']+=1;return action\r\n\r\n# EXP335: empty buyable-product sales are holes too, when no purchases exist.\r\n_E335_ORIGINAL_COMPACT=_e334_compact\r\n\r\ndef _e334_compact(obs,action):\r\n    market=action.get('market') or []\r\n    if int(obs['step'])<144 or len(market)<2:return action\r\n    if not all(len(o)>=3 and o[0]=='SELL' for o in market):return _E335_ORIGINAL_COMPACT(obs,action)\r\n    _,private=_r127_fields(obs,action);remaining=dict(private['shed']);effective=[]\r\n    for o in market:\r\n        item=o[1];q=min(max(0,int(o[2])),max(0,int(remaining.get(item,0))))\r\n        remaining[item]=max(0,int(remaining.get(item,0))-q)\r\n        effective.append(['SELL',item,q] if q else [])\r\n    new=[list(o) for o in effective];i=0;removed=0\r\n    while i<len(effective):\r\n        if effective[i] and effective[i][1] not in _E334_ITEMS:i+=1;continue\r\n        j=i+1\r\n        while j<len(effective) and (not effective[j] or effective[j][1] in _E334_ITEMS):j+=1\r\n        order=",
    b"[];qty={}\r\n        for o in effective[i:j]:\r\n            if not o:continue\r\n            if o[1] not in qty:order.append(o[1]);qty[o[1]]=0\r\n            qty[o[1]]+=o[2]\r\n        kept=[['SELL',item,qty[item]] for item in order]\r\n        new[i:j]=kept+[[] for _ in range(j-i-len(kept))];removed+=j-i-len(kept);i=j\r\n    if new==market:return action\r\n    _E334_REPORT['changed']+=1;_E334_REPORT['removed']+=removed\r\n    return dict(action,market=new)\r\n\r\ndef _e335_agent(observation,configuration=None):\r\n    return _e334_agent(observation,configuration)\r\n\r\n\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# v11 entry normalisation: Kaggle's loader takes the last callable in the module\r\n# namespace.  Rebind it to a fresh `agent` so the packaged name matches the rest\r\n# of the v9 releases (and so validate.py's name check passes).\r\n# ---------------------------------------------------------------------------\r\n_V11_ENTRY = [v for v in list(globals().values()) if callable(v)][-1]\r\n\r\n\r\ndef ag",
    b"ent(observation, configuration=None):\r\n    return _V11_ENTRY(observation, configuration)\r\n\r\n\r\nagent.telemetry = getattr(_V11_ENTRY, 'telemetry', {})\r\nagent = globals().pop('agent')\r\n\r\n\r\n# ---------------------------------------------------------------------------\r\n# v13 V219SKIP (review suggestion #1): V219 hires a tomato crew every day from\r\n# day 19, but tomatoes planted on day 18 only produce at the refreshes ending\r\n# days 25-28.  Before that, watering only keeps them alive, and a plant\r\n# survives one dry day (it dies after two consecutive dry refreshes).  On the\r\n# chosen skip days, when every committed tomato was watered yesterday\r\n# (consecutive_unwatered == 0), no crew is hired and nobody waters them today;\r\n# the next day's crew waters them again.\r\n# ---------------------------------------------------------------------------\r\nV13V_SKIP_DAYS = (19, 21, 23)\r\n_V13V_REPORT = dict(v_skipped=0, v_blocked=0, v_errors=0)\r\n_V13V_ORIG_REQUEST = _v219_request\r\n\r\n\r\ndef _v219_request(obs, action, state, native):",
    b'\r\n    try:\r\n        day = int(obs["step"]) // 24\r\n        if day in V13V_SKIP_DAYS and state.get("committed") and state.get("requested_day") != day:\r\n            farm = obs["farms"][int(obs["player"])]\r\n            tomatoes = 0\r\n            safe = True\r\n            for x, y in state.get("targets", []):\r\n                tile = farm["tiles"][y][x]\r\n                if isinstance(tile, dict) and tile.get("crop") == "TOMATO":\r\n                    tomatoes += 1\r\n                    if int(tile.get("consecutive_unwatered", 1)) != 0:\r\n                        safe = False\r\n            if tomatoes and safe:\r\n                state["requested_day"] = day\r\n                _V13V_REPORT["v_skipped"] += 1\r\n                return action\r\n            if tomatoes:\r\n                _V13V_REPORT["v_blocked"] += 1\r\n    except Exception:\r\n        _V13V_REPORT["v_errors"] += 1\r\n    return _V13V_ORIG_REQUEST(obs, action, state, native)\r\n\r\n\r\n_V13V_PARENT = agent\r\n\r\n\r\ndef agent(observation, configuration=None):\r\n    if int(observation[',
    b'"step"]) == 0:\r\n        for k in _V13V_REPORT:\r\n            _V13V_REPORT[k] = 0\r\n    return _V13V_PARENT(observation, configuration)\r\n\r\n\r\nagent.telemetry = _V13V_REPORT\r\nagent = globals().pop("agent")\r\n\r\n\r\n# ===========================================================================\r\n# AXIS 1.1: Weed Lag Replay Recovery (_e343_weedlag)\r\n# ===========================================================================\r\n_E343_WL_REPORT = dict(events=0, replays=0, errors=0)\r\n_E343_WL_STATE = {}\r\n_E343_WL_BASE = agent\r\n_E343_WL_STEPS = 8\r\n_E343_WL_OPS = (\'BUILD_PASTURE\', \'BUILD_COOP\')\r\n\r\ndef _e343_tape_units(player, t):\r\n    native = _IMPL.chassis.players.get(player)\r\n    if not native or t < 0 or t > 718: return []\r\n    route = 2 if t >= 648 else native.get(\'route\', 0)\r\n    e = _IMPL.chassis.routes[route][t]\r\n    return [e.get(\'farmer\') or [\'PASS\'], *(e.get(\'hands\') or [])]\r\n\r\ndef _e343_weedlag(obs, action):\r\n    step = int(obs[\'step\']); player = int(obs[\'player\']); st = _E343_WL_STATE.setdefault(player, {\'step\': -1',
    b", 'active': {}})\r\n    if step <= st['step']: st['active'] = {}\r\n    st['step'] = step\r\n    if step % 24 == 0: st['active'] = {}   # hands are re-hired each day; lag state cannot survive day boundary\r\n    farm = obs['farms'][player]; positions = [tuple(farm['farmer']), *map(tuple, farm['hands'])]; tiles = farm['tiles']\r\n    units = [list(action.get('farmer') or ['PASS']), *[list(h) if isinstance(h, list) else ['PASS'] for h in (action.get('hands') or [])]]\r\n    tape_now = _e343_tape_units(player, step); tape_prev = _e343_tape_units(player, step - 1)\r\n    changed = False\r\n    # 1. continue active lags\r\n    for k, tx in list(st['active'].items()):\r\n        if k >= len(units) or k >= len(positions): st['active'].pop(k, None); continue\r\n        age = step - tx['start']\r\n        if age == 1: units[k] = list(tx['intended']); changed = True; _E343_WL_REPORT['replays'] += 1\r\n        elif 2 <= age <= 1 + _E343_WL_STEPS:\r\n            prev = tape_prev[k] if k < len(tape_prev) else ['PASS']\r\n            units[k] = list(pr",
    b"ev) if isinstance(prev, list) and prev else ['PASS']; changed = True; _E343_WL_REPORT['replays'] += 1\r\n        else: st['active'].pop(k, None)\r\n    # 2. detect new weed-blocked build/plant events (chassis has turned them into DIG)\r\n    for k in range(min(len(units), len(positions), len(tape_now))):\r\n        if k in st['active']: continue\r\n        intent = tape_now[k]\r\n        if not (isinstance(intent, list) and intent and intent[0] in _E343_WL_OPS): continue\r\n        x, y = positions[k]; tile = tiles[y][x]\r\n        if not (isinstance(tile, dict) and tile.get('kind') == 'WEED'): continue\r\n        if units[k] != ['DIG']: units[k] = ['DIG']; changed = True\r\n        st['active'][k] = {'start': step, 'intended': list(intent)}; _E343_WL_REPORT['events'] += 1\r\n    if not changed: return action\r\n    return dict(action, farmer=units[0], hands=units[1:])\r\n\r\ndef agent(observation, configuration=None):\r\n    if int(observation.get('step', 0)) == 0:\r\n        for k in _E343_WL_REPORT: _E343_WL_REPORT[k] = 0\r\n        _E343_",
    b"WL_STATE.clear()\r\n    action = _E343_WL_BASE(observation, configuration)\r\n    try:\r\n        standard = configuration is None or all(configuration.get(k, v) == v for k, v in [('boardSize', 10), ('turnsPerDay', 24), ('shedCapacity', 100), ('maxMarketOrdersPerTurn', 10)])\r\n        if standard: action = _e343_weedlag(observation, action)\r\n    except Exception:\r\n        _E343_WL_REPORT['errors'] += 1\r\n    return action\r\n\r\nagent.telemetry = _E343_WL_REPORT\r\nkaggle_agent = agent\r\n\r\n\r\n# ===========================================================================\r\n# AXIS 2.1: Ready-Stock Sale Advancing & Frontloading (_adv_apply, _adv_frontload)\r\n# ===========================================================================\r\n_ADV_PARENT = agent\r\n_ADV_LOOK = 3\r\n_ADV_FROM = 216\r\n_ADV_TO = 718\r\n_ADV_PROTECT = True\r\n_ADV_FRONT = False\r\n_ADV_BOOK = False\r\n_ADV_SUBTRACT_DEBTS = True\r\n_ADV_ITEMS = ('STRAWBERRY', 'WOOL', 'EGG', 'MILK', 'MELON', 'CARROT', 'TOMATO')\r\n_ADV_REPORT = dict(adv_turns=0, adv_units=0, adv_errors=0, fron",
    b"t_turns=0)\r\n\r\ndef _adv_future(player, t):\r\n    native = _IMPL.chassis.players.get(player)\r\n    if not native or t < 0 or t > 718: return []\r\n    route = 2 if t >= 648 else native.get('route', 0)\r\n    return _IMPL.chassis.routes[route][t].get('market', []) or []\r\n\r\ndef _adv_apply(obs, action):\r\n    step = int(obs['step']); player = int(obs['player'])\r\n    if step % 24 == 23 or not _ADV_FROM <= step < _ADV_TO: return action\r\n    native = _IMPL.chassis.players.get(player)\r\n    if not native: return action\r\n    debts = native.setdefault('sell_state', {}).setdefault('r36_debts', {})\r\n    plan = []; first = None\r\n    for off in range(1, _ADV_LOOK + 1):\r\n        t = step + off\r\n        if t > 718: break\r\n        for o in _adv_future(player, t):\r\n            if not o or len(o) < 3: continue\r\n            if first is None: first = o\r\n            if o[0] == 'SELL' and o[1] in _ADV_ITEMS:\r\n                try: q = max(0, int(o[2]))\r\n                except Exception: q = 0\r\n                if _ADV_SUBTRACT_DEBTS: q -= deb",
    b"ts.get(t, {}).get(o[1], 0)\r\n                if q > 0: plan.append((t, o[1], q))\r\n    protected = first[1] if _ADV_PROTECT and first is not None and first[0] == 'SELL' else None\r\n    plan = [(t, item, q) for t, item, q in plan if item != protected]\r\n    if not plan: return action\r\n    market = [list(o) for o in (action.get('market') or [])]\r\n    if any(len(o) > 1 and o[0] == 'BUY_PRODUCT' for o in market): return action\r\n    stock = projected_shed(action, FarmView(obs))\r\n    selling = {}\r\n    for o in market:\r\n        if len(o) >= 3 and o[0] == 'SELL':\r\n            try: selling[o[1]] = selling.get(o[1], 0) + max(0, int(o[2]))\r\n            except Exception: return action\r\n    commands = [action.get('farmer') or ['PASS'], *(action.get('hands') or [])]\r\n    picked = {c[1] for c in commands if len(c) > 1 and c[0] == 'PICKUP'}\r\n    prices = obs['market']['prices']; added = 0; extra = []; booked = []\r\n    for item in sorted({it for _, it, _ in plan}, key=lambda it: -int(prices.get(it, 0))):\r\n        if item in picke",
    b"d or int(prices.get(item, 0)) < 2: continue\r\n        avail = int(stock.get(item, 0)) - selling.get(item, 0)\r\n        if avail < 1: continue\r\n        hit = next((o for o in market if len(o) >= 3 and o[0] == 'SELL' and o[1] == item), None)\r\n        if hit is None and len(market) + len(extra) >= 10: continue\r\n        n = 0\r\n        for t, it, q in plan:\r\n            if it != item or avail <= 0: continue\r\n            take = min(q, avail); booked.append((t, item, take)); n += take; avail -= take\r\n        if n < 1: continue\r\n        if hit is not None: hit[2] = int(hit[2]) + n\r\n        else: extra.append(['SELL', item, n])\r\n        added += n\r\n    if not added: return action\r\n    for t, item, take in (booked if _ADV_BOOK else []):\r\n        d = debts.setdefault(t, {}); d[item] = d.get(item, 0) + take\r\n    _ADV_REPORT['adv_turns'] += 1; _ADV_REPORT['adv_units'] += added\r\n    return dict(action, market=extra + market)\r\n\r\ndef _adv_frontload(obs, action):\r\n    market = [list(o) for o in (action.get('market') or []) if o",
    b"]\r\n    if len(market) < 2 or int(obs['step']) < _ADV_FROM: return action\r\n    buys = {o[1] for o in market if len(o) > 1 and o[0] == 'BUY_PRODUCT'}\r\n    front = [o for o in market if len(o) >= 3 and o[0] == 'SELL' and o[1] not in buys]\r\n    mid = [o for o in market if (len(o) >= 3 and o[0] == 'BUY_PRODUCT') or (len(o) >= 3 and o[0] == 'SELL' and o[1] in buys)]\r\n    rest = [o for o in market if o not in front and o not in mid]\r\n    new = front + mid + rest\r\n    if new == market: return action\r\n    _ADV_REPORT['front_turns'] = _ADV_REPORT.get('front_turns', 0) + 1\r\n    return dict(action, market=new)\r\n\r\ndef agent(observation, configuration=None):\r\n    action = _ADV_PARENT(observation, configuration)\r\n    try:\r\n        if int(observation.get('step', 0)) == 0:\r\n            _ADV_REPORT.update(adv_turns=0, adv_units=0, adv_errors=0, front_turns=0)\r\n        standard = configuration is None or all(configuration.get(k, v) == v for k, v in [('boardSize', 10), ('turnsPerDay', 24), ('shedCapacity', 100), ('maxMarketOrder",
    b"sPerTurn', 10)])\r\n        if standard: action = _adv_apply(observation, action)\r\n        if standard and _ADV_FRONT: action = _adv_frontload(observation, action)\r\n        st = _RACE_STATE.get(int(observation['player']))\r\n        if st is not None and st.get('prev_action') is not None and st.get('step') == int(observation['step']):\r\n            st['prev_action'] = action\r\n    except Exception:\r\n        _ADV_REPORT['adv_errors'] += 1\r\n    return action\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    return agent(observation, configuration)\r\n\r\ncha20_entry_agent.telemetry = _ADV_REPORT\r\nkaggle_agent = cha20_entry_agent\r\n\r\n\r\n# ===========================================================================\r\n# AXIS 6.2A: Terminal 7-Turn Zero-Waste Liquidation (T_TERMINAL_CLEAR)\r\n# ===========================================================================\r\n_T62A_PARENT = agent\r\n_T62A_REPORT = dict(term_sells=0, term_units=0)\r\n\r\ndef _t62a_agent(observation, configuration=None):\r\n    action = _T62A_PARENT(",
    b"observation, configuration)\r\n    try:\r\n        step = int(observation.get('step', 0))\r\n        if step >= 712 and action and isinstance(action, dict):\r\n            priv = observation.get('private', {})\r\n            shed = priv.get('shed', {})\r\n            prices = observation.get('market', {}).get('prices', {})\r\n            # Find all non-zero shed goods\r\n            items_to_sell = [it for it, q in shed.items() if int(q) > 0 and int(prices.get(it, 0)) >= 1]\r\n            if items_to_sell:\r\n                # Sort by price descending\r\n                items_to_sell.sort(key=lambda it: -int(prices.get(it, 0)))\r\n                # Build market orders strictly selling these goods\r\n                market = []\r\n                for it in items_to_sell[:10]:\r\n                    market.append(['SELL', it, int(shed[it])])\r\n                    _T62A_REPORT['term_units'] += int(shed[it])\r\n                action['market'] = market\r\n                _T62A_REPORT['term_sells'] += 1\r\n    except Exception:\r\n        pass\r\n    ret",
    b"urn action\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    return _t62a_agent(observation, configuration)\r\n\r\ncha20_entry_agent.telemetry = _T62A_REPORT\r\nkaggle_agent = cha20_entry_agent\r\n\r\n# ===========================================================================\r\n# PIPE16 HybridOpening port onto cha20 (mode=HybridOpening)\r\n# ===========================================================================\r\n_PIPE_MODE = 'EarlyCycle'\r\n_PIPE_RAW = copy.deepcopy(_IMPL.chassis.routes[0][:96])\r\n_PIPE_CT = dict(CT_TABLE)\r\n_PIPE_STATE = {}\r\n_PIPE_REPORT = {}\r\n\r\ndef _pipe_install(mode):\r\n    global CT_TABLE\r\n    tape=_IMPL.chassis.routes[0]\r\n    tape[:96]=copy.deepcopy(_PIPE_RAW)\r\n    CT_TABLE=dict(_PIPE_CT)\r\n    if mode=='Original': return\r\n    if mode=='HybridOpening':\r\n        tape[0]['market']=list(tape[0]['market'])+[['BUY_SEED','WHEAT',1]]\r\n    else:\r\n        CT_TABLE={}\r\n        tape[0]['market']=[['BUY_PRODUCT','WHEAT',5],['BUY_SEED','WHEAT',1]]\r\n        tape[1]['market']=[o for o in tape[1]['mar",
    b"ket'] if not (len(o)>=3 and o[0] in ('BUY_PRODUCT','SELL') and o[1]=='WHEAT')]\r\n    # Day-zero idle hand 1 walks from (5,4) to the future (2,4) pasture.\r\n    for step,command in {2:['WEST'],3:['WEST'],4:['WEST'],5:['PLANT','WHEAT'],6:['WATER']}.items():\r\n        assert tape[step]['hands'][1]==['PASS']\r\n        tape[step]['hands'][1]=command\r\n    # On day one, water instead of building the not-yet-needed pasture.\r\n    assert tape[29]['hands'][2]==['BUILD_PASTURE']\r\n    tape[29]['hands'][2]=['WATER']\r\n    # Day-two idle hand 0: water, harvest, restore pasture, deliver.\r\n    commands=[['WEST'],['WEST'],['WEST'],['WATER'],['HARVEST'],['BUILD_PASTURE'],['EAST'],['EAST'],['DROP']]\r\n    for step,command in zip(range(49,58),commands):\r\n        assert tape[step]['hands'][0]==['PASS']\r\n        tape[step]['hands'][0]=command\r\n    # OMW port (notebook 2743, Dmitrii Gluzdov): mature the temporary wheat one\r\n    # extra day, then harvest three instead of two and deliver at step 91.\r\n    if mode in ('EarlyCycle','HybridOpen",
    b"ing'):\r\n        for step in range(53,58):\r\n            tape[step]['hands'][0]=['PASS']\r\n        omw=[['EAST'],['EAST'],['WATER'],['HARVEST'],['BUILD_PASTURE'],['EAST'],['EAST'],['DROP']]\r\n        for step,command in zip(range(84,92),omw):\r\n            assert tape[step]['hands'][0]==['PASS'], ('OMW tape mismatch at step %d' % step)\r\n            tape[step]['hands'][0]=command\r\n    if mode=='TomatoInsteadOfCow':\r\n        tape[0]['market'].append(['BUY_SEED','TOMATO',1])\r\n        cow=next(o for o in tape[1]['market'] if o[:2]==['BUY_ANIMAL','COW'])\r\n        assert cow[2]==2; cow[2]=1\r\n        # The first carrier still owns the bought cow; the other cow's site\r\n        # becomes one tomato plant, with its worker route left in place.\r\n        assert tape[2]['hands'][4][:2]==['PICKUP','COW']\r\n        assert tape[3]['hands'][4]==['BUILD_PASTURE']\r\n        assert tape[4]['hands'][4][:2]==['PLACE','COW']\r\n        tape[2]['hands'][4]=['PASS']\r\n        tape[3]['hands'][4]=['PASS']\r\n        tape[4]['hands'][4]=['PLANT','T",
    b"OMATO']\r\n        tape[5]['hands'][4]=['WATER']\r\n    assert max(len(a.get('market',[])) for a in tape[:96])<=10\r\n\r\ndef _pipe_sell_extra(action,item,n):\r\n    if n<=0:return action\r\n    orders=[list(o) for o in action.get('market',[])]\r\n    sell=next((o for o in orders if len(o)>=3 and o[:2]==['SELL',item]),None)\r\n    if sell is not None:sell[2]+=n\r\n    elif len(orders)<10:orders.append(['SELL',item,n])\r\n    else:return action\r\n    return dict(action,market=orders)\r\n\r\n_PIPE_PARENT = cha20_entry_agent\r\n\r\ndef _pipe_agent(observation, configuration=None):\r\n    seat, step = int(observation['player']), int(observation['step'])\r\n    state = _PIPE_STATE.get(seat)\r\n    if state is None or step <= state['step']:\r\n        mode = _PIPE_MODE\r\n        _pipe_install(mode)\r\n        state = _PIPE_STATE[seat] = {'step': -1, 'mode': mode}\r\n        _PIPE_REPORT.clear()\r\n        _PIPE_REPORT.update(selected_opening=mode, temporary_crop_seen=0, temporary_crop_harvested=0,\r\n                           restored_pasture_seen=0, delivere",
    b"d_extra_wheat=0, tomato_seen=0,\r\n                           tomato_water_requests=0, tomato_harvest_requests=0, tomato_units_harvest_requested=0,\r\n                           extension_errors=0)\r\n    action = _PIPE_PARENT(observation, configuration)\r\n    try:\r\n        mode = state['mode']\r\n        if mode != 'Original':\r\n            farm = observation['farms'][seat]; private = observation['private']\r\n            site = farm['tiles'][4][2]\r\n            if step == 6:\r\n                _PIPE_REPORT['temporary_crop_seen'] = int(isinstance(site, dict) and site.get('crop') == 'WHEAT')\r\n            if step == 54:\r\n                _PIPE_REPORT['temporary_crop_harvested'] = int(private['inventories'][1].get('WHEAT', 0))\r\n            if step == 55:\r\n                _PIPE_REPORT['restored_pasture_seen'] = int(isinstance(site, dict) and site.get('kind') == 'PASTURE')\r\n            if step in (57, 91) and len(farm['hands']) >= 1:\r\n                if tuple(farm['hands'][0]) == (4, 4) and action.get('hands', [[]])[0] == ['DROP",
    b"']:\r\n                    n = int(private['inventories'][1].get('WHEAT', 0))\r\n                    action = _pipe_sell_extra(action, 'WHEAT', n)\r\n                    _PIPE_REPORT['delivered_extra_wheat'] = n\r\n            if mode == 'TomatoInsteadOfCow':\r\n                tomato = farm['tiles'][4][4]\r\n                if isinstance(tomato, dict) and tomato.get('crop') == 'TOMATO' and tomato.get('planted_day') == 0:\r\n                    _PIPE_REPORT['tomato_seen'] = 1\r\n                    units = [list(action.get('farmer') or ['PASS'])] + [list(c) for c in action.get('hands', [])]\r\n                    positions = [tuple(farm['farmer'])] + [tuple(p) for p in farm['hands']]\r\n                    watered = bool(tomato.get('watered_today')); yield_left = int(tomato.get('yield_units', 0))\r\n                    for actor, pos in enumerate(positions):\r\n                        if pos != (4, 4) or actor >= len(units):\r\n                            continue\r\n                        if units[actor][0] not in ('FEED', 'CARE', 'CO",
    b"LLECT_FERTILIZER', 'HARVEST', 'WATER', 'PASS'):\r\n                            continue\r\n                        if not watered:\r\n                            units[actor] = ['WATER']; watered = True\r\n                            _PIPE_REPORT['tomato_water_requests'] += 1\r\n                        elif yield_left > 0:\r\n                            units[actor] = ['HARVEST']\r\n                            _PIPE_REPORT['tomato_harvest_requests'] += 1\r\n                            _PIPE_REPORT['tomato_units_harvest_requested'] += yield_left\r\n                            yield_left = 0\r\n                        else:\r\n                            units[actor] = ['PASS']\r\n                    action = dict(action, farmer=units[0], hands=units[1:])\r\n                stock = int(private['shed'].get('TOMATO', 0))\r\n                scheduled = sum(int(o[2]) for o in action.get('market', []) if len(o) >= 3 and o[:2] == ['SELL', 'TOMATO'])\r\n                action = _pipe_sell_extra(action, 'TOMATO', max(0, stock - scheduled))\r\n    exc",
    b"ept Exception:\r\n        _PIPE_REPORT['extension_errors'] += 1\r\n    state['step'] = step\r\n    return action\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    return _pipe_agent(observation, configuration)\r\n\r\ncha20_entry_agent.telemetry = _PIPE_REPORT\r\nkaggle_agent = cha20_entry_agent\r\n\r\n\r\n\r\n\r\n# ===========================================================================\r\n# MIRROR-LIKE CLASSIFIER (behavior only; no recorded streams)\r\n# ===========================================================================\r\n_MA_REPORT = dict(classified=0, mirror_like=0, market_drop=0, errors=0)\r\n_MA_STATE = dict(prior=None, mirror_like=False)\r\n_MA_INNER = cha20_entry_agent\r\n\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    step = int(observation.get('step', 0))\r\n    if step == 0:\r\n        _MA_STATE.update(prior=None, mirror_like=False)\r\n        _MA_REPORT.update(classified=0, mirror_like=0, market_drop=0, errors=0)\r\n    if step == 1:\r\n        try:\r\n            _MA_STATE['prior'] = int(observation",
    b"['market']['inventory']['WHEAT'])\r\n        except Exception:\r\n            _MA_REPORT['errors'] += 1\r\n    if step == 2:\r\n        try:\r\n            prior = _MA_STATE['prior']\r\n            if prior is not None:\r\n                fall = prior - int(observation['market']['inventory']['WHEAT'])\r\n                _MA_STATE['mirror_like'] = fall >= 15\r\n                _MA_REPORT['classified'] = 1\r\n                _MA_REPORT['mirror_like'] = int(fall >= 15)\r\n                _MA_REPORT['market_drop'] = fall\r\n        except Exception:\r\n            _MA_REPORT['errors'] += 1\r\n    return _MA_INNER(observation, configuration)\r\n\r\n\r\nimport collections as _ma_collections\r\ncha20_entry_agent.telemetry = _ma_collections.ChainMap(_MA_REPORT, _MA_INNER.telemetry)\r\nkaggle_agent = cha20_entry_agent\r\n_MA_ENTRY = cha20_entry_agent\r\n\r\n# ===========================================================================\r\n# WHEAT BUY-FIRST (gated): profitable pairs + mirror-like opponent\r\n# ==========================================================",
    b'=================\r\n_WB3_REPORT = dict(reordered=0)\r\n_WB3_INNER = cha20_entry_agent\r\n_WB3_PAIRS = {("BRUNCH_SPOT", "BRUNCH_SPOT"), ("BAKERY", "BRUNCH_SPOT"), ("PIZZA_SHOP", "SMOOTHIE_SHOP")}\r\n\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    action = _WB3_INNER(observation, configuration)\r\n    try:\r\n        shops = tuple((observation.get("town", {}) or {}).get("unlocked_shops", [])[:2])\r\n        mirror = bool(_MA_STATE.get("mirror_like"))\r\n        target = (mirror and shops in _WB3_PAIRS) or (shops in {("BRUNCH_SPOT", "BRUNCH_SPOT"), ("BAKERY", "BRUNCH_SPOT")})\r\n        if target:\r\n            market = [list(o) for o in (action.get("market") or [])]\r\n            if len(market) > 1:\r\n                buys = [o for o in market if len(o) >= 3 and o[0] == "BUY_PRODUCT" and o[1] == "WHEAT"]\r\n                if buys and market[0] not in buys:\r\n                    rest = [o for o in market if o not in buys]\r\n                    _WB3_REPORT["reordered"] += 1\r\n                    action = dict(action, mar',
    b'ket=buys + rest)\r\n    except Exception:\r\n        pass\r\n    return action\r\n\r\n\r\nimport collections as _wb3_coll\r\ncha20_entry_agent.telemetry = _wb3_coll.ChainMap(_WB3_REPORT, _WB3_INNER.telemetry)\r\nkaggle_agent = cha20_entry_agent\r\n_WB3_ENTRY = cha20_entry_agent\r\n\r\n\r\n\r\n\r\n# ===========================================================================\r\n# FLOWPX: live rival-flow lead-sell (general; no recorded opponent data)\r\n#\r\n# Rival sales are recovered each turn from public market inventory deltas, the\r\n# town draw and our own submitted sells (same arithmetic RACE uses).  When the\r\n# rival has been active in an item over the last few turns and the item\'s quote\r\n# sits at or above its trailing average, planned sells of that item from the next\r\n# turns are pulled forward into this turn, capped by shed stock and the planned\r\n# quantity.  Raises our realized rate in endgame sale races without any\r\n# opponent-specific data.\r\n# ===========================================================================\r\n_FX_ITEMS = ("',
    b'CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL")\r\n_FX_FLOW_WIN = 8        # turns: rival considered active if >= _FX_FLOW_MIN units\r\n_FX_FLOW_MIN = 999        # units sold by the rival in the window to trigger a lead\r\n_FX_LEAD_H = 12         # turns of planned sells pulled forward\r\n_FX_QUOTE_WIN = 12      # turns of quote history for the relative-strength test\r\n_FX_MIN_STEP = 96       # keep away from the scripted opening\r\n_FX_STATE = {}\r\n_FX_REPORT = dict(fx_fires=0, fx_units=0, fx_rival_seen=0, fx_errors=0)\r\n\r\n\r\ndef _fx_update(observation, player, st):\r\n    """Recover rival sales for the previous turn into st[\'flow\']."""\r\n    prev = st.get("prev")\r\n    step = int(observation["step"])\r\n    if not prev or prev["step"] != step - 1:\r\n        return\r\n    inv = observation["market"]["inventory"]\r\n    draw = _v9_town_draw(prev["shops"], prev["step"])\r\n    for item in _FX_ITEMS:\r\n        if prev["prices"].get(item, 0) <= 3:\r\n            continue\r\n        sold = inv[item] - prev["inventory"][item] + ',
    b'draw.get(item, 0) - prev["own"].get(item, 0)\r\n        if sold > 0:\r\n            st["flow"][(prev["step"], item)] = sold\r\n            _FX_REPORT["fx_rival_seen"] += 1\r\n    if len(st["flow"]) > 512:\r\n        for k in sorted(st["flow"])[:256]:\r\n            st["flow"].pop(k, None)\r\n\r\n\r\ndef _fx_apply(observation, action, st):\r\n    step = int(observation["step"])\r\n    shops = list(observation["town"]["unlocked_shops"])\r\n    prices = observation["market"]["prices"]\r\n    st["quotes"][step] = {i: int(prices.get(i, 0)) for i in _FX_ITEMS}\r\n    if len(st["quotes"]) > 96:\r\n        for k in sorted(st["quotes"])[:48]:\r\n            st["quotes"].pop(k, None)\r\n    if step < _FX_MIN_STEP or step >= 700:\r\n        return action\r\n    if _MA_STATE.get("mirror_like"):\r\n        return action\r\n    market = [list(o) for o in (action.get("market") or [])]\r\n    if len(market) >= MAX_ORDERS:\r\n        return action\r\n    already = {o[1] for o in market if len(o) > 1 and o[0] in ("SELL", "BUY_PRODUCT")}\r\n    stock = projected_shed(action, F',
    b'armView(observation))\r\n    native = _IMPL.chassis.players.get(int(observation["player"]))\r\n    if not native or native.get("route") not in _IMPL.chassis.routes:\r\n        return action\r\n    tape = _IMPL.chassis.routes[native["route"]]\r\n    hist = st["quotes"]\r\n    recent_turns = [t for t in hist if step - _FX_QUOTE_WIN <= t < step]\r\n    added = False\r\n    for item in _FX_ITEMS:\r\n        if item in already or len(market) >= MAX_ORDERS:\r\n            continue\r\n        flow = sum(q for (t, i), q in st["flow"].items()\r\n                   if i == item and step - _FX_FLOW_WIN <= t < step)\r\n        if flow < _FX_FLOW_MIN:\r\n            continue\r\n        q = int(prices.get(item, 0))\r\n        if q <= 3:\r\n            continue\r\n        vals = [hist[t].get(item, 0) for t in recent_turns if hist[t].get(item, 0) > 0]\r\n        if vals and q < sum(vals) / len(vals):\r\n            continue\r\n        planned = 0\r\n        for t in range(step + 1, min(len(tape), step + _FX_LEAD_H + 1)):\r\n            for o in (tape[t] or {}).get("mark',
    b'et") or []:\r\n                if len(o) >= 3 and o[0] == "SELL" and o[1] == item:\r\n                    planned += max(0, int(o[2]))\r\n        qty = min(int(stock.get(item, 0)), planned)\r\n        if qty <= 0:\r\n            continue\r\n        market.insert(0, ["SELL", item, qty])\r\n        _FX_REPORT["fx_fires"] += 1\r\n        _FX_REPORT["fx_units"] += qty\r\n        added = True\r\n    if not added:\r\n        return action\r\n    result = dict(action)\r\n    result["market"] = market[:MAX_ORDERS]\r\n    return result\r\n\r\n\r\n_EV_HOURS = (15, 16, 17, 18, 19, 20)\r\n_EV_H = 8\r\n_EV_STATE = {}\r\n\r\n\r\ndef _ev_apply(observation, action, st):\r\n    """Evening-window chunked lead-sell: pull up to half of the next _EV_H turns\'\r\n    planned sells of premium items into the current turn while the quote is at or\r\n    above its trailing average."""\r\n    step = int(observation["step"])\r\n    if step < _FX_MIN_STEP or step >= 700:\r\n        return action\r\n    if (step % 24) not in _EV_HOURS:\r\n        return action\r\n    market = [list(o) for o in (actio',
    b'n.get("market") or [])]\r\n    if len(market) >= MAX_ORDERS:\r\n        return action\r\n    already = {o[1] for o in market if len(o) > 1 and o[0] in ("SELL", "BUY_PRODUCT")}\r\n    prices = observation["market"]["prices"]\r\n    stock = projected_shed(action, FarmView(observation))\r\n    native = _IMPL.chassis.players.get(int(observation["player"]))\r\n    if not native or native.get("route") not in _IMPL.chassis.routes:\r\n        return action\r\n    tape = _IMPL.chassis.routes[native["route"]]\r\n    hist = _FX_STATE.get(int(observation["player"]), {}).get("quotes", {})\r\n    recent = [t for t in hist if step - _FX_QUOTE_WIN <= t < step]\r\n    added = False\r\n    for item in _FX_ITEMS:\r\n        if item in already or len(market) >= MAX_ORDERS:\r\n            continue\r\n        q = int(prices.get(item, 0))\r\n        if q <= 3:\r\n            continue\r\n        vals = [hist[t].get(item, 0) for t in recent if hist[t].get(item, 0) > 0]\r\n        if vals and q < sum(vals) / len(vals):\r\n            continue\r\n        planned = 0\r\n        for',
    b' t in range(step + 1, min(len(tape), step + _EV_H + 1)):\r\n            for o in (tape[t] or {}).get("market") or []:\r\n                if len(o) >= 3 and o[0] == "SELL" and o[1] == item:\r\n                    planned += max(0, int(o[2]))\r\n        if planned <= 0:\r\n            continue\r\n        qty = min(int(stock.get(item, 0)), max(1, (3 * planned + 3) // 4))\r\n        if qty <= 0:\r\n            continue\r\n        market.insert(0, ["SELL", item, qty])\r\n        _FX_REPORT["ev_fires"] = _FX_REPORT.get("ev_fires", 0) + 1\r\n        _FX_REPORT["ev_units"] = _FX_REPORT.get("ev_units", 0) + qty\r\n        added = True\r\n    if not added:\r\n        return action\r\n    result = dict(action)\r\n    result["market"] = market[:MAX_ORDERS]\r\n    return result\r\n\r\n\r\n_FX_PARENT = cha20_entry_agent\r\n\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    player = int(observation.get("player", 0))\r\n    step = int(observation.get("step", 0))\r\n    st = _FX_STATE.get(player)\r\n    if st is None or step <= st["step"]:\r\n        st = _FX_S',
    b'TATE[player] = {"step": -1, "flow": {}, "quotes": {}, "prev": None}\r\n        if step == 0:\r\n            _FX_REPORT.update(fx_fires=0, fx_units=0, fx_rival_seen=0, fx_errors=0)\r\n    st["step"] = step\r\n    try:\r\n        _fx_update(observation, player, st)\r\n    except Exception:\r\n        _FX_REPORT["fx_errors"] += 1\r\n    action = _FX_PARENT(observation, configuration)\r\n    try:\r\n        out = _fx_apply(observation, action, st)\r\n    except Exception:\r\n        _FX_REPORT["fx_errors"] += 1\r\n        out = action\r\n    try:\r\n        out = _ev_apply(observation, out, st)\r\n    except Exception:\r\n        _FX_REPORT["fx_errors"] += 1\r\n    try:\r\n        stock = projected_shed(action, FarmView(observation))\r\n        own = {}\r\n        for o in (out.get("market") or []):\r\n            if o and o[0] == "SELL" and len(o) >= 3:\r\n                own[o[1]] = own.get(o[1], 0) + max(0, int(o[2]))\r\n        st["prev"] = {"step": step, "inventory": dict(observation["market"]["inventory"]),\r\n                      "prices": dict(observati',
    b'on["market"]["prices"]), "own": own,\r\n                      "shops": list(observation["town"]["unlocked_shops"])}\r\n    except Exception:\r\n        _FX_REPORT["fx_errors"] += 1\r\n        st["prev"] = None\r\n    return out\r\n\r\n\r\nimport collections as _fx_coll\r\ncha20_entry_agent.telemetry = _fx_coll.ChainMap(_FX_REPORT, _FX_PARENT.telemetry)\r\nkaggle_agent = cha20_entry_agent\r\n_FX_ENTRY = cha20_entry_agent\r\n\r\n\r\n# ===========================================================================\r\n# DAWNPX: dawn-window chunked lead-sell (AXIS2 #8; general, no opponent data)\r\n# ===========================================================================\r\n_DP_HOURS = (0, 1, 2)\r\n_DP_H = 8\r\n_DP_REPORT = dict(dp_fires=0, dp_units=0, dp_errors=0)\r\n\r\n\r\ndef _dp_apply(observation, action):\r\n    step = int(observation["step"])\r\n    if step < 96 or step >= 700:\r\n        return action\r\n    if (step % 24) not in _DP_HOURS:\r\n        return action\r\n    market = [list(o) for o in (action.get("market") or [])]\r\n    if len(market) >= MAX_ORDERS',
    b':\r\n        return action\r\n    already = {o[1] for o in market if len(o) > 1 and o[0] in ("SELL", "BUY_PRODUCT")}\r\n    prices = observation["market"]["prices"]\r\n    stock = projected_shed(action, FarmView(observation))\r\n    native = _IMPL.chassis.players.get(int(observation["player"]))\r\n    if not native or native.get("route") not in _IMPL.chassis.routes:\r\n        return action\r\n    tape = _IMPL.chassis.routes[native["route"]]\r\n    hist = _FX_STATE.get(int(observation["player"]), {}).get("quotes", {})\r\n    recent = [t for t in hist if step - _FX_QUOTE_WIN <= t < step]\r\n    added = False\r\n    for item in _FX_ITEMS:\r\n        if item in already or len(market) >= MAX_ORDERS:\r\n            continue\r\n        q = int(prices.get(item, 0))\r\n        if q <= 3:\r\n            continue\r\n        vals = [hist[t].get(item, 0) for t in recent if hist[t].get(item, 0) > 0]\r\n        if vals and q < sum(vals) / len(vals):\r\n            continue\r\n        planned = 0\r\n        for t in range(step + 1, min(len(tape), step + _DP_H + 1)):\r',
    b'\n            for o in (tape[t] or {}).get("market") or []:\r\n                if len(o) >= 3 and o[0] == "SELL" and o[1] == item:\r\n                    planned += max(0, int(o[2]))\r\n        if planned <= 0:\r\n            continue\r\n        qty = min(int(stock.get(item, 0)), max(1, (3 * planned + 3) // 4))\r\n        if qty <= 0:\r\n            continue\r\n        market.insert(0, ["SELL", item, qty])\r\n        _DP_REPORT["dp_fires"] += 1\r\n        _DP_REPORT["dp_units"] += qty\r\n        added = True\r\n    if not added:\r\n        return action\r\n    return dict(action, market=market[:MAX_ORDERS])\r\n\r\n\r\n_DP_PARENT = cha20_entry_agent\r\n\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    step = int(observation.get("step", 0))\r\n    if step == 0:\r\n        _DP_REPORT.update(dp_fires=0, dp_units=0, dp_errors=0)\r\n    action = _DP_PARENT(observation, configuration)\r\n    try:\r\n        return _dp_apply(observation, action)\r\n    except Exception:\r\n        _DP_REPORT["dp_errors"] += 1\r\n        return action\r\n\r\n\r\nimport collecti',
    b'ons as _dp_coll\r\ncha20_entry_agent.telemetry = _dp_coll.ChainMap(_DP_REPORT, _DP_PARENT.telemetry)\r\nkaggle_agent = cha20_entry_agent\r\n_DP_ENTRY = cha20_entry_agent\r\n\r\n\r\n_MP_HOURS = (10, 11, 12, 13)\r\n_MP_H = 8\r\n_MP_REPORT = dict(mp_fires=0, mp_units=0, mp_errors=0)\r\n\r\n\r\ndef _mp_apply(observation, action):\r\n    step = int(observation["step"])\r\n    if step < 96 or step >= 700:\r\n        return action\r\n    if (step % 24) not in _MP_HOURS:\r\n        return action\r\n    market = [list(o) for o in (action.get("market") or [])]\r\n    if len(market) >= MAX_ORDERS:\r\n        return action\r\n    already = {o[1] for o in market if len(o) > 1 and o[0] in ("SELL", "BUY_PRODUCT")}\r\n    prices = observation["market"]["prices"]\r\n    stock = projected_shed(action, FarmView(observation))\r\n    native = _IMPL.chassis.players.get(int(observation["player"]))\r\n    if not native or native.get("route") not in _IMPL.chassis.routes:\r\n        return action\r\n    tape = _IMPL.chassis.routes[native["route"]]\r\n    hist = _FX_STATE.get(int(observat',
    b'ion["player"]), {}).get("quotes", {})\r\n    recent = [t for t in hist if step - _FX_QUOTE_WIN <= t < step]\r\n    added = False\r\n    for item in _FX_ITEMS:\r\n        if item in already or len(market) >= MAX_ORDERS:\r\n            continue\r\n        q = int(prices.get(item, 0))\r\n        if q <= 3:\r\n            continue\r\n        vals = [hist[t].get(item, 0) for t in recent if hist[t].get(item, 0) > 0]\r\n        if vals and q < sum(vals) / len(vals):\r\n            continue\r\n        planned = 0\r\n        for t in range(step + 1, min(len(tape), step + _MP_H + 1)):\r\n            for o in (tape[t] or {}).get("market") or []:\r\n                if len(o) >= 3 and o[0] == "SELL" and o[1] == item:\r\n                    planned += max(0, int(o[2]))\r\n        if planned <= 0:\r\n            continue\r\n        qty = min(int(stock.get(item, 0)), max(1, (3 * planned + 3) // 4))\r\n        if qty <= 0:\r\n            continue\r\n        market.insert(0, ["SELL", item, qty])\r\n        _MP_REPORT["mp_fires"] += 1\r\n        _MP_REPORT["mp_units"] += qty',
    b'\r\n        added = True\r\n    if not added:\r\n        return action\r\n    return dict(action, market=market[:MAX_ORDERS])\r\n\r\n\r\n_MP_PARENT = cha20_entry_agent\r\n\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    step = int(observation.get("step", 0))\r\n    if step == 0:\r\n        _MP_REPORT.update(mp_fires=0, mp_units=0, mp_errors=0)\r\n    action = _MP_PARENT(observation, configuration)\r\n    try:\r\n        return _mp_apply(observation, action)\r\n    except Exception:\r\n        _MP_REPORT["mp_errors"] += 1\r\n        return action\r\n\r\n\r\nimport collections as _mp_coll\r\ncha20_entry_agent.telemetry = _mp_coll.ChainMap(_MP_REPORT, _MP_PARENT.telemetry)\r\nkaggle_agent = cha20_entry_agent\r\n_MP_ENTRY = cha20_entry_agent\r\n\r\n\r\n# ===========================================================================\r\n# BUYDIP: buy-the-dip for large wheat buys (general, no opponent data)\r\n# ===========================================================================\r\n_BD_ITEM = "WHEAT"\r\n_BD_MIN = 8\r\n_BD_WIN = 12\r\n_BD_DEADLINE = 6\r\n_BD_',
    b'CAP = 64\r\n_BD_STATE = {}\r\n_BD_REPORT = dict(bd_split=0, bd_pending_units=0, bd_released=0, bd_errors=0)\r\n\r\n\r\ndef _bd_apply(observation, action):\r\n    step = int(observation["step"])\r\n    if step < 24 or step >= 690 or not isinstance(action, dict):\r\n        return action\r\n    player = int(observation["player"])\r\n    st = _BD_STATE.setdefault(player, {"hist": [], "pending": 0, "since": None})\r\n    price = int((observation["market"]["prices"] or {}).get(_BD_ITEM, 0))\r\n    if price > 0:\r\n        st["hist"].append((step, price))\r\n        if len(st["hist"]) > 48:\r\n            del st["hist"][:24]\r\n    market = [list(o) for o in (action.get("market") or [])]\r\n\r\n    if st["pending"] > 0:\r\n        vals = [p for (s, p) in st["hist"] if step - _BD_WIN <= s < step]\r\n        avg = (sum(vals) / len(vals)) if vals else price\r\n        due = st["since"] is not None and step - st["since"] >= _BD_DEADLINE\r\n        if price > 0 and (price <= avg or due):\r\n            has = any(len(o) >= 3 and o[0] == "BUY_PRODUCT" and o[1] == _BD',
    b'_ITEM for o in market)\r\n            if len(market) < 10 and not has:\r\n                qty = min(st["pending"], 8)\r\n                market.append(["BUY_PRODUCT", _BD_ITEM, qty])\r\n                st["pending"] -= qty\r\n                _BD_REPORT["bd_released"] += qty\r\n            if st["pending"] <= 0:\r\n                st["since"] = None\r\n\r\n    if st["pending"] < _BD_CAP:\r\n        vals = [p for (s, p) in st["hist"] if step - _BD_WIN <= s < step]\r\n        avg = (sum(vals) / len(vals)) if vals else price\r\n        for o in market:\r\n            if len(o) >= 3 and o[0] == "BUY_PRODUCT" and o[1] == _BD_ITEM and int(o[2]) >= _BD_MIN:\r\n                q = int(o[2])\r\n                if price > 0 and price > avg:\r\n                    hold = min(q - 1, _BD_CAP - st["pending"], q // 2)\r\n                    if hold > 0:\r\n                        o[2] = q - hold\r\n                        st["pending"] += hold\r\n                        if st["since"] is None:\r\n                            st["since"] = step\r\n                      ',
    b'  _BD_REPORT["bd_split"] += 1\r\n                        _BD_REPORT["bd_pending_units"] += hold\r\n    return dict(action, market=market)\r\n\r\n\r\n_BD_PARENT = cha20_entry_agent\r\n\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    if int(observation.get("step", 0)) == 0:\r\n        _BD_REPORT.update(bd_split=0, bd_pending_units=0, bd_released=0, bd_errors=0)\r\n        _BD_STATE.clear()\r\n    action = _BD_PARENT(observation, configuration)\r\n    try:\r\n        return _bd_apply(observation, action)\r\n    except Exception:\r\n        _BD_REPORT["bd_errors"] += 1\r\n        return action\r\n\r\n\r\nimport collections as _bd_coll\r\ncha20_entry_agent.telemetry = _bd_coll.ChainMap(_BD_REPORT, _BD_PARENT.telemetry)\r\nkaggle_agent = cha20_entry_agent\r\n_BD_ENTRY = cha20_entry_agent\r\n\r\n\r\n# ===========================================================================\r\n# MODELPX: model-based lead seller for MILK/STRAWBERRY (general)\r\n# ===========================================================================\r\n_MPX_ITEMS = ("MILK", "STR',
    b'AWBERRY", "WOOL")\r\n_MPX_HOURS = tuple(range(12, 23))\r\n_MPX_REPORT = dict(mpx_fires=0, mpx_units=0, mpx_errors=0)\r\n_MPX_HIST = {}\r\n\r\n\r\ndef _mpx_draw_units(item, step):\r\n    # town draw cadence: shops every 4 steps, center every 24\r\n    draw = 0\r\n    if step % 4 == 0:\r\n        draw += 1\r\n    if step % 24 == 0:\r\n        draw += 1\r\n    return draw\r\n\r\n\r\ndef _mpx_apply(observation, action):\r\n    step = int(observation["step"])\r\n    if step < 144 or step >= 696 or (step % 24) not in _MPX_HOURS:\r\n        return action\r\n    player = int(observation["player"])\r\n    market = observation["market"]\r\n    inv_all = market.get("inventory") or {}\r\n    prices = market.get("prices") or {}\r\n    hist = _MPX_HIST.setdefault(player, {})\r\n    # record rival cadence from public inventory deltas\r\n    prev = hist.get("prev")\r\n    inv_now = {i: int(inv_all.get(i, 0)) for i in _MPX_ITEMS}\r\n    if prev and prev["step"] == step - 1:\r\n        for i in _MPX_ITEMS:\r\n            d = inv_now[i] - prev["inv"][i] + _mpx_draw_units(i, step - 1) - ',
    b'prev["own"].get(i, 0)\r\n            hist.setdefault(i, []).append(max(0, d))\r\n            if len(hist[i]) > 12:\r\n                del hist[i][:6]\r\n    hist["prev"] = {"step": step, "inv": inv_now, "own": {}}\r\n    tape = None\r\n    market_orders = [list(o) for o in (action.get("market") or [])]\r\n    for o in market_orders:\r\n        if len(o) >= 3 and o[0] == "SELL" and o[1] in _MPX_ITEMS:\r\n            hist["prev"]["own"][o[1]] = hist["prev"]["own"].get(o[1], 0) + int(o[2])\r\n    already = {o[1] for o in market_orders if len(o) > 1 and o[0] == "SELL"}\r\n    native = _IMPL.chassis.players.get(player)\r\n    if not native or native.get("route") not in _IMPL.chassis.routes:\r\n        return action\r\n    tape = _IMPL.chassis.routes[native["route"]]\r\n    stock = projected_shed(action, FarmView(observation))\r\n    added = False\r\n    for item in _MPX_ITEMS:\r\n        if item in already or len(market_orders) >= 10:\r\n            continue\r\n        avail = int(stock.get(item, 0))\r\n        if avail <= 0:\r\n            continue\r\n      ',
    b'  p_now = int(prices.get(item, 0))\r\n        if p_now <= 1:\r\n            continue\r\n        rival = hist.get(item) or []\r\n        rival_avg = (sum(rival[-4:]) / len(rival[-4:])) if rival else 0.0\r\n        planned = 6\r\n        try:\r\n            inv = int(inv_all.get(item, 0))\r\n            p_cur = float(_r37_market_price(item, inv))\r\n            inv_next = inv + rival_avg + planned - _mpx_draw_units(item, step)\r\n            p_next = float(_r37_market_price(item, max(0, int(inv_next))))\r\n        except Exception:\r\n            continue\r\n        if p_next < p_cur - 0.5:\r\n            take = min(avail, max(1, planned // 2))\r\n            if take <= 0:\r\n                continue\r\n            market_orders.insert(0, ["SELL", item, take])\r\n            _MPX_REPORT["mpx_fires"] += 1\r\n            _MPX_REPORT["mpx_units"] += take\r\n            hist["prev"]["own"][item] = hist["prev"]["own"].get(item, 0) + take\r\n            added = True\r\n    if not added:\r\n        return action\r\n    return dict(action, market=market_orders[:10])',
    b'\r\n\r\n\r\n_MPX_PARENT = cha20_entry_agent\r\n\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    if int(observation.get("step", 0)) == 0:\r\n        _MPX_REPORT.update(mpx_fires=0, mpx_units=0, mpx_errors=0)\r\n        _MPX_HIST.clear()\r\n    action = _MPX_PARENT(observation, configuration)\r\n    try:\r\n        return _mpx_apply(observation, action)\r\n    except Exception:\r\n        _MPX_REPORT["mpx_errors"] += 1\r\n        return action\r\n\r\n\r\nimport collections as _mpx_coll\r\ncha20_entry_agent.telemetry = _mpx_coll.ChainMap(_MPX_REPORT, _MPX_PARENT.telemetry)\r\nkaggle_agent = cha20_entry_agent\r\n_MPX_ENTRY = cha20_entry_agent\r\n\r\n\r\n# ===========================================================================\r\n# SHIELD-MILK: fund failing animal/land buys only in milk-shop worlds\r\n# ===========================================================================\r\n_SM_ANIMAL = {"COW": 400, "SHEEP": 500, "GOOSE": 300}\r\n_SM_LAND = (1000, 2000, 4000)\r\n_SM_MILK_SHOP = ("PIZZA_SHOP", "ICE_CREAM_SHOP", "SMOOTHIE_SHOP")\r\n_SM_PRODUC',
    b'TS = ("MELON", "STRAWBERRY", "MILK", "WOOL", "EGG", "TOMATO", "CARROT", "WHEAT", "FERTILIZER")\r\n_SM_REPORT = dict(sm_shields=0, sm_units=0, sm_errors=0)\r\n\r\n\r\ndef _sm_apply(observation, action):\r\n    if not isinstance(action, dict):\r\n        return action\r\n    shops = list((observation.get("town") or {}).get("unlocked_shops") or [])\r\n    if not any(s in _SM_MILK_SHOP for s in shops):\r\n        return action\r\n    market = [list(o) for o in (action.get("market") or [])]\r\n    if not market:\r\n        return action\r\n    seat = int(observation["player"])\r\n    farm = observation["farms"][seat]\r\n    shed = dict((observation.get("private") or {}).get("shed") or {})\r\n    prices = observation["market"].get("prices") or {}\r\n    quads = len(farm.get("unlocked_quadrants") or [])\r\n    cash = float(farm.get("money", 0) or 0)\r\n    sold = {}\r\n    for o in market:\r\n        if not o:\r\n            continue\r\n        if o[0] == "SELL" and len(o) >= 3:\r\n            item = o[1]\r\n            take = min(max(0, int(o[2])), int(shed.get(it',
    b'em, 0)) - sold.get(item, 0))\r\n            if take > 0:\r\n                cash += take * float(prices.get(item, 0))\r\n                sold[item] = sold.get(item, 0) + take\r\n        elif o[0] == "BUY_ANIMAL" and len(o) >= 3:\r\n            cash -= _SM_ANIMAL.get(o[1], 400) * max(0, int(o[2]))\r\n        elif o[0] == "BUY_SEED" and len(o) >= 3:\r\n            cash -= {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}.get(o[1], 100) * max(0, int(o[2]))\r\n        elif o[0] == "BUY_PRODUCT" and len(o) >= 3:\r\n            cash -= float(prices.get(o[1], 0)) * max(0, int(o[2]))\r\n        elif o[0] == "BUY_LAND":\r\n            extra = quads - 1\r\n            cash -= _SM_LAND[extra] if 0 <= extra < 3 else 4000\r\n    need = -cash\r\n    if need <= 0:\r\n        return action\r\n    candidates = []\r\n    for item in _SM_PRODUCTS:\r\n        price = float(prices.get(item, 0))\r\n        avail = int(shed.get(item, 0)) - sold.get(item, 0)\r\n        if price > 1 and avail > 0:\r\n            candidates.append((price, item, avail))',
    b'\r\n    candidates.sort(reverse=True)\r\n    added = []\r\n    for price, item, avail in candidates:\r\n        if need <= 0 or len(market) + len(added) >= 10:\r\n            break\r\n        q = min(avail, int(need / price) + 1)\r\n        if q <= 0:\r\n            continue\r\n        added.append(["SELL", item, q])\r\n        need -= q * price\r\n        _SM_REPORT["sm_units"] += q\r\n    if not added:\r\n        return action\r\n    _SM_REPORT["sm_shields"] += 1\r\n    idx = next((i for i, o in enumerate(market) if o and str(o[0]).startswith("BUY")), len(market))\r\n    market = market[:idx] + added + market[idx:]\r\n    return dict(action, market=market[:10])\r\n\r\n\r\n_SM_PARENT = cha20_entry_agent\r\n\r\n\r\ndef cha20_entry_agent(observation, configuration=None):\r\n    if int(observation.get("step", 0)) == 0:\r\n        _SM_REPORT.update(sm_shields=0, sm_units=0, sm_errors=0)\r\n    action = _SM_PARENT(observation, configuration)\r\n    try:\r\n        return _sm_apply(observation, action)\r\n    except Exception:\r\n        _SM_REPORT["sm_errors"] += 1\r\n     ',
    b'   return action\r\n\r\n\r\nimport collections as _sm_coll\r\ncha20_entry_agent.telemetry = _sm_coll.ChainMap(_SM_REPORT, _SM_PARENT.telemetry)\r\nkaggle_agent = cha20_entry_agent\r\n_SM_ENTRY = cha20_entry_agent\r\n\r\n\r\n# ==== F4: Layer D - exact best-response ordering (_CXD from elo_2615 order-book) ====\r\nimport itertools as _cxd_it\r\n_CXD_HOST = cha20_entry_agent\r\n_CXD_FIXED = (\'HIRE\', \'BUY_SEED\', \'BUY_ANIMAL\', \'BUY_LAND\')\r\n_CXD_BUDGET = 800\r\n_CXD_FROM = 0\r\n_CXD_REPORT = {\'cxd_turns\': 0, \'cxd_gain\': 0.0, \'cxd_evals\': 0, \'cxd_budget_hits\': 0, \'cxd_errors\': 0}\r\n_CXD_MODELS = []\r\n_CXD_PARENT_ORDERS = []\r\n\r\n\r\ndef _cxd_candidates(orders, slots, sells, fixed):\r\n    """Orderings of `sells` over `slots`, fixed-price orders filling the rest in their own order."""\r\n    for positions in _cxd_it.permutations(slots, len(sells)):\r\n        out = list(orders)\r\n        rest = [i for i in slots if i not in positions]\r\n        for i, order in zip(positions, sells):\r\n            out[i] = order\r\n        for i, order in zip(rest, fixed):\r\n    ',
    b"        out[i] = order\r\n        yield out\r\n\r\n\r\ndef _cxd_reorder(obs, action):\r\n    market = action.get('market') or []\r\n    if len(market) < 2:\r\n        return action\r\n    orders = [list(o) if isinstance(o, (list, tuple)) else o for o in market]\r\n    bought = {o[1] for o in orders if o and len(o) > 1 and o[0] == 'BUY_PRODUCT'}\r\n    slots, sells, fixed = [], [], []\r\n    for i, o in enumerate(orders):\r\n        if not o:\r\n            continue\r\n        if o[0] in _CXD_FIXED:\r\n            slots.append(i); fixed.append(o)\r\n        elif o[0] == 'SELL' and len(o) > 1 and o[1] not in bought:\r\n            slots.append(i); sells.append(o)\r\n    if not sells or len(slots) < 2:\r\n        return action\r\n    params = _v44y_params(obs)\r\n    stock = {k: max(0, int(v)) for k, v in projected_shed(action, FarmView(obs)).items()}\r\n    inv0 = {k: int(v) for k, v in obs['market']['inventory'].items()}\r\n    _CXD_PARENT_ORDERS[:] = [list(o) for o in orders if o]\r\n    models = [m for m in _CXD_MODELS if m] or [orders]\r\n    margins = [_v",
    b"44y_factor_margin(m, inv0, stock, params) for m in models]\r\n\r\n    def margin(cand):\r\n        return min(f(cand) for f in margins)\r\n    base = best = margin(orders)\r\n    best_orders = None\r\n    evals = 0\r\n    for cand in _cxd_candidates(orders, slots, sells, fixed):\r\n        if cand == orders:\r\n            continue\r\n        evals += 1\r\n        if evals > _CXD_BUDGET:\r\n            _CXD_REPORT['cxd_budget_hits'] += 1\r\n            break\r\n        value = margin(cand)\r\n        if value > best + 0.5:\r\n            best, best_orders = value, cand\r\n    _CXD_REPORT['cxd_evals'] += evals\r\n    if best_orders is None:\r\n        return action\r\n    _CXD_REPORT['cxd_turns'] += 1\r\n    _CXD_REPORT['cxd_gain'] += best - base\r\n    return dict(action, market=best_orders)\r\n\r\n\r\ndef cxd_agent(observation, configuration=None):\r\n    action = _CXD_HOST(observation, configuration)\r\n    try:\r\n        if int(observation.get('step', 0)) == 0:\r\n            _CXD_REPORT.update(cxd_turns=0, cxd_gain=0.0, cxd_evals=0, cxd_budget_hits=0, cxd_error",
    b"s=0)\r\n        if int(observation.get('step', 0)) >= _CXD_FROM:\r\n            return _cxd_reorder(observation, action)\r\n    except Exception:\r\n        _CXD_REPORT['cxd_errors'] += 1\r\n    return action\r\n\r\n\r\nimport collections as _cxd_coll\r\ncxd_agent.telemetry = _cxd_coll.ChainMap(_CXD_REPORT, _CXD_HOST.telemetry)\r\ncha20_entry_agent = cxd_agent\r\nkaggle_agent = cha20_entry_agent\r\n\r\n\r\n# ==== F5: EXP410 fertilizer guard (pipe18 `e410_agent`) ====\r\n_E410_REPORT = dict(skips=0, covered=0, capped=0, errors=0)\r\n_E410_PARENT = cha20_entry_agent\r\n\r\n\r\ndef e410_agent(observation, configuration=None):\r\n    action = _E410_PARENT(observation, configuration)\r\n    try:\r\n        step = int(observation['step']); seat = int(observation['player']); day = step // 24\r\n        if step == 0:\r\n            for k in _E410_REPORT: _E410_REPORT[k] = 0\r\n        units = [action.get('farmer') or ['PASS']] + list(action.get('hands') or [])\r\n        if not any(c == ['FERTILIZE'] for c in units): return action\r\n        farm, private = _PLANNER_NS[",
    b"'_clone_state'](observation['farms'][seat], observation['private'])\r\n        positions = [farm['farmer']] + list(farm['hands']); changed = False\r\n        native = _IMPL.chassis.players[seat]\r\n        expected = max(len(a.get('hands', [])) for a in _v219_native_day(native, day))\r\n        reactive = set(_R51_INPUT_STATES.get(seat, {}).get('workers', {}))\r\n        for i, cmd in enumerate(units[:len(positions)]):\r\n            pos = tuple(positions[i]); tile = farm['tiles'][pos[1]][pos[0]]\r\n            if cmd == ['FERTILIZE'] and isinstance(tile, dict) and tile.get('crop') in ('WHEAT', 'CARROT') and private['inventories'][i].get('FERTILIZER', 0) > 0:\r\n                until = int(tile.get('fertilized_until_day', -1)); covered = until >= day + 2; skip = covered\r\n                if not skip and i <= expected and i not in reactive:\r\n                    visits = _ca_visits(observation, action, pos, min(718, (int(tile['planted_day']) + 6) * 24), start=step + 1)\r\n                    kw = dict(y0=int(tile['yield_units']),",
    b" watered_day=day if tile.get('watered_today') else -1, now_step=step)\r\n                    old = _ca_yield_path(tile['crop'], int(tile['planted_day']), visits, fert_until=until, **kw)[0]\r\n                    new = _ca_yield_path(tile['crop'], int(tile['planted_day']), visits, fert_until=max(until, day + 2), **kw)[0]\r\n                    skip = old > 0 and old == new\r\n                if skip:\r\n                    units[i] = cmd = ['PASS']; changed = True\r\n                    _E410_REPORT['skips'] += 1; _E410_REPORT['covered' if covered else 'capped'] += 1\r\n            _PLANNER_NS['_apply_unit_action'](farm, private, i, cmd, len(farm['tiles']), day, 24, 100)\r\n        if changed: return dict(action, farmer=units[0], hands=units[1:])\r\n    except Exception:\r\n        _E410_REPORT['errors'] += 1\r\n    return action\r\n\r\n\r\nimport collections as _e410_coll\r\ne410_agent.telemetry = _e410_coll.ChainMap(_E410_REPORT, _E410_PARENT.telemetry)\r\ncha20_entry_agent = e410_agent\r\nkaggle_agent = cha20_entry_agent\r\n\r\n\r\n# ==== F6: EXP",
    b"402 late seed cap (pipe18 `e402_agent`) ====\r\n_E402_PARENT = cha20_entry_agent\r\n_E402_CACHE = {}\r\n_E402_REPORT = dict(cut_units=0, saved_cost=0, changed_turns=0, errors=0)\r\n\r\n\r\ndef _e402_remaining(native, step):\r\n    route = native['route']; key = (route, step)\r\n    if key in _E402_CACHE: return _E402_CACHE[key]\r\n    need = 0\r\n    for t in range(step + 1, 719):\r\n        tape = _IMPL.chassis.routes[2 if t >= 648 else route]\r\n        act = tape[t]\r\n        need += sum(1 for c in [act.get('farmer') or ['PASS']] + list(act.get('hands') or [])\r\n                    if len(c) > 1 and c[0] == 'PLANT' and c[1] in ('WHEAT', 'CARROT'))\r\n    _E402_CACHE[key] = need\r\n    return need\r\n\r\n\r\ndef e402_agent(observation, configuration=None):\r\n    action = _E402_PARENT(observation, configuration)\r\n    try:\r\n        step = int(observation['step']); seat = int(observation['player'])\r\n        if step == 0:\r\n            _E402_CACHE.clear()\r\n            for k in _E402_REPORT: _E402_REPORT[k] = 0\r\n        if step < 624: return action\r",
    b"\n        market = action.get('market', [])\r\n        if not any(len(o) >= 3 and o[0] == 'BUY_SEED' and o[1] in ('WHEAT', 'CARROT') for o in market): return action\r\n        native = _IMPL.chassis.players[seat]\r\n        remaining = _e402_remaining(native, step)\r\n        # Queued retries can outlive their original schedule; reserve for them too.\r\n        remaining += sum(1 for queue in native['pending'].values() for pos, c in queue\r\n                         if len(c) > 1 and c[0] == 'PLANT' and c[1] in ('WHEAT', 'CARROT'))\r\n        units = [action.get('farmer') or ['PASS']] + list(action.get('hands') or [])\r\n        available = {p: max(0, int(observation['private']['seeds'].get(p, 0)) - sum(c[:2] == ['PLANT', p] for c in units)) for p in ('WHEAT', 'CARROT')}\r\n        out = []; changed = False\r\n        for o in market:\r\n            if len(o) >= 3 and o[0] == 'BUY_SEED' and o[1] in available:\r\n                p = o[1]; qty = max(0, int(o[2])); keep = min(qty, max(0, remaining - available[p])); available[p] += keep\r",
    b"\n                if keep < qty:\r\n                    cut = qty - keep; changed = True\r\n                    _E402_REPORT['cut_units'] += cut; _E402_REPORT['saved_cost'] += cut * (10 if p == 'WHEAT' else 20)\r\n                    if p == 'CARROT':\r\n                        st = _CA_STATE.get(seat)\r\n                        if st is not None: st['spare_carrot'] = max(0, st.get('spare_carrot', 0) - cut)\r\n                    o = [o[0], p, keep] if keep else []\r\n            out.append(o)\r\n        if changed:\r\n            _E402_REPORT['changed_turns'] += 1\r\n            action = dict(action, market=out)\r\n    except Exception:\r\n        _E402_REPORT['errors'] += 1\r\n    return action\r\n\r\n\r\nimport collections as _e402_coll\r\ne402_agent.telemetry = _e402_coll.ChainMap(_E402_REPORT, _E402_PARENT.telemetry)\r\ncha20_entry_agent = e402_agent\r\nkaggle_agent = cha20_entry_agent\r\n\r\n\r\n# ==== MERGE: same-item SELL compaction (port of 2695's E334) ====\r\n_MG_REPORT = dict(mg_turns=0, mg_merged=0, mg_errors=0)\r\n_MG_PARENT = cha20_entry_agen",
    b't\r\n\r\n\r\ndef _mg_apply(observation, action):\r\n    if not isinstance(action, dict):\r\n        return action\r\n    market = [list(o) for o in (action.get("market") or [])]\r\n    if len(market) < 2:\r\n        return action\r\n    seen = {}\r\n    out = []\r\n    changed = False\r\n    for o in market:\r\n        if o and len(o) >= 3 and o[0] in ("SELL", "BUY_PRODUCT", "BUY_SEED"):\r\n            it = (o[0], o[1])\r\n            if it in seen:\r\n                out[seen[it]][2] = int(out[seen[it]][2]) + int(o[2])\r\n                changed = True\r\n                continue\r\n            seen[it] = len(out)\r\n        out.append(o)\r\n    if not changed:\r\n        return action\r\n    _MG_REPORT["mg_turns"] += 1\r\n    _MG_REPORT["mg_merged"] += len(market) - len(out)\r\n    return dict(action, market=out)\r\n\r\n\r\ndef mg_agent(observation, configuration=None):\r\n    if int(observation.get("step", 0)) == 0:\r\n        _MG_REPORT.update(mg_turns=0, mg_merged=0, mg_errors=0)\r\n    action = _MG_PARENT(observation, configuration)\r\n    try:\r\n        return _mg_a',
    b'pply(observation, action)\r\n    except Exception:\r\n        _MG_REPORT["mg_errors"] += 1\r\n        return action\r\n\r\n\r\nimport collections as _mg_coll\r\nmg_agent.telemetry = _mg_coll.ChainMap(_MG_REPORT, _MG_PARENT.telemetry)\r\ncha20_entry_agent = mg_agent\r\nkaggle_agent = cha20_entry_agent\r\n\r\n\r\n# ==== IG (bundled): queue hole-closure ====\r\n_IG_CASH = frozenset(("CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL"))\r\n_IG_REPORT = {"queue_changed_turns": 0, "zeroed_orders": 0, "pulled_orders": 0, "pulled_slots": 0, "errors": 0}\r\n_IG_PARENT = cha20_entry_agent\r\n\r\n\r\ndef _ig_close_queue(observation, action):\r\n    if not isinstance(action, dict):\r\n        return action\r\n    market = action.get("market") or []\r\n    if len(market) < 2:\r\n        return action\r\n    projected = dict(projected_shed(action, FarmView(observation)))\r\n    remaining = {item: max(0, int(projected.get(item, 0))) for item in _IG_CASH}\r\n    revised = []\r\n    zeroed = 0\r\n    for raw in market:\r\n        order = list(raw) if isinstance(raw, (li',
    b'st, tuple)) else raw\r\n        if (isinstance(order, list) and len(order) >= 3 and order[0] == "SELL" and order[1] in _IG_CASH):\r\n            requested = max(0, int(order[2]))\r\n            executed = min(requested, remaining[order[1]])\r\n            remaining[order[1]] -= executed\r\n            if executed <= 0:\r\n                revised.append([])\r\n                zeroed += 1\r\n            else:\r\n                revised.append(order)\r\n        else:\r\n            revised.append(order)\r\n    holes = []\r\n    pulled = 0\r\n    distance = 0\r\n    for index, order in enumerate(revised):\r\n        if not order:\r\n            holes.append(index)\r\n            continue\r\n        movable = (isinstance(order, list) and len(order) >= 3 and order[0] == "SELL" and order[1] in _IG_CASH and int(order[2]) > 0)\r\n        if not movable or not holes:\r\n            continue\r\n        target = holes.pop(0)\r\n        revised[target] = order\r\n        revised[index] = []\r\n        holes.append(index)\r\n        pulled += 1\r\n        distance += index - ',
    b'target\r\n    if revised == market:\r\n        return action\r\n    _IG_REPORT["queue_changed_turns"] += 1\r\n    _IG_REPORT["zeroed_orders"] += zeroed\r\n    _IG_REPORT["pulled_orders"] += pulled\r\n    _IG_REPORT["pulled_slots"] += distance\r\n    return dict(action, market=revised)\r\n\r\n\r\ndef ig_agent(observation, configuration=None):\r\n    if int(observation.get("step", 0)) == 0:\r\n        for key in _IG_REPORT:\r\n            _IG_REPORT[key] = 0\r\n    action = _IG_PARENT(observation, configuration)\r\n    try:\r\n        return _ig_close_queue(observation, action)\r\n    except Exception:\r\n        _IG_REPORT["errors"] += 1\r\n        return action\r\n\r\n\r\nimport collections as _ig_coll\r\nig_agent.telemetry = _ig_coll.ChainMap(_IG_REPORT, _IG_PARENT.telemetry)\r\ncha20_entry_agent = ig_agent\r\nkaggle_agent = cha20_entry_agent\r\n',
))
assert hashlib.sha256(SOURCE_BYTES).hexdigest() == EXPECTED_MAIN_SHA256, 'Embedded source is incomplete or modified; import the fixed notebook again.'
MAIN = WORKDIR / 'main.py'
MAIN.write_bytes(SOURCE_BYTES)
assert MAIN.read_bytes() == SOURCE_BYTES, 'main.py was not written correctly.'
# Kaggle's notebook-submission flow looks for submission.py in the output.
SUBMISSION = WORKDIR / 'submission.py'
SUBMISSION.write_bytes(SOURCE_BYTES)
assert SUBMISSION.read_bytes() == SOURCE_BYTES, 'submission.py was not written correctly.'
print('Wrote verified main.py and submission.py:', len(SOURCE_BYTES), 'bytes each', flush=True)


In [ ]:
import ast, gzip, io, tarfile, hashlib
source_bytes = MAIN.read_bytes()
assert hashlib.sha256(source_bytes).hexdigest() == EXPECTED_MAIN_SHA256
assert SUBMISSION.read_bytes() == source_bytes
compile(source_bytes, 'main.py', 'exec')
assert [n.name for n in ast.parse(source_bytes).body if isinstance(n, ast.FunctionDef)][-1] == 'ig_agent'
ARCHIVE = OUTPUT_ROOT / 'submission_cha22.tar.gz'
buffer = io.BytesIO()
with tarfile.open(fileobj=buffer, mode='w') as tar:
    info = tarfile.TarInfo('main.py')
    info.size = len(source_bytes); info.mtime = 0; info.mode = 0o644
    tar.addfile(info, io.BytesIO(source_bytes))
with ARCHIVE.open('wb') as f:
    with gzip.GzipFile(fileobj=f, mode='wb', mtime=0, filename='') as z:
        z.write(buffer.getvalue())
with tarfile.open(ARCHIVE) as tar:
    assert tar.getnames() == ['main.py']
    assert tar.extractfile('main.py').read() == source_bytes
print('Ready:', ARCHIVE)
print('Agent SHA256:', EXPECTED_MAIN_SHA256)
print('No competition submission was made.')
